### Loading the libraries

In [ ]:
import json
import os
import random
import re
import time
import ast
from datetime import datetime
from typing import Dict, List, Union, Tuple, Set, Optional

import gseapy as gp
import pandas as pd
import requests
from tqdm import tqdm

In [ ]:
corum = pd.read_csv("CORUM_dat.csv")
corum.columns = corum.columns.str.strip()

print(f"Loaded CORUM: {len(corum)} complexes")
gene_to_complexes = {}

for _, row in corum.iterrows():

    genes = str(row["subunits.Gene.name."]).split(";")

    for gene in genes:

        gene = gene.strip()

        if not gene or gene.lower() == "nan":
            continue

        gene_to_complexes.setdefault(
            gene,
            set()
        ).add(row["ComplexID"])

print(
    f"Genes represented in CORUM: "
    f"{len(gene_to_complexes)}"
)

input_text = """
Refined Community 1: ['ACSL3', 'ACTA2', 'ACTG2', 'ACTN1', 'ACVR1', 'ADAM12', 'ADAMTS12', 'AGPAT2', 'AHNAK', 'ANO1', 'AOC3', 'AOPEP', 'APBA1', 'AQP1', 'ATL1', 'ATP13A4', 'BGN', 'BOC', 'C1orf116', 'CACHD1', 'CADM1', 'CALD1', 'CAPN2', 'CAPN5', 'CARD19', 'CARMN', 'CAV1', 'CAVIN1', 'CBR3-AS1', 'CCDC15', 'CD81', 'CEP41', 'CHMP3', 'CNIH3', 'CNN2', 'CNPY4', 'COL8A1', 'COPS8', 'COPZ2', 'CTNND2', 'CX3CR1', 'DAAM1', 'DACT3', 'DDAH1', 'DDR2', 'DIPK1B', 'DLC1', 'DNAAF9', 'EDN2', 'EDNRA', 'EFNA3', 'EFR3B', 'ELAPOR2', 'EMC3', 'ENDOD1', 'ENPP1', 'ENSG00000278932', 'EPHA3', 'ERBB4', 'EXTL2', 'FAM210B', 'FAP', 'FBXO32', 'FBXW2', 'FERMT2', 'FHL1', 'FKBP9', 'FLNA', 'FLNC', 'FSTL1', 'GPC1', 'GSTA1', 'GTF2I', 'GUCY1A2', 'H19', 'H2AC6', 'H2AC8', 'H3C4', 'H4C12', 'H4C8', 'HSPG2', 'IRAG1', 'ITGB5', 'KANK2', 'KATNAL1', 'KCNMB1', 'KCTD1', 'KIDINS220', 'KLHDC10', 'LAMA2', 'LGALSL', 'LGR6', 'LIMS2', 'LIN28A', 'LINC03072', 'LMCD1', 'LSP1P5', 'MAGI3', 'MAPK7', 'MBOAT2', 'MDFI', 'MMGT1', 'MPRIP', 'MSRB3', 'MTSS1', 'MXRA7', 'MXRA8', 'MYH10', 'MYH9', 'MYL9', 'MYLK', 'NAV3', 'NCOA1', 'NDUFV1', 'NME7', 'NORAD', 'NOSTRIN', 'NPR2', 'NPR3', 'OBSL1', 'PALM2AKAP2', 'PARVA', 'PCDH18', 'PDE5A', 'PDGFD', 'PERP', 'PICALM', 'PLSCR3', 'PPP1R1B', 'PRICKLE1', 'PTPN14', 'PTPN21', 'PTPRZ1', 'RABGAP1', 'RCAN1', 'RFLNB', 'RGS4', 'RGS5', 'RIT1', 'RTN4', 'RUNX1', 'SAV1', 'SEMA5A', 'SH3PXD2A', 'SLC4A3', 'SLIT3', 'SLX4IP', 'SMIM10L2A', 'SMTN', 'SNAI2', 'SNHG14', 'SNRPN', 'SNX21', 'SORT1', 'SPEG', 'SPOCK1', 'SPRED2', 'SRPX2', 'STARD13', 'TAGLN', 'TGFB1I1', 'THBS4', 'TIMP3', 'TLN2', 'TMEM134', 'TMEM192', 'TMEM61', 'TOP6BL', 'TPM2', 'TTC28', 'TTC3', 'TUBG2', 'UACA', 'VPS37D', 'VSNL1', 'ZBTB43', 'ZCCHC24', 'ZFYVE1', 'ZNF20', 'ZNF385D', 'ZNF426', 'ZNF727']
Refined Community 2: ['ABCA3', 'ABCC4', 'ACSL5', 'ADAM8', 'ADORA2A', 'AFG2B', 'AGMAT', 'AIFM2', 'AIM2', 'ANKRD22', 'ANXA7', 'APOE', 'ASPHD2', 'ATP8A1', 'ATXN7', 'BHLHA15', 'BIRC3', 'BMAL2', 'BTN3A1', 'BTN3A3', 'BUB3', 'C1orf174', 'C1RL', 'C1RL-AS1', 'C1S', 'C2CD2', 'C9orf72', 'CALHM6', 'CASP10', 'CASS4', 'CBS', 'CCDC69', 'CCDC88C', 'CD274', 'CD38', 'CDCA3', 'CDCP1', 'CEBPD', 'CENPA', 'CEP55', 'CGAS', 'CHCHD1', 'CLEC2D', 'CLEC7A', 'CTSS', 'CUL2', 'CYBA', 'DAZAP1', 'DDX50', 'DESI1', 'DHX34', 'DNAJC9', 'DRAM1', 'DTX3L', 'E2F5', 'EEF1AKMT4', 'ELL', 'EME2', 'EMG1', 'EPB41', 'ERAP1', 'ERO1B', 'ETV6', 'ETV7', 'FAM53B', 'FBL', 'FBRSL1', 'FERRY3', 'FOXJ2', 'FOXM1', 'GABBR1', 'GABPB1', 'GALM', 'GBP4', 'GBP5', 'GLRX', 'GRK3', 'GRWD1', 'GTSE1', 'GYG1', 'GZMA', 'HCP5', 'HEATR1', 'HERPUD1', 'HLA-DOB', 'HSH2D', 'HSPA4', 'ICAM1', 'IFI30', 'IGLC2', 'IL12RB1', 'IL15', 'IL15RA', 'IL27RA', 'IL32', 'INTS13', 'IQCG', 'IRF1', 'ISCU', 'ITGB7', 'KBTBD8', 'KLRC1', 'LAG3', 'LAP3', 'LARS1', 'LINC00869', 'M6PR', 'MAGOHB', 'MAP2K1', 'MCM10', 'MCMBP', 'ME2', 'MICB', 'MINPP1', 'MIR155HG', 'MKI67', 'MRTO4', 'MYNN', 'NBN', 'NCF1', 'NFE2L3', 'NFKB2', 'NFKBIE', 'NME3', 'NOC3L', 'NOP16', 'NOP2', 'NR1H3', 'NSMCE4A', 'NUP93', 'PARP8', 'PARVB', 'PCOLCE2', 'PCSK6', 'PHB2', 'PHF19', 'PIM2', 'PLCL2', 'PLGRKT', 'PNKD', 'POGLUT1', 'PPA1', 'PPIF', 'PPP1R15B', 'PQBP1', 'PRR13', 'PSAP', 'PSMA5', 'PSMB10', 'PSMB2', 'PSMB8', 'PSMB9', 'PSMG4', 'PTPN6', 'PUS1', 'RAP2B', 'RARS1', 'RASGEF1A', 'RCL1', 'RELB', 'RFX5', 'RGS1', 'RILPL2', 'RIMKLB', 'RNF207', 'RPS21', 'SCML1', 'SEC23IP', 'SGPL1', 'SIAH2', 'SKAP1', 'SLC2A6', 'SLC4A1AP', 'SLC7A1', 'SNHG12', 'SNHG26', 'SOCS3', 'SOD2', 'SRGN', 'STAMBPL1', 'STAT1', 'STYK1', 'TAGAP', 'TAP1', 'TAPBPL', 'TCOF1', 'TEAD4', 'TIAL1', 'TLR2', 'TMX2', 'TNFAIP2', 'TNFRSF14', 'TNFRSF1B', 'TRABD', 'TRAF3', 'TRAFD1', 'TREX1', 'TYMP', 'TYSND1', 'VCAM1', 'WARS1', 'XPNPEP1', 'YBX3', 'ZCCHC9', 'ZFP42', 'ZMIZ2', 'ZNF512B']
Refined Community 3: ['FAT2', 'FLT1', 'HEY1', 'HEY2', 'HEYL', 'JAG1', 'KIT', 'KRT14', 'KRT5', 'MT1X', 'NPR3', 'NRARP', 'PDGFRB', 'PLAT', 'PRELP', 'RHOV', 'SEMA5B', 'TNFRSF19', 'VSNL1', 'ZNF469']
Refined Community 4: ['ADAMTS12', 'AK5', 'ALPK2', 'APLP1', 'ARHGAP17', 'ARHGEF40', 'ARNT2', 'BCAT1', 'BDNF', 'CCDC74A', 'CD99', 'CIBAR1', 'CMTM3', 'CNKSR3', 'COL6A2', 'DPY19L1', 'EDIL3', 'FAM171A1', 'FLNC', 'FZD2', 'GFPT2', 'GNG11', 'GRB10', 'GRK5', 'GULP1', 'IL11', 'KIF3C', 'KLF2', 'LOXL2', 'MAP1B', 'MB21D2', 'MMP16', 'NUAK1', 'PDCL3', 'PFKP', 'PNMA2', 'RBM24', 'RGS7', 'RNF182', 'SCG2', 'SEMA3A', 'SH2B3', 'SHOX2', 'STRADB', 'TNIP1', 'TRAF3', 'TRIO', 'TTC28', 'TUBB2A', 'UTRN', 'ZCCHC24', 'ZEB1']
Refined Community 5: ['ACSL1', 'ADGRF1', 'AKR1B10', 'APRT', 'ATP1A1', 'BLNK', 'BSPRY', 'C1orf116', 'C6orf132', 'CBLC', 'CCNL1', 'CD24', 'CD24P2', 'CD24P4', 'CDH1', 'CDH3', 'CDS1', 'CKMT1B', 'CLCA2', 'COX5B', 'CRYBG1', 'DMAC1', 'DMKN', 'DRAM2', 'DSG3', 'DUSP16', 'EHF', 'ELF1', 'ELF3', 'ELMO3', 'EMC10', 'ENTPD2', 'EPB41L4A', 'EPHA1', 'EPN3', 'ESRP1', 'ESRP2', 'F11R', 'FAM83B', 'FGFBP1', 'FOSB', 'FXYD3', 'GAREM1', 'GJB3', 'GRAMD2A', 'GRHL2', 'GSN', 'HOOK2', 'IRF6', 'ITGB6', 'ITPA', 'KCNK1', 'KIAA0040', 'KLK5', 'KLK6', 'KRT15', 'KRT16', 'KRT5', 'LAD1', 'LCN2', 'LIMK2', 'LITAF', 'MACC1', 'MAF', 'MAL2', 'MARVELD2', 'MBNL3', 'MIR205', 'MPP7', 'MPZL3', 'MRFAP1L2', 'MUC1', 'MYO5B', 'NHSL3', 'OSER1', 'OVOL1', 'OVOL2', 'PDZK1IP1', 'PKP3', 'PPL', 'PPP1R14C', 'PROM2', 'PRR15L', 'PRRG4', 'PRSS8', 'PTPN6', 'RAB25', 'RBL2', 'RHOD', 'RNF144B', 'RNF39', 'RNF43', 'RTKN', 'S100A14', 'S100A8', 'S100P', 'SCNN1A', 'SERPINA3', 'SETD6', 'SH2D3A', 'SH3YL1', 'SKIC8', 'SLC11A2', 'SLC49A4', 'SLPI', 'SPINT1', 'SPINT1-AS1', 'SPINT2', 'SSH3', 'ST14', 'STEAP4', 'SUSD6', 'SYTL1', 'TACSTD2', 'TC2N', 'TLCD1', 'TMEM134', 'TMEM183A', 'TMEM191A', 'TMEM30B', 'TMEM87A', 'TRIM29', 'TSC22D3', 'TSTD1', 'ZFP36', 'ZNF165']
Refined Community 6: ['ABCD3', 'ACTN1', 'ADA', 'ADD3', 'ADGRE5', 'ADM', 'ADORA2B', 'ADRB2', 'AGPS', 'AKR1B1', 'AKR1B10', 'AKR1C1', 'AKR1C3', 'AKT3', 'ALDH1A3', 'ALDH3A2', 'AMD1', 'ANKH', 'ANKRD33B', 'ANTXR1', 'ANXA1', 'ANXA2', 'ANXA2P2', 'ANXA3', 'ANXA4', 'ANXA8', 'APOL6', 'ARAP3', 'ARHGAP23', 'ARHGAP5', 'ARHGEF28', 'ARPC2', 'ASXL1', 'ATP10D', 'ATP1A1', 'ATP1B3', 'AXL', 'B2M', 'B3GNT5', 'BICC1', 'BICD2', 'BIN1', 'BIRC3', 'BMAL2', 'BMP1', 'BNC1', 'BTG3', 'BTN3A2', 'BTN3A3', 'C1R', 'C1S', 'C3', 'CALD1', 'CAMTA1', 'CARD6', 'CASP1', 'CASP4', 'CAV1', 'CAV2', 'CAVIN1', 'CAVIN3', 'CBR1', 'CCDC28A', 'CCDC80', 'CCDC82', 'CCDC88A', 'CCDC9B', 'CCNA1', 'CCNYL1', 'CD109', 'CD14', 'CD44', 'CD58', 'CD59', 'CDC42EP3', 'CDCP1', 'CDH3', 'CDK6', 'CEBPD', 'CFB', 'CFI', 'CFL2', 'CFLAR', 'CHMP1B', 'CHST3', 'CIBAR1', 'CLIC4', 'CLIP4', 'CLMP', 'CMPK1', 'COL4A1', 'COL4A2', 'COL8A1', 'COPS8', 'CORO1C', 'COTL1', 'CRIPT', 'CRK', 'CRYAB', 'CRYBG1', 'CSNK2A2', 'CTSC', 'CXCL1', 'CXCL2', 'CXCL3', 'CYB5R3', 'CYBRD1', 'CYLD', 'DCBLD1', 'DCBLD2', 'DCTD', 'DEPP1', 'DGKA', 'DIPK1A', 'DMD', 'DNAJB4', 'DOCK5', 'DPYD', 'DSC3', 'DSE', 'DSG2', 'DSG3', 'DST', 'DUOX1', 'EGFR', 'EHBP1', 'ELK3', 'ELL2', 'EMP1', 'EMP3', 'EPHA2', 'ERAP2', 'EREG', 'ESYT2', 'ETF1', 'ETS1', 'ETS2', 'EVA1A', 'EVA1C', 'EXT1', 'F2RL1', 'F3', 'FAM171A1', 'FAM83A', 'FAM83D', 'FAP', 'FAS', 'FBLIM1', 'FDFT1', 'FERMT1', 'FGF2', 'FGFBP1', 'FHL1', 'FKBP1A', 'FMNL2', 'FNDC3B', 'FOSL1', 'FOXQ1', 'FRMD6', 'FSCN1', 'FST', 'FSTL1', 'FXYD5', 'FZD6', 'GABRE', 'GALNT2', 'GART', 'GAS1', 'GBP1', 'GBP3', 'GFOD1', 'GJB3', 'GJC1', 'GLIPR1', 'GM2A', 'GNA15', 'GNAI1', 'GNAL', 'GNG12', 'GPM6B', 'GPSM2', 'GPX8', 'GSTP1', 'GTF2B', 'HIF1A', 'HLA-E', 'HMGA2', 'HOTAIRM1', 'HOXA1', 'HOXA3', 'HOXA5', 'HP1BP3', 'HRCT1', 'HRH1', 'HSD17B11', 'HTRA1', 'ICAM1', 'IFI16', 'IFI27', 'IFI44', 'IFIT3', 'IFITM3P7', 'IFNGR1', 'IGF2BP2', 'IGF2BP3', 'IGFBP6', 'IGFBP7', 'IL15', 'IL18', 'IL1A', 'IL1RAP', 'IL20RB', 'IL7R', 'INHBA', 'INPP1', 'IRS2', 'IRX1', 'ITGA1', 'ITGA6', 'ITGB1', 'ITGB8', 'ITM2C', 'ITPRID2', 'JAG1', 'KIF1B', 'KIRREL1', 'KLF5', 'KLHL29', 'KLK10', 'KLK5', 'KPNA1', 'KRT14', 'KRT15', 'KRT16', 'KRT17', 'KRT5', 'KRT6A', 'KRT6B', 'LAMA3', 'LAMB3', 'LAMC1', 'LAMC2', 'LARP6', 'LBH', 'LINC00511', 'LINC01133', 'LIPG', 'LOX', 'LOXL2', 'LUZP1', 'LY6K', 'LYN', 'MAML2', 'MAP4K4', 'MAP7D3', 'MBNL1', 'MBNL2', 'MBP', 'MDFIC', 'MDH1', 'MET', 'MFGE8', 'MICALL1', 'MIR100HG', 'MIR22HG', 'MIR31HG', 'MMADHC', 'MME', 'MMP14', 'MPZL1', 'MSN', 'MT1F', 'MT1G', 'MT1H', 'MT1X', 'MT2A', 'MTMR2', 'MYL12A', 'MYO1B', 'MYO1E', 'NAB1', 'NABP1', 'NAMPT', 'NAP1L1', 'NAV2', 'NCK1', 'NDEL1', 'NDFIP2', 'NFAT5', 'NFE2L2', 'NKX1-2', 'NMI', 'NNMT', 'NOB1', 'NR3C1', 'NRP1', 'NSFL1C', 'NT5E', 'NUDT15', 'NUP50', 'NXN', 'OGFRL1', 'ORMDL1', 'OSBPL3', 'OSBPL9', 'OSMR', 'OXR1', 'P3H2', 'PALM2AKAP2', 'PARP4', 'PDGFC', 'PDP1', 'PDZK1IP1', 'PERP', 'PGM2', 'PHLDB2', 'PI3', 'PIK3CD', 'PKN2', 'PKP2', 'PLA2G4A', 'PLAT', 'PLAU', 'PLS3', 'PLSCR1', 'PM20D2', 'PNLIPRP3', 'PPP1R14C', 'PPP4R1', 'PRKD3', 'PRNP', 'PRSS12', 'PSAT1', 'PSMB8', 'PSMB9', 'PTGS2', 'PTK7', 'PTPN2', 'PTPRM', 'PXDC1', 'RAC2', 'RALB', 'RALBP1', 'RBFOX2', 'RBM7', 'RBMS1', 'RBMS3', 'RETSAT', 'REXO2', 'RFLNB', 'RGCC', 'RGL1', 'RGS2', 'RGS20', 'RHOA', 'RIOK3', 'RIPK4', 'RND3', 'RNF145', 'RRAS2', 'RUNX3', 'S100A2', 'SAA1', 'SAMD9', 'SCHIP1', 'SCPEP1', 'SEL1L3', 'SELENOF', 'SEPTIN10', 'SERPINB2', 'SERPINB5', 'SERPINE2', 'SFN', 'SFRP1', 'SGK1', 'SH3D19', 'SH3GLB1', 'SH3KBP1', 'SIRPA', 'SIRPAP1', 'SKAP2', 'SLC16A1', 'SLC16A7', 'SLC1A3', 'SLC25A37', 'SLC49A4', 'SLC6A15', 'SLC9A6', 'SLPI', 'SMAD3', 'SMCHD1', 'SNAI2', 'SNX7', 'SOAT1', 'SOX7', 'SP100', 'SPATS2L', 'SPRY2', 'SPTBN1', 'SPX', 'SRI', 'SRPX', 'SRPX2', 'ST3GAL6', 'STAMBPL1', 'STAT3', 'STAT4', 'STING1', 'STK17A', 'SVIL', 'TAP2', 'TAX1BP3', 'TBC1D1', 'TBPL1', 'TCEAL9', 'TGFA', 'TGFBI', 'TGFBR2', 'TKT', 'TLE4', 'TLR2', 'TM2D1', 'TMED5', 'TMEM154', 'TMEM245', 'TMEM30A', 'TMEM35B', 'TNFAIP3', 'TNFAIP8', 'TNFRSF10D', 'TOX2', 'TP63', 'TRIM22', 'TRIM29', 'TRIP10', 'TRMT6', 'TUBA4A', 'TUBB6', 'TWIST2', 'TWSG1', 'UBASH3B', 'UBE2E3', 'UPP1', 'VAMP3', 'VSNL1', 'WDR1', 'WLS', 'WWTR1', 'YAP1', 'YBX1', 'YBX3', 'YES1', 'ZBTB16', 'ZBTB38', 'ZC3H12C', 'ZDHHC2']
Refined Community 7: ['ABCA3', 'ABCG1', 'ABHD11', 'ABHD12', 'ACVR1B', 'ADCY6', 'ADGRB2', 'AFF3', 'AGR2', 'AMZ1', 'ANKRD13D', 'ANKRD30A', 'ANXA6', 'ANXA9', 'AR', 'ARF3', 'ARFGEF3', 'ARFIP2', 'ARHGEF26', 'ARID2', 'ARID3A', 'ARRB1', 'ASB8', 'ASH1L', 'ASTN2', 'ATP2C2', 'ATP6AP1', 'ATP6V0E2', 'ATP8B1', 'ATXN7L3B', 'AVL9', 'BAZ2A', 'BCAS1', 'BCOR', 'BLNK', 'BPTF', 'C14orf132', 'C17orf58', 'C4orf19', 'C9orf152', 'CA12', 'CACFD1', 'CACNA1D', 'CACNA2D2', 'CACNB3', 'CACNG4', 'CACYBP', 'CADM1', 'CAMSAP3', 'CANT1', 'CAPN9', 'CCDC117', 'CCND1', 'CCT6P3', 'CDC42SE1', 'CDYL2', 'CEP350', 'CERS2', 'CERS6', 'CFD', 'CHDH', 'CHN2', 'CHTOP', 'CIRBP', 'CISH', 'CLSTN2', 'CRACD', 'CREB3L1', 'CREB3L4', 'CRNKL1', 'CSNK1D', 'CSRNP2', 'CTNND2', 'CTXN1', 'CXXC5', 'CYB561', 'CYBC1', 'DAAM1', 'DACH1', 'DDAH2', 'DDX42', 'DEGS2', 'DENND1A', 'DENND4B', 'DEPTOR', 'DHRS13', 'DIP2C', 'DLG3', 'DNAJA4', 'DNAJC1', 'DNALI1', 'DOP1B', 'DSCAM-AS1', 'DUSP8', 'EFR3B', 'EIF3B', 'ELAPOR1', 'ELL3', 'ELOVL2', 'EMP2', 'ENPP1', 'ENSG00000280119', 'ENTR1', 'EPN3', 'EPS8L1', 'ERBB3', 'ERGIC1', 'ESR1', 'ETNK2', 'EVL', 'F7', 'FAM110B', 'FAM234B', 'FBRSL1', 'FGFR4', 'FKBP4', 'FOXA1', 'FRMD4A', 'FRS2', 'FTX', 'FUS', 'FZD4', 'GALNT6', 'GAMT', 'GARNL3', 'GARS1', 'GART', 'GATA3', 'GATA3-AS1', 'GGA1', 'GGA3', 'GOLT1A', 'GP1BB', 'GPD1L', 'GPR160', 'GPRC5C', 'GRAMD4', 'GSE1', 'GSPT1', 'GTF3C1', 'HECTD4', 'HEXD', 'HID1', 'HIP1R', 'HK2', 'HMG20B', 'HMGCS2', 'HNRNPA2B1', 'HPN', 'HPX', 'ICA1', 'INHBB', 'INPP5J', 'INTS15', 'IQCE', 'IQSEC1', 'IRGQ', 'ISG20', 'IVD', 'KAT6B', 'KATNIP', 'KCTD15', 'KDELR2', 'KDM4B', 'KDM7A', 'KIAA0040', 'KIAA0232', 'KIF12', 'KIFC2', 'KLF2', 'KLHDC9', 'KLHL22', 'KMT2D', 'KRT19', 'LARGE1', 'LARP4B', 'LCOR', 'LFNG', 'LIN7A', 'LINC01128', 'LLGL2', 'LMCD1', 'LMNTD2-AS1', 'LNX1', 'LONRF2', 'LRP3', 'LRRN1', 'LUC7L3', 'LZTR1', 'MACO1', 'MAGED2', 'MAPK9', 'MAPT', 'MARS1', 'MB21D2', 'MCCC2', 'MDM4', 'MEGF9', 'MGAT4A', 'MGRN1', 'MIF4GD', 'MLPH', 'MTCL2', 'MTERF2', 'MXRA8', 'MYB', 'MYCN', 'MYEF2', 'MYO5B', 'MYO6', 'NACA', 'NDUFS8', 'NECTIN2', 'NEK5', 'NHERF1', 'NKAIN1', 'NLK', 'NME3', 'NPDC1', 'NUCB2', 'NUDT4', 'ONECUT2', 'P4HTM', 'PATZ1', 'PBX1', 'PCBP2', 'PCK2', 'PCP4', 'PDCL3', 'PGGT1B', 'PGR', 'PI4KA', 'PKP4', 'PLA2G12A', 'PLCXD1', 'PLEKHH1', 'PLXNA3', 'POGZ', 'POLE', 'POMT1', 'PPFIA1', 'PPP1R16A', 'PPP2R2C', 'PREX1', 'PRKAG1', 'PRLR', 'PRR14', 'PRR36', 'PRRC2C', 'PRRT2', 'PRRT3', 'PYCR1', 'RAB11FIP3', 'RAB17', 'RAB3D', 'RAB40C', 'RABEP2', 'RALGAPA1', 'RALGPS1', 'RBAK', 'RDH13', 'REEP5', 'RGL2', 'RHBDF1', 'RHOB', 'RHPN1', 'RIIAD1', 'RIPOR3', 'RND1', 'RNF103', 'RNU6-1016P', 'RSAD1', 'RSPH1', 'RUBCN', 'RUSC1', 'SBK1', 'SCUBE2', 'SCYL3', 'SEC16A', 'SECISBP2', 'SERF2', 'SFI1', 'SFMBT2', 'SH3GLB2', 'SHANK2', 'SHTN1', 'SIDT1', 'SIDT2', 'SLC16A6', 'SLC1A4', 'SLC24A3', 'SLC25A29', 'SLC25A44', 'SLC26A11', 'SLC27A3', 'SLC2A10', 'SLC35A1', 'SLC37A1', 'SLC38A1', 'SLC38A10', 'SLC44A4', 'SLC4A8', 'SLC7A8', 'SMARCC2', 'SMIM14', 'SNED1', 'SNX27', 'SOX12', 'SOX13', 'SPATA2L', 'SPDEF', 'SPMIP5', 'SPTLC2', 'SRRM2', 'STARD10', 'STRADA', 'STRBP', 'SYCP2', 'SYNE4', 'SYNGR2', 'TADA2B', 'TAPT1', 'TBC1D16', 'TBC1D30', 'TBL1X', 'TBX3', 'TC2N', 'TCAF1', 'TENT5C', 'TESK1', 'TESMIN', 'TFF1', 'TFF3', 'TGFB3', 'TGIF2', 'THSD4', 'THUMPD1', 'TJP3', 'TLE3', 'TMBIM6', 'TMEM150C', 'TMEM184A', 'TMEM229B', 'TMEM268', 'TMEM276', 'TMEM80', 'TNIP1', 'TNRC18', 'TOB1', 'TOMM70', 'TRAPPC9', 'TRIL', 'TRIM3', 'TRPS1', 'TSPAN13', 'TSPAN15', 'TTC3', 'TTC39A', 'TTC6', 'TTC9', 'UAP1L1', 'UBN1', 'ULK1', 'USP3', 'USP42', 'USP7', 'VIPR1', 'VPS37C', 'VPS72', 'WFS1', 'ZBTB42', 'ZFYVE16', 'ZMIZ1', 'ZNF12', 'ZNF24', 'ZNF296', 'ZNF398', 'ZNF444', 'ZNF467', 'ZNF703', 'ZNF704', 'ZNF74', 'ZNF84', 'ZSWIM8']
Refined Community 8: ['AAK1', 'ABCC4', 'ACTA2', 'ACTG1', 'ACTN1', 'ACTR2', 'ADAMTS12', 'ADAMTS5', 'ADD3', 'ADGRE5', 'ADORA2B', 'AGPS', 'AHNAK2', 'AIDA', 'AK5', 'AKAP12', 'AKR1B1', 'AKR1C3', 'AKT3', 'ALKBH5', 'ALPK2', 'ANKH', 'ANKRD28', 'ANKRD33B', 'ANLN', 'ANTXR1', 'ANTXR2', 'ANXA1', 'ANXA2', 'ANXA2P2', 'ANXA5', 'AP1S2', 'APCDD1L', 'ARAP3', 'ARHGAP21', 'ARHGAP23', 'ARHGEF40', 'ARSJ', 'ASAP1', 'ASB1', 'ASXL1', 'ATM', 'ATP10D', 'AXL', 'B3GNT5', 'BACH1', 'BAG2', 'BASP1', 'BCAT1', 'BCHE', 'BDNF', 'BICC1', 'BICD2', 'BIN1', 'BIRC5', 'BMAL2', 'BNC2', 'BUB1', 'C17orf49', 'C1S', 'CALD1', 'CASP1', 'CAV1', 'CAV2', 'CAVIN1', 'CAVIN2', 'CAVIN3', 'CCBE1', 'CCDC50', 'CCDC82', 'CCDC88A', 'CD44', 'CD99', 'CDC20', 'CDC27', 'CDH11', 'CDK6', 'CDKN2C', 'CENPV', 'CEP170', 'CFL2', 'CHST2', 'CHST3', 'CHSY1', 'CIBAR1', 'CKAP2L', 'CLIP2', 'CLMP', 'CMTM3', 'CMTM7', 'CNRIP1', 'COL13A1', 'COL4A1', 'COL4A2', 'COL5A1', 'COL6A1', 'COL6A2', 'COL6A3', 'CORO1C', 'CORO2B', 'CPNE2', 'CRIM1', 'CRTAP', 'CSRP2', 'CTNNAL1', 'CTSC', 'CUL4B', 'CXCL2', 'CYP26B1', 'DAB2', 'DCBLD2', 'DENND5A', 'DEPDC1', 'DHX33', 'DIPK1A', 'DKK3', 'DMD', 'DNAJB4', 'DNM1', 'DOCK10', 'DPY19L1', 'DPYD', 'DSE', 'DZIP1', 'ECHDC1', 'EDIL3', 'EFEMP2', 'EGFR', 'EHBP1', 'EIF4A1', 'EIF5A2', 'ELAVL1', 'ELK3', 'ELP5', 'EMILIN2', 'EMP3', 'ENG', 'EPB41L2', 'EPHA2', 'ERFE', 'ETS1', 'ETV5', 'EVA1A', 'EXT1', 'EXTL2', 'EYA4', 'F2RL2', 'FAM171A1', 'FAM200B', 'FAM20C', 'FAS', 'FAT4', 'FBLIM1', 'FBN1', 'FER', 'FEZ2', 'FHL1', 'FHOD3', 'FLI1', 'FLNC', 'FLRT2', 'FMNL2', 'FOSL1', 'FOSL2', 'FOXC1', 'FOXF2', 'FOXQ1', 'FSCN1', 'FST', 'FSTL1', 'FXYD5', 'FZD2', 'FZD7', 'G0S2', 'GADD45A', 'GALNT2', 'GFPT2', 'GID4', 'GJA1', 'GLIPR1', 'GLIPR2', 'GLIS3', 'GLS', 'GNAI1', 'GNG11', 'GNG12', 'GNG2', 'GPR161', 'GPR176', 'GPRC5B', 'GPX8', 'GRK5', 'GSDME', 'GSTP1', 'GULP1', 'GXYLT2', 'HACD1', 'HAS2', 'HEG1', 'HJURP', 'HMGA2', 'HOTAIRM1', 'HRCT1', 'HRH1', 'HSD17B11', 'HTRA1', 'HYCC1', 'IFI16', 'IFNGR2', 'IGF2BP2', 'IGF2BP3', 'IGFBP6', 'IGFBP7', 'IGFBPL1', 'IKBIP', 'IL11', 'IL15', 'IL6', 'IL7R', 'ILF3', 'INCENP', 'INHBA', 'INPP1', 'IPO5', 'IRAK2', 'ITGA1', 'ITGAV', 'ITM2C', 'ITPRID2', 'ITPRIP', 'ITSN1', 'JAG1', 'KANK2', 'KATNAL1', 'KCTD12', 'KIF18B', 'KIF3C', 'KIRREL1', 'KLHL29', 'KPNB1', 'LAMB1', 'LAMC1', 'LARP6', 'LAYN', 'LBH', 'LDHB', 'LHFPL6', 'LIX1L', 'LOXL1', 'LOXL2', 'LPAR1', 'LRRK1', 'LTBP2', 'LUZP1', 'LYN', 'MALT1', 'MAML2', 'MAP1B', 'MAP4K4', 'MAP7D1', 'MAPRE1', 'MARVELD1', 'MCAM', 'MCFD2', 'MDFIC', 'MET', 'MFGE8', 'MID1', 'MIR100HG', 'MMP2', 'MPP1', 'MPRIP', 'MRC2', 'MSANTD3', 'MSN', 'MSRB3', 'MTCL1', 'MTMR2', 'MXRA7', 'MYLK', 'NAB1', 'NAP1L1', 'NAP1L5', 'NAV1', 'NAV3', 'NCAPG', 'NDC80', 'NDEL1', 'NECTIN3', 'NEXN', 'NID2', 'NMNAT2', 'NMT2', 'NNMT', 'NOL7', 'NOL9', 'NONO', 'NR2F1', 'NR2F1-AS1', 'NR3C1', 'NRP1', 'NT5E', 'NUP155', 'NUP88', 'OSMR', 'P3H1', 'P3H2', 'PABPC4', 'PAFAH1B1', 'PALM2AKAP2', 'PAPPA', 'PARVA', 'PDE3A', 'PDE4A', 'PDE7B', 'PDGFA', 'PDGFC', 'PDP1', 'PFAS', 'PFKP', 'PHLDB2', 'PICALM', 'PIK3CD', 'PIM1', 'PLAT', 'PLAU', 'PLK1', 'PLP2', 'PMP22', 'PNMA2', 'POGLUT3', 'POPDC3', 'PPP1R18', 'PPP4R1', 'PRKCA', 'PRKD3', 'PRNP', 'PROS1', 'PRR16', 'PRSS12', 'PSMB2', 'PSMB6', 'PSMB9', 'PSMD1', 'PTGS2', 'PTPRG', 'PTPRM', 'PTTG1', 'PTX3', 'PXDN', 'PYGL', 'QKI', 'RAB31', 'RAB34', 'RAB8B', 'RAI14', 'RALBP1', 'RBMS3', 'RFLNB', 'RFTN1', 'RGS2', 'RGS20', 'RGS4', 'RNF145', 'RNF182', 'RTN4', 'RUNX2', 'S100A2', 'SACS', 'SDCBP', 'SEMA3A', 'SEPTIN6', 'SERPINB1', 'SERPINE1', 'SERPINE2', 'SFPQ', 'SFRP1', 'SGCB', 'SGK1', 'SGTB', 'SH2B3', 'SH2D5', 'SH3GLB1', 'SH3KBP1', 'SHOX2', 'SKP2', 'SLC16A7', 'SLC25A37', 'SLC35B4', 'SLIT2', 'SMAD3', 'SMAD9', 'SNAI2', 'SNRPD1', 'SNX7', 'SOAT1', 'SOCS3', 'SPAG9', 'SPARC', 'SPATS2L', 'SPDL1', 'SPRY2', 'SPTBN1', 'SRGN', 'SRPX', 'ST3GAL6', 'STAMBPL1', 'STIL', 'STK10', 'STK17A', 'STRADB', 'STX2', 'SV2A', 'SYDE1', 'TAP2', 'TBC1D1', 'TCEAL9', 'TEAD1', 'TFPI', 'TGFB1', 'TGFB1I1', 'TGFB2', 'TGFBI', 'TGFBR2', 'TIMP1', 'TIMP2', 'TLCD3A', 'TMEM158', 'TMEM245', 'TMEM35B', 'TNFAIP3', 'TNFRSF10D', 'TOX', 'TOX2', 'TPM1', 'TRAF3', 'TRAM2', 'TRIO', 'TRNP1', 'TTL', 'TUBB', 'TUBB6', 'TWIST2', 'TWSG1', 'TYMS', 'UBASH3B', 'UBE2G1', 'UBE2S', 'UBLCP1', 'UPP1', 'VCL', 'VIM', 'VPS54', 'WASHC4', 'WIPF1', 'WLS', 'WNT5B', 'WRAP53', 'WWTR1', 'ZC3H12C', 'ZCCHC24', 'ZDHHC2', 'ZEB1', 'ZIC2', 'ZYG11B']
Refined Community 9: ['AAGAB', 'ABAT', 'ABCA12', 'ABCC11', 'ABCG1', 'ABHD11', 'ACVR1B', 'ADCY6', 'ADGRG1', 'ADIRF', 'AGR2', 'ALDH3B2', 'ALDH4A1', 'ALDH6A1', 'ANK3', 'ANKRD22', 'ANKRD30A', 'ANO1', 'ANXA9', 'AP1M2', 'APH1A', 'ARF3', 'ARFGAP2', 'ARFGEF3', 'ARFIP2', 'ARHGAP8', 'ARHGEF5', 'ARRDC1', 'ARRDC4', 'ASB8', 'ATOSA', 'ATP2C2', 'ATP6AP1', 'AZGP1', 'BBIP1', 'BCAS1', 'BCR', 'BEX5', 'BICDL1', 'BICDL2', 'BIK', 'BLNK', 'BLOC1S1', 'BLVRB', 'BOLA1', 'BSPRY', 'C11orf52', 'C1orf43', 'C4orf19', 'C6orf132', 'C9orf152', 'CA12', 'CACFD1', 'CACNA1D', 'CACNB3', 'CACNG4', 'CADPS2', 'CAMK2N1', 'CAMSAP3', 'CAPN8', 'CASZ1', 'CBLC', 'CCDC6', 'CCDC97', 'CD24', 'CD24P2', 'CD24P4', 'CDC42SE1', 'CDH1', 'CDH3', 'CDS1', 'CEACAM6', 'CEBPA', 'CERS6', 'CGN', 'CHASERR', 'CHMP2A', 'CHMP4C', 'CHN2', 'CIRBP', 'CKMT1B', 'CLDN3', 'CLDN4', 'CLDN7', 'CLN3', 'CNNM4', 'COG7', 'COMMD3', 'COP1', 'CRABP2', 'CRACDL', 'CREB3L4', 'CRIP1', 'CRIP2', 'CRNDE', 'CRYBG2', 'CSAD', 'CTNND2', 'CYB561', 'CYB5A', 'CYP4B1', 'DAAM1', 'DBP', 'DCAF11', 'DDI2', 'DDR1', 'DEGS2', 'DENND2D', 'DHRS13', 'DHRS4-AS1', 'DLG3', 'DNAJA4', 'DNAJC1', 'DOK7', 'DSCAM-AS1', 'DUSP16', 'EEIG1', 'EFHD1', 'EFNA4', 'EHF', 'ELAPOR1', 'ELF3', 'ELL3', 'ELMO3', 'EMP2', 'ENPP5', 'ENSA', 'ENSG00000280119', 'ENTPD2', 'EOLA1', 'EPB41L4A', 'EPB41L5', 'EPCAM', 'EPHA1', 'EPHB3', 'EPN3', 'EPPK1', 'EPS8L1', 'ERBB2', 'ERBB3', 'ERP29', 'ESR1', 'ESRP1', 'ESRP2', 'EXPH5', 'F11R', 'FA2H', 'FAAH', 'FAM110A', 'FAM174B', 'FAM83H', 'FANCF', 'FBP1', 'FCSK', 'FEM1B', 'FGD3', 'FKBP4', 'FLAD1', 'FOXA1', 'FRAT2', 'FREM2', 'FUT1', 'FXYD3', 'GALNT6', 'GAREM1', 'GATA3', 'GGCT', 'GGT6', 'GOLT1A', 'GPD1L', 'GPR157', 'GPR160', 'GPR89B', 'GRAMD4', 'GRB7', 'GRHL1', 'GRHL2', 'GRHL2-DT', 'GSPT1', 'GSTO2', 'H2AC18', 'H2AJ', 'HID1', 'HILPDA', 'HIP1R', 'HOOK1', 'HOOK2', 'ICA1', 'IDH2', 'IFT20', 'IGFBP2', 'INHBB', 'INPP5J', 'IQCE', 'IRF6', 'IRX5', 'ISG20L2', 'ITPK1', 'JMJD8', 'JTB', 'JUP', 'KATNIP', 'KDM7A', 'KIAA0040', 'KIAA0319L', 'KIFC2', 'KLHL28', 'KRT19', 'KRT23', 'LAD1', 'LIMK2', 'LLGL2', 'LMTK3', 'LNX1', 'LSR', 'LYPD3', 'MACC1', 'MAL2', 'MAP7', 'MARVELD2', 'MARVELD3', 'MB', 'MCCC2', 'MCF2L', 'MCRIP2', 'MED28', 'MEGF9', 'MGAT4A', 'MGMT', 'MINDY1', 'MIR29B2', 'MKNK2', 'MLPH', 'MMEL1', 'MPP7', 'MPZL3', 'MREG', 'MRFAP1L1', 'MRFAP1L2', 'MRPL41', 'MSX2', 'MUC1', 'MUCL1', 'MYB', 'MYH14', 'MYLIP', 'MYO1D', 'MYO5B', 'MYO5C', 'MYO6', 'NEBL', 'NECTIN4', 'NHERF1', 'NHSL3', 'NME3', 'NOL3', 'NPDC1', 'NR2F6', 'OCLN', 'OVOL1', 'OVOL2', 'P4HTM', 'PADI2', 'PAGR1', 'PAQR4', 'PATJ', 'PATZ1', 'PCDH1', 'PCK2', 'PDCD4', 'PDXDC1', 'PER2', 'PEX11B', 'PGAP2', 'PGAP3', 'PIP4K2C', 'PKP3', 'PLA2G12A', 'PLCH1', 'PLEKHA6', 'PLEKHF2', 'PLXNA3', 'PLXNB1', 'PPCS', 'PPDPF', 'PPFIBP2', 'PPM1H', 'PPP1R16A', 'PPP1R3D', 'PPP2R5A', 'PREX1', 'PRKD2', 'PRLR', 'PRODH', 'PROM2', 'PRR15', 'PRR15L', 'PRRG4', 'PRSS8', 'PTPN6', 'PWWP2B', 'RAB11FIP4', 'RAB17', 'RAB25', 'RAB27B', 'RAB3D', 'RAB40C', 'RAB5B', 'RABEP2', 'RALGPS1', 'RASEF', 'RBM47', 'REEP5', 'RERG', 'RGL2', 'RHBDF1', 'RHOB', 'RHOH', 'RHPN1', 'RIPOR3', 'RMND5B', 'RNF103', 'RNF144B', 'ROGDI', 'RPS6KA5', 'RSPH1', 'RTL10', 'RUSC1', 'S100A14', 'S100A8', 'S100P', 'SCAMP2', 'SCYL3', 'SELENBP1', 'SELENOP', 'SEMA3F', 'SEMA4A', 'SEPHS2', 'SERF2', 'SERINC2', 'SETD6', 'SFI1', 'SH2D3A', 'SH3YL1', 'SHANK2', 'SIDT1', 'SIGIRR', 'SLC16A14', 'SLC25A29', 'SLC29A2', 'SLC35A1', 'SLC37A1', 'SLC44A2', 'SLC52A3', 'SLC7A8', 'SMPDL3B', 'SORL1', 'SOX13', 'SPATA2L', 'SPDEF', 'SPINT1', 'SPINT1-AS1', 'SPINT2', 'SPOPL', 'SPTBN2', 'SPTLC2', 'SSBP2', 'SSH3', 'SSR4', 'ST14', 'ST6GALNAC2', 'STARD10', 'SUCO', 'SULT2B1', 'SUN1', 'SUOX', 'SUSD6', 'SYCP2', 'SYMPK', 'SYNE4', 'SYNGR1', 'SYNGR2', 'SYT7', 'SYTL1', 'TADA2B', 'TBC1D30', 'TC2N', 'TENT5C', 'TFF1', 'TFF3', 'TJP3', 'TLE3', 'TM7SF2', 'TMBIM4', 'TMBIM6', 'TMC4', 'TMED3', 'TMEM125', 'TMEM134', 'TMEM183A', 'TMEM184A', 'TMEM191A', 'TMEM229B', 'TMEM241', 'TMEM268', 'TMEM276', 'TMEM30B', 'TMEM41A', 'TMEM45B', 'TMEM79', 'TMPRSS13', 'TMPRSS2', 'TMT1A', 'TNK2', 'TNKS1BP1', 'TOB1', 'TOP6BL', 'TOX3', 'TP53TG1', 'TPD52L1', 'TRAF4', 'TRAPPC6A', 'TRIB3', 'TRIL', 'TRIM3', 'TRPM4', 'TRPS1', 'TSC22D3', 'TSPAN1', 'TSPAN13', 'TSTD1', 'TTC39A', 'TTC9', 'TUFT1', 'UBQLN4', 'ULK1', 'ULK3', 'USP18', 'VAV3', 'VIPR1', 'VPS45', 'VRK3', 'WFS1', 'ZBTB42', 'ZBTB7B', 'ZCCHC8', 'ZG16B', 'ZHX2', 'ZKSCAN1', 'ZNF165', 'ZNF24', 'ZNF385A', 'ZNF467', 'ZNF74', 'ZXDC']
Refined Community 10: ['ANKRD11', 'CDH1', 'CDH13', 'CDK11B', 'CDKN1C', 'CTSB', 'FANCA', 'HRAS', 'INS', 'LLGL1', 'PDIA3', 'PRKCZ', 'TOP3A']
Refined Community 11: ['AURKA', 'CCND1', 'CSE1L', 'CTTN', 'CYP24A1', 'E2F5', 'ERBB2', 'EXT1', 'FGF3', 'FGF4', 'LAMC2', 'LRRC32', 'MED1', 'MOS', 'MT-CO2', 'MYBL2', 'MYC', 'NCOA3', 'PAK1', 'PEBP1', 'PTGS2', 'PTK2', 'PTPN1', 'TGFB2', 'THRA', 'TNFRSF6B', 'TOP2A', 'ZNF217']
Refined Community 12: ['CASP9', 'CHRNA9', 'KIF13B', 'MTUS1', 'NRG1', 'PDGFRL', 'PPP2R1B', 'SOX7']
Refined Community 13: ['BCL6', 'CCND1', 'CTTN', 'EBAG9', 'ERBB2', 'ESRRG', 'EXT1', 'FGF3', 'FGFR1', 'JAG1', 'KCNQ3', 'MTSS1', 'MYC', 'PAK1', 'PREX1', 'PTPRT', 'RBBP5', 'SLA', 'SOX9', 'STK3', 'TG', 'WNT3', 'WNT9B']
Refined Community 14: ['ANP32E', 'ARTN', 'ATP11A', 'C16orf95', 'CALU', 'CHST3', 'CLIP4', 'CTSV', 'DEK', 'DSC2', 'EHBP1', 'FERMT1', 'FOXC1', 'FSCN1', 'KRT16', 'LMO4', 'MARCHF3', 'NNT', 'PAPSS1', 'PARVB', 'PLAUR', 'PLOD2', 'PMAIP1', 'POPDC3', 'PYGL', 'SGCB', 'SLC16A1', 'SMO', 'SPIDR', 'SPP1', 'TIMP2', 'WDR33', 'WWTR1']
Refined Community 15: ['ABCA12', 'ABCA8', 'ABCC6', 'AGR2', 'ALCAM', 'ALDH3B2', 'ALDH4A1', 'APOD', 'AR', 'AZGP1', 'BMERB1', 'BRINP3', 'C1orf115', 'CCNDBP1', 'CDK12', 'CEACAM6', 'CERS4', 'CIRBP', 'CLCA2', 'CPD', 'CRAT', 'CRISP3', 'CYB5A', 'CYP4F8', 'DCXR', 'DHRS2', 'ECHDC2', 'FA2H', 'FAM174B', 'FASN', 'FGFR4', 'GALNT6', 'GALNT7', 'GGT1', 'GPD1L', 'HGD', 'HMGCS2', 'HPX', 'KCNMA1', 'KMO', 'LASP1', 'LRIG1', 'LRRC31', 'MLPH', 'MPHOSPH6', 'NEIL1', 'NFIA', 'PBLD', 'PDCD4', 'PEX11A', 'PIP', 'PRLR', 'RASL10A', 'REEP5', 'RHOB', 'RND1', 'SCUBE2', 'SERHL', 'SERHL2', 'SIDT1', 'SLC16A2', 'SPDEF', 'SULT1A2', 'TBC1D9', 'TFAP2B', 'TFF3', 'TNIK', 'TNXA', 'TOX3', 'TRGC1', 'TRIL', 'TRIM3']
Refined Community 16: ['ART3', 'ASS1', 'BBOX1', 'BCL11A', 'BMAL2', 'CHST3', 'COL9A3', 'CRYAB', 'CXCL8', 'DSC2', 'EGFR', 'ELF5', 'EN1', 'FABP7', 'FOXC1', 'GABBR2', 'GABRP', 'GAL', 'GJB3', 'IGF2BP3', 'KRT16', 'KRT5', 'KRT6B', 'LY6D', 'MIA', 'MID1', 'MMP7', 'MSLN', 'PALS2', 'PAX6', 'PGBD5', 'PLAAT1', 'PSAT1', 'RARRES1', 'ROPN1', 'S100A8', 'SCRG1', 'SFRP1', 'SLPI', 'SOD2', 'SOX11', 'TTLL4', 'TTYH1', 'UGT8', 'UPP1', 'VGLL1', 'ZIC1']
Refined Community 17: ['ABAT', 'ACADSB', 'ACOX2', 'ADIRF', 'AGR2', 'ALCAM', 'ANXA9', 'AR', 'AREG', 'ARMT1', 'AZGP1', 'BCL2', 'BMPR1B', 'C1orf21', 'C4A', 'CA12', 'CACNA1D', 'CACNG4', 'CCDC170', 'CELSR1', 'CERS4', 'CFB', 'CFD', 'CHAD', 'CLGN', 'CLSTN2', 'CRIP1', 'CYBRD1', 'CYP2B6', 'CYP2B7P', 'DACH1', 'DHRS2', 'DNAJC12', 'DNALI1', 'DUSP4', 'EEF1A2', 'ELAPOR1', 'ELOVL2', 'EPS8L1', 'ERBB4', 'ESR1', 'EVL', 'FBP1', 'GALNT7', 'GAMT', 'GATA3', 'GDF15', 'GFRA1', 'GPC1-AS1', 'GPRC5A', 'GREB1', 'HEMK1', 'IFT122', 'IFT74', 'INPP4B', 'KCNE4', 'KRT18', 'MAP3K12', 'MAPT', 'MAST4', 'MLPH', 'MUC1', 'MYB', 'MYO6', 'NAT1', 'PBX1', 'PDZK1', 'PEX11A', 'PGR', 'PIERCE1', 'PIP', 'PNPLA4', 'PRR36', 'RET', 'RHOB', 'RND1', 'SCCPDH', 'SCGB1D2', 'SCGB2A2', 'SCNN1A', 'SCUBE2', 'SELENBP1', 'SEMA3B', 'SEMA3C', 'SERPINA5', 'SIDT1', 'SLC16A6', 'SLC27A2', 'SLC39A6', 'SLC44A4', 'SLC4A8', 'SLC7A8', 'SPDEF', 'SSH3', 'STC2', 'STK32B', 'SYBU', 'SYT1', 'SYT17', 'TBC1D9', 'TFF1', 'TFF3', 'TJP3', 'TMC5', 'TNNT1', 'TOX3', 'TPSAB1', 'TSPAN1', 'TTC39A', 'UGCG', 'VAV3']
Refined Community 18: ['ABCA12', 'ABCC5', 'ABCC6', 'ABHD11', 'ACACA', 'ACAD8', 'ACE2', 'ACOX2', 'ACSL1', 'ACSL3', 'ACSS3', 'ADCY7', 'AGPS', 'AK4', 'AKAP9', 'AKR1A1', 'AKR1D1', 'ALCAM', 'ALDH3B2', 'AMACR', 'ANKLE2', 'ANKRD27', 'ANP32E', 'APLP2', 'APOBEC3A', 'APOD', 'AR', 'ARFIP2', 'ATP11B', 'ATP6V1G1', 'ATP7A', 'AZGP1', 'BBS1', 'BCAP29', 'BCL11A', 'BLVRB', 'BMAL2', 'BMERB1', 'BRINP3', 'BTBD3', 'BTG3', 'C1QBP', 'CADPS2', 'CASD1', 'CCDC88A', 'CCL2', 'CCNDBP1', 'CDC20', 'CDC42EP3', 'CDH1', 'CDKN2A', 'CEP55', 'CERS4', 'CERS6', 'CES1', 'CHST1', 'CIRBP', 'CLCA2', 'CLDN8', 'CLDND1', 'CLGN', 'CLN3', 'CLU', 'CNTNAP2', 'CRAT', 'CRIPT', 'CRISP3', 'CROT', 'CTBS', 'CX3CL1', 'CYB561', 'CYB5A', 'CYP2J2', 'CYP4F8', 'CYP51A1', 'DALRD3', 'DAP', 'DBI', 'DCAF6', 'DCXR', 'DHCR24', 'DHRS2', 'DMD', 'DSC3', 'DUSP10', 'DUSP4', 'E2F3', 'ECHDC1', 'ECHDC2', 'EFHD1', 'EGF', 'ELAPOR1', 'ELOVL5', 'EN1', 'ENOSF1', 'ENSG00000291006', 'ERBB4', 'ERLIN2', 'ESRRG', 'ETFA', 'ETFB', 'FABP5', 'FAH', 'FAM174B', 'FAM216A', 'FAM234B', 'FASN', 'FBL', 'FCN2', 'FDFT1', 'FEM1C', 'FJX1', 'FMO5', 'FMR1', 'FOXC1', 'G6PD', 'GABRP', 'GALE', 'GALNT7', 'GFUS', 'GGT1', 'GGTLC1', 'GHR', 'GINS1', 'GLS', 'GNMT', 'GPD1L', 'GPM6B', 'GPR161', 'GPX7', 'GSE1', 'GSPT2', 'GSTZ1', 'HACD3', 'HERC5', 'HGD', 'HMGCR', 'HMGCS2', 'HMGN4', 'HOXB2', 'HPRT1', 'HSPA6', 'IDH1', 'IDH2', 'IDI1', 'IGF2BP2', 'INPP4B', 'KANK1', 'KCNMA1', 'KDM4B', 'KIF11', 'KIF14', 'KIF18B', 'KIF20A', 'KIF23', 'KIT', 'KLF5', 'KMO', 'KRT16', 'KRT18', 'KRT6A', 'KRT6B', 'KRT7', 'KYNU', 'LAMP3', 'LBR', 'LDHB', 'LIMA1', 'LIMCH1', 'LMO4', 'LONP2', 'LRBA', 'LRRC8D', 'LYN', 'MAGED2', 'MAOA', 'MAPT', 'MELK', 'MFAP2', 'MID1', 'MKI67', 'MKNK2', 'MLF1', 'MLPH', 'MPHOSPH6', 'MSMO1', 'MSX2', 'MTUS1', 'MYC', 'MYL6B', 'NAAA', 'NANS', 'NAV2', 'NCK1', 'NEDD4', 'NHERF1', 'NR2F1', 'NRAS', 'NT5C2', 'NUP88', 'OBI1', 'OPTN', 'P2RY2', 'P4HA1', 'PAM', 'PAPSS2', 'PCLO', 'PDCD5', 'PDLIM1', 'PELI1', 'PEX11A', 'PFDN4', 'PGRMC2', 'PHKB', 'PIK3R1', 'PIP', 'PKP1', 'PLAAT2', 'PLAGL1', 'PLCL2', 'PLEKHB1', 'PLSCR1', 'PMAIP1', 'PNISR', 'PON2', 'PPIC', 'PREP', 'PRKAA1', 'PRKACB', 'PRKAR2B', 'PRKX', 'PRLR', 'PROM1', 'PRR15L', 'PSIP1', 'PTK6', 'PUM3', 'QKI', 'QSOX1', 'RABEP2', 'RASGRP1', 'RBBP8', 'RBM47', 'REEP5', 'RETSAT', 'RGS10', 'RHOB', 'RND1', 'RNF138', 'RNF141', 'S100A13', 'S100A2', 'S100A4', 'S100A6', 'SAR1B', 'SC5D', 'SCD', 'SCHIP1', 'SCNN1A', 'SEC23B', 'SEL1L3', 'SELENOP', 'SEPHS2', 'SERHL', 'SERHL2', 'SH3BP4', 'SIDT1', 'SIGLEC15', 'SIK1', 'SLC16A1', 'SLC19A2', 'SLC22A18', 'SLC26A3', 'SLC2A10', 'SLC35A3', 'SLC38A1', 'SNRPD1', 'SOCS5', 'SPDEF', 'SPRED2', 'SPTLC2', 'SREBF1', 'STIL', 'STK26', 'STOML2', 'SWAP70', 'SYCP1', 'SYNCRIP', 'TAF1A', 'TAF5', 'TASOR2', 'TCF7L2', 'TFAP2B', 'TMEM135', 'TMEM158', 'TMEM41B', 'TMEM87A', 'TMPRSS2', 'TNFRSF21', 'TOP2A', 'TP53BP2', 'TP53TG1', 'TPD52', 'TPGS2', 'TRAK2', 'TRIL', 'TRIM36', 'TRMT11', 'TRPV6', 'TSC22D3', 'TSPAN13', 'TSPAN3', 'TTC13', 'TTLL12', 'TUBB6', 'TYMS', 'UBE2C', 'UBE2E3', 'UCP2', 'UGDH', 'UGT2B11', 'UPF3B', 'UTP25', 'VWA5A', 'YEATS2', 'ZBTB16', 'ZKSCAN1', 'ZMYND8', 'ZWILCH']
Refined Community 19: ['ABCA12', 'ABCC5', 'ABCC6', 'ABHD2', 'ACAA2', 'ACAD8', 'ACADM', 'ACE2', 'ACOT7', 'ACSL1', 'ACSL3', 'ADAM9', 'ADM', 'AGFG1', 'AGPS', 'AK4', 'AKR1A1', 'AKR1B10', 'AKR1D1', 'ALCAM', 'ALDH1A3', 'ALDH3B2', 'AMACR', 'ANKLE2', 'ANKRD27', 'ANP32E', 'ANXA9', 'APLP2', 'APOD', 'AR', 'ARL3', 'ARNT2', 'ASB13', 'ASS1', 'ATP7A', 'AZGP1', 'BAG1', 'BCL2', 'BLNK', 'BMI1', 'BNIP3', 'BRINP3', 'BST2', 'BTG2', 'C14orf132', 'C2CD5', 'C3orf52', 'CACNA2D2', 'CCDC6', 'CCNDBP1', 'CDKN1B', 'CES1', 'CHN2', 'CHST1', 'CHST15', 'CLCA2', 'CLDN1', 'CLDN8', 'CLIC3', 'CLSTN2', 'CLU', 'COL9A3', 'COMT', 'CPA3', 'CPQ', 'CRABP2', 'CROT', 'CRYBG1', 'CSGALNACT1', 'CSTB', 'CTBS', 'CXCL8', 'CYB5A', 'CYP2J2', 'DAP', 'DBI', 'DBNDD2', 'DCAF10', 'DDAH2', 'DDC', 'DHCR24', 'DHCR7', 'DIP2C', 'DNALI1', 'DUSP14', 'ECHDC1', 'EFHD1', 'EGFR', 'EGR3', 'EIF2AK2', 'EIF4EBP1', 'ELOVL2', 'ENO2', 'ERGIC2', 'ERO1A', 'ESR1', 'ETFA', 'ETFB', 'EVL', 'FABP7', 'FAM106A', 'FAR2', 'FASN', 'FCHSD2', 'FCN2', 'FDFT1', 'FGFR2', 'FKBP5', 'G6PD', 'GALE', 'GALNT3', 'GATA3', 'GFRA1', 'GFUS', 'GGT1', 'GGTLC1', 'GHR', 'GMPS', 'GNMT', 'GPRC5B', 'GREB1', 'HDGFL3', 'HEBP2', 'HGD', 'HMGCS1', 'HMGN4', 'HPRT1', 'HSD17B2', 'HSD17B4', 'IDH1', 'IDH2', 'IFI35', 'IGF1R', 'IGFBP4', 'IKBKB', 'IL6ST', 'INPP5F', 'KCNMA1', 'KIT', 'KMO', 'KPNB1', 'KRT7', 'KYNU', 'LAMB2', 'LBP', 'LDOC1', 'LIMCH1', 'MACIR', 'MANSC1', 'MAOA', 'MAOB', 'MAP2K6', 'MAP3K5', 'MARCHF3', 'MCCC1', 'MCF2L', 'MDM2', 'MEAK7', 'MED13', 'MED13L', 'MEIS3P1', 'MIA3', 'MICB', 'MKNK2', 'MLF1', 'MORC4', 'MPHOSPH6', 'MPHOSPH9', 'MPZL1', 'MSMO1', 'MSX2', 'MTFR1', 'MTUS1', 'MYH10', 'MYL6B', 'MYO10', 'MZT2A', 'NAAA', 'NAMPT', 'NBR1', 'NDNF', 'NDRG1', 'NME3', 'NOTCH2NLA', 'NPEPPS', 'NPY1R', 'NRAS', 'NUDT15', 'OASL', 'OAZ3', 'OPN3', 'OPTN', 'ORM1', 'P4HA1', 'PALS2', 'PAM', 'PAPSS2', 'PCDH7', 'PCSK6', 'PDLIM1', 'PDZD2', 'PDZK1IP1', 'PER3', 'PEX3', 'PFKM', 'PGK1', 'PGRMC1', 'PHKB', 'PI3', 'PIK3CB', 'PIP', 'PKP2', 'PLAAT1', 'PLAAT3', 'PLCH1', 'PLCL1', 'PLEKHB1', 'PMAIP1', 'PNISR', 'PNMA1', 'PNP', 'POLG2', 'PON2', 'PPIF', 'PRKAA1', 'PRKAR2B', 'PSIP1', 'PSME4', 'QPRT', 'QSOX1', 'RAB38', 'RABGAP1L', 'RALGPS1', 'RAMP1', 'RBBP8', 'REEP5', 'RESF1', 'RIOX2', 'RLN2', 'RMND1', 'RNF11', 'RNF141', 'RNF24', 'RTN1', 'RTP4', 'S100A13', 'S100A14', 'S100A8', 'S100A9', 'SAR1B', 'SCCPDH', 'SCD', 'SCGB1D2', 'SCUBE2', 'SEC23B', 'SEPHS2', 'SERHL', 'SERHL2', 'SERPINA5', 'SETMAR', 'SGMS1', 'SLC31A1', 'SLF2', 'SLPI', 'SMAD3', 'SMARCD3', 'SMARCE1', 'SMCO4', 'SMPDL3A', 'SPRED2', 'SPTLC2', 'SRD5A1', 'SREK1', 'SSR1', 'ST6GALNAC2', 'STEAP3', 'STK32B', 'STK39', 'STYK1', 'SUZ12', 'SWAP70', 'SYCP1', 'SYNGR3', 'TACC2', 'TASOR2', 'TBC1D9', 'TBL1X', 'TBX3', 'TESMIN', 'TFAP2B', 'TFF1', 'TFF3', 'THBS1', 'TK1', 'TLE1', 'TMEM87A', 'TMPRSS2', 'TMSB15A', 'TMT1A', 'TMX4', 'TNFSF4', 'TPGS2', 'TPSAB1', 'TRAF5', 'TRIM36', 'TRMT1L', 'TRPS1', 'TRPV6', 'TSPAN3', 'TSPAN8', 'TTYH1', 'TUBA4A', 'U2SURP', 'UCHL3', 'UCP2', 'UGT2B11', 'ULK2', 'VEGFA', 'VEZF1', 'VLDLR', 'WDR19', 'WIPF2', 'WWC3', 'YAP1', 'YBX1', 'ZDHHC17', 'ZDHHC4']
Refined Community 20: ['ABAT', 'ACADSB', 'ACOT2', 'ACTL6A', 'ADD2', 'ADIRF', 'ADM', 'AGR2', 'AHNAK', 'AKR7A3', 'ALDH6A1', 'AMACR', 'AMD1', 'ANXA9', 'AP1AR', 'APBB2', 'APPBP2', 'AR', 'AREG', 'ARHGEF9', 'ARMT1', 'ART3', 'ASF1A', 'ATG5', 'ATP7B', 'BBOF1', 'BBOX1', 'BBS1', 'BCL11A', 'BCL2', 'BCL2A1', 'BLVRA', 'BTG3', 'BUB1', 'CA12', 'CACNA2D2', 'CANT1', 'CAPN9', 'CASP8AP2', 'CCL2', 'CCNC', 'CCNE1', 'CD24P2', 'CD44', 'CDC20', 'CDK17', 'CDKN2A', 'CEBPB', 'CEBPG', 'CELSR1', 'CENPA', 'CERS6', 'CHEK1', 'CHI3L1', 'CHMP2A', 'CHST2', 'CILP', 'CIRBP', 'CLCN4', 'CLIC4', 'CORO1C', 'COX16', 'CPB1', 'CPD', 'CRIP1', 'CSRP2', 'CX3CL1', 'CXCL5', 'CXCL8', 'CYP2B6', 'CYP2B7P', 'DACH1', 'DALRD3', 'DCXR', 'DHRS2', 'DHRS7', 'DKC1', 'DMD', 'DNAJC1', 'DNAJC12', 'DNALI1', 'DSC2', 'DSC3', 'E2F3', 'ECI2', 'EFCAB11', 'EFHC1', 'EIF1AX', 'ELAPOR1', 'EN1', 'ERBB4', 'ESR1', 'EVL', 'FABP5', 'FAM171A1', 'FAM174B', 'FAM234B', 'FASN', 'FBP1', 'FBXL5', 'FBXL7', 'FMO5', 'FOXC1', 'FOXM1', 'FUT8', 'FYCO1', 'GABBR2', 'GABRP', 'GAL', 'GALNT10', 'GALNT6', 'GALNT7', 'GAMT', 'GATA3', 'GDF15', 'GFRA1', 'GLS', 'GMPS', 'GPC1-AS1', 'GPD1L', 'GPM6B', 'GPR161', 'GREB1', 'GSE1', 'GSTM3', 'GSTP1', 'GSTZ1', 'GTPBP4', 'H1-1', 'HDAC2', 'HEBP2', 'HHAT', 'HS3ST1', 'HSD17B4', 'HSPB1', 'IFRD1', 'IGF2BP2', 'IGFBP4', 'IL6ST', 'INPP4B', 'IRS1', 'ITGB5', 'KDM4B', 'KIAA0232', 'KIF14', 'KIF16B', 'KIF4A', 'KLF5', 'KLHL7', 'KRT16', 'KRT18', 'KRT6A', 'KRT6B', 'KRT8', 'LASP1', 'LBR', 'LDHB', 'LGALS8', 'LIMA1', 'LMO4', 'LONP2', 'LRBA', 'LRP12', 'LRRC17', 'LRRC8D', 'LYN', 'MACIR', 'MAGED2', 'MAGOH', 'MAPT', 'MARCO', 'MCCC2', 'MCM10', 'MCM5', 'MELK', 'MICALL1', 'MID1', 'MLPH', 'MMP12', 'MPZL2', 'MRFAP1L1', 'MRTFB', 'MSN', 'MTHFD2', 'MTMR2', 'MYB', 'MYC', 'MYO10', 'MYO5C', 'NAT1', 'NBEA', 'NCK1', 'NDC80', 'NDUFAF4', 'NFE2L3', 'NFIB', 'NFIL3', 'NHERF1', 'NME3', 'NQO1', 'NRTN', 'NT5C2', 'NUDT4', 'P4HTM', 'PAAF1', 'PALS2', 'PBX1', 'PDGFD', 'PDSS1', 'PDZK1', 'PELI1', 'PEX11A', 'PFKP', 'PHGDH', 'PI3', 'PKP1', 'PLAAT1', 'PLIN2', 'PNMA1', 'PPP1CB', 'PRDX4', 'PRKAR1A', 'PRKD3', 'PRKX', 'PRNP', 'PRR15L', 'PSME4', 'PTGER3', 'PTTG1', 'PTX3', 'PUM3', 'QKI', 'RABEP1', 'RAD51AP1', 'RALGAPA1', 'RARA', 'RARRES1', 'RBM47', 'REEP1', 'REEP5', 'RET', 'RETSAT', 'RGS5', 'RHOB', 'RIF1', 'RND1', 'RNF138', 'RNF43', 'RSU1', 'S100B', 'SCCPDH', 'SCHIP1', 'SCNN1A', 'SCUBE2', 'SEMA3C', 'SEMA3F', 'SEPHS1', 'SERHL', 'SERPINA5', 'SIDT1', 'SKP2', 'SLC16A1', 'SLC16A6', 'SLC19A2', 'SLC22A18', 'SLC2A10', 'SLC2A5', 'SLC35E2B', 'SLC39A6', 'SLC43A3', 'SLC44A4', 'SLC49A3', 'SLC9A6', 'SMCO4', 'SNRPD1', 'SOD2', 'SPATA20', 'SPDEF', 'SPRED2', 'SPTLC2', 'SREBF1', 'SRPK1', 'ST6GALNAC2', 'ST8SIA1', 'STEAP3', 'STIL', 'SYBU', 'SYNC', 'SYNCRIP', 'TBC1D12', 'TBC1D9', 'TBX3', 'TCEAL1', 'TCEAL4', 'TCF7L1', 'TESMIN', 'TFAP2A', 'TFF1', 'TFF3', 'TGFB3', 'TM7SF2', 'TMEM123', 'TMEM135', 'TOB1', 'TOM1L1', 'TPBG', 'TPX2', 'TRIM2', 'TRIM29', 'TRMT11', 'TSC22D3', 'TSPAN1', 'TSPAN13', 'TTC39A', 'TTK', 'TTLL4', 'TUBB6', 'U2SURP', 'UGCG', 'UGT8', 'VEZF1', 'VGLL1', 'VPS37C', 'WDR19', 'WFS1', 'WWP1', 'YBX1', 'YBX3', 'YEATS2']
Refined Community 21: ['CCR2', 'CCR5', 'CD2', 'CD38', 'CD3D', 'CD48', 'CD8A', 'CYTIP', 'FKBP11', 'GIMAP4', 'GMFG', 'GPR171', 'GPR18', 'GZMA', 'GZMK', 'HERC5', 'HERC6', 'IFI44', 'IFI44L', 'IFI6', 'IFIT1', 'IFIT3', 'IGHA1', 'IGHG1', 'IGHM', 'IGKV1D-13', 'IGKV3-20', 'IGLC2', 'IGLL3P', 'IGLV2-14', 'IL10RA', 'ISG15', 'ITK', 'JCHAIN', 'LCK', 'LCP2', 'MX1', 'MZB1', 'OAS1', 'OAS2', 'OAS3', 'OASL', 'POU2AF1', 'RSAD2', 'SH2D1A', 'TNFRSF17', 'TRBC2', 'XAF1']
Refined Community 22: ['ARMC1', 'ASPM', 'ATAD2', 'BUB1', 'BUB1B', 'CCNA2', 'CCNB2', 'CENPA', 'CENPE', 'CEP55', 'ESRP1', 'KIF11', 'KIF15', 'KIF20A', 'KIF4A', 'MAD2L1', 'MCM4', 'MELK', 'MRPL15', 'MTERF3', 'MTFR1', 'NCAPG', 'NDC80', 'NEK2', 'NUSAP1', 'PRC1', 'PRKDC', 'PTTG1', 'RAD54B', 'RB1CC1', 'RRS1', 'SLC25A32', 'TPX2']
Refined Community 23: ['ADM', 'AGFG1', 'CLDN1', 'EGFR', 'ERO1A', 'KYNU', 'MALL', 'NAMPT', 'NDRG1', 'PALS2', 'PDZK1IP1', 'PEX3', 'SLC31A1', 'SRD5A1', 'STEAP3', 'VLDLR']
Refined Community 24: ['ADAM12', 'AEBP1', 'CDH11', 'COL1A2', 'COL3A1', 'COL5A1', 'COL5A2', 'CTSK', 'DACT1', 'FAP', 'FBN1', 'FILIP1L', 'HTRA1', 'NID2', 'PCOLCE', 'SERPINF1', 'SPARC', 'SRPX2', 'THBS2']
Refined Community 25: ['CYB561', 'DCAF7', 'DHX40', 'FTSJ3', 'MED13', 'MTMR4', 'POLG2', 'PPM1D', 'PTRH2', 'RAD51C', 'RNF43', 'RPS6KB1', 'SMARCD2', 'SMG8', 'SUPT4H1', 'TACO1', 'TRIM37', 'TUBD1', 'VMP1']
Refined Community 26: ['AGR2', 'CERS6', 'ESR1', 'GATA3', 'GPD1L', 'KIAA0232', 'MLPH', 'RBM47', 'REEP5', 'RHOB', 'SLC39A6', 'TBC1D9', 'TSPAN13', 'TTC39A']
Refined Community 27: ['ALCAM', 'ALDH3B2', 'AR', 'BLVRB', 'CDH1', 'DHRS2', 'ECI2', 'ESRP2', 'FASN', 'GHR', 'GSE1', 'IRX5', 'MSX2', 'RND1', 'SCD', 'SLC35A3', 'SREBF1', 'TMEM135', 'TMEM41B', 'TRIM36']
Refined Community 28: ['CDK12', 'ERBB2', 'GRB7', 'GSDMB', 'MED1', 'PGAP3', 'PNMT']
Refined Community 29: ['ADM', 'ADRA2A', 'CD247', 'CD48', 'CD52', 'CD8A', 'CXCL14', 'F2RL2', 'FAXC', 'FRZB', 'GIMAP5', 'GZMA', 'HOXA10', 'ITGBL1', 'LCP1', 'MIR99AHG', 'OGN', 'PLAAT1', 'PLEK', 'RAI2', 'RUNX3', 'SLC40A1', 'SNTG2', 'SPP1', 'TRBV5-4', 'VGLL1']
Refined Community 30: ['ACVR1B', 'BMPR1B', 'DCLK1', 'EPHA10', 'ERBB3', 'ERBB4', 'FGFR3', 'GAK', 'IGF1R', 'IKBKB', 'MAP3K12', 'NEK11', 'NEK9', 'NRK', 'RET', 'STK32B', 'STK36', 'STK39', 'TBCK', 'ULK4', 'WNK4']
Refined Community 31: ['CDK17', 'CHUK', 'CSNK1A1', 'CSNK1G3', 'DYRK1A', 'HIPK1', 'MAP4K3', 'NEK4', 'PRP4K', 'SCYL2', 'SLK', 'SNRK', 'TLK1', 'TRIM33', 'TWF1']
Refined Community 32: ['BTK', 'FGR', 'HCK', 'ITK', 'JAK3', 'LCK', 'LYN', 'MAP4K1', 'MLKL', 'PIM2', 'PRKCB', 'PRKCQ', 'STK10', 'STK17B', 'SYK', 'ZAP70']
Refined Community 33: ['AURKA', 'AURKB', 'BUB1', 'BUB1B', 'CDC7', 'CDK1', 'CHEK1', 'MASTL', 'MELK', 'NEK2', 'PBK', 'PLK1', 'PLK4', 'SRPK1', 'TTK', 'VRK1']
Refined Community 34: ['ACE', 'ACSL5', 'AGO4', 'AKAP13', 'AMY2B', 'ANKLE2', 'ARHGAP17', 'ASMTL-AS1', 'ATXN7', 'BAG6', 'BRD2', 'BTAF1', 'BTN2A1', 'C2orf68', 'CALML4', 'CAMK2G', 'CAPN15', 'CASP4', 'CCDC77', 'CCDC93', 'CCNJ', 'CCNL1', 'CDC42SE1', 'CHD2', 'CHEK1', 'CHTOP', 'CLASRP', 'CLK3', 'CNTRL', 'COG3', 'CSNK2B', 'CYBA', 'CYTH1', 'DAPK2', 'DAZAP1', 'DDX11', 'DDX39B', 'DENND11', 'DGKZ', 'DHX16', 'DHX37', 'DMD', 'DNHD1', 'DNM2', 'DOCK7', 'DPH7', 'DXO', 'ECHDC1', 'EHBP1L1', 'EHMT2', 'EIF4G1', 'ENGASE', 'EPM2AIP1', 'EWSR1', 'EXOC3', 'FBXL18', 'FBXL19', 'FNBP4', 'FUBP1', 'FURIN', 'GBA2', 'GPC2', 'GRAMD4', 'GRIPAP1', 'GTPBP2', 'HCFC1', 'HINT1', 'HLX', 'HNRNPD', 'HNRNPH3', 'HSPBAP1', 'ITFG2', 'ITPR3', 'ITSN2', 'KDM4A', 'KHDC4', 'KIFC1', 'KMT2E-AS1', 'LARP1', 'LENG8', 'LIMD2', 'LINC01355', 'MAD2L1BP', 'MADD', 'MBD4', 'MCM3', 'MCM5', 'MIR3682', 'MIRLET7D', 'MKI67', 'MLXIP', 'MON2', 'MSH5', 'MSL2', 'MYO9B', 'NAA16', 'NPIPB3', 'NSUN4', 'NSUN5', 'NSUN5P1', 'NUP214', 'NUP62', 'OFD1', 'PABPN1', 'PARP6', 'PASK', 'PCGF6', 'PGS1', 'PHACTR4', 'PHF11', 'PLXND1', 'PML', 'PNN', 'PODXL', 'POGLUT1', 'POLR1A', 'POLR1HASP', 'PPARD', 'PPHLN1', 'PROSER1', 'PRRC2A', 'PSPC1', 'PTBP1', 'PTOV1-AS2', 'PUM1', 'QSOX2', 'QTRT2', 'RAB35', 'RABGAP1', 'RAPGEF1', 'RBM26', 'RNF220', 'RNPS1', 'RPS6KA1', 'RUBCN', 'SBNO2', 'SET', 'SF1', 'SFPQ', 'SH3BP5-AS1', 'SLC15A4', 'SLC18B1', 'SLC23A2', 'SLC25A25-AS1', 'SMARCD1', 'SMC4', 'SMC6', 'SNAPC4', 'SNHG26', 'SRSF2', 'ST14', 'SUPT20H', 'SUPT7L', 'SZT2', 'TAFAZZIN', 'TAPBP', 'TBRG1', 'TCIRG1', 'TCOF1', 'TEPSIN', 'TINAGL1', 'TJAP1', 'TLN1', 'TMC6', 'TMEM147-AS1', 'TNFAIP2', 'TNFRSF25', 'TNIP1', 'TOPBP1', 'TRA2A', 'TRRAP', 'TSPOAP1', 'TSPOAP1-AS1', 'TYW5', 'UQCC2', 'WDTC1', 'XPO5', 'XPO6', 'ZMIZ2', 'ZNF142', 'ZNF316', 'ZNF318', 'ZNF436-AS1', 'ZNF532']
Refined Community 35: ['ADCY1', 'ADD1', 'ADH4', 'ADNP', 'AP1AR', 'ARFGEF2', 'ARFGEF3', 'ATP2B4', 'ATP5F1E', 'AZI2', 'BAIAP3', 'BCAS4', 'BLOC1S6', 'BLVRB', 'BNIP3L', 'C20orf141', 'CACNG4', 'CAVIN2', 'CDS1', 'CFD', 'CTNNBL1', 'DAAM1', 'DDX27', 'DIXDC1', 'DMAC2', 'DMAC2L', 'DNAJB14', 'DNAJC21', 'DNAL1', 'DPM1', 'DSCR10', 'EFCAB11', 'ELP2', 'EMILIN3', 'ENPP1', 'ERGIC3', 'ESR1', 'F2R', 'FITM2', 'FNIP1', 'GINM1', 'GRID1', 'HSP90AA1', 'HSPB1', 'IFT22', 'ITPRIPL2', 'KCTD20', 'KIF5B', 'KLF9', 'KLHL8', 'LEPROT', 'LIN52', 'LINC00491', 'MAGT1', 'MAP3K12', 'MAPT', 'MLLT10', 'MORF4L2', 'MTG2', 'NCOA3', 'ODR4', 'PARD6B', 'PDP1', 'PDP2', 'PFDN1', 'POSTN', 'PPP1R3D', 'PRRC1', 'PSMA7', 'RAPH1', 'RBM3', 'RBM39', 'RDH11', 'RHOBTB3', 'RNASE4', 'ROMO1', 'RSPH3', 'SEC62', 'SERINC3', 'SLC39A6', 'SLC4A7', 'SMAD5', 'SMIM14', 'SNORA71B', 'SPATA24', 'SPPL2A', 'STAG2', 'STAU1', 'STX16', 'SYDE2', 'TAF1', 'TALAM1', 'TCAF1', 'TMBIM4', 'TMED4', 'TMEM218', 'TOX4', 'TPR', 'TSHZ2', 'UBBP1', 'UHMK1', 'UQCC1', 'YIPF5', 'YWHAB', 'ZBTB8A', 'ZNF148', 'ZNF217', 'ZNF552', 'ZNF587', 'ZNF91']
Refined Community 36: ['ABCA7', 'ABCC10', 'ABCF3', 'ACAD10', 'ACAP1', 'ACIN1', 'AGRN', 'AMPD2', 'ANKRD13C-DT', 'ANKRD13D', 'ANKRD54', 'AP5S1', 'AP5Z1', 'APBA1', 'APBA3', 'ARHGAP17', 'ARHGAP30', 'ARHGEF11', 'ARHGEF2', 'ARMC6', 'ARPC5L', 'ATAD3A', 'ATAD3B', 'ATAT1', 'ATF5', 'ATN1', 'ATXN2L', 'B4GALT1', 'BAZ1B', 'BICRA', 'BRD1', 'CACTIN', 'CALR', 'CAMK2B', 'CAPN15', 'CC2D1B', 'CCAR2', 'CCDC134', 'CCDC93', 'CDC42EP1', 'CDC42SE1', 'CDK3', 'CFAP410', 'CHRNB2', 'CIC', 'CLASRP', 'CMTR1', 'CNOT3', 'CNTROB', 'COPS7B', 'CPSF7', 'CRYGS', 'CSNK1E', 'CSNK1G2', 'CSNK2B', 'CTNND1', 'CUL7', 'CUL9', 'CYBA', 'CYP2D6', 'DAGLB', 'DAZAP1', 'DCAF8', 'DCLRE1C', 'DDX11', 'DDX39B', 'DENND4B', 'DENND5A', 'DEPDC5', 'DHX33', 'DMWD', 'DOT1L', 'DPH7', 'DRG2', 'DUOXA2', 'DXO', 'EHMT2', 'EIF3B', 'ENGASE', 'EPN1', 'ERCC2', 'EVI5L', 'EYA3', 'FANCA', 'FBRSL1', 'FBXL18', 'FBXL19', 'FBXO44', 'FBXO46', 'FKBP1A', 'FNBP4', 'FOXO4', 'FURIN', 'FXR2', 'GAS5', 'GATAD2A', 'GBA2', 'GFER', 'GGA1', 'GIGYF1', 'GLP1R', 'GPC2', 'GRAMD4', 'GRIP2', 'GRIPAP1', 'GTF2H4', 'HAUS8', 'HCFC1', 'HDAC10', 'HDAC7', 'HEYL', 'HIP1R', 'HIRA', 'HNRNPD', 'HNRNPU', 'HPCA', 'HSF4', 'IFRD2', 'IL17C', 'IL18BP', 'ILF3', 'INTS1', 'INTS15', 'IQCG', 'KAT8', 'KCNK5', 'KCTD13', 'KDM4A', 'KHDC4', 'KHSRP', 'KLF16', 'KMT2D', 'KMT2E-AS1', 'L3MBTL2', 'LARP1', 'LEMD2', 'LENG8', 'LIMD2', 'LIMK2', 'LINC00115', 'LINC01355', 'LINC02627', 'LINC02693', 'LLGL1', 'LMNA', 'LPAR2', 'LRCH4', 'LRRC74A', 'LSM14B', 'LTB4R', 'MAP2K2', 'MAPK11', 'MAPK8IP3', 'MAPRE3', 'MARK2', 'MBD3', 'MCTP2', 'MCTS1', 'MDN1', 'MEF2D', 'MICALL1', 'MIIP', 'MINK1', 'MIRLET7D', 'MOGS', 'MRPS25', 'MSTO1', 'MTOR', 'MTSS2', 'MYL10', 'MYO9B', 'MYPOP', 'NCKAP5L', 'NEU4', 'NEURL4', 'NFKBIL1', 'NOL8', 'NOTUM', 'NPIPB3', 'NPM3', 'NSUN4', 'NSUN5P1', 'NUP214', 'NUP62', 'NUTM2E', 'PABPN1', 'PBX2', 'PDXP', 'PHF21A', 'PHF5A', 'PHKG2', 'PHPT1', 'PICK1', 'PIP5K1A', 'POLM', 'POLR2F', 'PPARD', 'PPM1G', 'PPP1R12C', 'PPP6R2', 'PPRC1', 'PQBP1', 'PRRC2A', 'PTBP1', 'PTOV1-AS2', 'PUS1', 'QTRT1', 'R3HDM4', 'RABGAP1', 'RAD54L2', 'RANBP3', 'RANGAP1', 'RAP1GAP2', 'RASSF7', 'RBBP6', 'RBM10', 'RBM14', 'RBM15B', 'RBM33', 'RCC1', 'REXO1', 'RHBG', 'RNF207', 'RNF214', 'RNPS1', 'RPL18', 'RTKN', 'RUNX3', 'SAFB2', 'SAP25', 'SBNO2', 'SCAF1', 'SCUBE1', 'SEC31B', 'SEC61A2', 'SEMA6B', 'SEMA6C', 'SETD1A', 'SETD5', 'SETDB1', 'SF1', 'SFSWAP', 'SFTPC', 'SGSM3', 'SH3BP1', 'SH3BP2', 'SH3BP5L', 'SLC12A9', 'SLC23A2', 'SLC39A3', 'SLC4A11', 'SLC52A3', 'SMARCA4', 'SMARCD1', 'SMG5', 'SMG7', 'SMG9', 'SNAPC4', 'SNHG3', 'SNRNP70', 'SPHK2', 'SPON2', 'SPPL2B', 'SRCAP', 'SRL', 'SRRM2', 'SRRT', 'STK11', 'SUGP2', 'SUN1', 'SUPT7L', 'SUV39H1', 'SYMPK', 'SZT2', 'TAF15', 'TAF1C', 'TARDBP', 'TBC1D22A-DT', 'TBC1D22B', 'TCF3', 'TCHP', 'TCOF1', 'TEAD3', 'TEAD4', 'TECPR1', 'TEPSIN', 'THOC2', 'TICAM1', 'TLCD3B', 'TLN1', 'TMEM201', 'TMEM259', 'TMEM63A', 'TMEM86B', 'TNK2', 'TNPO2', 'TNPO3', 'TPM2', 'TRABD', 'TRAF2', 'TRMU', 'TRRAP', 'TSPAN14', 'TTC13', 'TTC17', 'TYW5', 'U2AF2', 'UBAP2L', 'UNC13D', 'UPB1', 'UQCC2', 'URB2', 'USP21', 'VARS1', 'XPO5', 'ZBTB5', 'ZBTB7B', 'ZC3H7B', 'ZDHHC24', 'ZMIZ2', 'ZMYND19', 'ZNF133', 'ZNF202', 'ZNF316', 'ZNF358', 'ZNF436-AS1', 'ZNF444', 'ZNF473', 'ZNF524', 'ZNF580', 'ZNF598', 'ZNF611', 'ZNF638', 'ZNF646', 'ZNF692', 'ZNF793']
Refined Community 37: ['ADD1', 'AGAP4', 'ANAPC5', 'ANG', 'APBB2', 'ATG12', 'ATP6V1A', 'BAIAP3', 'BBX', 'BLOC1S6', 'BMPR2', 'BTD', 'C22orf39', 'CHURC1', 'COMMD1', 'CREB1', 'CYP20A1', 'DNAJB14', 'DNAJC21', 'DNAL1', 'EMILIN3', 'ESR1', 'F2R', 'F8', 'FAM114A1', 'FKBP14', 'GINM1', 'GNB4', 'GNB5', 'HEATR3', 'ICE2', 'INIP', 'KCTD20', 'KLHL20', 'LACTB', 'LEPROT', 'LIN52', 'LPP', 'MAP3K12', 'MINDY2', 'MORF4L2', 'N4BP2', 'PDP2', 'POLK', 'PWWP2A', 'RAD1', 'RSBN1L', 'RSPH3', 'SCAF11', 'SF3B1', 'SIAH1', 'SLC17A5', 'SLC25A36', 'SLC35A3', 'SLU7', 'SMIM14', 'SNX9', 'SP1', 'SPPL2A', 'SRSF10', 'STAG2', 'STX16', 'TRAK2', 'TRIP11', 'UFL1', 'WASHC4', 'WSB1', 'YIPF5', 'ZBTB38']
Refined Community 38: ['AFDN', 'ARHGEF7', 'BANP', 'CASD1', 'CEPT1', 'DLEU2', 'EHD1', 'EP300', 'EVI5', 'FKBP8', 'FTCD', 'HPS5', 'METAP2', 'NAP1L1', 'OCIAD1', 'PRPF6', 'SRSF11', 'SVIL', 'SYNCRIP', 'UBR5']
Refined Community 39: ['ADNP', 'ATP5F1B', 'CHPF', 'COL6A2', 'CYP1A1', 'JUNB', 'KDM6A', 'MCF2', 'PNOC', 'RPL15', 'RPL19', 'RPL31', 'RPP30', 'RPS13', 'RPS4X', 'RPS5', 'SPRY1', 'TANK', 'UBA52']
Refined Community 40: ['BCL2', 'BIRC5', 'CCNA2', 'CCNB1', 'CCND1', 'CCNE1', 'EGFR', 'EREG', 'IFI27', 'KRT8', 'MDM2', 'MKI67', 'PGR', 'RAD50', 'SKP2', 'TOP2A', 'VIM']
Refined Community 41: ['APEX1', 'ARHGEF6', 'BABAM2', 'BMP6', 'CD36', 'CD83', 'CSNK1E', 'CSRP2', 'FOXO1', 'GABRP', 'GFM2', 'GOLGA1', 'IL7', 'ITGAE', 'KDR', 'MAST1', 'MMP13', 'NCOA1', 'PDE6A', 'PDGFRB', 'PLXNA2', 'PMEPA1', 'POLR2A', 'PON1', 'PTEN', 'PTPRM', 'RCL1', 'RGL2', 'RGS1', 'RGS16', 'RYBP', 'SBNO1', 'SFRP4', 'SRSF11', 'SUGP2', 'SYT17', 'TAL1', 'TRMT6', 'WNT2', 'YTHDC2', 'ZBTB14', 'ZNF211']
Refined Community 42: ['ACTR1A', 'AFP', 'AKT1', 'BAD', 'CALU', 'COPA', 'CTNNBL1', 'CXCL5', 'DLGAP5', 'ELOB', 'ENOX2', 'FDFT1', 'GART', 'GCAT', 'GNB2', 'GTPBP1', 'HARS1', 'IL17RA', 'IL18R1', 'IL1B', 'KDM5A', 'LOX', 'MAP2K3', 'MAPRE1', 'MNAT1', 'MPI', 'MPV17L2', 'NAGA', 'NCSTN', 'NOC2L', 'NT5DC2', 'PAF1', 'PAK2', 'PEF1', 'PISD', 'PPP1CB', 'PPP2R5A', 'PPY2P', 'RALY', 'RBBP4', 'RNF167', 'RUNX1', 'SF3B4', 'SLC9A1', 'SPCS3', 'STRN3', 'TATDN2', 'TUFM', 'UBAP2L']
Refined Community 43: ['ACAA1', 'ADIG', 'ADIPOR2', 'AHNAK', 'ALAD', 'ALAS1', 'ANGPTL2', 'BCKDHB', 'BNIP2', 'CAVIN3', 'CCDC80', 'CD34', 'CIDEC', 'CLEC3B', 'COL18A1', 'COL1A1', 'COL3A1', 'COL4A1', 'COL5A1', 'COL6A3', 'CRIP1', 'CXCL12', 'DECR1', 'DERL1', 'DGAT1', 'DHRS7', 'DPEP1', 'DPT', 'EPHX2', 'ETFB', 'FEZ2', 'FSTL1', 'FZD4', 'GAS6', 'GHR', 'GJA1', 'GNAI1', 'GPAM', 'H6PD', 'HEPH', 'HSPB8', 'HTRA1', 'IDH1', 'IGFBP6', 'KLF4', 'LAMB1', 'LUM', 'MAN1A1', 'MAP4', 'MFNG', 'MRC1', 'NR1H3', 'NRP1', 'PDE8A', 'PENK', 'PGM1', 'PHYH', 'PLAC8', 'PPP2R5A', 'QKI', 'RAB34', 'S100A6', 'SORBS1', 'SQOR', 'SRPX', 'ST3GAL6', 'SULT1A1', 'TGFBI', 'THBD', 'UCK1', 'VWF', 'ZEB1']
Refined Community 44: ['ACSL4', 'ALDOC', 'ARHGEF5', 'ATP6AP2', 'ATP6V1A', 'CD82', 'CHMP2B', 'CLDN3', 'CLDN7', 'CMAS', 'CRYBG1', 'CYB561', 'DAP', 'DSG2', 'EBP', 'EHF', 'ELL2', 'FXYD3', 'GALNT3', 'GOLPH3', 'HAVCR1', 'ID2', 'IRF6', 'IRX3', 'KCNN4', 'LCN2', 'LITAF', 'LRRFIP1', 'LSR', 'NUCB2', 'PLET1', 'RETREG1', 'RNF149', 'SERP1', 'SHROOM3', 'SLC66A2', 'SOX4', 'SPINT1', 'SRP19', 'SS18L2', 'STRBP', 'TFAP2C', 'TPD52', 'VDR', 'WWC1', 'XBP1']
Refined Community 45: ['ANP32E', 'CHD9', 'CTSB', 'DHX16', 'EIF1', 'ERCC2', 'FAM242E', 'FAN1', 'FAU', 'FNDC5', 'FTH1P5', 'GGTLC1', 'GIT2', 'HENMT1', 'HNRNPA0', 'HNRNPA1', 'IRAG1-AS1', 'ITGB1', 'KASH5', 'KPNB1', 'MLPH', 'MT-ND5', 'NCOR1', 'NRN1', 'NYX', 'PCYT1A', 'PTMA', 'RASSF8', 'SCN8A', 'SIGLEC1', 'TUBB6', 'ZNF470']
Refined Community 46: ['ADAM10', 'ADAM9', 'AGRN', 'ANGPTL4', 'C1QA', 'C1QC', 'CCN2', 'COL22A1', 'COL24A1', 'COL4A6', 'CST3', 'CTSB', 'CTSC', 'CTSF', 'EFEMP2', 'EGLN1', 'F10', 'F13B', 'HABP2', 'HCFC2', 'HTRA1', 'IGFBP4', 'ITIH4', 'LOXL2', 'LTBP3', 'MFGE8', 'P3H3', 'P4HTM', 'PAPLN', 'PLXNB2', 'PRG4', 'S100A10', 'S100A2', 'SERPINC1', 'SERPINE2', 'SERPINF1', 'SNED1', 'SRPX', 'TIMP1', 'TINAGL1', 'VWF']
Refined Community 47: ['ADAM10', 'ADAM9', 'AGRN', 'ANGPTL4', 'CCN2', 'COL22A1', 'COL24A1', 'CST3', 'CTSB', 'CTSC', 'CTSF', 'EFEMP2', 'EGLN1', 'HCFC2', 'HTRA1', 'IGFBP4', 'LOXL2', 'LTBP3', 'MFGE8', 'P3H3', 'P4HTM', 'PRG4', 'S100A10', 'S100A2', 'SERPINE2', 'SNED1', 'SRPX', 'TIMP1', 'TINAGL1']
Refined Community 48: ['ASPN', 'COL17A1', 'COL19A1', 'COL26A1', 'COL28A1', 'COL6A6', 'EMILIN2', 'FBLN5', 'FLG2', 'HMCN1', 'ITIH3', 'LAMA2', 'LUM', 'MMP1', 'MMP19', 'NGLY1', 'OGN', 'PLAT', 'PRELP', 'S100A16', 'S100A8', 'THBS2', 'TIMP3', 'TNXB', 'VWA1']
Refined Community 49: ['COL17A1', 'COL19A1', 'COL26A1', 'FBLN5', 'FLG2', 'MMP1', 'MMP19', 'NGLY1', 'OGN', 'PLAT', 'S100A16', 'S100A8']
Refined Community 50: ['ANXA5', 'C1S', 'CEBPD', 'CLDN1', 'CLK4', 'DCN', 'FBLN1', 'IFT81', 'MCEE', 'NANOG', 'OMD', 'REXO2', 'SHOX2', 'SMOC2', 'SPARCL1', 'SSR2', 'TGFBR3', 'TXNIP']
Refined Community 51: ['AMH', 'AQP5', 'ASPM', 'ATP2C2', 'BAALC', 'BUB1', 'CARTPT', 'CDCA5', 'CDCA8', 'CENPN', 'CEP55', 'CSAG2', 'CTPS1', 'DSCC1', 'EBP', 'EXO1', 'FA2H', 'FADD', 'GSE1', 'HSPA14', 'IL20RB', 'KIF20A', 'KLRG2', 'MELK', 'MLLT1', 'MTFR2', 'MYBL2', 'NDC80', 'PDX1', 'PMCH', 'POLD1', 'PRAME', 'PSMD14', 'PSMD2', 'PTTG1', 'RACGAP1', 'RASIP1', 'RNASE9', 'RNFT2', 'SCLT1', 'SEMA4C', 'SHMT2', 'SMARCA4', 'SOX11', 'SPAG5', 'TIMELESS', 'TMEM208', 'TNNC2', 'TRIM24', 'UBE2T', 'UHRF1']
Refined Community 52: ['DLG5', 'EIF5AL1', 'KCNMA1', 'POLR3A', 'PPIF', 'RPS24', 'ZCCHC24', 'ZMIZ1']
Refined Community 53: ['AAMDC', 'ACER3', 'ACTN3', 'ACY3', 'AIP', 'ALDH3B2', 'ALG8', 'ANKRD13D', 'ANO1', 'AQP11', 'ARRB1', 'B3GNT6', 'B4GAT1', 'BBS1', 'BRMS1', 'CABP2', 'CABP4', 'CAPN5', 'CARNS1', 'CCDC81', 'CCDC83', 'CCDC87', 'CCDC89', 'CCND1', 'CCS', 'CD248', 'CD5', 'CD6', 'CDK2AP2', 'CHRDL2', 'CLCF1', 'CLNS1A', 'CNIH2', 'CORO1B', 'CPSF7', 'CPT1A', 'CREBZF', 'CTSF', 'CTTN', 'CYB561A3', 'DAGLA', 'DDB1', 'DGAT2', 'DHCR7', 'DPP3', 'EED', 'EMSY', 'FADD', 'FGF19', 'FGF3', 'FGF4', 'GAB2', 'GAL', 'GDPD4', 'GDPD5', 'GPR152', 'GSTP1', 'HIKESHI', 'IGHMBP2', 'INTS4', 'KCNE3', 'KCTD14', 'KCTD21', 'KLC2', 'KLHL35', 'KRTAP5-10', 'KRTAP5-11', 'KRTAP5-7', 'KRTAP5-8', 'KRTAP5-9', 'LRFN4', 'LRRC32', 'LTO1', 'MAP6', 'ME3', 'MOGAT2', 'MRGPRD', 'MRGPRF', 'MRPL11', 'MRPL21', 'MYEOV', 'MYO7A', 'NADSYN1', 'NARS2', 'NDUFC2', 'NDUFV1', 'NDUFV1-DT', 'NEU3', 'NPAS4', 'NUDT8', 'OMP', 'OR2AT4', 'P4HA3', 'PAK1', 'PELI3', 'PGA3', 'PGA4', 'PGA5', 'PGM2L1', 'PICALM', 'PITPNM1', 'POLD3', 'POLD4', 'PPFIA1', 'PPME1', 'PPP1CA', 'PPP6R3', 'PTPRCAP', 'RAB1B', 'RAD9A', 'RBM14', 'RBM4', 'RBM4B', 'RCE1', 'RIN1', 'RNF169', 'RPS3', 'RPS6KB2', 'RSF1', 'SAXO4', 'SDHAF2', 'SERPINH1', 'SHANK2', 'SLC15A3', 'SLC29A2', 'SLCO2B1', 'SPCS2', 'SPCS2P4', 'SPTBN2', 'SSH3', 'SYT7', 'SYTL2', 'TBC1D10C', 'TBX10', 'TENM4', 'TESMIN', 'THAP12', 'THRSP', 'TKFC', 'TMEM109', 'TMEM126A', 'TMEM126B', 'TMEM132A', 'TMEM134', 'TMEM138', 'TMEM151A', 'TMEM216', 'TOP6BL', 'TPCN2', 'TSKU', 'USP35', 'UVRAG', 'VPS37C', 'VWCE', 'WNT11', 'XRRA1', 'YIF1A', 'ZDHHC24']
Refined Community 54: ['AGAP2', 'AVIL', 'BEST3', 'C12orf56', 'CAND1', 'CCT2', 'CDK4', 'CNOT2', 'CPM', 'CPSF6', 'CTDSP2', 'CYP27B1', 'DYRK2', 'EEF1AKMT3', 'FRS2', 'GNS', 'IFNG', 'IL22', 'IL26', 'KCNMB4', 'KICS2', 'LGR5', 'LRRC10', 'LYZ', 'MARCHF9', 'MDM1', 'MDM2', 'METTL1', 'NUP107', 'OS9', 'PTPRB', 'PTPRR', 'RAB3IP', 'RAP1B', 'RAP1BL', 'RASSF3', 'SLC35E3', 'TBK1', 'THAP2', 'TMEM19', 'TSFM', 'TSPAN31', 'TSPAN8', 'XPOT', 'YEATS4', 'ZFC3H1']
Refined Community 55: ['ANKLE2', 'CHFR', 'DDX51', 'EP400', 'GALNT9', 'GOLGA3', 'MMP17', 'NOC4L', 'P2RX2', 'PGAM5', 'POLE', 'PUS1', 'PXMP2', 'SFSWAP', 'ULK1']
Refined Community 56: ['DDHD1', 'ERO1A', 'FERMT2', 'FRMD6', 'GNG2', 'GNPNAT1', 'GPR137C', 'NID2', 'PSMC6', 'PTGDR', 'PTGER2', 'RTRAF', 'STYX', 'TXNDC16']
Refined Community 57: ['ADAMTS17', 'ALDH1A3', 'ARRDC4', 'ASB7', 'CERS3', 'CHSY1', 'FAM169BP', 'IGF1R', 'LINS1', 'LRRC28', 'LRRK1', 'LYSMD4', 'MEF2A', 'OR4F15', 'OR4F6', 'PCSK6', 'SELENOS', 'SNRPA1', 'SYNM', 'TARS3', 'TM2D3', 'TTC23']
Refined Community 58: ['ANTKMT', 'ARHGDIG', 'AXIN1', 'BAIAP3', 'C1QTNF8', 'CACNA1H', 'CAPN15', 'CCDC78', 'CHTF18', 'CIAO3', 'CLCN7', 'CLDN6', 'CLDN9', 'CRAMP1', 'DECR2', 'ELOB', 'EME2', 'FAHD1', 'FAM234A', 'FBXL16', 'FLYWCH1', 'FLYWCH2', 'GFER', 'GNG13', 'GNPTG', 'HAGH', 'HAGHL', 'HBA1', 'HBA2', 'HBM', 'HBQ1', 'HBZ', 'HCFC1R1', 'HS3ST6', 'IFT140', 'IGFALS', 'IL32', 'JMJD8', 'JPT2', 'KCTD5', 'KREMEN2', 'LMF1', 'LUC7L', 'MAPK8IP3', 'MCRIP2', 'MEIOB', 'METRN', 'METTL26', 'MMP25', 'MPG', 'MRPL28', 'MRPS34', 'MSLN', 'MSLNL', 'MSRB1', 'NDUFB10', 'NHERF2', 'NHLRC4', 'NME3', 'NME4', 'NOXO1', 'NPRL3', 'NPW', 'NTHL1', 'NUBP2', 'PAQR4', 'PDIA2', 'PGAP6', 'PIGQ', 'PKD1', 'PKMYT1', 'POLR3K', 'PRR35', 'PRSS21', 'PRSS22', 'PRSS27', 'PRSS33', 'PTX4', 'RAB11FIP3', 'RAB26', 'RAB40C', 'RGS11', 'RHBDF1', 'RHBDL1', 'RHOT2', 'RNF151', 'RPL3L', 'RPS2', 'RPUSD1', 'SNRNP25', 'SOX8', 'SPSB3', 'SRRM2', 'SSTR5', 'STUB1', 'SYNGR3', 'TBL3', 'TELO2', 'THOC6', 'TMEM204', 'TNFRSF12A', 'TPSAB1', 'TPSB2', 'TPSD1', 'TPSG1', 'TRAF7', 'TSC2', 'TSR3', 'UBE2I', 'UNKL', 'UQCC4', 'WDR24', 'WDR90', 'WFIKKN1', 'ZG16B', 'ZNF205', 'ZNF213', 'ZNF598', 'ZSCAN10']
Refined Community 59: ['ACSF3', 'ANKRD11', 'APRT', 'BANP', 'C16orf74', 'CA5A', 'CBFA2T3', 'CDH15', 'CDK10', 'CDT1', 'CENPBD1P', 'CHMP1A', 'COX4I1', 'CPNE7', 'CTU2', 'CYBA', 'DBNDD1', 'DEF8', 'DPEP1', 'EMC8', 'FANCA', 'GALNS', 'GAS8', 'GAS8-AS1', 'GINS2', 'GSE1', 'IL17C', 'JPH3', 'KLHDC4', 'MAP1LC3B', 'MC1R', 'MVD', 'PABPN1L', 'PIEZO1', 'PRDM7', 'RNF166', 'RPL13', 'SLC7A5', 'SNAI3', 'SPATA2L', 'SPATA33', 'SPG7', 'SPIRE2', 'TCF25', 'TRAPPC2L', 'TUBB3', 'VPS9D1', 'ZC3H18', 'ZCCHC14', 'ZFPM1', 'ZNF276', 'ZNF778']
Refined Community 60: ['ALDH3A1', 'ALDH3A2', 'B9D1', 'EPN2', 'MAPK7', 'MFAP4', 'RNF112', 'SLC47A1', 'SLC47A2', 'ULK2']
Refined Community 61: ['ABHD15', 'ACACA', 'ALDOC', 'ANKRD13B', 'ASIC2', 'BLMH', 'BLTP2', 'C17orf78', 'CACNB1', 'CASC3', 'CCL18', 'CCL2', 'CCL23', 'CCL3', 'CCL4', 'CCR7', 'CDC6', 'CDK12', 'CORO6', 'CPD', 'CRYBA1', 'CSF3', 'CWC25', 'DDX52', 'DHRS13', 'DUSP14', 'EFCAB5', 'ERAL1', 'ERBB2', 'FAM222B', 'FBXL20', 'FBXO47', 'FLOT2', 'FOXN1', 'GIT1', 'GJD3', 'GOSR1', 'GPR179', 'GRB7', 'GSDMA', 'GSDMB', 'HNF1B', 'IGFBP4', 'IKZF3', 'KRT10', 'KRT10-AS1', 'KRT12', 'KRT20', 'KRT222', 'KRT23', 'KRT24', 'KRT25', 'KRT26', 'KRT27', 'KRT28', 'KRTAP4-1', 'KRTAP4-2', 'KRTAP4-3', 'KRTAP4-4', 'KRTAP4-5', 'KRTAP9-2', 'KRTAP9-3', 'KRTAP9-4', 'KRTAP9-8', 'LASP1', 'MED1', 'MED24', 'MIEN1', 'MLLT6', 'MRPL45', 'MSL1', 'MYO18A', 'MYO1D', 'NEK8', 'NEUROD2', 'NR1D1', 'NSRP1', 'NUFIP2', 'ORMDL3', 'PCGF2', 'PGAP3', 'PHF12', 'PIGS', 'PIP4K2B', 'PIPOX', 'PLXDC1', 'PNMT', 'PPP1R1B', 'PROCA1', 'PSMB3', 'PSMD3', 'RAB34', 'RAPGEFL1', 'RARA', 'RPL19', 'RPL23', 'RPL23A', 'RPL23AP42', 'RSKR', 'SARM1', 'SDF2', 'SEZ6', 'SLC13A2', 'SLC46A1', 'SLC6A4', 'SMARCE1', 'SOCS7', 'SPACA3', 'SPAG5', 'SPMAP1', 'SRCIN1', 'SSH2', 'STAC2', 'STARD3', 'SUPT6H', 'SYNRG', 'TADA2A', 'TAOK1', 'TBC1D29P', 'TBC1D3F', 'TCAP', 'THRA', 'TLCD1', 'TMEM98', 'TMIGD1', 'TNS4', 'TOP2A', 'TP53I13', 'TRAF4', 'UNC119', 'WIPF2', 'ZPBP2']
Refined Community 62: ['AATK', 'ABCA10', 'ABCA5', 'ABCA6', 'ABCA8', 'ABCA9', 'ABCC3', 'ABI3', 'ACOX1', 'ACSF2', 'ACTG1', 'AFMID', 'ALYREF', 'AMZ2', 'ANKFN1', 'ANKRD40', 'APOH', 'APPBP2', 'ARHGDIA', 'ARL16', 'ARMC7', 'ARSG', 'ASPSCR1', 'ATP5MC1', 'ATP5PD', 'AXIN2', 'B3GNTL1', 'B4GALNT2', 'BAHCC1', 'BAIAP2', 'BCAS3', 'BIRC5', 'BPTF', 'BRIP1', 'BTBD17', 'C17orf58', 'C17orf67', 'C1QTNF1', 'CA10', 'CA4', 'CACNA1G', 'CACNG1', 'CACNG4', 'CACNG5', 'CALCOCO2', 'CANT1', 'CASKIN2', 'CBX1', 'CCDC137', 'CCDC47', 'CCDC57', 'CD300A', 'CD300C', 'CD300E', 'CD300LB', 'CD300LD-AS1', 'CD300LF', 'CD7', 'CD79B', 'CDC42EP4', 'CDK3', 'CDK5RAP3', 'CDR2L', 'CENPX', 'CEP112', 'CEP131', 'CEP95', 'CHAD', 'CHCT1', 'CHMP6', 'CLTC', 'COG1', 'COIL', 'COL1A1', 'COPZ2', 'COX11', 'CSH1', 'CSH2', 'CSHL1', 'CSNK1D', 'CYBC1', 'CYTH1', 'DCXR', 'DDX42', 'DDX5', 'DGKE', 'DHX40', 'DLX3', 'DLX4', 'DNAI2', 'DUS1L', 'EFCAB3', 'EME1', 'ENGASE', 'EPN3', 'EPX', 'ERN1', 'EVPL', 'EXOC7', 'FAAP100', 'FADS6', 'FAM117A', 'FAM20A', 'FASN', 'FBF1', 'FDXR', 'FN3K', 'FN3KRP', 'FOXJ1', 'FOXK2', 'FSCN2', 'FTSJ3', 'GALK1', 'GALR2', 'GDPD1', 'GGA3', 'GH1', 'GH2', 'GIP', 'GNA13', 'GNGT2', 'GPR142', 'GPRC5C', 'GPS1', 'GRB2', 'GRIN2C', 'H3-3B', 'HEATR6', 'HELZ', 'HEXD', 'HGS', 'HID1', 'HLF', 'HOXB1', 'HOXB13', 'HOXB2', 'HOXB3', 'HOXB4', 'HOXB5', 'HOXB6', 'HOXB7', 'HOXB8', 'HOXB9', 'HSF5', 'ICAM2', 'IGF2BP1', 'INTS2', 'ITGA3', 'ITGB4', 'JPT1', 'KAT7', 'KCNJ16', 'KCNJ2', 'KCTD2', 'KIF19', 'KIF2B', 'LGALS3BP', 'LIMD2', 'LINC00469', 'LINC00482', 'LINC01973', 'LINC02875', 'LLGL2', 'LPO', 'LRRC37A3', 'LRRC45', 'LRRC59', 'LUC7L3', 'MAP2K6', 'MAP3K3', 'MARCHF10', 'MBTD1', 'MED13', 'METRNL', 'METTL2A', 'MIF4GD', 'MILR1', 'MKS1', 'MMD', 'MPO', 'MRC2', 'MRPL12', 'MRPL27', 'MRPL38', 'MRPL58', 'MRPS7', 'MTMR4', 'MTNAP1', 'MYCBPAP', 'NACA2', 'NARF', 'NAT9', 'NDUFAF8', 'NFE2L1', 'NGFR', 'NHERF1', 'NME1', 'NME1-NME2', 'NME2', 'NOG', 'NOL11', 'NOTUM', 'NPLOC4', 'NT5C', 'NUP85', 'NXPH3', 'OGFOD3', 'OR4D2', 'OTOP2', 'OTOP3', 'OXLD1', 'PCTP', 'PDE6G', 'PDK2', 'PECAM1', 'PGS1', 'PHB1', 'PHOSPHO1', 'PITPNC1', 'PNPO', 'POLG2', 'PPM1D', 'PPM1E', 'PPP1R27', 'PPP1R9B', 'PRAC1', 'PRKAR1A', 'PRKCA', 'PRR11', 'PRR15L', 'PSMC5', 'PSMD12', 'PTRH2', 'RAB37', 'RAB40B', 'RAC3', 'RAD51C', 'RECQL5', 'RFNG', 'RGS9', 'RNF157', 'RNF43', 'RNFT1', 'RPL38', 'RPS6KB1', 'RPTOR', 'RSAD1', 'SAMD14', 'SAP30BP', 'SCN4A', 'SCPEP1', 'SDK2', 'SECTM1', 'SEPTIN4', 'SGCA', 'SKA2', 'SKAP1', 'SLC16A3', 'SLC16A5', 'SLC16A6', 'SLC25A10', 'SLC25A19', 'SLC35B1', 'SLC38A10', 'SLC39A11', 'SMARCD2', 'SMG8', 'SMURF2', 'SNF8', 'SNX11', 'SOCS3', 'SOX9', 'SP2', 'SPAG9', 'SPATA20', 'SPOP', 'SRP68', 'SSTR2', 'STRADA', 'STXBP4', 'SUMO2', 'SUPT4H1', 'SYNGR2', 'TAC4', 'TACO1', 'TBCD', 'TBX2', 'TBX4', 'TEPSIN', 'TEX14', 'TEX19', 'TEX2', 'TIMP2', 'TK1', 'TLK2', 'TMC6', 'TMC8', 'TMEM100', 'TMEM104', 'TMEM105', 'TMEM92', 'TMEM94', 'TNRC6C', 'TOB1', 'TOM1L1', 'TRIM25', 'TRIM37', 'TRIM47', 'TRIM65', 'TSEN54', 'TSPAN10', 'TSPOAP1', 'TTLL6', 'TTYH2', 'TUBD1', 'UBE2Z', 'UNC13D', 'UNK', 'USH1G', 'USP32', 'USP36', 'UTP18', 'UTS2R', 'VCF1', 'VMP1', 'WBP2', 'WDR45B', 'WFIKKN2', 'WIPI1', 'XYLT2', 'YPEL2', 'ZACN', 'ZNF652', 'ZNF750']
Refined Community 63: ['MYO9B', 'NR2F6', 'OCEL1', 'USE1', 'USHBP1']
Refined Community 64: ['ZFP30', 'ZNF260', 'ZNF345', 'ZNF382', 'ZNF383', 'ZNF420', 'ZNF461', 'ZNF527', 'ZNF529', 'ZNF540', 'ZNF567', 'ZNF568', 'ZNF569', 'ZNF570', 'ZNF571', 'ZNF585A', 'ZNF585B', 'ZNF781', 'ZNF790', 'ZNF793', 'ZNF829', 'ZNF875']
Refined Community 65: ['CACNG6', 'CACNG8', 'NDUFA3', 'OSCAR', 'PRPF31', 'TFPT', 'VSTM1']
Refined Community 66: ['ADAM15', 'CELF3', 'CKS1B', 'CLK2', 'DCST1', 'DCST2', 'DPM3', 'EFNA1', 'EFNA3', 'EFNA4', 'ENTREP3', 'FDPS', 'FLAD1', 'GBA1', 'HCN3', 'KRTCAP2', 'LENEP', 'LINGO4', 'MRPL9', 'MTX1', 'MUC1', 'OAZ3', 'PBXIP1', 'PKLR', 'PMVK', 'PYGO2', 'RORC', 'RUSC1', 'RUSC1-AS1', 'SCAMP3', 'SHC1', 'SLC50A1', 'TDRKH', 'THBS3', 'THEM4', 'THEM5', 'TRIM46', 'ZBTB7B']
Refined Community 67: ['DYRK3', 'EIF2D', 'FCAMR', 'FCMR', 'IKBKE', 'IL10', 'IL19', 'IL20', 'IL24', 'MAPKAPK2', 'PIGR', 'RASSF5', 'SRGAP2']
Refined Community 68: ['ADAM33', 'ADISSP', 'ATRN', 'CENPB', 'GFRA4', 'HSPA12B', 'SIGLEC1', 'SPEF1']
Refined Community 69: ['AAR2', 'ACSS2', 'CEP250', 'CNBD2', 'CPNE1', 'DLGAP4', 'DYNLRB1', 'EDEM2', 'EIF6', 'EPB41L1', 'ERGIC3', 'FAM83C', 'GDF5', 'GGT7', 'GSS', 'MAP1LC3A', 'MMP24', 'MYH7B', 'NCOA6', 'NFS1', 'PHF20', 'PIGU', 'PROCR', 'RBM12', 'RBM39', 'ROMO1', 'SCAND1', 'SPAG4', 'TP53INP2', 'TRPC4AP', 'UQCC1']
Refined Community 70: ['ABHD16B', 'ADNP', 'ADRM1', 'APCDD1L', 'ARFGAP1', 'ARFGEF2', 'ARFRP1', 'ATP5F1E', 'ATP9A', 'AURKA', 'BCAS1', 'BCAS4', 'BHLHE23', 'BIRC7', 'BMP7', 'C20orf204', 'CABLES2', 'CASS4', 'CBLN4', 'CDH26', 'CDH4', 'CEBPB', 'CHRNA4', 'CIMIP1', 'COL20A1', 'COL9A3', 'CRMA', 'CSTF1', 'CTCFL', 'CTSZ', 'CYP24A1', 'DIDO1', 'DNAJC5', 'DOK5', 'DPM1', 'EDN3', 'EEF1A2', 'FAM209A', 'FAM209B', 'FAM210B', 'FAM217B', 'FNDC11', 'GATA5', 'GID8', 'GMEB2', 'GNAS', 'HELZ2', 'HMGB1P1', 'HRH3', 'KCNG1', 'KCNQ2', 'KCNS1', 'LAMA5', 'LIME1', 'LKAAEAR1', 'LSM14B', 'MATN4', 'MC3R', 'MIR1-1HG', 'MIR646HG', 'MOCS3', 'MRGBP', 'MTG2', 'MYT1', 'NCOA3', 'NELFCD', 'NFATC2', 'NKAIN4', 'NPBWR2', 'NPEPL1', 'NTSR1', 'OGFR', 'OPRL1', 'OSBPL2', 'PARD6B', 'PCK1', 'PCMTD2', 'PEDS1', 'PEDS1-UBE2V1', 'PFDN4', 'PHACTR3', 'PI3', 'PMEPA1', 'PPDPF', 'PPP1R3D', 'PRELID3B', 'PREX1', 'PRPF6', 'PSMA7', 'PTK6', 'PTPN1', 'RAB22A', 'RAE1', 'RBBP8NL', 'RBM38', 'RBPJL', 'RGS19', 'RIPOR3', 'RNF114', 'RPS21', 'RTEL1', 'RTF2', 'SALL4', 'SAMD10', 'SDC4', 'SEMG1', 'SEMG2', 'SLC17A9', 'SLC2A4RG', 'SLC9A8', 'SLCO4A1', 'SLPI', 'SNAI1', 'SOX18', 'SPATA2', 'SPO11', 'SRMS', 'SS18L1', 'STK4', 'STMN3', 'STX16', 'SULF2', 'SYCP2', 'SYS1', 'TAF4', 'TCEA2', 'TCFL5', 'TFAP2C', 'TNFRSF6B', 'TOMM34', 'TP53TG5', 'TPD52L2', 'TSHZ2', 'TUBB1', 'UBE2V1', 'UCKL1', 'VAPB', 'WFDC12', 'WFDC5', 'YTHDF1', 'YWHAB', 'ZBP1', 'ZBTB46', 'ZFP64', 'ZGPAT', 'ZMYND8', 'ZNF217', 'ZNF512B', 'ZNF831']
Refined Community 71: ['ADARB1', 'C21orf58', 'COL18A1', 'COL18A1-AS1', 'COL6A1', 'COL6A2', 'DIP2A', 'FTCD', 'LSS', 'MCM3AP', 'PCBP3', 'PCNT', 'POFUT2', 'SLC19A1', 'SPATC1L', 'YBEY']
Refined Community 72: ['ALG12', 'BRD1', 'CRELD2', 'HDAC10', 'IL17REL', 'MAPK11', 'MAPK12', 'MLC1', 'MOV10L1', 'PANX2', 'PIM3', 'PLXNB2', 'SELENOO', 'TRABD', 'TTLL8', 'TUBGCP6', 'ZBED4']
Refined Community 73: ['AHRR', 'BRD9', 'CCDC127', 'CEP72', 'CLPTM1L', 'EXOC3', 'IRX2', 'IRX2-DT', 'IRX4', 'LPCAT1', 'LRRC14B', 'MRPL36', 'NDUFS6', 'NKD2', 'PDCD6', 'PLEKHG4B', 'SDHA', 'SLC12A7', 'SLC6A18', 'SLC6A19', 'SLC6A3', 'SLC9A3', 'TERT', 'TPPP', 'TRIP13', 'ZDHHC11']
Refined Community 74: ['CAP2', 'CD83', 'DEK', 'FAM8A1', 'GFOD1', 'ID4', 'KDM1B', 'KIF13A', 'MBOAT1', 'MCUR1', 'NHLRC1', 'NOL7', 'NUP153', 'PHACTR1', 'RANBP9', 'RBM24', 'RNF144B', 'RNF182', 'SIRT5', 'TBC1D7', 'TPMT']
Refined Community 75: ['HOXA1', 'HOXA10', 'HOXA11', 'HOXA13', 'HOXA2', 'HOXA3', 'HOXA4', 'HOXA5', 'HOXA6', 'HOXA7', 'HOXA9']
Refined Community 76: ['ACTB', 'ADAP1', 'AMZ1', 'AP5Z1', 'BRAT1', 'C7orf50', 'CHST12', 'COX19', 'CYP2W1', 'EIF3B', 'FAM20C', 'FBXL18', 'FOXK1', 'GET4', 'GNA12', 'GPER1', 'GPR146', 'GRIFIN', 'INTS1', 'IQCE', 'LFNG', 'MAD1L1', 'MAFK', 'MICALL2', 'MRM2', 'NUDT1', 'PAPOLB', 'PSMG3', 'RADIL', 'SLC29A4', 'SNX8', 'SUN1', 'TMEM184A', 'TNRC18', 'TTYH3', 'UNCX', 'WIPI2', 'ZFAND2A']
Refined Community 77: ['AKAP9', 'ANKIB1', 'ARPC1A', 'ARPC1B', 'ASB4', 'ASNS', 'ATP5MF', 'AZGP1', 'BAIAP2L1', 'BET1', 'BHLHA15', 'BRI3', 'BUD31', 'CALCR', 'CASD1', 'CDK14', 'CDK6', 'CFAP69', 'CLDN12', 'COL1A2', 'CPSF4', 'CYP3A4', 'CYP3A43', 'CYP3A5', 'CYP3A7', 'CYP51A1', 'DLX5', 'DLX6', 'DYNC1I1', 'FAM133B', 'FAM200A', 'FZD1', 'GATAD1', 'GJC3', 'GNG11', 'GNGT1', 'GTPBP10', 'HEPACAM2', 'KRIT1', 'LMTK2', 'LRRD1', 'MTERF1', 'NPTX2', 'OCM2', 'OR2AE1', 'PDAP1', 'PDK4', 'PEG10', 'PEX1', 'PON1', 'PON2', 'PON3', 'PPP1R9A', 'PTCD1', 'RBM48', 'SAMD9', 'SAMD9L', 'SDHAF3', 'SEM1', 'SGCE', 'SLC25A13', 'SMURF1', 'TAC1', 'TECPR1', 'TFPI2', 'TMEM130', 'TRIM4', 'TRRAP', 'VPS50', 'ZKSCAN1', 'ZKSCAN5', 'ZNF394', 'ZNF655', 'ZNF789', 'ZSCAN25']
Refined Community 78: ['ADAM18', 'ADAM2', 'ADAM32', 'ADAM9', 'ADGRA2', 'ADRB3', 'ANK1', 'AP3M2', 'ASH2L', 'BAG4', 'BRF2', 'CHRNA6', 'CHRNB3', 'DDHD2', 'DKK4', 'EIF4EBP1', 'ERLIN2', 'FGFR1', 'FNTA', 'GINS4', 'GOLGA7', 'GOT1L1', 'GPAT4', 'HGSNAT', 'HOOK3', 'HTRA4', 'IDO1', 'IDO2', 'IKBKB', 'KAT6A', 'LETM2', 'LINC03042', 'LSM1', 'NKX6-3', 'NSD3', 'PLAT', 'PLEKHA2', 'PLPBP', 'PLPP5', 'POLB', 'POMK', 'POTEA', 'RAB11FIP1', 'RNF170', 'SFRP1', 'SLC20A2', 'SMIM19', 'STAR', 'TACC1', 'TCIM', 'THAP1', 'TM2D2', 'TPT1P8', 'UNC5D', 'VDAC3', 'ZMAT4', 'ZNF703']
Refined Community 79: ['ANKRD46', 'ASPH', 'ATP6V1C1', 'AZIN1', 'BAALC', 'CALB1', 'CCNE2', 'CDH17', 'CFAP418', 'CHD7', 'CHMP4C', 'CIBAR1', 'CLVS1', 'CNBD1', 'COX6C', 'CPQ', 'CRISPLD1', 'CTHRC1', 'CYP7A1', 'DCAF13', 'DCAF4L2', 'DECR1', 'DPY19L4', 'ELOC', 'ERICH5', 'ESRP1', 'EYA1', 'FABP4', 'FABP5', 'FABP9', 'FBXO43', 'FZD6', 'GDAP1', 'GDF6', 'GEM', 'GGH', 'GRHL2', 'HEY1', 'HNF4G', 'IL7', 'IMPA1', 'INTS8', 'JPH1', 'KCNB2', 'KCNS2', 'KLF10', 'LACTB2', 'LAPTM4B', 'LY96', 'MATN2', 'MMP16', 'MRPS28', 'MSC', 'MTDH', 'MTERF3', 'NBN', 'NCOA2', 'NDUFAF6', 'NECAB1', 'NIPAL2', 'NKAIN3', 'NSMAF', 'ODF1', 'OSGIN2', 'OSR2', 'OTUD6B', 'PABPC1', 'PAG1', 'PDP1', 'PEX2', 'PI15', 'PIP4P2', 'PKIA', 'PLEKHF2', 'PMP2', 'POLR2K', 'POP1', 'PRDM14', 'PTDSS1', 'RAB2A', 'RAD54B', 'RALYL', 'RBM12B', 'RDH10', 'RGS22', 'RIDA', 'RIMS2', 'RIPK2', 'RNF19A', 'RPL30', 'RPL7', 'RPL7P9', 'RPSAP47', 'RUNX1T1', 'SBSPON', 'SDC2', 'SDCBP', 'SLC10A5', 'SLC25A32', 'SLC26A7', 'SLCO5A1', 'SNX16', 'SNX31', 'SPAG1', 'STAU2', 'STK3', 'STMN2', 'SULF1', 'TERF1', 'TMEM64', 'TMEM67', 'TMEM70', 'TP53INP1', 'TPD52', 'TRAM1', 'TRPA1', 'TSPYL5', 'TTPA', 'UBE2W', 'UBXN2B', 'UQCRB', 'VIRMA', 'VPS13B', 'XKR9', 'YTHDF3', 'YWHAZ', 'ZBTB10', 'ZC2HC1A', 'ZFAND1', 'ZFHX4', 'ZNF704', 'ZNF706']
Refined Community 80: ['AARD', 'ADCK5', 'ADGRB1', 'AGO2', 'ANXA13', 'ARC', 'ARHGAP39', 'ATAD2', 'BOP1', 'C8orf33', 'C8orf76', 'C8orf82', 'CCN3', 'CCN4', 'CHRAC1', 'COL14A1', 'COLEC10', 'COMMD5', 'CPSF1', 'CSMD3', 'CYC1', 'CYP11B1', 'CYP11B2', 'DENND3', 'DEPTOR', 'DERL1', 'DNAAF11', 'DSCC1', 'EBAG9', 'EEF1D', 'EIF3H', 'ENPP2', 'ENY2', 'EPPK1', 'EXOSC4', 'EXT1', 'FAM83A', 'FAM83H', 'FAM91A1', 'FBXL6', 'FBXO32', 'FER1L6', 'FOXH1', 'GFUS', 'GLI4', 'GML', 'GPAA1', 'GPIHBP1', 'GPR20', 'GPT', 'GRINA', 'GSDMD', 'HAS2', 'HGH1', 'HSF1', 'JRK', 'KCNK9', 'KCNQ3', 'KCNV1', 'KIFC2', 'KLHL38', 'LRATD2', 'LRRC14', 'LRRC24', 'LY6D', 'LY6E', 'LY6H', 'LY6K', 'LY6S-AS1', 'LYNX1', 'LYPD2', 'MAF1', 'MAFA', 'MAL2', 'MAPK15', 'MED30', 'MFSD3', 'MROH1', 'MROH5', 'MRPL13', 'MTBP', 'MTSS1', 'MYC', 'NAPRT', 'NDRG1', 'NDUFB9', 'NRBP2', 'NSMCE2', 'NTAQ1', 'NUDCD1', 'OPLAH', 'PARP10', 'PHF20L1', 'PKHD1L1', 'PLEC', 'PPP1R16A', 'PSCA', 'PTK2', 'PTP4A3', 'PUF60', 'PYCR3', 'RAD21', 'RECQL4', 'RHPN1', 'RNF139', 'RPL8', 'SAMD12', 'SCRIB', 'SCX', 'SHARPIN', 'SLA', 'SLC30A8', 'SLC39A4', 'SLC45A4', 'SLC52A2', 'SLURP1', 'SNTB1', 'SPATC1', 'SQLE', 'ST3GAL1', 'SYBU', 'TAF2', 'TATDN1', 'TBC1D31', 'TG', 'THEM6', 'TIGD5', 'TMEM65', 'TMEM71', 'TMEM74', 'TNFRSF11B', 'TONSL', 'TOP1MT', 'TRAPPC9', 'TRHR', 'TRIB1', 'TRMT12', 'TRPS1', 'TSNARE1', 'UTP23', 'VPS28', 'WASHC5', 'ZC3H3', 'ZFP41', 'ZFTRAF1', 'ZHX1', 'ZHX2', 'ZNF16', 'ZNF250', 'ZNF251', 'ZNF34', 'ZNF517', 'ZNF572', 'ZNF623', 'ZNF696', 'ZNF7', 'ZNF707']
Refined Community 81: ['ADGRB1', 'AGAP2', 'AHRR', 'AKAP9', 'ANK1', 'ARC', 'ARFGEF2', 'ARFRP1', 'ARRB1', 'ATRN', 'BLTP2', 'CACNA1G', 'CACNA1H', 'CHCT1', 'CHD7', 'CLPTM1L', 'CSMD3', 'CSNK1D', 'DCAF13', 'DGKE', 'DNAH17', 'DYRK2', 'ENTREP3', 'EP400', 'FABP4', 'FAM217B', 'FANCA', 'FGFR1', 'GDF6', 'GGA3', 'GOLGA7', 'GRIN2C', 'GSDMB', 'HCN3', 'HEPACAM2', 'HOOK3', 'HOXA3', 'HOXA4', 'IKBKB', 'JMJD8', 'KCNQ3', 'LPO', 'MSLNL', 'MYH7B', 'MYO7A', 'MYO9B', 'NCOA6', 'NDUFA3', 'NID2', 'NSMCE2', 'NUFIP2', 'OR4D2', 'PANX2', 'PCNT', 'PDCD6', 'PIGS', 'PKHD1L1', 'POP1', 'PPM1E', 'PUS1', 'RADIL', 'RCE1', 'RGS22', 'RIMS2', 'RSKR', 'SAMD9', 'SEPTIN4', 'SIGLEC1', 'SLC6A3', 'SLCO2B1', 'SPCS2', 'SPO11', 'SULF2', 'SYTL2', 'TAC4', 'TBK1', 'TESMIN', 'TG', 'THBS3', 'TRIM25', 'UBE2I', 'UQCC1', 'UTS2R', 'VPS13B', 'ZCCHC14', 'ZCCHC24', 'ZFC3H1', 'ZFHX4', 'ZFP64', 'ZMIZ1', 'ZNF461', 'ZNF529', 'ZNF569', 'ZNF707']
Refined Community 82: ['ATF2', 'CRX', 'CUX1', 'ETV5', 'FOXP2', 'HNF1A', 'HOXA4', 'IRF8', 'MEF2C', 'MYOD1', 'NFIX', 'NFKB1', 'NFYC', 'POU2F1', 'POU4F2', 'SOX15', 'STAT1', 'STAT4', 'TCF7L1', 'TP53', 'XBP1']
Refined Community 83: ['ACADSB', 'ACSM1', 'AK5', 'ALDH4A1', 'ANKH', 'ANPEP', 'ARHGEF5', 'ARMT1', 'AZGP1', 'BMP5', 'BST2', 'CA12', 'CACNA1D', 'CAPN3', 'CCDC170', 'CD151', 'CD59', 'CDK10', 'CELSR1', 'CELSR2', 'CHI3L1', 'CITED1', 'CLEC3B', 'CLMN', 'CNN1', 'COL4A5', 'CPB1', 'CRLF1', 'CYP1A1', 'CYP4B1', 'DACH1', 'DCT', 'DKK3', 'DNAJC12', 'DST', 'ECE1', 'EDN3', 'ERBB4', 'EREG', 'ESR1', 'EXOC7', 'FABP7', 'FBXO2', 'FGF2', 'FGFR3', 'FLNB', 'FRMD4A', 'FRZB', 'GATA3', 'GFRA1', 'GOLGA2P5', 'GRIA2', 'GRP', 'GSTT1', 'H2AC6', 'HBA1', 'HBB', 'HMGCS2', 'IFT140', 'IGF1R', 'IGFBP2', 'IGSF1', 'IL17RB', 'INPP4B', 'ITGA7', 'ITPR1', 'KCTD2', 'KDM4B', 'KLK10', 'KRT17', 'KRT5', 'KRT6B', 'LAMB3', 'LGALS4', 'LTBP3', 'LTBP4', 'LTF', 'MAPT', 'MATN2', 'MGP', 'MIA', 'MPPED2', 'MSX2', 'MYB', 'MYLK', 'NEDD4L', 'NPY1R', 'NPY2R', 'NQO1', 'OXTR', 'PDGFA', 'PDZK1', 'PEX6', 'PGGHG', 'PIK3R1', 'PKP1', 'PLAT', 'PPIP5K1', 'PPP1R3C', 'PTN', 'RABEP1', 'RABGAP1', 'RPL29P17', 'SAA1', 'SCUBE2', 'SDC4', 'SERPINA3', 'SERPINA5', 'SEZ6L2', 'SLC25A37', 'SLC27A6', 'SLC39A6', 'SSH3', 'STC1', 'STC2', 'SYMPK', 'SYNM', 'SYT17', 'TAPT1', 'TDRD12', 'TESC', 'TF', 'TIMM44', 'TMEM101', 'TPM2', 'TRA2A', 'UGT2B11', 'VTCN1', 'WFDC2', 'WIF1', 'WNT5A', 'XYLT2', 'ZDHHC11']
Refined Community 84: ['ABCC4', 'ADAM12', 'ADAMDEC1', 'ADCY7', 'ADGRE2', 'AHI1', 'AIF1', 'AIM2', 'APOBEC3B', 'APOC1', 'ARHGAP15', 'ARHGAP25', 'ASPM', 'AURKA', 'BANK1', 'BCAT1', 'BCL2A1', 'BCL2L11', 'BIRC5', 'BLM', 'BPNT2', 'BTK', 'BUB1B', 'CALCRL', 'CALU', 'CCDC88A', 'CCL11', 'CCL18', 'CCL3', 'CCL4', 'CCL8', 'CCN4', 'CCNA2', 'CCNB2', 'CCNE2', 'CCR1', 'CCR7', 'CD163', 'CD19', 'CD1E', 'CD2', 'CD27', 'CD38', 'CD3D', 'CD40', 'CD48', 'CD52', 'CD53', 'CD6', 'CD69', 'CD74', 'CD79A', 'CD79B', 'CD86', 'CDC20', 'CDH11', 'CDK1', 'CDKN3', 'CEACAM5', 'CEACAM6', 'CEMIP', 'CENPA', 'CENPF', 'CENPU', 'CEP55', 'CH25H', 'CHIT1', 'CHST11', 'CKS2', 'CLEC7A', 'CNTNAP2', 'COL10A1', 'COL11A1', 'COL5A1', 'COL5A2', 'COL6A3', 'COTL1', 'CPM', 'CPNE7', 'CPPED1', 'CREB3L1', 'CSF2RB', 'CST7', 'CTSC', 'CTSS', 'CXCL10', 'CXCL11', 'CXCL13', 'CXCL9', 'CXCR3', 'CXCR4', 'CYLD', 'CYTIP', 'DIO1', 'DLGAP5', 'DOCK2', 'DSC2', 'DSCC1', 'DTL', 'ECM1', 'EGFL6', 'EPPK1', 'EPYC', 'EVI2A', 'EVI2B', 'FANCI', 'FAP', 'FCER1G', 'FCGR1A', 'FCGR2A', 'FCGR3A', 'FCGR3B', 'FCMR', 'FKBP11', 'FN1', 'FPR3', 'FYB1', 'GAB2', 'GBP1', 'GDF15', 'GINS1', 'GINS2', 'GLIPR1', 'GNLY', 'GPR171', 'GPR183', 'GPR65', 'GPSM2', 'GRB2', 'GREM1', 'GZMA', 'GZMB', 'GZMK', 'HCLS1', 'HHIPL2', 'HLA-DMB', 'HLA-DQA1', 'HLA-DRB1', 'HLA-DRB6', 'HOPX', 'HS3ST3A1', 'HSD17B1', 'HYAL1', 'IFI30', 'IGHG1', 'IGHM', 'IGHV1-69', 'IGHV3-20', 'IGHV3-21', 'IGHV3-23', 'IGHV3-33', 'IGHV3-47', 'IGHV3-7', 'IGHV4-34', 'IGHV4-61', 'IGKV1D-13', 'IGKV1D-8', 'IGKV1OR2-108', 'IGLV3-10', 'IGLV3-19', 'IGLV4-60', 'IL18', 'IL18R1', 'IL2RG', 'IL32', 'INHBA', 'IRAG2', 'IRF8', 'ISG20', 'ITGA4', 'ITGAX', 'ITGB2', 'ITK', 'KDELR3', 'KIF11', 'KIF15', 'KIF20A', 'KIF2C', 'KMO', 'LAMP3', 'LAPTM5', 'LAT2', 'LCK', 'LCP2', 'LGALS2', 'LGALS9', 'LILRB1', 'LILRB2', 'LILRB4', 'LMO3', 'LOXL1', 'LPXN', 'LRP8', 'LRRC15', 'LST1', 'LTB', 'LY96', 'LYZ', 'MAD2L1', 'MAFB', 'MAN1A1', 'MAP4K1', 'MCM4', 'MELK', 'MMP1', 'MMP11', 'MMP12', 'MMP13', 'MMP19', 'MMP3', 'MMP9', 'MNDA', 'MS4A1', 'NARS2', 'NCAPG', 'NCF2', 'NDC80', 'NEK2', 'NET1', 'NID2', 'NPL', 'NUSAP1', 'OIP5', 'PBK', 'PCLAF', 'PIGR', 'PLA2G7', 'PLAC8', 'PLAUR', 'PLEK', 'PRC1', 'PRKCB', 'PRR16', 'PRUNE2', 'PSTPIP1', 'PTPRC', 'PTTG1', 'RAD51AP1', 'RASSF2', 'RET', 'RGS16', 'RGS4', 'RHOF', 'RRM2', 'RSAD2', 'S100P', 'SAMSN1', 'SELL', 'SELPLG', 'SERPINA6', 'SERPINE1', 'SLA', 'SLAMF8', 'SLC16A3', 'SLC18A2', 'SLC7A5', 'SLC7A7', 'SOAT1', 'SOX11', 'SP140', 'SPC25', 'SQLE', 'SRGN', 'STAG3', 'STAT4', 'STK17B', 'SULF1', 'TBC1D31', 'TCL1A', 'TDO2', 'TFEC', 'THEMIS2', 'TK1', 'TNC', 'TNFAIP6', 'TNFSF4', 'TOP2A', 'TPK1', 'TPX2', 'TRAC', 'TRAF1', 'TRAF3IP3', 'TRAT1', 'TRBC1', 'TRBC2', 'TRDC', 'TRGC1', 'TRIP13', 'TYMS', 'UTS2', 'VCAM1', 'VCAN']
Refined Community 85: ['ABL1', 'ACACA', 'AKT1', 'AR', 'ATF1', 'ATM', 'ATR', 'AURKA', 'BACH1', 'BAP1', 'BARD1', 'BCCIP', 'BLM', 'BRAP', 'BRCA2', 'BUB1B', 'CCNA2', 'CCNB1', 'CCND1', 'CCNE1', 'CDC25A', 'CDC25C', 'CDK1', 'CDK2', 'CDK4', 'CHEK1', 'CHEK2', 'CLSPN', 'CREBBP', 'CSNK2A1', 'CTBP1', 'DHX9', 'E2F1', 'E2F4', 'ELK1', 'EMSY', 'EP300', 'ESR1', 'FANCA', 'FANCD2', 'FANCG', 'FHL2', 'FLNA', 'H2AX', 'HDAC1', 'HDAC2', 'HMG20B', 'JAK1', 'JAK2', 'JUNB', 'KAT2B', 'KPNA2', 'LMO4', 'MAP2K3', 'MCM3', 'MDC1', 'MDM2', 'MED1', 'MLH1', 'MRE11', 'MSH2', 'MSH6', 'MYC', 'NBN', 'NELFB', 'NFKBIA', 'NMI', 'NPM1', 'NUFIP1', 'PEX5', 'PLK1', 'PLK3', 'PRKDC', 'RAD50', 'RAD51', 'RAD9A', 'RB1', 'RBBP4', 'RBBP7', 'RBBP8', 'RELA', 'RFC1', 'RFC2', 'RFC4', 'RPA1', 'RPA2', 'RPA3', 'SEM1', 'SMAD3', 'SMARCA4', 'SMC1A', 'SP1', 'STAT1', 'STAT5A', 'TERF1', 'TERF2', 'TP53', 'TP53BP1', 'TUBG1', 'XRCC6', 'ZNF350']
Refined Community 86: ['BUB3', 'CD47', 'DDX39B', 'NEMP1', 'PAXIP1', 'SSBP2', 'TBCA', 'TOP1', 'ZNF330']
Refined Community 87: ['ASF1A', 'AURKB', 'BLM', 'CAD', 'CDC20', 'CDC7', 'CDK1', 'CDKN2C', 'CENPA', 'CEP57', 'CNOT9', 'DDX39A', 'DEK', 'DNA2', 'DNMT1', 'EXOSC8', 'EZH2', 'FOXM1', 'HMGN4', 'HMMR', 'ILF3', 'KATNA1', 'LBR', 'MCM2', 'MCM4', 'MCM5', 'MCM6', 'MRE11', 'MSH2', 'MTF2', 'MYBL2', 'NASP', 'NCAPD2', 'NCK1', 'NDC80', 'PNN', 'RAD51AP1', 'RAD54L', 'RFC3', 'RFC4', 'RPA1', 'RPIA', 'SH2D1A', 'SKP2', 'SMC4', 'SNRPA', 'SRSF10', 'SRSF11', 'STMN1', 'TCERG1', 'TFDP1', 'TOPBP1', 'TTF2', 'UBE2S', 'USP1']
Refined Community 88: ['ANKRD26P1', 'C16orf87', 'CAPNS2', 'CDH8', 'CDYL2', 'CES1', 'CES1P1', 'CES5A', 'CNTNAP4', 'DNAJA2', 'DYNLRB2', 'GNAO1', 'GPT2', 'IRX6', 'ITFG1', 'LPCAT2', 'MAF', 'MMP2', 'MYLK3', 'NETO2', 'ORC6', 'PHKB', 'SHCBP1', 'SLC6A2', 'VPS35', 'WWOX']
Refined Community 89: ['ADGRG1', 'ADGRG3', 'ADGRG5', 'AFG3L1P', 'AKTIP', 'ANKRD26P1', 'C16orf78', 'C16orf87', 'CBLN1', 'CCL17', 'CCL22', 'CDH8', 'CETP', 'CFAP20', 'CFAP263', 'CHD9', 'CIAPIN1', 'CMIP', 'CNGB1', 'CNOT1', 'COQ9', 'CPNE2', 'CSNK2A2', 'CX3CL1', 'DBNDD1', 'DEF8', 'DOK4', 'DRC7', 'ENSG00000187185', 'FANCA', 'FTO', 'GAN', 'GAS8', 'GINS3', 'GOT2', 'GPT2', 'IL4', 'IRX3', 'KATNB1', 'KIFC3', 'MC1R', 'MPHOSPH10P1', 'MYLK3', 'NDRG4', 'NLRC5', 'OAZ1', 'ORC6', 'PLCG2', 'PLLP', 'POLR2C', 'PRDM7', 'PRSS54', 'PSMD7', 'PSME3IP1', 'RBL2', 'RPGRIP1L', 'RSPRY1', 'SHCBP1', 'SLC38A7', 'SPIRE2', 'TCF25', 'TUBB4A', 'VPS35', 'ZNF276']
Refined Community 90: ['ABAT', 'ABCD3', 'ABHD11', 'ACADSB', 'ACTG2', 'ALDH3B2', 'AMY1A', 'ANXA9', 'AREG', 'ARHGAP29', 'ASNS', 'ATP6V0A4', 'AUTS2', 'CADPS2', 'CD24P2', 'CD24P4', 'CEBPD', 'CELSR1', 'CELSR2', 'CFD', 'CLDN3', 'DEFB1', 'DNAJC12', 'DSC3', 'DST', 'DTNA', 'ELAPOR1', 'EPN3', 'ERBB3', 'ERBB4', 'FAM174B', 'FAM234B', 'GALNT7', 'GDF15', 'GPR87', 'GRAMD1C', 'GRB14', 'HOOK1', 'IER5', 'IL20RA', 'ISOC1', 'KIAA0040', 'KRT14', 'LAMA3', 'LETM1', 'LRBA', 'MAGED2', 'MAST4', 'MDK', 'MLPH', 'MMP10', 'MRPS30', 'MYB', 'MYO6', 'OPN3', 'PADI2', 'PDCD4', 'PDZK1IP1', 'PERP', 'PEX7', 'PLK2', 'PRKAR2B', 'PROM1', 'PTPRF', 'RAB11FIP1', 'RAPGEF5', 'SCNN1A', 'SERPINA1', 'SHANK2', 'SLC19A2', 'SLC1A4', 'SLC27A2', 'SORT1', 'STC2', 'TBC1D8', 'TBX3', 'TJP3', 'TM4SF1', 'TMED5', 'TP63', 'TTC39A', 'VAV3', 'WFS1', 'ZDHHC11', 'ZNF750']
Refined Community 91: ['ABCC4', 'ACKR3', 'ACSL4', 'ADAM12', 'ADAM19', 'ADCY7', 'ADGRF5', 'ADGRL4', 'ADORA3', 'AEBP1', 'AHNAK2', 'AKR1B1', 'AKT3', 'ALOX5', 'ANGPTL2', 'ANKRD6', 'ANXA1', 'APOE', 'ARHGEF40', 'ARL4C', 'ASPN', 'ATP10D', 'AXL', 'BGN', 'BICC1', 'C1orf54', 'C1QA', 'C1QB', 'C1R', 'C1S', 'C3AR1', 'CALD1', 'CASP1', 'CAVIN1', 'CAVIN3', 'CCL5', 'CCN1', 'CCN2', 'CCN4', 'CCND2', 'CCR1', 'CD163', 'CD53', 'CD55', 'CD74', 'CD93', 'CDH11', 'CELF2', 'CEMIP', 'CFH', 'CH25H', 'CILP', 'CLEC11A', 'CLEC2B', 'CLEC4A', 'CLEC7A', 'COL10A1', 'COL11A1', 'COL14A1', 'COL15A1', 'COL16A1', 'COL1A1', 'COL1A2', 'COL3A1', 'COL4A1', 'COL4A2', 'COL5A1', 'COL5A2', 'COL6A1', 'COL6A2', 'COL6A3', 'COL8A2', 'COLEC12', 'COMP', 'COPZ2', 'CPA3', 'CPQ', 'CPVL', 'CRISPLD2', 'CSF2RB', 'CTSK', 'CTSS', 'DAB2', 'DACT1', 'DCN', 'DNM3', 'DPY19L1', 'DPYD', 'DPYSL3', 'DSE', 'ECM2', 'EDNRA', 'EFEMP2', 'ELK3', 'EML1', 'EMP1', 'EMP3', 'ENPEP', 'ENTPD1', 'ERG', 'ESM1', 'ETV1', 'ETV5', 'EVI2A', 'EVI2B', 'F2R', 'FAP', 'FAS', 'FBLN2', 'FBN1', 'FBN2', 'FCER1G', 'FCGR1A', 'FCGR2A', 'FCGR3B', 'FERMT2', 'FGL2', 'FKBP11', 'FLI1', 'FN1', 'FOXN3', 'FSTL1', 'FTL', 'FXYD5', 'FYN', 'FZD2', 'GAS1', 'GBP1', 'GEM', 'GFPT2', 'GIMAP4', 'GIMAP6', 'GLIPR1', 'GLRX', 'GLS', 'GPNMB', 'GPR65', 'GPX7', 'GREM1', 'GRK5', 'HCLS1', 'HEG1', 'HEY1', 'HLA-DMA', 'HLA-DMB', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DQB2', 'HLA-DRA', 'HLA-DRB1', 'HLA-DRB4', 'HLA-DRB6', 'HMOX1', 'HPGDS', 'HSD17B11', 'HSD17B6', 'HTR2B', 'HTRA1', 'IFFO1', 'IFI16', 'IFI30', 'IGFBP3', 'IGFBP7', 'IGLC2', 'IL18', 'IL1R1', 'INHBA', 'IRAK3', 'IRF8', 'ISLR', 'ITGBL1', 'JAM3', 'JCAD', 'KCND2', 'KCNJ2', 'KCTD12', 'KDR', 'LAMA4', 'LAMB1', 'LAMC1', 'LAPTM5', 'LCP2', 'LDB2', 'LDLRAD4', 'LGALS1', 'LHFPL6', 'LMCD1', 'LMO2', 'LOX', 'LOXL1', 'LOXL2', 'LPXN', 'LRRC15', 'LRRC17', 'LUM', 'LY96', 'LYN', 'MAF', 'MAFB', 'MAP1B', 'MAP4K4', 'MEF2C', 'MEIS2', 'MERTK', 'MFAP2', 'MFAP5', 'MICAL2', 'MME', 'MMP1', 'MMP11', 'MMP13', 'MMP2', 'MMP3', 'MNDA', 'MRC2', 'MS4A4A', 'MS4A6A', 'MSN', 'MSR1', 'MXRA5', 'MXRA7', 'MXRA8', 'MYL9', 'NBL1', 'NDN', 'NID1', 'NID2', 'NNMT', 'NOX4', 'NREP', 'NRP1', 'NUAK1', 'OLFML1', 'OLFML2B', 'OLFML3', 'OLR1', 'P2RY14', 'PALLD', 'PAM', 'PARVA', 'PDE10A', 'PDGFRA', 'PDGFRB', 'PDLIM2', 'PECAM1', 'PKD2', 'PLAU', 'PLAUR', 'PLCG2', 'PLN', 'PLS3', 'PLSCR4', 'PLXDC1', 'PLXNC1', 'PMEPA1', 'POSTN', 'PPIC', 'PRNP', 'PROS1', 'PRRX1', 'PSMB9', 'PTEN', 'PTPRC', 'PTPRM', 'QKI', 'RARRES2', 'RASSF2', 'RCAN2', 'RCBTB2', 'RECK', 'RGCC', 'RGS1', 'RGS2', 'RGS4', 'RNASE6', 'ROBO1', 'ROR1', 'S100A4', 'SAMD4A', 'SAMSN1', 'SCG5', 'SDC2', 'SEC23A', 'SEPTIN11', 'SEPTIN6', 'SERPINE2', 'SERPINF1', 'SERPING1', 'SERPINH1', 'SGCB', 'SGCD', 'SGK1', 'SHOX2', 'SKAP2', 'SLAMF8', 'SLC16A3', 'SLC2A3', 'SLC7A7', 'SLC9A6', 'SMAD7', 'SMCO4', 'SNAI2', 'SOBP', 'SPARC', 'SPARCL1', 'SPHK1', 'SPOCK1', 'SPON1', 'SPON2', 'SRGN', 'SRPX2', 'SSPN', 'ST3GAL6', 'SULF1', 'TACC1', 'TCF4', 'TDO2', 'TFEC', 'TGFB1I1', 'TGFBI', 'TGFBR2', 'TGM2', 'THBS2', 'THSD7A', 'THY1', 'TIMP3', 'TLE4', 'TLR7', 'TM6SF1', 'TMEFF1', 'TMEM158', 'TMEM204', 'TMEM47', 'TNFAIP6', 'TNFSF4', 'TNS3', 'TP53I3', 'TRDC', 'TRIM22', 'TRPC1', 'TUBB6', 'TWIST1', 'TYROBP', 'VCAN', 'WIPF1', 'WWTR1', 'XYLT1', 'ZEB1', 'ZEB2', 'ZFPM2']
Refined Community 92: ['AAGAB', 'AAMDC', 'ABAT', 'ABCA12', 'ABCA3', 'ABCA8', 'ABCB1', 'ABCC3', 'ABCC6', 'ABCC8', 'ABCG1', 'ABLIM3', 'ACACA', 'ACACB', 'ACADSB', 'ACKR1', 'ACKR3', 'ACOX2', 'ACSL3', 'ACSM5', 'ADCY1', 'ADCY9', 'ADGRB2', 'ADH1B', 'ADIPOQ', 'ADIRF', 'ADRA2A', 'AFP', 'AGL', 'AGR2', 'AGTR1', 'AHNAK', 'AK5', 'AKR7A3', 'ALB', 'ALCAM', 'ALDH1A1', 'ALDH3B2', 'ALDH4A1', 'ALG12', 'ALG8', 'ALOX15B', 'AMDHD2', 'AMIGO2', 'ANG', 'ANO1', 'ANOS1', 'ANXA9', 'AOX1', 'APBB2', 'APOD', 'APPBP2', 'AQR', 'AR', 'AREG', 'ARHGAP32', 'ARHGAP35', 'ARL3', 'ARMT1', 'ASAH1', 'ASCL1', 'ASPA', 'ASPH', 'ASPN', 'ASTN2', 'ATP1A2', 'ATP2A3', 'ATP7B', 'ATP8A1', 'ATP8A2', 'ATRNL1', 'AZGP1', 'B9D1', 'BAIAP3', 'BATF', 'BBS4', 'BCAM', 'BCAS1', 'BCAS4', 'BCL2', 'BEX1', 'BIK', 'BLVRA', 'BLVRB', 'BMERB1', 'BMI1', 'BMP4', 'BMPR1B', 'BRINP2', 'BRINP3', 'BTG2', 'C14orf132', 'C1orf21', 'C1QTNF3', 'C2CD2L', 'C3orf52', 'C4A', 'CA12', 'CACNA1D', 'CACNA1H', 'CACNA2D2', 'CACNA2D3', 'CACNG1', 'CACNG4', 'CAMK2B', 'CAMK2N1', 'CAMP', 'CANT1', 'CAPN9', 'CASD1', 'CATSPERB', 'CBFA2T3', 'CCDC106', 'CCDC170', 'CCL14', 'CCN5', 'CCND1', 'CCNG2', 'CCNO', 'CD36', 'CDK12', 'CDS1', 'CEACAM5', 'CEACAM6', 'CELSR1', 'CEP15', 'CERS4', 'CERS6', 'CFAP45', 'CFAP69', 'CFB', 'CFD', 'CHAD', 'CHN2', 'CHRD', 'CHST1', 'CILP', 'CITED1', 'CLBA1', 'CLCA2', 'CLDN5', 'CLEC3B', 'CLGN', 'CLSTN2', 'CNR1', 'CNTNAP2', 'COG7', 'COL14A1', 'COL4A5', 'COL4A6', 'COMP', 'COQ4', 'COQ7', 'CPA3', 'CPB1', 'CPD', 'CPE', 'CRAT', 'CREB3L1', 'CRIP1', 'CRIP2', 'CRISP3', 'CROT', 'CRY2', 'CSAD', 'CSF3R', 'CST3', 'CST5', 'CX3CR1', 'CXCL12', 'CXCL14', 'CXXC4', 'CYB561', 'CYB5A', 'CYBRD1', 'CYP21A2', 'CYP26A1', 'CYP2A6', 'CYP2B6', 'CYP2B7P', 'CYP2E1', 'CYP4B1', 'CYP4F8', 'DACH1', 'DCAF10', 'DCLK1', 'DCN', 'DCXR', 'DDO', 'DENND1B', 'DEPTOR', 'DGKD', 'DGLUCY', 'DHCR24', 'DHRS2', 'DIO1', 'DLG3', 'DLG5', 'DNAAF1', 'DNAAF11', 'DNAI7', 'DNAJC1', 'DNAJC12', 'DNALI1', 'DNMBP', 'DPT', 'DUSP4', 'DUSP5', 'DUSP6', 'DZANK1', 'DZIP3', 'ECM1', 'EEF1A2', 'EFCAB2', 'EFHC1', 'EGR3', 'ELAPOR1', 'ELF1', 'ELOVL2', 'ELOVL5', 'EMCN', 'ENPP1', 'ENSG00000301761', 'ENTPD3', 'EPHA3', 'EPHX2', 'EPOR', 'EPS8L1', 'ERBB2', 'ERBB3', 'ERBB4', 'ERLIN2', 'ESR1', 'ESRRG', 'EVA1B', 'EVL', 'EXOC7', 'F13A1', 'F7', 'FAAH', 'FABP4', 'FAH', 'FAIM2', 'FAM110B', 'FAM131B', 'FAM174B', 'FAM234B', 'FAM83E', 'FASN', 'FAXDC2', 'FBLN1', 'FBP1', 'FBXL7', 'FCER1A', 'FCMR', 'FDXR', 'FGB', 'FGFR3', 'FGG', 'FLRT3', 'FMO5', 'FOSB', 'FRY', 'FUT8', 'G6PC3', 'GALNT10', 'GALNT6', 'GALNT7', 'GAMT', 'GASK1B', 'GATA2', 'GATA3', 'GATB', 'GDF15', 'GFRA1', 'GGT1', 'GJA1', 'GJA4', 'GLRB', 'GNA14', 'GNG13', 'GNMT', 'GOLGA2P5', 'GP2', 'GPC1-AS1', 'GPC3', 'GPD1', 'GPD1L', 'GPRC5A', 'GPRC5C', 'GREB1', 'GREB1L', 'GRIA2', 'GRP', 'GSE1', 'GSTM3', 'GSTT2', 'GSTZ1', 'GTF2H2C', 'GUSBP14', 'GUSBP3', 'HBA1', 'HCFC1R1', 'HDAC11', 'HEATR6', 'HEBP1', 'HEMK1', 'HGD', 'HGF', 'HHEX', 'HIGD1B', 'HMGCS2', 'HNMT', 'HOXB2', 'HOXB6', 'HPGD', 'HPN', 'HPX', 'HSDL2', 'HSPA2', 'HSPB1', 'HSPB8', 'ICA1', 'IFT122', 'IFT140', 'IGF1', 'IGF1R', 'IGFBP4', 'IKBKB', 'IL20RA', 'IL33', 'IL6ST', 'INPP4B', 'INPP5J', 'IQGAP2', 'IRS1', 'IRX5', 'ISYNA1', 'ITGA7', 'ITGBL1', 'ITM2A', 'ITPR1', 'IVD', 'JMJD7-PLA2G4B', 'KAT6B', 'KAZALD1', 'KCNAB1', 'KCND3', 'KCNE4', 'KCNJ3', 'KCNMA1', 'KCTD3', 'KDM4B', 'KIAA0040', 'KIAA0319L', 'KIF13B', 'KIF16B', 'KIF5C', 'KMO', 'KRT18', 'KRT19', 'KRT8', 'LAMP5', 'LDLRAD4', 'LEP', 'LGALS8', 'LIAS', 'LIN7A', 'LINC01503', 'LINC01949', 'LIPE', 'LMF1', 'LONP2', 'LRBA', 'LRIG1', 'LRRC17', 'LRRC31', 'LRRN3', 'MACIR', 'MAGED2', 'MAK', 'MAN1A1', 'MAN1C1', 'MAOA', 'MAP3K12', 'MAP9', 'MAPT', 'MAST4', 'MATN3', 'MCCC2', 'MCF2L', 'MDM1', 'MED13L', 'MED24', 'MEGF9', 'MEOX2', 'METRN', 'MFAP4', 'MFAP5', 'MIPEP', 'MISP', 'MKNK2', 'MLPH', 'MMP17', 'MON2', 'MREG', 'MRPS30', 'MS4A2', 'MSL1', 'MSMB', 'MSX2', 'MTARC1', 'MTUS1', 'MUC1', 'MUC6', 'MYB', 'MYH11', 'MYO6', 'MYT1', 'MZF1', 'N4BP2L2', 'NADSYN1', 'NAIP', 'NAT1', 'NAT2', 'NAV3', 'NBEA', 'NCOA3', 'NEBL', 'NEDD4L', 'NELL2', 'NF1', 'NFIA', 'NHERF1', 'NKAIN1', 'NME3', 'NME5', 'NOVA1', 'NPDC1', 'NPIPB3', 'NPY1R', 'NQO1', 'NR2F1', 'NR4A2', 'NRIP1', 'NTRK2', 'NUCB2', 'NUDT4', 'OGN', 'OLFML3', 'OMD', 'PAMR1', 'PAN2', 'PATJ', 'PAX2', 'PBLD', 'PBX1', 'PCBP2', 'PCDHA9', 'PCK1', 'PCLO', 'PCSK5', 'PCSK6', 'PDCD4', 'PDGFD', 'PDS5B', 'PDZK1', 'PEX11A', 'PGAP3', 'PGGHG', 'PGR', 'PIDD1', 'PIERCE1', 'PIEZO2', 'PIP', 'PLA1A', 'PLA2G10', 'PLAAT2', 'PLAAT3', 'PLAAT4', 'PLAT', 'PLCL1', 'PLEKHF2', 'PLIN1', 'PLXNB1', 'PNMT', 'PNPLA2', 'PNPLA4', 'POLD4', 'POLM', 'PPDPF', 'PPIP5K1', 'PPP1R3C', 'PRKACB', 'PRLR', 'PRSS23', 'PSD3', 'PTGER3', 'PTHLH', 'PTK6', 'PTP4A2', 'PTPRN2', 'PTPRT', 'PVALB', 'QDPR', 'RAB11FIP1', 'RAB17', 'RAB26', 'RAB27B', 'RAB38', 'RABEP1', 'RAI2', 'RALGPS1', 'RALGPS2', 'RAPGEF3', 'RAPGEFL1', 'RARA', 'RASSF1', 'RBM19', 'RBM47', 'RBP4', 'RDH16', 'REEP1', 'REEP5', 'REPS2', 'RET', 'RETREG1', 'RGS11', 'RGS5', 'RHBDL1', 'RHOB', 'RHOH', 'RMND1', 'RNASE4', 'RND1', 'RNF41', 'RPS20P22', 'RTN1', 'SCCPDH', 'SCD', 'SCGB1D2', 'SCGB2A2', 'SCGN', 'SCNN1A', 'SCUBE2', 'SEC14L2', 'SELENBP1', 'SELENOP', 'SEMA3B', 'SEMA3C', 'SEMA3G', 'SEPTIN8', 'SERHL', 'SERHL2', 'SERPINA5', 'SERPINA6', 'SERPINI1', 'SFRP4', 'SGK3', 'SGSH', 'SH3BGRL', 'SIDT1', 'SIGIRR', 'SIGLEC15', 'SIL1', 'SIX1', 'SLC16A6', 'SLC18A2', 'SLC19A2', 'SLC1A1', 'SLC1A4', 'SLC22A18', 'SLC24A1', 'SLC26A3', 'SLC27A2', 'SLC2A10', 'SLC2A8', 'SLC35E3', 'SLC38A1', 'SLC39A6', 'SLC44A4', 'SLC48A1', 'SLC49A3', 'SLC4A7', 'SLC4A8', 'SLC6A4', 'SLC7A8', 'SLC9A1', 'SLIT3', 'SMPD3', 'SNCG', 'SNED1', 'SNHG14', 'SNPH', 'SNX1', 'SOCS2', 'SORD', 'SORL1', 'SPAG16', 'SPARCL1', 'SPATA7', 'SPDEF', 'SPHK2', 'SPRR3', 'SPTLC2', 'SREBF1', 'SRI', 'SSH3', 'SSTR2', 'ST6GALNAC2', 'STARD3', 'STAU2', 'STC2', 'STK32B', 'STK39', 'STS', 'SUPT6H', 'SVEP1', 'SYBU', 'SYT1', 'SYT17', 'SYTL2', 'TAT', 'TBC1D19', 'TBC1D30', 'TBC1D9', 'TBX3', 'TCEAL4', 'TCN1', 'TESMIN', 'TFAP2B', 'TFF1', 'TFF3', 'TFPI2', 'THBS4', 'TIMP4', 'TJP3', 'TK2', 'TKFC', 'TM7SF2', 'TMC5', 'TMEM101', 'TMEM143', 'TMEM265', 'TMEM30B', 'TNFSF10', 'TNIK', 'TNNT1', 'TNXA', 'TOB1', 'TOM1L1', 'TOX3', 'TP53TG1', 'TPPP3', 'TPSAB1', 'TRAF5', 'TRAK1', 'TREM2', 'TRGC1', 'TRIL', 'TRIM3', 'TRIM36', 'TRIM68', 'TSKU', 'TSPAN1', 'TSPAN13', 'TSPAN8', 'TTC12', 'TTC39A', 'TTC9', 'TUBA3E', 'UBA7', 'UBL3', 'UCN', 'UCP2', 'UGCG', 'UGDH', 'UGT2B11', 'UGT2B15', 'UNC119', 'VAV3', 'VIPR1', 'WIPF2', 'WWOX', 'WWP1', 'XIST', 'XRCC4', 'YIPF6', 'ZBTB16', 'ZBTB18', 'ZCCHC24', 'ZMYND8', 'ZNF329', 'ZNF385D', 'ZNF43', 'ZNF446', 'ZNF552', 'ZNF587', 'ZNF652', 'ZNF839', 'ZNF91', 'ZSCAN18']
Refined Community 93: ['ACAN', 'ACP1', 'ACTA1', 'ACTG2', 'ACTL8', 'ACTN1', 'ADAM17', 'ADCY2', 'ADD2', 'ADGRG2', 'ADGRG6', 'ADM', 'ADORA2B', 'AGBL5', 'AK2', 'AMD1', 'ANGPT1', 'ANGPTL4', 'ANP32E', 'ANXA3', 'ANXA8', 'APBA2', 'APOBEC3A', 'APOBEC3B', 'AQP5', 'AQP9', 'ARHGEF9', 'ARL4C', 'ARNT', 'ARPC4', 'ART3', 'ARTN', 'ASNS', 'ASPM', 'ATAD2', 'ATP11A', 'ATP13A3', 'BACE2', 'BAG2', 'BARX1', 'BBOX1', 'BCL11A', 'BCL2A1', 'BIRC5', 'BMAL2', 'BOP1', 'BTG3', 'BUB1', 'BYSL', 'C16orf95', 'CA5BP1', 'CA6', 'CA9', 'CACNA1A', 'CALB2', 'CALD1', 'CALML5', 'CALU', 'CAND2', 'CAPN6', 'CBS', 'CCDC88A', 'CCKBR', 'CCL18', 'CCL2', 'CCL20', 'CCL5', 'CCL7', 'CCL8', 'CCN2', 'CCNA2', 'CCNB2', 'CCNE1', 'CD24P2', 'CD24P4', 'CD38', 'CD44', 'CD59', 'CDC20', 'CDC25A', 'CDC42EP1', 'CDC45', 'CDC5L', 'CDCA3', 'CDCA8', 'CDH19', 'CDH2', 'CDH3', 'CDK1', 'CDKAL1', 'CDKN2A', 'CDT1', 'CDYL', 'CEACAM1', 'CENPA', 'CENPE', 'CENPF', 'CENPN', 'CEP170', 'CEP55', 'CHAC1', 'CHAF1A', 'CHAF1B', 'CHEK1', 'CHI3L1', 'CHI3L2', 'CHML', 'CHODL', 'CHRM3', 'CHST11', 'CHST3', 'CLCN4', 'CLDN1', 'CLDN10', 'CLEC7A', 'CLIC4', 'CLIP4', 'CNN3', 'COCH', 'COL11A2', 'COL2A1', 'COL4A2', 'COL9A3', 'COTL1', 'CP', 'CRABP1', 'CRLF1', 'CRYAB', 'CSPG4', 'CSRP2', 'CTAG1B', 'CTSV', 'CWH43', 'CX3CL1', 'CXADR', 'CXCL1', 'CXCL10', 'CXCL11', 'CXCL5', 'CXCL8', 'CYB5R2', 'CYP26B1', 'CYP39A1', 'DBN1', 'DCPS', 'DCX', 'DDIT4', 'DDX17', 'DEFB1', 'DENND2A', 'DEPP1', 'DKC1', 'DKK1', 'DLAT', 'DLGAP5', 'DLX5', 'DMRT1', 'DNAH17', 'DNAJB4', 'DNAJC6', 'DNM3', 'DSC2', 'DSC3', 'DSG1', 'DSG2', 'DSG3', 'DUOX1', 'DUSP9', 'DYNC1I1', 'DZIP1', 'E2F3', 'ECHDC1', 'EDN1', 'EDN2', 'EGFL6', 'EGFR', 'ELF4', 'ELF5', 'ELN', 'ELOVL4', 'EMC1', 'EN1', 'ENO1', 'EPB41L2', 'EPHB3', 'EPHX3', 'EPRS1', 'EXO1', 'EXTL1', 'FABP5', 'FABP7', 'FAM107A', 'FAM171A1', 'FANCA', 'FAT1', 'FAT2', 'FBXO17', 'FERMT1', 'FGF9', 'FGFBP1', 'FLNA', 'FNDC3B', 'FNDC4', 'FOLH1', 'FOLR1', 'FOSL1', 'FOXC1', 'FOXD1', 'FOXM1', 'FSCN1', 'FUT9', 'FXYD5', 'FZD7', 'FZD9', 'GABBR1', 'GABBR2', 'GABRP', 'GAL', 'GALNT12', 'GALNT14', 'GART', 'GATA6', 'GBP1', 'GDI2', 'GGH', 'GJB3', 'GJC1', 'GLDC', 'GLS', 'GMDS', 'GPM6B', 'GPR161', 'GPR19', 'GPR37', 'GPRC5B', 'GPSM2', 'GSTA1', 'GSTP1', 'GTSE1', 'H2BC12L', 'HACD1', 'HAPLN1', 'HBEGF', 'HELLS', 'HEPH', 'HLA-G', 'HMGA1', 'HNRNPU', 'HOMER3', 'HOXA9', 'HPSE', 'HRK', 'HS3ST1', 'HSPA4L', 'HSPA6', 'HSPB2', 'HTN1', 'HYAL1', 'ICAM1', 'ID4', 'IDO1', 'IFI16', 'IFIH1', 'IFNAR2', 'IFRD1', 'IGF2BP2', 'IGF2BP3', 'IGHG1', 'IGHV1-69', 'IGHV3-23', 'IGHV3-33', 'IGHV3-7', 'IGHV4-34', 'IGHV4-61', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-37', 'IGKV1D-39', 'IGKV1OR1-1', 'IGKV1OR2-108', 'IGKV2D-28', 'IGLC2', 'IGLL3P', 'IGLV2-14', 'IGLV3-10', 'IL12RB2', 'IL1R2', 'IL27RA', 'IL32', 'ILRUN', 'IMPA2', 'INAVA', 'ING1', 'IQCG', 'IRX4', 'ISG20', 'ITGA6', 'ITGB8', 'ITM2C', 'ITPKB', 'JRKL', 'KCNG1', 'KCNK5', 'KCNN4', 'KCTD14', 'KHDRBS3', 'KIF18B', 'KIF1A', 'KIFC1', 'KIRREL1', 'KIT', 'KLF5', 'KLF6', 'KLHL24', 'KLHL7', 'KLK10', 'KLK5', 'KLK6', 'KLK7', 'KLK8', 'KRT14', 'KRT15', 'KRT16', 'KRT17', 'KRT23', 'KRT4', 'KRT5', 'KRT6A', 'KRT6B', 'KRT81', 'KRT83', 'L1CAM', 'LAD1', 'LALBA', 'LAMB3', 'LAMC2', 'LAMP3', 'LBR', 'LCN2', 'LDHB', 'LEFTY2', 'LGALSL', 'LGR5', 'LHX2', 'LMO4', 'LOX', 'LRFN4', 'LRP12', 'LRP8', 'LTBP1', 'LY6D', 'LYN', 'LYPD1', 'LZTS3', 'MACROH2A2', 'MAGEA2', 'MAGEA3', 'MAGEA4', 'MAGEA9', 'MAGED4B', 'MAGOH', 'MALL', 'MAP2', 'MAPK8', 'MARCO', 'MCF2L-AS1', 'MCM10', 'MCM2', 'MCM4', 'MCM5', 'MCM7', 'MCOLN3', 'MDC1', 'MDFI', 'MEAK7', 'MELK', 'MET', 'MFAP2', 'MFGE8', 'MIA', 'MICALL1', 'MID1', 'MKI67', 'MLC1', 'MLLT10', 'MLLT11', 'MMP1', 'MMP12', 'MMP14', 'MMP7', 'MOB3B', 'MOCOS', 'MPZL2', 'MRAS', 'MRPS12', 'MSH6', 'MSLN', 'MSN', 'MT1X', 'MTAP', 'MTMR2', 'MUC16', 'MYBL1', 'MYBL2', 'MYC', 'MYLK', 'MYO10', 'NAP1L1', 'NCALD', 'NCAM1', 'NCAN', 'NCAPH', 'NDC80', 'NDRG1', 'NDRG2', 'NDUFA4L2', 'NEFH', 'NEK2', 'NES', 'NFE2L3', 'NFIB', 'NFIL3', 'NFIX', 'NKX2-5', 'NMU', 'NPR3', 'NPTX2', 'NR6A1', 'NRTN', 'NSD2', 'NT5DC2', 'NUDT1', 'NUDT11', 'NUP93', 'NUSAP1', 'NXN', 'OBP2B', 'OCA2', 'ODAM', 'ODC1', 'OGFRL1', 'ORC1', 'P3H2', 'PADI2', 'PAK1IP1', 'PALS2', 'PART1', 'PAX6', 'PBK', 'PCDHB3', 'PCOLCE2', 'PCP4', 'PCSK1N', 'PDAP1', 'PDE9A', 'PDGFRA', 'PDK1', 'PDPN', 'PDXK', 'PDZK1IP1', 'PEG10', 'PEG3', 'PELI1', 'PERP', 'PFKP', 'PFN2', 'PGBD5', 'PHGDH', 'PI3', 'PKP1', 'PLA2G4A', 'PLAAT1', 'PLAGL1', 'PLCB4', 'PLCH1', 'PLEKHB1', 'PLIN2', 'PLOD2', 'PLSCR1', 'PML', 'POU5F1', 'PPM1E', 'PPP1R14B', 'PRAME', 'PRDM13', 'PRELP', 'PRG2', 'PRKCA', 'PRKD3', 'PRKX', 'PRKY', 'PROM1', 'PRRX2', 'PRSS16', 'PSAT1', 'PTGFR', 'PTGS2', 'PTK7', 'PTP4A3', 'PTPN14', 'PTPRZ1', 'PTTG1', 'PTX3', 'PVR', 'PXDN', 'PYGB', 'QKI', 'QPCT', 'RAB23', 'RAB6B', 'RAD21', 'RAD51AP1', 'RAD54L', 'RAP2A', 'RARRES1', 'RASA2', 'RASAL1', 'RBMS1', 'RBP1', 'RCAN1', 'REG1A', 'RGS2', 'RGS20', 'RIPK4', 'RMND5A', 'RND3', 'ROPN1', 'RPL39L', 'RRAGD', 'RRAS2', 'RTP4', 'RUNX3', 'RYR1', 'S100A1', 'S100A10', 'S100A2', 'S100A4', 'S100A6', 'S100A8', 'S100A9', 'S100B', 'SCHIP1', 'SCRG1', 'SDC2', 'SERPINB2', 'SERPINB3', 'SERPINB4', 'SERPINH1', 'SET', 'SFN', 'SFRP1', 'SHCBP1', 'SHOX2', 'SIGMAR1', 'SIK1', 'SIM1', 'SIX3', 'SKP2', 'SLAMF8', 'SLC15A1', 'SLC16A1', 'SLC25A37', 'SLC26A2', 'SLC27A6', 'SLC2A3', 'SLC2A6', 'SLC34A2', 'SLC43A3', 'SLC6A14', 'SLC6A15', 'SLC7A5', 'SLPI', 'SMCO4', 'SMPDL3B', 'SNTB1', 'SOCS3', 'SOD2', 'SOSTDC1', 'SOX10', 'SOX11', 'SOX15', 'SOX9', 'SPIB', 'SPP1', 'SRD5A1', 'SSRP1', 'ST3GAL6', 'ST8SIA1', 'STEAP1B', 'STIP1', 'STXBP6', 'SUSD5', 'SYNCRIP', 'SYNM', 'TAGLN2', 'TAP1', 'TBX19', 'TCEAL2', 'TCF7L1', 'TCP11L1', 'TDRD12', 'TEAD4', 'TFCP2L1', 'TFDP1', 'TFF2', 'THEMIS2', 'TIMM44', 'TM4SF1', 'TMCC2', 'TMEFF1', 'TMEM100', 'TMEM158', 'TMEM45A', 'TMSB15A', 'TMSB15B', 'TNFRSF11B', 'TNFRSF21', 'TNNI2', 'TOX', 'TP53', 'TPM2', 'TPX2', 'TRA2B', 'TREM1', 'TRIM2', 'TRIM29', 'TRIP13', 'TSN', 'TSPYL5', 'TTK', 'TTLL4', 'TTYH1', 'TUBB2A', 'TUBB2B', 'TUSC3', 'TXNL4A', 'TYMS', 'UBE2E3', 'UBE2S', 'UCHL1', 'UCK2', 'UGT8', 'UPP1', 'VASH2', 'VEGFA', 'VGLL1', 'VLDLR', 'VNN1', 'WARS1', 'WIF1', 'WNT5B', 'WNT6', 'WWTR1', 'YBX1', 'YBX3', 'YES1', 'YWHAE', 'YWHAZ', 'ZIC1', 'ZNF124', 'ZNF532', 'ZNF750']
Refined Community 94: ['FGFR2', 'GABRP', 'NPY1R', 'PMAIP1', 'SOX10']
Refined Community 95: ['ABCA12', 'ABCC2', 'ABCC3', 'ABCC6', 'ACE2', 'ACSL1', 'AKR1B10', 'ALCAM', 'ALDH3B2', 'ALDH4A1', 'ANP32A', 'APOD', 'AQP3', 'AR', 'ASPH', 'ASS1', 'ATP2C2', 'BIK', 'BLTP2', 'BRINP3', 'C2orf72', 'CAMP', 'CATSPERB', 'CD164', 'CD24P4', 'CD46', 'CDK10', 'CDK12', 'CEACAM6', 'CLCA2', 'CLIC3', 'CORO1B', 'CPD', 'CRISP2', 'CRISP3', 'CRYBG1', 'CYB561', 'DDC', 'DUSP6', 'EFNA3', 'EIF5A', 'ELL2', 'EPYC', 'ERBB2', 'FA2H', 'FAIM2', 'FASN', 'FFAR2', 'FGB', 'FGG', 'FUT3', 'G6PD', 'GALNT6', 'GCAT', 'GGT1', 'GOT1', 'GRB7', 'GSDMB', 'GSE1', 'HMGCS2', 'HPX', 'IGF2R', 'IGHV3-23', 'IGHV3-7', 'IGHV4-61', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-37', 'IGKV1D-39', 'IGKV1OR2-108', 'IGKV3-20', 'IGKV4-1', 'IGLC2', 'IL13RA1', 'ITGB6', 'KIAA0319L', 'KMO', 'KRT24', 'KYNU', 'LAMA4', 'LBP', 'LDLR', 'LPCAT3', 'LRP1', 'LRRC31', 'MAB21L4', 'MED1', 'MED24', 'MMP15', 'MPHOSPH6', 'MUC1', 'NCOA3', 'NOL3', 'NQO1', 'NUDT4', 'NVL', 'ORM2', 'P4HB', 'PADI3', 'PAPSS2', 'PEX11A', 'PGAP3', 'PIP', 'PLEKHA6', 'PLPBP', 'PNMT', 'PPP1R10', 'PRMT7', 'PRODH', 'PSMD3', 'QPRT', 'RRP7A', 'S100A7', 'S100A8', 'S100A9', 'SCD', 'SCGB1D2', 'SCGB2A2', 'SCNN1A', 'SEC24D', 'SEPTIN8', 'SERHL', 'SERHL2', 'SLC12A1', 'SLC2A10', 'SLC37A1', 'SLC44A4', 'SLPI', 'SNPH', 'SPTLC2', 'SQSTM1', 'SRPK3', 'SRSF6', 'STARD3', 'SUPT6H', 'TFAP2B', 'TM7SF2', 'TMC5', 'TMED2', 'TMPRSS2', 'TRAF4', 'TRIM3', 'TRPM4', 'TRPV6', 'TSKU', 'TSPAN1', 'TSPAN8', 'UBE2M', 'UCP2', 'UGT2B11', 'WIPF2', 'WWC2']
Refined Community 96: ['ANP32E', 'CCNE1', 'CDC20', 'CDC45', 'CDCA3', 'CDCA8', 'CENPN', 'DKC1', 'KRT16', 'LRP8', 'MYBL2', 'NCAPH', 'NDC80', 'SLC7A5', 'TOP2A', 'TPX2', 'TTLL4', 'VGLL1']
Refined Community 97: ['ABCA8', 'ADH1B', 'ADIPOQ', 'ADIRF', 'ADRA2A', 'AREG', 'ARMT1', 'ASPN', 'ATP1A2', 'ATRNL1', 'BTG2', 'C1orf21', 'CCN5', 'CFD', 'CHRDL1', 'CIDEC', 'CILP', 'COL14A1', 'CPA3', 'CXCL14', 'CYP4B1', 'DHRS2', 'DPT', 'DUSP1', 'ELOVL2', 'ENPP2', 'ESR1', 'F13A1', 'FABP4', 'FAXDC2', 'FBLN1', 'FHL1', 'FOSB', 'G0S2', 'GFRA1', 'GPD1', 'HBB', 'HLF', 'HMGCS2', 'HOXB2', 'HOXB6', 'IGF1', 'ITGA7', 'ITIH5', 'LAMA2', 'LAMP5', 'LEP', 'LEPR', 'LIPE', 'LMOD1', 'LRRC17', 'MAOA', 'MAP3K12', 'MFAP4', 'MYB', 'MYH11', 'NAT1', 'NKX3-1', 'OGN', 'OMD', 'PCSK5', 'PDCD4', 'PGR', 'PLIN1', 'PNMA2', 'PPARG', 'PSD3', 'PTGER3', 'PTHLH', 'PTN', 'RAI2', 'RBP4', 'RTN1', 'RUNX1', 'SCUBE2', 'SFRP4', 'SLC26A3', 'SPTAN1', 'STC2', 'SVEP1', 'TAT', 'TIMP4', 'TNN', 'TNXA', 'ZBTB16']
Refined Community 98: ['ABCA8', 'ABCC2', 'ABCC4', 'ACADL', 'ACAP1', 'ACE2', 'ACKR1', 'ACTG2', 'ADAMDEC1', 'ADAMTS1', 'ADCY2', 'ADD2', 'ADD3', 'ADGRF5', 'ADGRG2', 'ADGRG6', 'ADH1B', 'ADM', 'AKR1B10', 'AKR1C1', 'ALDH1A1', 'ALDH1A3', 'AMD1', 'ANGPT1', 'ANGPTL4', 'ANK2', 'ANP32E', 'ANXA1', 'ANXA3', 'ANXA8', 'APBA2', 'APOD', 'APP', 'AQP1', 'ARHGAP25', 'ARL4C', 'ART3', 'ASS1', 'B3GNT3', 'BACE2', 'BAG2', 'BANK1', 'BBOX1', 'BCL11A', 'BCL11B', 'BCL2A1', 'BIN1', 'BIN2', 'BIRC3', 'BTN3A2', 'C1S', 'C3', 'C7', 'CA9', 'CALB2', 'CALD1', 'CALML5', 'CAPN6', 'CAV1', 'CAV2', 'CCL13', 'CCL14', 'CCL18', 'CCL19', 'CCL2', 'CCL20', 'CCL21', 'CCL5', 'CCN1', 'CCN2', 'CCN3', 'CCR2', 'CCR7', 'CD19', 'CD1C', 'CD1E', 'CD2', 'CD247', 'CD24P4', 'CD27', 'CD37', 'CD38', 'CD3D', 'CD3G', 'CD48', 'CD52', 'CD69', 'CD7', 'CD79A', 'CD79B', 'CD96', 'CDH3', 'CDKN1C', 'CDKN2A', 'CFI', 'CH25H', 'CHAC1', 'CHI3L1', 'CHI3L2', 'CHODL', 'CHRDL1', 'CHRM3', 'CHST3', 'CHST7', 'CLCA2', 'CLCN4', 'CLDN1', 'CLDN8', 'CLEC10A', 'CLEC4A', 'CLIC2', 'CLIC3', 'CLIC4', 'COL14A1', 'COL4A2', 'COL9A3', 'CORO1A', 'COTL1', 'CR2', 'CRABP1', 'CRLF1', 'CRYAB', 'CRYBG1', 'CSF2RA', 'CSGALNACT1', 'CSRP2', 'CTSC', 'CTSW', 'CX3CL1', 'CXCL1', 'CXCL10', 'CXCL11', 'CXCL12', 'CXCL2', 'CXCL8', 'CXCL9', 'CYBA', 'CYP1B1', 'CYP26B1', 'CYP39A1', 'CYRIA', 'CYTIP', 'CYTL1', 'DCN', 'DEFB1', 'DEPP1', 'DKK1', 'DLK1', 'DLX5', 'DMD', 'DNAJB4', 'DNM3', 'DOCK2', 'DPT', 'DSC2', 'DSC3', 'DSG3', 'DST', 'DTNB-AS1', 'DTX4', 'DZIP1', 'EDN1', 'EDN2', 'EFEMP1', 'EGFL6', 'EGFR', 'ELF5', 'ELN', 'EMCN', 'EN1', 'ENPP2', 'EPB41L2', 'EXTL1', 'FABP5', 'FABP7', 'FAM107A', 'FAM171A1', 'FAS', 'FAT1', 'FBLN1', 'FBLN2', 'FBLN5', 'FBXO17', 'FCER1A', 'FERMT1', 'FERMT2', 'FHL1', 'FHOD3', 'FLNA', 'FMO2', 'FNDC3B', 'FNDC4', 'FOLH1', 'FOLR1', 'FOXC1', 'FOXO3', 'FRZB', 'FSCN1', 'FYN', 'FZD7', 'GABBR1', 'GABBR2', 'GABRE', 'GABRP', 'GAD2', 'GAL', 'GALNT12', 'GALNT3', 'GBP1', 'GEM', 'GIMAP5', 'GJB3', 'GLIPR1', 'GNLY', 'GPM6B', 'GPR161', 'GPR171', 'GPR18', 'GPR183', 'GPR19', 'GRAMD2B', 'GRB14', 'GSDMB', 'GSTA1', 'GSTP1', 'GUCY1A1', 'GZMA', 'GZMB', 'GZMH', 'GZMK', 'HACD1', 'HBA1', 'HBB', 'HLA-DOB', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB4', 'HLA-G', 'HOXA5', 'HOXA9', 'HSD17B2', 'HSPB2', 'ICAM1', 'ICAM2', 'ICOS', 'ID1', 'ID4', 'IDO1', 'IFI16', 'IGF1', 'IGF2', 'IGF2BP2', 'IGF2BP3', 'IGFBP3', 'IGHA1', 'IGHD', 'IGHG1', 'IGHM', 'IGHV1-69', 'IGHV3-21', 'IGHV3-23', 'IGHV3-33', 'IGHV3-47', 'IGHV3-7', 'IGHV4-34', 'IGHV4-61', 'IGKC', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-37', 'IGKV1D-39', 'IGKV1OR1-1', 'IGKV1OR2-108', 'IGKV2D-28', 'IGKV3-20', 'IGKV4-1', 'IGLC2', 'IGLL3P', 'IGLV2-14', 'IGLV3-10', 'IGLV3-19', 'IGLV3-25', 'IL12RB2', 'IL1R2', 'IL21R', 'IL27RA', 'IL2RG', 'IL32', 'IL33', 'IL6', 'IL7R', 'IMPA2', 'INAVA', 'IRAG2', 'ISG20', 'ITGA6', 'ITGA7', 'ITGB7', 'ITK', 'ITM2A', 'ITM2C', 'JCHAIN', 'JRKL', 'KCNG1', 'KCNK5', 'KCNN4', 'KHDRBS3', 'KIT', 'KLF5', 'KLF6', 'KLK10', 'KLK5', 'KLK6', 'KLK7', 'KLRB1', 'KLRC1', 'KLRK1', 'KRT14', 'KRT15', 'KRT16', 'KRT17', 'KRT23', 'KRT5', 'KRT6A', 'KRT6B', 'KRT7', 'KRT81', 'KRT86', 'KYNU', 'L1CAM', 'LAD1', 'LAMB3', 'LAMP3', 'LBP', 'LBR', 'LCK', 'LCN2', 'LDHB', 'LEPR', 'LGALS2', 'LGALS7', 'LILRA4', 'LMO4', 'LPL', 'LTB', 'LTBP1', 'LTF', 'LY6D', 'LY9', 'LYN', 'LYVE1', 'LYZ', 'MALL', 'MAP1B', 'MAP4K1', 'MAPRE2', 'MARCO', 'MCM5', 'MEAK7', 'MEOX2', 'MET', 'MFAP2', 'MFAP4', 'MFGE8', 'MIA', 'MICAL3', 'MICALL1', 'MID1', 'MME', 'MMP1', 'MMP14', 'MMP3', 'MMP7', 'MPPED2', 'MPZL2', 'MRAS', 'MRC1', 'MS4A1', 'MSN', 'MT1F', 'MT1M', 'MT1X', 'MUC16', 'MYCN', 'MYH11', 'MYLK', 'MZB1', 'NCF1', 'NDRG1', 'NDRG2', 'NES', 'NFIB', 'NFIL3', 'NFKBIL1', 'NINJ2', 'NKG7', 'NKX2-5', 'NLRP1', 'NMU', 'NNMT', 'NPR3', 'NPTX2', 'NSG1', 'NUDT11', 'NUP93', 'ORM2', 'P2RX5', 'P2RY14', 'PADI2', 'PALS2', 'PAMR1', 'PAPSS2', 'PAX6', 'PCOLCE2', 'PDE9A', 'PDGFRA', 'PDPN', 'PDZK1IP1', 'PECAM1', 'PELI1', 'PERP', 'PFKP', 'PGBD5', 'PGRMC1', 'PHGDH', 'PI3', 'PIK3CD', 'PKP1', 'PLA2G2A', 'PLA2G4A', 'PLAAT1', 'PLAC8', 'PLAGL1', 'PLCH1', 'PLIN1', 'PLPP3', 'PLSCR3', 'PLTP', 'PML', 'POU2AF1', 'PPP1R16B', 'PPP1R1A', 'PRELP', 'PRKCB', 'PRKD3', 'PRKX', 'PROM1', 'PROS1', 'PRRX2', 'PSAT1', 'PTCH1', 'PTGER4', 'PTGFR', 'PTGIS', 'PTGS2', 'PTK7', 'PTN', 'PTP4A3', 'PTPN22', 'PTPRC', 'PTPRCAP', 'PTPRZ1', 'PTX3', 'PVRIG', 'RAB23', 'RAC2', 'RARRES1', 'RASGRP2', 'RBMS1', 'RBP4', 'RCAN1', 'RGCC', 'RGS2', 'RIPK4', 'RIPOR2', 'RND3', 'ROPN1', 'RRAGD', 'RRAS2', 'RUNX3', 'RYR1', 'S100A1', 'S100A2', 'S100A7', 'S100A8', 'S100A9', 'S100B', 'SAA1', 'SAT1', 'SCRG1', 'SEL1L3', 'SELE', 'SELL', 'SERPINE2', 'SERPING1', 'SERPINH1', 'SFN', 'SFRP1', 'SFRP4', 'SH2D1A', 'SHOX2', 'SIT1', 'SLC16A1', 'SLC25A37', 'SLC2A3', 'SLC34A2', 'SLC43A3', 'SLC6A14', 'SLC6A15', 'SLPI', 'SMCO4', 'SNCAIP', 'SOD2', 'SOSTDC1', 'SOX10', 'SOX11', 'SPIB', 'SRD5A1', 'SRGN', 'SRPX', 'ST8SIA1', 'STAP1', 'STEAP1B', 'STK10', 'STXBP6', 'SV2B', 'SVEP1', 'SYNGR1', 'SYNM', 'TAGLN', 'TAP1', 'TBX19', 'TCEAL2', 'TCF7L1', 'TCF7L2', 'TCL1A', 'TDO2', 'TET3', 'TFPI', 'TIE1', 'TM4SF1', 'TMEM100', 'TMEM158', 'TMEM45A', 'TMSB15A', 'TNC', 'TNFAIP3', 'TNFAIP8', 'TNFRSF11B', 'TNFRSF17', 'TNFRSF21', 'TNFRSF4', 'TNNI2', 'TNXA', 'TRAC', 'TRAF3IP3', 'TRAT1', 'TRBC1', 'TRBC2', 'TRDC', 'TRIM2', 'TRIM29', 'TSPAN8', 'TTLL4', 'TTYH1', 'UBE2E3', 'UGT8', 'VCAM1', 'VCAN', 'VGLL1', 'VNN1', 'VNN2', 'VPREB3', 'WARS1', 'WIF1', 'WLS', 'WNT6', 'WTAP', 'WWTR1', 'XCL1', 'YBX1', 'YBX3', 'ZAP70', 'ZIC1', 'ZNF532']
Refined Community 99: ['ABAT', 'ABCA3', 'ABCC8', 'ACADSB', 'ACOX2', 'AGL', 'AGR2', 'AGTR1', 'ALG8', 'AMIGO2', 'ANXA9', 'ARMT1', 'ARNT2', 'ASCL1', 'B9D1', 'BAIAP3', 'BMPR1B', 'C17orf75', 'CA12', 'CACNA1D', 'CACNA2D2', 'CACNG4', 'CAMK2B', 'CCDC106', 'CCND1', 'CEACAM5', 'CELSR1', 'CELSR3', 'CGA', 'CHGA', 'CHN2', 'CLSTN2', 'CRABP2', 'CRIP1', 'CYP2A6', 'CYP2B6', 'CYP2B7P', 'DACH1', 'DCAF10', 'DENND1B', 'DIO1', 'DNAI7', 'DNAJC1', 'DNAJC12', 'DNALI1', 'EEF1A2', 'ENPP1', 'EPOR', 'ERBB4', 'ESR1', 'EVL', 'FAM234B', 'FBP1', 'FGFR2', 'FLRT3', 'G6PC3', 'GATA3', 'GDF15', 'GFRA1', 'GLRB', 'GLS2', 'GNG13', 'GP2', 'GPR162', 'GREB1', 'GRIA2', 'GSTM3', 'GSTZ1', 'GTF2H2C', 'GUSBP14', 'GUSBP3', 'HDAC11', 'HEXIM2-AS1', 'HPN', 'IFT122', 'IL24', 'IL6ST', 'INSM1', 'ISYNA1', 'KCNE4', 'KCNJ3', 'KCTD3', 'KDM4B', 'KIF5C', 'KRT18', 'LDLRAD4', 'LINC01503', 'MAGI2', 'MAPT', 'MCCC2', 'METRN', 'MLPH', 'MRPS30', 'MSMB', 'MYT1', 'NAIP', 'NAT1', 'NELL2', 'NHERF1', 'NKAIN1', 'NOVA1', 'NPIPB3', 'NPY1R', 'NUDT6', 'PCLO', 'PCSK6', 'PIERCE1', 'PIEZO2', 'PLEKHF2', 'PPP1R3C', 'PTGES', 'PTPRT', 'PVALB', 'PYCR3', 'QDPR', 'RAB11FIP1', 'RAB26', 'RABEP1', 'RASSF1', 'RDH16', 'REPS2', 'RET', 'RETREG1', 'RHBDL1', 'RSF1', 'SCCPDH', 'SCUBE2', 'SEMA3B', 'SERPINA3', 'SERPINA5', 'SERPINA6', 'SERPINI1', 'SIAH2', 'SLC19A2', 'SLC1A1', 'SLC27A2', 'SLC39A6', 'SLC6A4', 'SLC8A2', 'SORD', 'SPAG16', 'SPINK4', 'SREBF1', 'SSTR2', 'STC1', 'STK32B', 'SYBU', 'SYT1', 'SYT17', 'TBC1D30', 'TBC1D9', 'TEDC2', 'TESMIN', 'TFF1', 'TFF3', 'TJP3', 'TMEM101', 'TNNI3', 'TNNT1', 'TOX3', 'TPPP3', 'TRIM9', 'TRPA1', 'TTC12', 'UGCG', 'WFDC2', 'WWP1', 'YBX2', 'ZDHHC11', 'ZNF652']
Refined Community 100: ['AP1M2', 'CDS1', 'COL11A1', 'EPN3', 'GPRC5A', 'KRT18']
Refined Community 101: ['AASS', 'ABCA6', 'ABCA8', 'ABCB1', 'ABLIM1', 'ACACB', 'ACAP1', 'ACKR1', 'ACTA2', 'ACVRL1', 'ADAM28', 'ADARB1', 'ADD3', 'ADGRF5', 'ADH1B', 'AIF1', 'ALDH1A1', 'ANGEL1', 'ANGPT2', 'ANK2', 'AOAH', 'AOC3', 'APLNR', 'APOBEC3G', 'AQP1', 'ARHGAP15', 'ARHGAP25', 'ARHGAP4', 'ARHGAP45', 'ARID5A', 'ARL4C', 'ATP2A3', 'BAALC', 'BANK1', 'BBOX1', 'BCL11A', 'BCL11B', 'BIN1', 'BIN2', 'BIRC3', 'BTK', 'BTN3A2', 'BTN3A3', 'C1R', 'C1S', 'C3', 'C7', 'CASP1', 'CASP10', 'CAV1', 'CCDC102B', 'CCDC69', 'CCDC88A', 'CCL14', 'CCL19', 'CCL2', 'CCL21', 'CCL5', 'CCR2', 'CCR6', 'CCR7', 'CD180', 'CD19', 'CD1C', 'CD1D', 'CD1E', 'CD2', 'CD22', 'CD244', 'CD247', 'CD27', 'CD302', 'CD37', 'CD38', 'CD3D', 'CD3G', 'CD4', 'CD40', 'CD40LG', 'CD48', 'CD52', 'CD6', 'CD69', 'CD7', 'CD72', 'CD74', 'CD79A', 'CD79B', 'CD8A', 'CD96', 'CDKN1C', 'CDO1', 'CELF2', 'CES1', 'CFH', 'CFI', 'CH25H', 'CHI3L1', 'CHL1', 'CHRDL1', 'CHRNB2', 'CIITA', 'CLDN5', 'CLEC10A', 'CLEC2B', 'CLEC3B', 'CLEC4A', 'CLIC2', 'CLU', 'CMA1', 'CNN1', 'COL14A1', 'COL17A1', 'COL18A1', 'CORO1A', 'COTL1', 'CPE', 'CPVL', 'CR2', 'CRY2', 'CRYAB', 'CSF2RA', 'CSGALNACT1', 'CSN3', 'CTSC', 'CTSG', 'CTSW', 'CXCL12', 'CXCL13', 'CXCL2', 'CXCR5', 'CYLD', 'CYP1B1', 'CYRIA', 'CYTIP', 'CYTL1', 'DCN', 'DDX28', 'DEF6', 'DEPP1', 'DES', 'DLK1', 'DLK2', 'DMD', 'DOC2B', 'DOCK10', 'DOCK2', 'DOK2', 'DPP4', 'DPT', 'DST', 'DTNB-AS1', 'EDN3', 'EFEMP1', 'ELMO1', 'EMCN', 'EMILIN1', 'ENPP2', 'ENSG00000293341', 'EPHA4', 'EVI2B', 'EZH1', 'F10', 'F13A1', 'FABP4', 'FAM107A', 'FBLN5', 'FCER1A', 'FCMR', 'FCRL2', 'FHL1', 'FLI1', 'FLRT2', 'FLT3LG', 'FMNL1', 'FMO2', 'FNDC4', 'FOLR2', 'FOXN3', 'FRZB', 'FXYD1', 'FYB1', 'FYN', 'FZD7', 'GDF7', 'GEM', 'GGT5', 'GIMAP4', 'GIMAP5', 'GIMAP6', 'GIT2', 'GJA4', 'GLIPR1', 'GMFG', 'GMIP', 'GNG11', 'GPR171', 'GPR18', 'GPR183', 'GVINP1', 'GZMA', 'GZMB', 'GZMK', 'GZMM', 'HBB', 'HGF', 'HHEX', 'HIVEP2', 'HLA-DOB', 'HLA-DPA1', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1', 'HOXA5', 'HOXA7', 'HOXA9', 'HSPB2', 'ICAM2', 'ICOS', 'ID1', 'ID3', 'ID4', 'IFI16', 'IGF1', 'IGHA1', 'IGHD', 'IGHG1', 'IGHM', 'IGKC', 'IGKV1D-17', 'IGKV2D-28', 'IGKV3-20', 'IGKV4-1', 'IGLC2', 'IGLL3P', 'IGLV2-14', 'IGLV3-10', 'IGLV3-25', 'IL18R1', 'IL1R1', 'IL1R2', 'IL21R', 'IL27RA', 'IL2RG', 'IL33', 'IL6', 'IL6R', 'IL7', 'IL7R', 'IRAG2', 'IRF8', 'ISG20', 'ITGA4', 'ITGA7', 'ITGAL', 'ITGB7', 'ITIH5', 'ITK', 'ITM2A', 'ITM2C', 'JCHAIN', 'KANK3', 'KCND3', 'KCTD12', 'KCTD7', 'KIT', 'KLK7', 'KLRB1', 'KLRK1', 'KRI1', 'KRT14', 'KRT5', 'LAMA2', 'LAMA4', 'LAMC3', 'LAMP3', 'LAT', 'LCK', 'LCP2', 'LDHB', 'LEF1', 'LEPR', 'LGALS2', 'LGALSL', 'LHFPL6', 'LILRA4', 'LILRB3', 'LIME1', 'LINC01140', 'LMOD1', 'LPCAT4', 'LPL', 'LRCH4', 'LRRN3', 'LST1', 'LTB', 'LY75', 'LY9', 'LY96', 'LYL1', 'LYN', 'LYVE1', 'LYZ', 'LZTS1', 'MALL', 'MAP4K1', 'MAP7D3', 'MAPRE2', 'MAST3', 'MATK', 'MATN2', 'MBP', 'MEF2C', 'MEOX1', 'MEOX2', 'MET', 'MFAP4', 'MGAT3', 'MME', 'MRC1', 'MS4A1', 'MS4A6A', 'MT1M', 'MTRF1', 'MYH11', 'MYLK', 'MZB1', 'NAP1L2', 'NCF1', 'NCF4', 'NCKAP1L', 'NCR3', 'NFAT5', 'NKG7', 'NLRP1', 'NLRP3', 'NPR1', 'NPR2', 'NR3C2', 'NRIP2', 'NRXN1', 'NRXN2', 'NSG1', 'NT5E', 'NTRK2', 'ODAM', 'OPRPN', 'OXTR', 'P2RX5', 'P2RY14', 'P3H2', 'PACRG', 'PALMD', 'PAX5', 'PCSK5', 'PDE2A', 'PDGFD', 'PDGFRA', 'PECAM1', 'PELI2', 'PIK3CD', 'PIK3R1', 'PLA2G2A', 'PLAC8', 'PLCB2', 'PLCL2', 'PLEKHO1', 'PLPP3', 'PLTP', 'PNOC', 'POU2AF1', 'POU5F1', 'POU6F1', 'PPP1R16B', 'PPP2R1B', 'PPP3CC', 'PRF1', 'PRKCB', 'PRKCQ', 'PROS1', 'PTGDS', 'PTGER4', 'PTN', 'PTPN22', 'PTPRC', 'PTPRCAP', 'PVRIG', 'RAC2', 'RASA4', 'RASGRP2', 'RASGRP3', 'RBMS1', 'RELN', 'RGS5', 'RIPOR2', 'RNASE6', 'RNF125', 'ROBO3', 'RRN3', 'RUBCNL', 'RUNX1T1', 'RUNX3', 'S100B', 'SAA1', 'SATB1', 'SEL1L3', 'SELE', 'SELENOP', 'SELL', 'SERPING1', 'SFRP1', 'SH2D1A', 'SIGLEC1', 'SIT1', 'SKAP2', 'SLA', 'SLC12A4', 'SLC16A7', 'SLC6A14', 'SLCO3A1', 'SLIT3', 'SNCAIP', 'SNX1', 'SORBS1', 'SOSTDC1', 'SP110', 'SP140', 'SPARCL1', 'SPIB', 'SPINK5', 'SPRY2', 'SRGN', 'SRPX', 'ST3GAL2', 'ST6GAL1', 'STAG3', 'STAP1', 'STK10', 'SVEP1', 'SYNE1', 'SYNE3', 'SYNM', 'TCF4', 'TCL1A', 'TEP1', 'TESC', 'TESPA1', 'TF', 'TFPI', 'TGFBR3', 'THBD', 'THEMIS2', 'TIE1', 'TLE4', 'TMOD1', 'TNFAIP8', 'TNFRSF17', 'TNFRSF1B', 'TNFRSF25', 'TNXA', 'TP63', 'TRAC', 'TRAF3IP3', 'TRAT1', 'TRBC1', 'TRBC2', 'TRDC', 'TRGC1', 'TSPAN7', 'TXNIP', 'VAMP1', 'VAMP5', 'VCAM1', 'VEGFD', 'VILL', 'VNN2', 'VPREB3', 'WIF1', 'WIPF1', 'XCL1', 'ZAP70', 'ZBP1', 'ZBTB20', 'ZEB1']
Refined Community 102: ['ACAN', 'ACE2', 'ACTG2', 'ACTL8', 'ADD2', 'AKR1C1', 'AMD1', 'ANP32E', 'APBA2', 'AQP5', 'ARL4C', 'ART3', 'ATP6V1A', 'BAG2', 'BBOX1', 'BCL11A', 'BGN', 'BMP1', 'C6orf62', 'CA9', 'CALU', 'CAPN6', 'CASK', 'CCL20', 'CD22', 'CD24P2', 'CD24P4', 'CD44', 'CD52', 'CDC5L', 'CDH19', 'CDKAL1', 'CDKN2A', 'CDV3', 'CEACAM1', 'CEP170', 'CHI3L2', 'CHRM3', 'CHST3', 'CLDN1', 'CLDN10', 'CLIC4', 'CLIP4', 'COCH', 'COL11A2', 'COL2A1', 'COL4A2', 'COL9A3', 'CP', 'CRLF1', 'CRYAB', 'CSN3', 'CTAG1B', 'CWF19L1', 'CXCL5', 'CXCL8', 'CXCR4', 'CYBA', 'CYP26B1', 'CYP39A1', 'DKK1', 'DLAT', 'DNAJB4', 'DSC2', 'DSC3', 'DSG1', 'EDN1', 'EGFR', 'EIF5A', 'ELF5', 'ELN', 'EN1', 'ENO1', 'EPHB3', 'EPHX3', 'EPRS1', 'EXOC5', 'FABP7', 'FBXO17', 'FERMT1', 'FGFBP1', 'FLNA', 'FNDC4', 'FOLH1', 'FOXC1', 'FOXO3', 'FSCN1', 'FYN', 'FZD7', 'FZD9', 'GABRP', 'GAD2', 'GAGE1', 'GAL', 'GALNT14', 'GART', 'GBP1', 'GDI2', 'GJB3', 'GLDC', 'GPM6B', 'GPR161', 'GPR19', 'GPSM2', 'GSTM4', 'HAPLN1', 'HLA-DOB', 'HLA-DQB2', 'HLA-G', 'HOXA9', 'HPCAL1', 'HSPD1', 'ID4', 'IFI16', 'IGF2BP2', 'IGF2BP3', 'IGHG1', 'IGHV1-69', 'IGHV3-23', 'IGHV3-33', 'IGHV3-7', 'IGHV4-34', 'IGHV4-61', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-37', 'IGKV1D-39', 'IGKV1OR1-1', 'IGKV1OR2-108', 'IGKV2D-28', 'IGKV3-20', 'IGKV4-1', 'IGLC2', 'IGLV2-14', 'IGLV3-10', 'IGLV3-19', 'IGLV3-25', 'IL12RB2', 'IL1R2', 'IL32', 'IMPA2', 'ITGA6', 'ITM2C', 'KCNG1', 'KCNK5', 'KCNN4', 'KHDRBS3', 'KLF5', 'KLF6', 'KLHL24', 'KLHL7', 'KLK10', 'KLK5', 'KLK6', 'KLK7', 'KRT14', 'KRT16', 'KRT17', 'KRT23', 'KRT5', 'KRT6A', 'KRT6B', 'KRT81', 'KRT83', 'KYNU', 'L1CAM', 'LAMP3', 'LDHB', 'LEFTY2', 'LGALS2', 'LGR5', 'LMO4', 'LRP12', 'LTBP1', 'MAGEA12', 'MAGEA2', 'MAGEA3', 'MAGEA4', 'MAGEA5P', 'MALL', 'MAP4K1', 'MAPK8', 'MARCO', 'MCL1', 'MCM5', 'MDC1', 'MFGE8', 'MIA', 'MICALL1', 'MMP14', 'MMP7', 'MRAS', 'MSH6', 'MSLN', 'MTAP', 'MUC16', 'NDUFA4L2', 'NFIB', 'NFIX', 'NFYC', 'NKX2-5', 'NLRP1', 'NMU', 'NPRL3', 'NPTX2', 'NSD2', 'NUP62', 'OBP2A', 'OBP2B', 'OCA2', 'ODAM', 'PADI2', 'PALS2', 'PAX6', 'PDAP1', 'PDE9A', 'PDK1', 'PDPN', 'PDZK1IP1', 'PEG10', 'PEG3', 'PERP', 'PGBD5', 'PGRMC1', 'PHGDH', 'PI3', 'PICALM', 'PKP1', 'PLA2G4A', 'PLEKHB1', 'PLOD2', 'POU2AF1', 'PRAME', 'PRDM13', 'PRELP', 'PRKD3', 'PRKDC', 'PRKX', 'PSAT1', 'PSPH', 'PTGFR', 'PTGS2', 'PTP4A3', 'PTPRF', 'PTPRZ1', 'PTX3', 'PYGB', 'QKI', 'QPCT', 'RAB23', 'RAC2', 'RARRES1', 'RBP1', 'RHEB', 'RIOK3', 'ROPN1', 'RRAGD', 'RUNX3', 'RYR1', 'S100A1', 'S100A2', 'S100A8', 'S100A9', 'S100B', 'SCRG1', 'SERPINB2', 'SFRP1', 'SIGMAR1', 'SIX3', 'SLC16A1', 'SLC16A3', 'SLC25A37', 'SLC2A3', 'SLC43A3', 'SLC6A14', 'SLC7A5', 'SLPI', 'SMPDL3B', 'SOD2', 'SOSTDC1', 'SOX10', 'SOX11', 'SOX9', 'SPIB', 'SRD5A1', 'SSRP1', 'ST8SIA1', 'STIP1', 'STXBP6', 'SYCP1', 'SYNM', 'TAGLN2', 'TBX19', 'TDO2', 'TFAP2C', 'TFCP2L1', 'TM4SF1', 'TMEM158', 'TMEM45A', 'TMSB15A', 'TNFRSF11B', 'TNFRSF21', 'TP53', 'TRIM2', 'TRIM29', 'TTLL4', 'TTYH1', 'TUBB2A', 'TUBB2B', 'UCHL1', 'USP34', 'VEGFA', 'VGLL1', 'WARS1', 'WIF1', 'WNT5B', 'WTAP', 'WWTR1', 'YBX1', 'YBX3', 'YWHAE', 'YWHAZ', 'ZFP36L2', 'ZIC1']
Refined Community 103: ['ABAT', 'ACADSB', 'ADH1B', 'AGR2', 'ANXA9', 'ASCL1', 'ATRNL1', 'BCAS1', 'CA12', 'CAMP', 'CAPN9', 'CCNO', 'CEACAM5', 'CEACAM6', 'CELSR1', 'CHAD', 'CITED1', 'CLGN', 'CPB1', 'CRIP1', 'CYP2B6', 'CYP2B7P', 'DCLK1', 'DNALI1', 'DUSP4', 'EPHA3', 'ESR1', 'EVL', 'FAIM2', 'FBP1', 'FGFR3', 'GALNT6', 'GALNT7', 'GATA3', 'GFRA1', 'GP2', 'GPC1-AS1', 'GSTT2', 'HBA1', 'HMGCS2', 'HPX', 'IL24', 'INPP4B', 'INPP5J', 'JMJD7-PLA2G4B', 'KAZALD1', 'KIF5C', 'LIN7A', 'LRP1B', 'MAST4', 'MATN3', 'MB', 'MGAM', 'MLPH', 'MSMB', 'MUC1', 'NAT1', 'NELL2', 'NR2F1', 'PAH', 'PDE4DIP', 'PGAP3', 'PGGHG', 'PIDD1', 'PIERCE1', 'PLA1A', 'PLK2', 'PSD3', 'PTPRT', 'RAPGEF3', 'REPS2', 'RETREG1', 'RND1', 'SCCPDH', 'SCGB1D2', 'SCGB2A1', 'SCGB2A2', 'SCUBE2', 'SEMA3B', 'SEPTIN8', 'SERPINA5', 'SIL1', 'SLC19A2', 'SLC1A1', 'SLC39A6', 'SLC44A4', 'SLC4A8', 'SPDEF', 'SYT1', 'TBC1D19', 'TFF1', 'TFF3', 'TFPI2', 'TIMP4', 'TMEM101', 'TOM1L1', 'TOX3', 'TSPAN1', 'TTC12']
Refined Community 104: ['ADIRF', 'AGR2', 'ALCAM', 'ARMT1', 'ASAH1', 'ASTN2', 'BIK', 'C4A', 'CA12', 'CCNO', 'CDS1', 'CELSR1', 'CHST8', 'CLCN3', 'CLGN', 'COL4A5', 'CRIP1', 'CXXC4', 'CYP2D6', 'CYP4B1', 'DACH1', 'DCAF10', 'DEPTOR', 'DNAJC12', 'DNALI1', 'ELAPOR1', 'ERBB3', 'ERBB4', 'ESR1', 'EVL', 'GALNT6', 'GATA3', 'GATB', 'GOLGA2P5', 'GPC1-AS1', 'GREB1', 'GSTM3', 'GUSBP3', 'IKBKB', 'IL11', 'IL6ST', 'INPP5J', 'ITPR1', 'IVD', 'JMJD7-PLA2G4B', 'KDM4B', 'LIN7A', 'MAP9', 'MAPKBP1', 'MAPT', 'MAST4', 'MCCC2', 'MLPH', 'N4BP2L2', 'NPDC1', 'NRIP1', 'PGRMC2', 'PSD3', 'QDPR', 'RAB27B', 'RBM47', 'RET', 'SELENBP1', 'SERPINA5', 'SIDT1', 'SLC19A2', 'SLC27A2', 'SLC44A4', 'SLC4A8', 'SLC7A8', 'SLC9A1', 'SPDEF', 'ST6GALNAC2', 'STK32B', 'TBC1D9', 'TFF3', 'TJP3', 'TOX3', 'TSFM', 'TTC9', 'VAV3', 'ZNF446', 'ZNF91']
Refined Community 105: ['ART3', 'BCL11A', 'CCKAR', 'CDH3', 'CENPF', 'CENPN', 'CHST3', 'COL9A3', 'CP', 'CXCL9', 'DLX5', 'EGFL6', 'EGFR', 'EN1', 'FOLR1', 'GDF5', 'HOXA9', 'IGHV3-23', 'IGHV4-61', 'IMPA2', 'KCNK5', 'KIFC1', 'KRT16', 'LDHB', 'MCM10', 'MCM5', 'MMP7', 'PRAME', 'PRKX', 'PTX3', 'RARRES1', 'SCRG1', 'SFRP1', 'ST8SIA1', 'TMSB15A', 'TNFRSF21', 'TRIM29', 'TTLL4', 'VGLL1', 'VSNL1', 'WWTR1']
Refined Community 106: ['ARNT2', 'CFB', 'CPB1', 'DNAJC12', 'DNALI1', 'ELOVL2', 'GLRB', 'PCSK6', 'RND2', 'STC2']
Refined Community 107: ['CLIC3', 'CRLF1', 'FGB', 'KYNU', 'PPP1R1A', 'S100A8']
Refined Community 108: ['ABCA8', 'ACKR1', 'ADH1B', 'ADIPOQ', 'ASPN', 'CILP', 'CLDN5', 'COL14A1', 'COMP', 'CXCL14', 'DNALI1', 'DPT', 'DUSP6', 'FABP4', 'HMGCS2', 'IGF1', 'ITGA7', 'LMOD1', 'LRRC31', 'LRRN3', 'NAT1', 'PDCD4', 'PPP1R1A', 'PRODH', 'RAPGEF3', 'SCGB1D2', 'SCGB2A1', 'SCGB2A2', 'SCUBE2', 'SFRP4', 'SLC1A1', 'TBC1D19', 'TFAP2B', 'TFF1', 'TIMP4', 'TNXA', 'TOX3', 'TTC12']
Refined Community 109: ['ADD2', 'ANP32E', 'CLIC4', 'COL2A1', 'CSPG4', 'CTSV', 'FOXD1', 'HAPLN1', 'KLK6', 'KLK7', 'KPNA4', 'KRT17', 'MMP14', 'MUC16', 'NMU', 'PRAME', 'PSPH', 'RYR1', 'SLC7A5', 'TNNI2', 'TTYH1', 'UCHL1']
Refined Community 110: ['BAMBI', 'CCL18', 'CDH3', 'CEACAM1', 'CHAC1', 'CP', 'CST1', 'CXCL9', 'FABP7', 'FRMPD1', 'FYB1', 'GABBR1', 'HCP5', 'HLA-DQB1', 'IGHV3-21', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-39', 'IGKV3-20', 'INAVA', 'LTF', 'MAGEA3', 'MIA', 'PEG10', 'POPDC3', 'PSAT1', 'S100A7', 'S100A9', 'S100P', 'SIX1', 'SLPI', 'SRD5A1']
Refined Community 111: ['CHN2', 'CXCL14', 'GJA1', 'SYNGR3', 'TENM3', 'TPPP3']
Refined Community 112: ['AASS', 'AKAP11', 'ARHGEF12', 'ARHGEF40', 'BBOF1', 'BBS1', 'CASP9', 'CFAP69', 'CIRBP', 'CRTC3', 'CRY2', 'CTDSP1', 'CX3CR1', 'CYBRD1', 'DEAF1', 'DIXDC1', 'DYNC2H1', 'DZANK1', 'ECHDC2', 'FOS', 'FRY', 'GFOD3P', 'IFT46', 'IFT88', 'JHY', 'KIF13B', 'LAMB2', 'LTBP3', 'MARCHF8', 'MPHOSPH8', 'NBR1', 'NF1', 'NME5', 'NYNRIN', 'PIGV', 'RGPD5', 'RNASE4', 'RUNX1', 'SESN1', 'SIRT3', 'SLC24A1', 'SMARCA2', 'SNX1', 'STARD13', 'STAT5B', 'SYNC', 'TBC1D17', 'TP53BP1', 'TPT1', 'WDR19', 'ZFP2', 'ZNF395', 'ZNF862']
Refined Community 113: ['APOBEC3B', 'ASPM', 'AURKA', 'AURKB', 'BIRC5', 'BLM', 'BOLA2', 'BUB1', 'BUB1B', 'BYSL', 'CCNA2', 'CCNB1', 'CCNB2', 'CCNE2', 'CCT5', 'CDC20', 'CDC25A', 'CDC45', 'CDCA3', 'CDCA8', 'CDK1', 'CDK2', 'CDKN3', 'CENPA', 'CENPE', 'CENPF', 'CENPI', 'CENPN', 'CENPU', 'CEP55', 'CHEK1', 'CKS2', 'CMC2', 'COX7B', 'DDX39A', 'DLGAP5', 'DNAJC9', 'DONSON', 'DSCC1', 'DSN1', 'E2F1', 'E2F8', 'ECT2', 'ELOC', 'ESPL1', 'EXO1', 'EZH2', 'FBXO5', 'FEN1', 'FOXM1', 'GINS1', 'GMPS', 'GTPBP4', 'GTSE1', 'H2AZ1', 'H2AZ2', 'H4C2', 'HCCS', 'HJURP', 'HMCES', 'HMGB3', 'HMMR', 'ITCH', 'JMJD6', 'JPT1', 'KIF11', 'KIF14', 'KIF15', 'KIF18B', 'KIF20A', 'KIF2C', 'KIF4A', 'KIFC1', 'KPNA2', 'LAGE3', 'LMNB1', 'MAD2L1', 'MARS1', 'MCM10', 'MCM2', 'MCM3', 'MCM4', 'MCM6', 'MELK', 'MIS18A', 'MKI67', 'MRPL12', 'MRPL15', 'MRPS17', 'MYBL2', 'NCAPG', 'NCAPG2', 'NCAPH', 'NDC80', 'NEK2', 'NME1', 'NUDT1', 'NUP93', 'NUSAP1', 'NUTF2', 'OIP5', 'ORMDL2', 'PARPBP', 'PDSS1', 'PFDN6', 'PIMREG', 'PKMYT1', 'PLK1', 'POLQ', 'POLR2K', 'PRC1', 'PSMA7', 'PSMD12', 'PTTG1', 'PTTG3P', 'QPRT', 'RAB5IF', 'RACGAP1', 'RAD51', 'RAD54B', 'RFC4', 'RNASEH2A', 'RRM2', 'SHMT2', 'SLC52A2', 'SLC7A5', 'SNRPC', 'SNRPF', 'SNRPG', 'SPAG5', 'STIL', 'STIP1', 'STMN1', 'TACC3', 'TIMELESS', 'TIMM10', 'TMPO', 'TOMM70', 'TOP2A', 'TPX2', 'TRIP13', 'TROAP', 'TTK', 'TUBA1A', 'TUBA1B', 'TUBA1C', 'TXN', 'TXNRD1', 'UBE2C', 'UBE2N', 'UBE2S', 'VRK1', 'XPOT', 'ZWINT']
Refined Community 114: ['ANKRD50', 'APPL2', 'BLTP2', 'BMPR2', 'CA2', 'CCDC74B', 'CDH11', 'CSNK1A1', 'CTSK', 'DKK3', 'DNALI1', 'ECI1', 'ETFB', 'FAM110C', 'FMOD', 'GLI3', 'GOLM1', 'HDGFL3', 'HSD11B2', 'ITGBL1', 'KLHL2', 'MID1IP1', 'NEURL1B', 'NQO1', 'PIGT', 'PRKD1', 'PRKG1', 'PRRX2', 'RGP1', 'RPL23A', 'SCAMP1', 'SH3BP4', 'SKP1P1', 'SLC9A2', 'SYNJ2BP', 'TPPP3', 'UBE2Q2', 'UNC119', 'VEZF1', 'ZFHX3', 'ZNF395']
Refined Community 115: ['ART3', 'ATAD3A', 'CCL2', 'CD19', 'DENND1B', 'DNTTIP2', 'EEF1D', 'EML4', 'GLMN', 'IGHM', 'KCNN4', 'KDM2B', 'MTF2', 'MYBL1', 'MYC', 'OGG1', 'PACSIN3', 'PIM2', 'POU2AF1', 'PPP1CB', 'PUM3', 'REXO2', 'RRP1', 'RRP1B', 'RTCA', 'SERBP1', 'SLC11A2', 'THEMIS2', 'TRIP6', 'UCHL5', 'VPREB3', 'VRK2', 'WDR43', 'ZPR1']
Refined Community 116: ['ABRACL', 'ADAMTS7', 'ADM', 'AGFG1', 'AMD1', 'ANLN', 'APOBEC3B', 'ASNS', 'ASS1', 'ATAD3A', 'ATL3', 'ATP1B3', 'BCL11A', 'BCL2A1', 'BIN1', 'BIRC2', 'BLM', 'BTG3', 'BUB1', 'C21orf91', 'CA9', 'CCL18', 'CCL5', 'CCNB2', 'CCNC', 'CD180', 'CDC20', 'CDCA8', 'CDH3', 'CDK2AP1', 'CEBPG', 'CENPA', 'CEP55', 'CHI3L1', 'CHIC2', 'CHST2', 'CLCN4', 'CORO1C', 'CREB3L2', 'CSRP2', 'CSTB', 'CTPS1', 'CTSC', 'CTSV', 'CX3CL1', 'CYB5R2', 'CYBA', 'CYBB', 'DAPK1', 'DEF8', 'DEFB1', 'DESI2', 'DSC2', 'DUSP2', 'DUSP9', 'DYSF', 'EGFL6', 'EML4', 'ENO1', 'EPHA2', 'FABP7', 'FAM171A1', 'FAM20A', 'FANCA', 'FDX1', 'FIRRM', 'FNDC3B', 'FOXC1', 'FSCN1', 'FZD9', 'GABRP', 'GAPDH', 'GAPDHS', 'GATAD2A', 'GBP1', 'GDF5', 'GLIPR1', 'GMPS', 'GOLT1B', 'GPRC5B', 'GPRIN2', 'GPSM2', 'H1-1', 'HDGF', 'HEBP2', 'HIF1A', 'HJURP', 'HK3', 'IDO1', 'IFNAR2', 'IFRD1', 'IL15RA', 'ILF2', 'IMPA2', 'INHBC', 'ITGB2', 'JRKL', 'KATNA1', 'KCNK5', 'KCNN4', 'KIF1B', 'KIF2C', 'KRT16', 'KRT23', 'KRT6B', 'KRT7', 'LAD1', 'LAMP3', 'LDHB', 'LGALSL', 'LILRB3', 'LMO4', 'LPIN1', 'LPXN', 'LYN', 'MALL', 'MAPRE2', 'MARCO', 'MCM5', 'MCM6', 'MEAK7', 'MELK', 'MEMO1', 'MID1', 'MMP7', 'MPZL1', 'MRAS', 'MSN', 'MTMR2', 'MYBL2', 'MYO10', 'NASP', 'NCAPD2', 'NCK1', 'NDRG1', 'NF2', 'NFKBIE', 'NMB', 'NMI', 'ODC1', 'OPTN', 'P2RY6', 'PADI2', 'PDCD5', 'PDIA6', 'PDXK', 'PDZK1IP1', 'PEDS1', 'PFKP', 'PGM1', 'PHACTR2', 'PHGDH', 'PIM1', 'PIMREG', 'PKP1', 'PLEKHG1', 'PLSCR1', 'PLTP', 'PPP1CB', 'PRIM2', 'PROM1', 'PSAT1', 'PSME4', 'PSMG1', 'RAB6B', 'RAD54L', 'RARRES1', 'RBMS1', 'RCAN1', 'RDX', 'RNF114', 'RRP1', 'RSRC1', 'RSU1', 'S100A10', 'SAMD4A', 'SCHIP1', 'SEM1', 'SERBP1', 'SFT2D2', 'SH3BP1', 'SLC2A5', 'SLC2A6', 'SLC35C1', 'SLC43A3', 'SMCO4', 'SNN', 'SOD2', 'SOX10', 'SRGN', 'SRSF7', 'ST14', 'ST8SIA1', 'STEAP3', 'STIL', 'SYNCRIP', 'TBC1D1', 'TBPL1', 'TCN2', 'TES', 'THEMIS2', 'TM4SF1', 'TMCC2', 'TMEM123', 'TMEM45A', 'TMSB10', 'TNF', 'TNFAIP3', 'TNFRSF21', 'TOPBP1', 'TRIM29', 'TRPV6', 'TTC7A', 'TTK', 'TTYH1', 'UBE2E3', 'UCHL3', 'UCK2', 'UGP2', 'UGT8', 'UQCRH', 'VGLL1', 'WARS1', 'YBX1']
Refined Community 117: ['ABAT', 'ABCA3', 'ACADSB', 'AFF1', 'APPL2', 'AQR', 'BAG1', 'BCL2', 'BHLHE40', 'BRD8', 'BTBD9', 'BTF3', 'C3orf18', 'C4A', 'CA12', 'CAMLG', 'CCDC106', 'CCDC74B', 'CCNG2', 'CELSR1', 'CERS2', 'CERS6', 'CHAD', 'CHD6', 'CHRD', 'CIRBP', 'CISH', 'COX6C', 'CXXC5', 'CYB5R1', 'DBNDD2', 'DELE1', 'DNAJC12', 'DNALI1', 'ECI1', 'EEIG1', 'ELOVL5', 'EML2', 'ERBB3', 'ERBB4', 'ESR1', 'EVA1B', 'EVL', 'FAAH', 'FAM110C', 'FAM174B', 'FAN1', 'FARP2', 'FBP1', 'FGD3', 'FOXA1', 'FUT8', 'GADD45G', 'GATA3', 'GFRA1', 'GLI3', 'GPD1L', 'GPR162', 'GREB1', 'HDAC11', 'HEXIM1', 'HHAT', 'HMGCL', 'HNRNPA1', 'HSD17B4', 'HTT', 'IGBP1', 'IGFBP4', 'IL6ST', 'INPP5J', 'IRS1', 'ITPK1', 'KCTD3', 'KDM4B', 'KIAA0232', 'KIF13B', 'KIF16B', 'KRT18', 'LAMB2', 'LONRF2', 'LRBA', 'LRIG1', 'LZTFL1', 'MACIR', 'MAGED2', 'MAN2B2', 'MAST4', 'MCCC2', 'MED13L', 'MEIS3P1', 'MINDY1', 'MREG', 'MRPS27', 'MYB', 'MYLIP', 'MYO5C', 'NAT1', 'NEDD4L', 'NEK9', 'NHERF1', 'NPDC1', 'OVOL2', 'P4HTM', 'PBX1', 'PCSK6', 'PEX19', 'PIGT', 'PPP1R26', 'PREX1', 'PTP4A2', 'PTPRT', 'QDPR', 'RAB11FIP3', 'RARA', 'RBM47', 'RETREG1', 'RNF103', 'SCAMP1', 'SCCPDH', 'SCUBE2', 'SEC14L2', 'SKP1P1', 'SLC16A6', 'SLC35E2B', 'SLC39A6', 'SNX1', 'SOX12', 'SPDEF', 'SPEF1', 'SREBF1', 'STARD10', 'SYBU', 'SYNGR1', 'TBC1D9', 'TCEAL1', 'TFF3', 'TMBIM4', 'TMBIM6', 'TOGARAM1', 'TP53INP1', 'TPBG', 'TSPAN13', 'TUSC2', 'UGCG', 'VAV3', 'WDR6', 'XBP1', 'ZBTB40', 'ZNF587']
Refined Community 118: ['ADGRG6', 'ADM', 'AGFG1', 'ARMC1', 'ASNS', 'ASPM', 'ATAD2', 'AURKA', 'BIRC5', 'BNIP3', 'BUB1', 'CA9', 'CCNB2', 'CCNE2', 'CDC25B', 'CDC42BPA', 'CDK16', 'CENPA', 'CENPN', 'CKS2', 'CMC2', 'COL4A2', 'CP', 'CTPS1', 'CTSV', 'DCK', 'DEGS1', 'DENND11', 'DEPDC1', 'DIAPH3', 'DLGAP5', 'DTL', 'ECT2', 'ESM1', 'EXT1', 'EZH2', 'FBXO5', 'FLT1', 'GBE1', 'GGH', 'GMPS', 'GNAZ', 'GPSM2', 'HACD2', 'HJURP', 'HMGB3', 'IGFBP5', 'INAVA', 'INTS7', 'IVNS1ABP', 'KIF14', 'KIF21A', 'LPCAT1', 'LRP12', 'MAD2L1', 'MAPRE2', 'MCCC1', 'MCM6', 'MELK', 'MGAT4A', 'MLLT10', 'MMP9', 'MRPL13', 'MTDH', 'MTMR2', 'NDC80', 'NDRG1', 'NMB', 'NMU', 'NUSAP1', 'ORC6', 'OXCT1', 'PALM2AKAP2', 'PAQR3', 'PFKP', 'PGK1', 'PIMREG', 'PIR', 'PITRM1', 'PLAAT1', 'PLEKHA1', 'PRAME', 'PRC1', 'PSMD2', 'PSMD7', 'PTDSS1', 'RAB6B', 'RAD21', 'RFC4', 'RRAGD', 'SACS', 'SERF1A', 'SLC2A3', 'SLC7A1', 'SMC4', 'SPC25', 'STK3', 'STMN1', 'STX1A', 'SYNCRIP', 'TFRC', 'TK1', 'TMEM45A', 'TMEM65', 'TMEM74B', 'TRIP13', 'TSPYL5', 'UCHL5', 'VEGFA']
Refined Community 119: ['ACADS', 'ALDH4A1', 'ALDH6A1', 'AP2B1', 'BBC3', 'BTG2', 'CCDC74B', 'CCN4', 'CHPT1', 'CIRBP', 'EBF4', 'ECI2', 'ELAPOR1', 'ERGIC1', 'EVL', 'FBP1', 'FGF18', 'FUT8', 'GSTM3', 'INPP5J', 'IP6K2', 'KIAA1217', 'KIF3B', 'KRT18', 'LAMP5', 'LETMD1', 'MATN3', 'MS4A7', 'MYLIP', 'MYRIP', 'NEO1', 'PCSK6', 'PDIA4', 'PEX12', 'QDPR', 'RAB27B', 'RBP3', 'RPS4X', 'SCUBE2', 'SEC14L2', 'SPEF1', 'STK32B', 'TBC1D9', 'TBX3', 'TGFB3']
Refined Community 120: ['ADGRG6', 'ALDH4A1', 'AP2B1', 'BBC3', 'CCN4', 'CCNE2', 'CDC42BPA', 'CENPA', 'CMC2', 'COL4A2', 'DCK', 'DIAPH3', 'DTL', 'EBF4', 'ECI2', 'ESM1', 'EXT1', 'FGF18', 'FLT1', 'GMPS', 'GNAZ', 'GSTM3', 'IGFBP5', 'LPCAT1', 'MCM6', 'MELK', 'MMP9', 'MS4A7', 'MTDH', 'NDC80', 'NMU', 'NUSAP1', 'ORC6', 'OXCT1', 'PALM2AKAP2', 'PITRM1', 'PLAAT1', 'PRC1', 'RAB6B', 'RFC4', 'SCUBE2', 'SERF1A', 'SLC2A3', 'STK32B', 'TGFB3', 'TMEM65', 'TMEM74B', 'TSPYL5', 'UCHL5']
Refined Community 121: ['ACOT11', 'ANAPC15', 'BCL2L14', 'CEP57', 'COL2A1', 'FUT3', 'GABRQ', 'GAS2', 'MYH2', 'PARP4', 'RFX7', 'RPS4XP3', 'TNFSF10', 'ZNF362', 'ZSWIM8']
Refined Community 122: ['ABLIM1', 'ACACB', 'AP2A2', 'ARHGDIB', 'BICD1', 'C3', 'CAPN2', 'CD44', 'CLN8', 'CNKSR1', 'DUSP4', 'EIF4EBP3', 'ETV2', 'FKBP2', 'GFOD2', 'GOLM1', 'IL18', 'LINC01482', 'LST1', 'MAP4', 'METTL25B', 'MMP23B', 'NEFL', 'NEURL1', 'OR12D2', 'ORC3', 'PHF11', 'SLC35A1', 'TACC2', 'TESPA1', 'TNFSF13', 'ZFP36L2']
Refined Community 123: ['ATAD2', 'CBX3', 'CCNE2', 'CENPU', 'EEF1A2', 'FEN1', 'GTSE1', 'H4C8', 'HDGFL3', 'KPNA2', 'MYRF', 'NCAPG2', 'PGAP6', 'PLK1', 'POLQ', 'PPP1CC', 'PSMC2', 'SMC4', 'SUPT16H', 'UCKL1', 'YIF1A', 'ZCCHC8']
Refined Community 124: ['CAV1', 'CAVIN1', 'CDC25A', 'CLC', 'CTNNA1', 'DKK1', 'DUSP1', 'EPHA2', 'ETV5', 'F3', 'FOSL1', 'FYN', 'GADD45A', 'GDF15', 'GJB3', 'HBEGF', 'INHBA', 'IRS2', 'KLF6', 'KRT17', 'MAFF', 'MYO1B', 'PTPRG', 'SERPINB8', 'SLC3A2', 'SLC7A5', 'SLC9A1', 'SMURF2', 'SPHK1', 'STK17A', 'THBS1', 'TNFRSF10B', 'TUBB', 'UPP1', 'ZYX']
Refined Community 125: ['GLS', 'PHGDH', 'PSAT1', 'PSPH', 'SLC1A5', 'SLC2A1', 'SLC7A11']
Refined Community 126: ['AKT1', 'AKT2', 'AKT3', 'APC', 'APC2', 'ARAF', 'ATM', 'ATR', 'AXIN1', 'AXIN2', 'BAK1', 'BAX', 'BRAF', 'BRCA1', 'BRCA2', 'CCND1', 'CDK4', 'CDK6', 'CDKN1A', 'CETN3', 'CSNK1A1', 'CSNK1A1L', 'CSNK2A1', 'CSNK2A2', 'CSNK2A3', 'CSNK2B', 'CTNNB1', 'DDB2', 'DLL1', 'DLL3', 'DLL4', 'DVL1', 'DVL2', 'DVL3', 'E2F1', 'E2F2', 'E2F3', 'EGF', 'EGFR', 'ERBB2', 'ESR1', 'ESR2', 'FGF1', 'FGF10', 'FGF16', 'FGF17', 'FGF18', 'FGF19', 'FGF2', 'FGF20', 'FGF21', 'FGF22', 'FGF23', 'FGF3', 'FGF4', 'FGF5', 'FGF6', 'FGF7', 'FGF8', 'FGF9', 'FGFR1', 'FLT4', 'FOS', 'FRAT1', 'FRAT2', 'FZD1', 'FZD10', 'FZD2', 'FZD3', 'FZD5', 'FZD6', 'FZD7', 'FZD8', 'FZD9', 'GADD45A', 'GADD45B', 'GADD45G', 'GRB2', 'GSK3B', 'HES1', 'HES5', 'HEY1', 'HEY2', 'HEYL', 'HRAS', 'IGF1', 'IGF1R', 'JAG2', 'JUN', 'KIT', 'KRAS', 'LEF1', 'LRP5', 'LRP6', 'MAP2K1', 'MAP2K2', 'MAPK1', 'MAPK3', 'MRE11', 'MTOR', 'MYC', 'NBN', 'NCOA1', 'NCOA3', 'NFKB2', 'NOTCH1', 'NOTCH2', 'NOTCH3', 'NOTCH4', 'NRAS', 'PARP1', 'PGR', 'PIK3CA', 'PIK3CD', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'POLK', 'PTEN', 'RAD50', 'RAD51', 'RAF1', 'RB1', 'RPS6KB1', 'RPS6KB2', 'SFRP4', 'SHC1', 'SHC2', 'SHC3', 'SHC4', 'SKP1', 'SOS1', 'SOS2', 'SP1', 'TCF7', 'TCF7L1', 'TCF7L2', 'TNFSF11', 'TP53', 'WNT1', 'WNT10A', 'WNT10B', 'WNT11', 'WNT16', 'WNT2', 'WNT2B', 'WNT3', 'WNT3A', 'WNT4', 'WNT5A', 'WNT5B', 'WNT6', 'WNT7A', 'WNT7B']
Refined Community 127: ['AKT1', 'AKT2', 'AKT3', 'CCND1', 'CCNE1', 'CDK2', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN1B', 'CYP19A1', 'EGFR', 'ERBB2', 'ESR1', 'HRAS', 'IGF1R', 'INS', 'IRS1', 'KRAS', 'MAPK1', 'MAPK3', 'MTOR', 'NRAS', 'PIK3C2A', 'PIK3C2B', 'PIK3C2G', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3CG', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PIK3R4', 'PIK3R5', 'PIK3R6', 'RAF1', 'RPTOR']
Refined Community 128: ['HK1', 'LDHA', 'PDK1', 'PDP1', 'PFKP', 'PKM', 'SLC16A1', 'SLC2A1']
Refined Community 129: ['ABL1', 'AHR', 'AKT1', 'ALKBH1', 'ANXA1', 'APOBEC3G', 'AR', 'ARAF', 'ATF1', 'ATM', 'ATR', 'AURKA', 'BACH1', 'BAD', 'BAK1', 'BARD1', 'BAX', 'BCL2', 'BID', 'BLM', 'BMPR1A', 'BMPR2', 'BRAF', 'BRCA1', 'BRCA2', 'CASP3', 'CASP8', 'CASP9', 'CCNB1IP1', 'CCND1', 'CDC25A', 'CDC25B', 'CDC42', 'CDH1', 'CDK2', 'CDK4', 'CDK7', 'CERK', 'CHEK1', 'CHEK2', 'CHUK', 'CREB1', 'CSNK1D', 'CTNNB1', 'CYP19A1', 'DAG1', 'DCAKD', 'DHTKD1', 'E2F1', 'EDAR', 'EGFR', 'EP300', 'ERAL1', 'ESR1', 'FADD', 'FAU', 'FER', 'FILIP1', 'FOSL1', 'FOSL2', 'FOXO1', 'GADD45A', 'GDI1', 'GRN', 'GSK3A', 'HDAC1', 'HIPK2', 'HMGCR', 'IMPA1', 'IRS1', 'ITPKC', 'JAK1', 'JAKMIP1', 'JUN', 'KRAS', 'LGALS13', 'MAP3K13', 'MAP3K7CL', 'MAPK1', 'MAX', 'MDM2', 'MIR21', 'MIR29B1', 'MIR29B2', 'MMP1', 'MRE11', 'MSH2', 'MSH6', 'MTOR', 'MYC', 'MYCBP2', 'MYT1', 'NAB1', 'NCOA3', 'NF1', 'NFKB1', 'NOXA1', 'NUP85', 'ODC1', 'PAK1', 'PHB1', 'PIAS1', 'PIGR', 'PIK3R2', 'PKIA', 'PLK1', 'PLK3', 'PML', 'PPP4R3A', 'PPP4R3B', 'PTEN', 'RAC1', 'RAD50', 'RAD51', 'RAD54L', 'RALA', 'RALGAPA1', 'RAP1A', 'RASGEF1A', 'RASGRP3', 'RB1', 'RHEB', 'RHO', 'RPP38', 'RRAS', 'SELENOK', 'SIRT1', 'SMAD1', 'SMAD2', 'SMAD4', 'SMAD6', 'SMAD7', 'SMARCA4', 'SP1', 'STAT1', 'STK11', 'TAB1', 'TFPI', 'TGFBR1', 'TGFBR2', 'TP53', 'TPR', 'TRADD', 'TSC1', 'TSC2', 'UBE2F', 'USP15', 'USP16', 'USP21', 'USP38', 'VEGFA', 'WEE1', 'XRCC3', 'ZMIZ1', 'ZMYND8', 'ZNF655']
Refined Community 130: ['AKT1', 'AKT1S1', 'AKT2', 'AKT3', 'AR', 'ARAF', 'BRAF', 'DEPTOR', 'EIF4EBP1', 'GRB2', 'HRAS', 'INPP4B', 'IRS1', 'KRAS', 'MAP2K1', 'MAP2K2', 'MAPK1', 'MAPK3', 'MAPKAP1', 'MLST8', 'MTOR', 'NRAS', 'PARP1', 'PARP2', 'PDK1', 'PIK3CA', 'PIK3R1', 'PRR5L', 'PTEN', 'RAF1', 'RICTOR', 'RPS6KB1', 'RPTOR', 'SOS1', 'SOS2', 'TELO2', 'TSC1', 'TSC2']
Refined Community 131: ['AKT1', 'AKT2', 'AKT3', 'EGFR', 'ERBB2', 'ERBB3', 'ERBB4', 'HRAS', 'KRAS', 'MAPK1', 'MAPK3', 'NRAS', 'PIK3C2A', 'PIK3C2B', 'PIK3C2G', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3CG', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PIK3R4', 'PIK3R5', 'PIK3R6', 'RAF1']
Refined Community 132: ['APBA2', 'CEBPB', 'DEFB1', 'DSE', 'GMPS', 'GTPBP4', 'KCNK5', 'LAD1', 'LPIN1', 'MAP3K13', 'NCK1', 'PFKP', 'PLAUR', 'PLIN2', 'RGS16', 'RIN3', 'ROPN1', 'SIRPA', 'TEX10', 'THEMIS2', 'TMEM123', 'TTK']
Refined Community 133: ['ABCG1', 'ACOT2', 'AGGF1', 'ARMT1', 'BLVRA', 'CELSR1', 'COX6C', 'CRIP1', 'DALRD3', 'DNALI1', 'EEF1A2', 'ELOVL5', 'FAM174B', 'FOXA1', 'GNB1', 'INPP5J', 'LONP2', 'MYO5C', 'PTP4A2', 'REEP5', 'RETREG1', 'SKP1', 'SLC39A6', 'TMBIM6', 'TSPAN1', 'TSPAN13']
Refined Community 134: ['ARHGEF9', 'BCL11A', 'BTG3', 'CDH3', 'CYB5R2', 'FABP5', 'GABRP', 'LBR', 'LDHB', 'MFAP2', 'PLCH1', 'PROM1', 'PUM3', 'RAP2B', 'RARRES1', 'SFRP1', 'SLC25A37', 'SLC9A6', 'SOX11', 'TBX19', 'TLE4', 'TRIM2', 'TUBB6', 'YEATS2']
Refined Community 135: ['ACAT2', 'ACOT9', 'AK2', 'ANP32E', 'ANXA1', 'ARHGEF4', 'BBOX1', 'BICD1', 'BMAL2', 'CHI3L1', 'CXCR4', 'CYP39A1', 'DSC2', 'FNDC3B', 'FOXC1', 'FSCN1', 'FXR1', 'GSTP1', 'HDAC2', 'HSPA14', 'IFNAR2', 'IFRD1', 'INAVA', 'KIF14', 'MCCC1', 'ODC1', 'PALS2', 'PLD1', 'PLS3', 'PPP1CB', 'PRKD3', 'PRNP', 'RRP1B', 'RSU1', 'SEC63', 'SEL1L3', 'SERBP1', 'SGCE', 'SMCO4', 'SNX3', 'SOS1', 'ST3GAL6', 'SYNCRIP', 'TCF7L2', 'TTLL4', 'UBE2J1', 'UGT8']
Refined Community 136: ['AGR2', 'ALAD', 'AR', 'BBOF1', 'CERS6', 'CHMP2A', 'COQ7', 'CPT1A', 'DACH1', 'DNAJC17', 'EVL', 'FRAT1', 'GOLGA1', 'GSTZ1', 'GTF3C1', 'GUSBP14', 'HMGCL', 'IFT46', 'IVD', 'KIAA0319L', 'MOAP1', 'MSX2', 'PATZ1', 'RAB17', 'RAB26', 'SIGIRR', 'SIRT3', 'SNRNP35', 'STK32B', 'STN1', 'TP53TG1', 'UGCG']
Refined Community 137: ['ABAT', 'AHNAK', 'ANXA9', 'APBB2', 'BBS4', 'CA12', 'CIRBP', 'CYP2B6', 'ERBB4', 'ESR1', 'FBP1', 'GATA3', 'GREB1', 'GSTM3', 'INPP4B', 'KCNK15', 'KDM4B', 'KIAA0232', 'KRT18', 'LRBA', 'MAGED2', 'MAPT', 'MLPH', 'NAT1', 'NUMA1', 'P4HTM', 'SCNN1A', 'SCUBE2', 'SLC7A8', 'TBC1D9', 'TFF1', 'TFF3', 'TNS2', 'WFS1', 'WWP1', 'XBP1']
Refined Community 138: ['A2M', 'ACAD11', 'AEBP1', 'AFG2A', 'ALDH18A1', 'ANAPC5', 'ANKHD1', 'ANTKMT', 'ANXA9', 'APH1A', 'ARHGDIA', 'ARMC8', 'ARRDC1', 'ATP5IF1', 'BANF1', 'BCAP31', 'BPHL', 'BSCL2', 'C17orf100', 'C1orf122', 'C4orf54', 'CACHD1', 'CACNG7', 'CAST', 'CCDC124', 'CCDC93', 'CDK5RAP3', 'CEACAM1', 'CGRRF1', 'CNDP2', 'COL1A1', 'COMMD5', 'COPG1', 'CPD', 'CRYZL1', 'CSF1', 'CSF1R', 'CSNK1D', 'CXCL13', 'CXCL16', 'DAG1', 'DCAF6', 'DDR2', 'DESI1', 'DNAJC13', 'DNPEP', 'EIF4E3', 'EYA3', 'FNDC3A', 'FTH1', 'GALNT1', 'GFPT2', 'GLCCI1', 'GTF2H5', 'GYG1', 'GZMA', 'H4C9', 'HCK', 'HDAC11', 'HELLS', 'HPS1', 'HSP90B1', 'IFI27', 'IL18', 'JMY', 'KIF23', 'KLHL25', 'LAMA5', 'LMNB1', 'MDFIC', 'MFSD5', 'MMP3', 'MTMR11', 'MXD4', 'NFKBIZ', 'NPRL3', 'NRIP2', 'NXPH3', 'OSBP', 'PAPOLA', 'PCMTD1', 'PFKFB4', 'PLOD3', 'PMP22', 'POFUT2', 'POLR3K', 'PPL', 'PPP1R13L', 'PPP1R2', 'PRKCD', 'PRPF39', 'PTGR2', 'PTPRB', 'RAMAC', 'RANBP1', 'RBM5', 'RENO1', 'REXO4', 'RGS10', 'RNF166', 'RORA', 'RSPH3', 'S100A1', 'SAP18', 'SCNN1A', 'SEC61A2', 'SEMA7A', 'SENP6', 'SETMAR', 'SF3B4', 'SOCS3', 'SOX10', 'SPECC1L', 'SPRY4', 'SRR', 'SRSF2', 'STAP2', 'STC1', 'SYNJ1', 'TAC1', 'TACSTD2', 'TAPBP', 'TIRAP', 'TLE4', 'TNFAIP2', 'TRMT13', 'TRPC4AP', 'TSHZ3', 'UGCG', 'UGT1A10', 'UQCRH', 'VAT1', 'WSB2', 'ZC3H11A', 'ZEB1', 'ZNF18', 'ZNF398', 'ZNRD2']
Refined Community 139: ['ABHD13', 'ACAT2', 'ACSL3', 'ACTL6A', 'ACTR3B', 'ACTR6', 'ACYP1', 'ADAMTS20', 'ADGRL3', 'ADH5', 'ADK', 'AEBP2', 'AHCY', 'AHR', 'AJUBA', 'AKAP9', 'ALG13', 'ANAPC1', 'ANKLE2', 'ANKRD26', 'ANLN', 'ANP32E', 'AP1S3', 'APPBP2', 'ARAP2', 'ARL6IP6', 'ARMC10', 'ARNT2', 'ARPP19', 'ASXL1', 'ATAD1', 'ATAD2', 'ATR', 'ATXN1', 'AURKA', 'AUTS2', 'AZIN1', 'BBIP1', 'BCAP29', 'BCL11A', 'BCL2', 'BCLAF3', 'BET1', 'BMI1', 'BRCA1', 'BRD7', 'BUB1', 'BUB1B', 'C11orf54', 'C1D', 'C1orf131', 'C1orf74', 'C21orf91', 'CA12', 'CALM2', 'CAPRIN1', 'CAPZA2', 'CASP8AP2', 'CAV2', 'CCDC90B', 'CCNA2', 'CD24', 'CD2AP', 'CD38', 'CDC25A', 'CDC40', 'CDC6', 'CDC73', 'CDCA7', 'CDCA8', 'CDKL5', 'CENPF', 'CEP164', 'CEP192', 'CEP43', 'CEP55', 'CETN3', 'CFAP97', 'CHCHD4', 'CHEK1', 'CHMP1B2P', 'CLDN12', 'CNOT6', 'COL25A1', 'COL4A5', 'CPSF6', 'CRLS1', 'CRYZ', 'CSTF2', 'CTCF', 'CTDSPL2', 'CTH', 'CTNND2', 'CTTNBP2', 'DBF4', 'DCK', 'DDR1', 'DEPDC1', 'DLAT', 'DNAJA1', 'DNAJC9', 'DPY30', 'DSG2', 'DUSP11', 'E2F7', 'ECT2', 'EDAR', 'EDARADD', 'EFCAB7', 'EFTUD2', 'EID1', 'EIF4B', 'EIF4H', 'EMID1', 'ENPP3', 'EPS8', 'ERCC6L2', 'ERI2', 'ESCO2', 'ETFRF1', 'EWSR1', 'EZH2', 'FAM13B', 'FAM199X', 'FAM76B', 'FARSB', 'FGFR1OP2', 'FH', 'FIGNL1', 'FIRRM', 'G3BP2', 'GEMIN6', 'GLRB', 'GMNN', 'GPM6B', 'GPSM2', 'GRHL2', 'GSDME', 'GTF2H2', 'GTPBP10', 'H2AZ1', 'HACE1', 'HARS1', 'HAT1', 'HELLS', 'HMGB2', 'HNRNPA2B1', 'HNRNPA3', 'HNRNPDL', 'HNRNPLL', 'HNRNPU', 'HOOK1', 'HUS1', 'IDH3A', 'IFT81', 'IFTAP', 'IK', 'ILF3', 'IMMP1L', 'ING3', 'INTS2', 'IPO8', 'ITGA2', 'ITGA6', 'JADE1', 'KANSL1L', 'KBTBD8', 'KCTD15', 'KIF16B', 'KIF2C', 'KITLG', 'LAMTOR5', 'LCORL', 'LMBRD2', 'LRBA', 'LRRC40', 'LSM5', 'LSM7', 'LZTFL1', 'MAP7D2', 'MARCHF6', 'MATR3', 'MBNL3', 'MCM2', 'MDH1', 'MED6', 'MELK', 'MET', 'METTL9', 'MIS18BP1', 'MLLT3', 'MME', 'MOB4', 'MOXD1', 'MPHOSPH8', 'MPZL1', 'MRPS18C', 'MRPS31', 'MRPS33', 'MTDH', 'MTHFD1', 'MYB', 'N4BP2', 'NAE1', 'NAF1', 'NANP', 'NCAPD2', 'NCAPD3', 'NCAPG', 'NCBP1', 'NDUFA4', 'NDUFC1', 'NDUFC2', 'NDUFS4', 'NEK1', 'NEO1', 'NFIA', 'NFU1', 'NFYB', 'NME7', 'NOL4L', 'NRAS', 'NUDCD1', 'NUP107', 'NUP155', 'NUP85', 'NVL', 'ORC5', 'OXCT1', 'PABPC4L', 'PACSIN2', 'PAICS', 'PAIP2', 'PANK1', 'PARK7', 'PARP1', 'PATZ1', 'PAXBP1', 'PBK', 'PCDH18', 'PCMTD2', 'PCYOX1', 'PDK1', 'PDS5B', 'PDZD8', 'PEX7', 'PFN2', 'PHB2', 'PHF14', 'PHF6', 'PIGF', 'PIN1', 'PKP4', 'PLCB1', 'PLEKHB1', 'POLA1', 'POLD3', 'POLR1C', 'POLR2B', 'POLR3H', 'POLR3K', 'POT1', 'PPAT', 'PPIP5K2', 'PPM1B', 'PPP2R1A', 'PPP4R3B', 'PRELID3B', 'PRIM1', 'PROM1', 'PRPS2', 'PRR14L', 'PSAT1', 'PSPC1', 'PTPN14', 'PUS7L', 'PWP1', 'RAD21', 'RAD51', 'RBAK', 'RBBP5', 'RBBP8', 'RBL1', 'RBM45', 'RELL1', 'RFC1', 'RFX3', 'RIDA', 'RIF1', 'RNF128', 'RNF44', 'RNF6', 'RNF7', 'RNMT', 'RPA1', 'RPA3', 'RPE', 'RPL15', 'RPRD2', 'RPS3', 'RPS6KA6', 'RRAGD', 'RRM2', 'SAP30', 'SBNO1', 'SEC22C', 'SEMA5A', 'SERBP1', 'SERTAD4', 'SESN3', 'SETX', 'SFPQ', 'SGO1', 'SGO2', 'SHCBP1', 'SHMT1', 'SKA2', 'SKP2', 'SLC20A2', 'SLC38A1', 'SLC39A10', 'SLC7A1', 'SLF1', 'SMARCA5', 'SMC2', 'SMC4', 'SMCHD1', 'SMIM15', 'SMNDC1', 'SNAPIN', 'SNRPB2', 'SNX5', 'SPIN1', 'SRSF10', 'SRSF3', 'SSB', 'SSBP1', 'ST6GAL1', 'STARD7', 'STK26', 'STMN1', 'STRAP', 'STRBP', 'STXBP6', 'SUCLA2', 'SYDE2', 'SYNCRIP', 'TAF1D', 'TAF2', 'TASP1', 'TAX1BP1', 'TCEA1', 'TCEANC', 'TFRC', 'TGFA', 'TIMM17A', 'TIMM21', 'TMEM117', 'TMEM209', 'TMEM230', 'TMEM33', 'TMPO', 'TMTC4', 'TNIK', 'TOMM70', 'TOP2A', 'TOPBP1', 'TP53BP1', 'TP63', 'TRAPPC10', 'TRAPPC2', 'TRDMT1', 'TRIM37', 'TRIM59', 'TRIP13', 'TRMT11', 'TRMT6', 'TRPM3', 'TRPS1', 'TRRAP', 'TSPAN13', 'TSPAN2', 'TTC3', 'TTC9C', 'TTF2', 'TULP3', 'UBA2', 'UBA3', 'UBE2B', 'UBE2D2', 'UBLCP1', 'UCHL5', 'UGDH', 'UHRF1', 'UNG', 'USP1', 'USP34', 'UTP15', 'VBP1', 'VDAC1', 'VDAC3', 'VPS26B', 'VPS54', 'WDR11', 'WDR3', 'WDR75', 'WEE1', 'WIF1', 'XPO1', 'XRCC5', 'YWHAB', 'YWHAQ', 'ZBTB44', 'ZC3HAV1L', 'ZCCHC8', 'ZFAND4', 'ZFAND6', 'ZIK1', 'ZNF136', 'ZNF148', 'ZNF160', 'ZNF22', 'ZNF239', 'ZNF24', 'ZNF267', 'ZNF317', 'ZNF362', 'ZNF367', 'ZNF532', 'ZNF770', 'ZNF846', 'ZNF878', 'ZRANB2']
        """
 

pattern = r"Refined Community \d+:\s*(\[[^\]]*\])"

communities = [
    ast.literal_eval(match)
    for match in re.findall(pattern, input_text)
]

print(f"Found {len(communities)} communities")
all_genes = list(
    set(
        gene
        for community in communities
        for gene in community
    )
)

print(f"Found {len(all_genes)} unique genes")
def shares_corum_complex(gene1, gene2):

    complexes1 = gene_to_complexes.get(gene1, set())
    complexes2 = gene_to_complexes.get(gene2, set())

    return len(
        complexes1.intersection(complexes2)
    ) > 0

random_communities = []

for community_idx, community in enumerate(
    communities,
    start=1
):

    community_size = len(community)

    random_community = []
    used_genes = set()

    attempts = 0
    max_attempts = 50000

    while len(random_community) < community_size:

        attempts += 1

        if attempts > max_attempts:

            raise ValueError(
                f"Failed to build community "
                f"{community_idx} "
                f"(size={community_size})"
            )

        candidate = random.choice(all_genes)

        if candidate in used_genes:
            continue

        valid = True

        for existing_gene in random_community:

            if shares_corum_complex(
                candidate,
                existing_gene
            ):
                valid = False
                break

        if valid:

            random_community.append(candidate)
            used_genes.add(candidate)

    random_communities.append(random_community)

print(
    f"Generated {len(random_communities)} "
    f"CORUM-disjoint random communities"
)

for idx, community in enumerate(
    random_communities,
    start=1
):

    for i in range(len(community)):
        for j in range(i + 1, len(community)):

            if shares_corum_complex(
                community[i],
                community[j]
            ):
                raise ValueError(
                    f"CORUM violation detected "
                    f"in community {idx}"
                )

print("Verification passed")

random_text = ""

for i, community in enumerate(
    random_communities,
    start=1
):

    genes = ", ".join(
        f"'{gene}'"
        for gene in community
    )

    random_text += (
        f"Refined Community {i}: "
        f"[{genes}]\n"
    )

output_file = (
    "BREAST_random_communities.txt"
)

with open(output_file, "w") as f:
    f.write(random_text)

print(
    f"Saved random communities to: "
    f"{output_file}"
)

### Annotation Pipeline

In [ ]:
class OptimizedPathwayAnalyzer:
    def __init__(self):
        self.api_key = None
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.checkpoint_file = f"checkpoint_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        self.results_csv = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        self.pathway_gene_mapping = {}
        
    def set_api_key(self, api_key: str = None):
        """
        Set the Deepseek API key
        """
        if api_key:
            self.api_key = api_key
        elif os.environ.get("DEEPSEEK_API_KEY"):
            self.api_key = os.environ.get("DEEPSEEK_API_KEY")
        else:
            raise ValueError("No Deepseek API key provided. Set it via argument or DEEPSEEK_API_KEY environment variable.")
    
    def _create_system_prompt(self) -> str:
        return """You are a bioinformatics expert. Focus on Breast Cancer. Provide detailed pathway analysis with scientific literature support. Always cite specific papers when discussing pathway relationships."""
    
    def _create_all_in_one_prompt(self, genes: List[str], enrichment_pathways: List[str]) -> str:
        enrichment_section = ""
        if enrichment_pathways and len(enrichment_pathways) > 0:
            enrichment_section = "\nEnrichment Analysis Results:\n" + "\n".join(f"- {pathway}" for pathway in enrichment_pathways)
        
        genes_str = ", ".join(genes)
        
        enrichment_instruction = "Using the enrichment pathways above (if any), provide an analysis of the gene set"
        
        return f"""You are a bioinformatics expert conducting pathway analysis of gene sets. For the gene set: [{genes_str}]{enrichment_section}

Please perform analysis in clearly labeled sections:

===== SECTION 1: ANALYSIS WITH ENRICHMENT =====
{enrichment_instruction}:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. Provide explicit reasoning for choosing this pathway/process
4. List the specific genes from the gene set that contribute to the process
5. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
6. If there isn't sufficient evidence or fewer than two genes are associated with a common pathway, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITH ENRICHMENT: [Process Name] ([Confidence Score])

PATHWAY REASONING WITH ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

CONTRIBUTING GENES WITH ENRICHMENT:
[Comma-separated list of contributing genes]

ANALYSIS TEXT WITH ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 2: ANALYSIS WITHOUT ENRICHMENT =====
Based SOLELY on your knowledge of these genes, without considering the enrichment results:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. List the specific genes from the gene set that are involved in the process
4. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
5. IMPORTANT: If there isn't sufficient evidence or fewer than two genes share a common biological function, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITHOUT ENRICHMENT: [Process Name] ([Confidence Score])

CONTRIBUTING GENES WITHOUT ENRICHMENT:
[Comma-separated list of contributing genes]

PATHWAY REASONING WITHOUT ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

ANALYSIS TEXT WITHOUT ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 3: FINAL PROCESS SELECTION =====
Compare both analyses and provide:
1. Which process (with or without enrichment) has higher confidence and why
2. Final reasoning for the chosen process

Output format for this section:
FINAL PROCESS REASONING:
[Detailed explanation of why you chose the final process, comparing both analyses and explaining which one provides stronger evidence]

Analytical Guidelines:
- Be concise and avoid unnecessary words
- Be factual without editorializing
- Be specific, avoiding overly general statements
- Avoid listing individual protein facts
- Group proteins by similar functions
- Discuss their interplay, synergistic or antagonistic effects
- Focus on functional integration within the system
- Include at least 2-3 specific paper citations when discussing established pathway relationships

Confidence Score Instructions:
- Assign a score from 0.00 to 1.00
- 0.00 indicates lowest confidence
- 1.00 reflects highest confidence
- Base the score on the proportion of genes participating in the identified process"""
    
    def _make_deepseek_request(self, prompt: str) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        data = {
            "model": "deepseek-chat",
            "messages": [
                {"role": "system", "content": self._create_system_prompt()},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0,
            "max_tokens": 4000
        }
        
        response = requests.post(self.api_url, headers=headers, json=data)
        
        if response.status_code == 200:
            result = response.json()
            return result['choices'][0]['message']['content']
        else:
            raise Exception(f"Deepseek API request failed: {response.status_code} - {response.text}")
    
    def parse_communities(self, input_text: str) -> Dict[str, List[str]]:
        communities = {}
        for line in input_text.strip().split('\n'):
            line = line.strip()
            if 'Refined Community' in line:
                match = re.search(r'Refined Community (\d+): \[(.*?)\]', line)
                if match:
                    community_num = match.group(1)
                    genes_str = match.group(2)
                    genes = [g.strip("'") for g in genes_str.split(', ')]
                    communities[community_num] = genes
        return communities
    
    def perform_enrichment(self, gene_list: List[str]) -> List[str]:

        try:
            databases = ['GO_Biological_Process_2021', 'Reactome_2022', 'KEGG_2021_Human']
            
            all_results = []
            for database in databases:
                enr = gp.enrichr(gene_list=gene_list,
                                gene_sets=[database],
                                organism='Human',
                                outdir=None,
                                no_plot=True,
                                cutoff=0.01)
                
                results_df = enr.results
                if not results_df.empty:
                    results_df = results_df.sort_values('Adjusted P-value')
                    for _, row in results_df.head(5).iterrows():
                        all_results.append(row['Term'])
            
            return list(set(all_results))
        
        except Exception as e:
            print(f"Enrichment analysis failed: {str(e)}")
            return []
    
    def _clean_markdown_formatting(self, text: str) -> str:

        if not text or not isinstance(text, str):
            return text
            
        text = re.sub(r'\*([^*]+)\*', r'\1', text)
        
        text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
        
        text = re.sub(r'(?<!\w)\*(?!\w)', '', text)
        
        return text.strip()
    
    def _extract_section(self, text: str, section_name: str) -> str:

        patterns = [
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\n[\w\s]+:|===|$)",  # Main pattern
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n[A-Z][A-Z\s]+:|$)",
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\*\*|$)" 
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                result = match.group(1).strip()
                if result:
                    return self._clean_markdown_formatting(result)
        
        return f"{section_name} not found"
    
    def _extract_process_info(self, text: str, prefix: str) -> Tuple[str, float]:

        patterns = [
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]+?)\s*\(([0-9.]+)\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]*?)\s*\(\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*Confidence Score:\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\*\*Confidence Score:\s*([0-9.]+)\*\*",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*\*\*Confidence Score:\s*([0-9.]+)\*\*\s*\)"
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                process_name = match.group(1).strip()
                try:
                    confidence_score = float(match.group(2))
                    process_name = self._clean_markdown_formatting(process_name)
                    process_name = process_name.strip()
                    if process_name and confidence_score >= 0:
                        return process_name, confidence_score
                except ValueError:
                    continue
        
        simple_pattern = rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)(?=\n|$)"
        match = re.search(simple_pattern, text, re.IGNORECASE)
        if match:
            process_name = match.group(1).strip()
            process_name = self._clean_markdown_formatting(process_name)
            process_name = re.sub(r'\s*\([^)]*$', '', process_name)
            process_name = process_name.strip()
            return process_name, 0.0
        
        return f"Unknown Process {prefix} Enrichment", 0.0
    
    def _extract_final_process(self, text: str) -> str:

        match = re.search(r"FINAL PROCESS REASONING:(.*?)(?=\n\n|$)", text, re.DOTALL | re.IGNORECASE)
        if match:
            result = match.group(1).strip()
            return self._clean_markdown_formatting(result)
        return "Final process reasoning not found"
    
    def analyze_community_optimized(self, comm_id: str, genes: List[str]) -> Dict:

        results = {
            "Community": comm_id,
            "Genes": genes,
            "Genes_String": ", ".join(genes)
        }
        
        try:
            enrichment_pathways = self.perform_enrichment(genes)
            results["Enrichment_Pathways"] = enrichment_pathways
            
            all_in_one_prompt = self._create_all_in_one_prompt(genes, enrichment_pathways)
            
            full_analysis = self._make_deepseek_request(all_in_one_prompt)
            results["Full_Analysis"] = full_analysis

            process_with_enrichment, confidence_with_enrichment = self._extract_process_info(full_analysis, "WITH")
            results["Process_With_Enrichment"] = process_with_enrichment
            results["Confidence_With_Enrichment"] = confidence_with_enrichment

            process_without_enrichment, confidence_without_enrichment = self._extract_process_info(full_analysis, "WITHOUT")
            results["Process_Without_Enrichment"] = process_without_enrichment
            results["Confidence_Without_Enrichment"] = confidence_without_enrichment
            
            try:
                results["Pathway_Reasoning_With_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITH ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Pathway_Reasoning_Without_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITHOUT ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Contributing_Genes_With_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITH ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Contributing_Genes_Without_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITHOUT ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Analysis_Text_With_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITH ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Analysis_Text_Without_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITHOUT ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                final_reasoning = self._extract_section(full_analysis, "FINAL PROCESS REASONING")
                results["Final_Process_Reasoning"] = final_reasoning
            except Exception as e:
                results["Final_Process_Reasoning"] = f"Extraction failed: {str(e)}"
            
            if confidence_with_enrichment >= confidence_without_enrichment:
                results["Final_Process"] = process_with_enrichment
                results["Final_Confidence"] = confidence_with_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_With_Enrichment", "Not available")
            else:
                results["Final_Process"] = process_without_enrichment
                results["Final_Confidence"] = confidence_without_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_Without_Enrichment", "Not available")
            
            return results
            
        except Exception as e:
            error_msg = f"Analysis failed: {str(e)}"
            print(f"Error analyzing community {comm_id}: {e}")
            results["Error"] = error_msg
            results.update({
                "Process_With_Enrichment": f"Error: {error_msg}",
                "Confidence_With_Enrichment": 0.0,
                "Process_Without_Enrichment": f"Error: {error_msg}",
                "Confidence_Without_Enrichment": 0.0,
                "Final_Process": f"Error: {error_msg}",
                "Final_Confidence": 0.0,
                "Full_Analysis": f"Error: {error_msg}",
                "Pathway_Reasoning_With_Enrichment": "Error occurred",
                "Pathway_Reasoning_Without_Enrichment": "Error occurred",
                "Contributing_Genes_With_Enrichment": "Error occurred",
                "Contributing_Genes_Without_Enrichment": "Error occurred",
                "Analysis_Text_With_Enrichment": "Error occurred",
                "Analysis_Text_Without_Enrichment": "Error occurred",
                "Final_Process_Reasoning": "Error occurred",
                "Final_Contributing_Genes": "Error occurred"
            })
            return results
    
    def analyze_communities_batch(self, communities: Dict[str, List[str]], batch_size: int = 1) -> List[Dict]:

        all_results = []
        community_items = list(communities.items())
        
        from tqdm import tqdm
        
        progress_bar = tqdm(total=len(community_items), desc="Analyzing communities", unit="community")
        
        for i in range(0, len(community_items), batch_size):
            batch = community_items[i:i+batch_size]
            
            for comm_id, genes in batch:
                result = self.analyze_community_optimized(comm_id, genes)
                all_results.append(result)

                progress_bar.update(1)

                processed = len(all_results)
                total = len(community_items)
                progress_bar.set_description(f"Processed {processed}/{total} communities")
                
                progress_bar.set_postfix({"Current": f"Community {comm_id}", "Genes": len(genes)})
        
        progress_bar.close()
        
        return all_results
    
    def create_detailed_dataframe(self, communities_text: str, batch_size: int = 1) -> pd.DataFrame:

        communities = self.parse_communities(communities_text)
        all_results = self.analyze_communities_batch(communities, batch_size)
        
        full_df = pd.DataFrame(all_results)
        
        requested_columns = [
            "Community",
            "Genes_String",
            "Enrichment_Pathways",
            "Process_With_Enrichment",
            "Confidence_With_Enrichment", 
            "Pathway_Reasoning_With_Enrichment",
            "Contributing_Genes_With_Enrichment",
            "Process_Without_Enrichment",
            "Confidence_Without_Enrichment",
            "Pathway_Reasoning_Without_Enrichment",
            "Contributing_Genes_Without_Enrichment", 
            "Final_Process",
            "Final_Confidence",
            "Final_Contributing_Genes",
            "Final_Process_Reasoning",
            "Full_Analysis"
        ]
        
        available_columns = [col for col in requested_columns if col in full_df.columns]
        return full_df[available_columns]

def run_optimized_analysis(input_text: str, api_key: str = None, detailed_csv: str = None, batch_size: int = 1) -> pd.DataFrame:

    analyzer = OptimizedPathwayAnalyzer()
    analyzer.set_api_key(api_key)
    
    detailed_df = analyzer.create_detailed_dataframe(input_text, batch_size)
    
    if detailed_csv:
        detailed_df.to_csv(detailed_csv, index=False)
    
    return detailed_df

if __name__ == "__main__":
    input_text = """
Refined Community 1: ['NAV1', 'MMP7', 'ERAP1', 'CLCF1', 'MRPL58', 'DYRK1A', 'SYCP1', 'GRINA', 'CENPA', 'ENSG00000301761', 'REXO4', 'USHBP1', 'MTERF3', 'BIN2', 'HNMT', 'NOSTRIN', 'MAP7D1', 'TET3', 'IGHA1', 'TCEAL4', 'PLA2G7', 'SDHAF2', 'GIMAP6', 'ODC1', 'CCL7', 'BEST3', 'WNK4', 'CARNS1', 'RAC1', 'CRYBG2', 'PEX5', 'CSNK1D', 'CALCOCO2', 'TOB1', 'CD14', 'ADGRE5', 'HCK', 'DBN1', 'CD36', 'CD40LG', 'CTSB', 'UBAP2L', 'PRLR', 'RFTN1', 'CDH4', 'TDRD12', 'EPHA1', 'SERINC3', 'CHRM3', 'RPE', 'CALML4', 'FAM131B', 'ING1', 'EHMT2', 'DIO1', 'LAMP5', 'MRPL9', 'IRF8', 'BAG4', 'TNNI3', 'SLC39A6', 'KIF21A', 'PPP2R5A', 'TRADD', 'TIMM21', 'ZNF74', 'CTCFL', 'KPNA1', 'LITAF', 'ZFC3H1', 'TMEM63A', 'WNT16', 'STXBP4', 'DIAPH3', 'SEPTIN6', 'OMD', 'SGCA', 'MAP7D3', 'PCBP2', 'SPRY1', 'IDO1', 'H2AZ2', 'UCHL3', 'CALU', 'GPR171', 'C3orf52', 'LMNA', 'CHEK1', 'TNFRSF11B', 'PDE5A', 'ADAMTS17', 'HOXA10', 'TSPYL5', 'CCN2', 'COQ9', 'PLXDC1', 'CTSS', 'ERAP2', 'ENOSF1', 'HRH3', 'CDK1', 'SPTAN1', 'MID1IP1', 'RUNX3', 'PON1', 'KHDC4', 'ZNF22', 'ANK1', 'RNF149', 'SPO11', 'GPR161', 'CLK4', 'PSMA7', 'TIGD5', 'SRPK1', 'MBP', 'GRIPAP1', 'RSU1', 'TRAF4', 'KRT8', 'FGFR1', 'HMGCS2', 'FIGNL1', 'MAL2', 'SS18L2', 'PRIM2', 'HSPB1', 'NSD3', 'TRIM3', 'CES5A', 'ATP11B', 'NHLRC1', 'MADD', 'CD74', 'IKBIP', 'CXCL12', 'ARHGEF28', 'CAV1', 'RTP4', 'ABL1', 'ALAD', 'CXXC4', 'FLT1', 'FN3K', 'CDKN1C', 'OPN3', 'IRS2', 'SPIRE2', 'ACTB', 'APOC1', 'PRNP', 'SOX13', 'SCD', 'UCK2', 'TUBG1', 'CDC42EP3', 'C4A', 'P2RX5', 'TMEM101', 'YIF1A', 'ITPK1', 'KDELR2', 'IL17RB', 'ZNF623', 'SLC37A1', 'ITSN1', 'POU2F1', 'NANS', 'BIRC2', 'EPM2AIP1', 'ZNF296', 'BRINP3', 'CLMP', 'CPT1A', 'CLDN12', 'PRPS2', 'PEX12', 'PTPN14', 'C2orf72', 'PHACTR3', 'BRMS1', 'CCDC88C', 'C1D', 'SERPINC1', 'SIAH2', 'TMEM123', 'KAT6B', 'TRAFD1', 'ACKR3', 'WLS', 'ZNF571', 'GSTT2', 'CTHRC1', 'PPP1R26', 'PTPN22', 'MRPS33', 'IQSEC1', 'ADRM1', 'ISG20', 'CLSPN', 'CDK14', 'CRIP2', 'MRPS12', 'BHLHE40', 'CDC42SE1', 'B3GNT3', 'ABHD15', 'ZNF470', 'CD72', 'DES', 'STK17B', 'ARL4C', 'SAP30', 'PTEN', 'SLC7A1', 'TPBG', 'RNASE6', 'EXT1', 'LIMD2', 'MLF1', 'EMC8', 'TFRC', 'NT5C', 'RNF128', 'ETV5', 'TGFA', 'PRKAR1A', 'CRELD2', 'DNAI7', 'PDE8A', 'RAB34', 'LINC01128', 'SLC12A4', 'SPDEF', 'PIK3CD', 'ZNF382', 'GAS8', 'GRHL2', 'ENSG00000280119', 'EMILIN1', 'SNX11', 'ASAH1', 'ZNF160', 'NEXN', 'PMVK', 'DALRD3', 'KANK3', 'RECQL4', 'PLAGL1', 'RNF138', 'SLC45A4', 'IFI27', 'PCDHA9', 'FFAR2', 'TBX2', 'MAGI2', 'SCAF1', 'LIPE', 'SLC35A3', 'CACNA1A', 'IGKC', 'HPS1', 'COL4A6', 'TMEM33', 'SCX', 'PDGFRA', 'NOXO1', 'CDKAL1', 'CCDC15', 'EIF4B', 'OCLN', 'MSRB1', 'TJAP1', 'CITED1', 'TLN2', 'MAP6', 'RPL8', 'RABGAP1', 'GIP', 'CAVIN3', 'KCNB2', 'RNF170', 'ASS1', 'SUPT16H', 'EN1', 'KRT16', 'HLA-DOB', 'SLC15A4', 'KRT26', 'ADGRF5', 'BARX1', 'DNHD1', 'ADH4', 'SMAD7', 'MDK', 'MAN1C1', 'EPHA3', 'IGKV1D-8', 'MSL2', 'NT5C2', 'HTRA4', 'DNAJC1', 'TTLL6', 'MLXIP', 'PAPPA', 'CHSY1', 'RALY', 'CSGALNACT1', 'PNP', 'FAM13B', 'CCN5', 'ID3', 'LRRC31', 'FANCG', 'PROS1', 'SELENOF', 'FLG2', 'ARHGAP23', 'ESPL1', 'ZBTB5', 'NME4', 'PLAAT2', 'SBK1', 'CD274']
Refined Community 2: ['TSR3', 'MAFK', 'GALNS', 'SEC23B', 'ERG', 'SAMD9L', 'UNCX', 'IRF6', 'LGALS7', 'ADM', 'WNT10B', 'EIF5A', 'XRCC4', 'ITIH3', 'METTL26']
Refined Community 3: ['VPS28', 'KCTD20', 'ARL3', 'TMEM30A', 'NECTIN2', 'ODF1', 'FCAMR', 'ARID3A', 'PALLD', 'CFAP263', 'H3C4', 'SPEG', 'SMC1A', 'BRAF', 'HENMT1', 'BCAS3', 'COL24A1', 'NKX2-5', 'FBLN5', 'NEDD4L', 'GPR176', 'SCG2', 'TMBIM6', 'CDK1', 'SRSF10', 'NOTCH1', 'MT-ND5', 'TRIM65', 'STAG3', 'GRINA', 'TSPAN2', 'FBXL18', 'CFAP69', 'NORAD', 'SP140', 'SERHL2', 'UBE2F', 'LYPD2', 'AFF3', 'LCP2', 'CNTRL', 'IFITM3P7', 'CTNNBL1', 'ADIPOQ', 'ZNF16', 'ABCA9', 'TMEM104', 'CD53', 'LRRC8D', 'YEATS2', 'UQCC4', 'GM2A', 'CRYBG2', 'TMEM209', 'MYPOP', 'ACTA2', 'TMPRSS2', 'KATNAL1', 'PRLR', 'FXYD3', 'RNF169', 'CCL4', 'FAAH', 'DUSP1', 'ANAPC5', 'TAFAZZIN', 'KLHL25', 'LRRC24', 'HOXA13', 'CYP7A1', 'GSK3B', 'IGHV4-61', 'PPP1R14B', 'BUB3', 'FMNL2', 'NELL2', 'FGF1', 'TLN1', 'GALNT7', 'RAB37', 'DMWD', 'GINS3', 'MARCHF6', 'TFF3', 'HECTD4']
Refined Community 4: ['IRF6', 'MMADHC', 'SYNRG', 'HBEGF', 'RB1', 'SPG7', 'FOXO1', 'FAM200B', 'FGL2', 'NLRC5', 'SYK', 'OSBPL3', 'MRPS34', 'ZNF133', 'CRYZL1', 'MAGEA2', 'GNS', 'LMO3', 'NUFIP2', 'PPAT', 'NCAPG', 'CHD6', 'ZKSCAN5', 'GMNN', 'HNRNPU', 'ZNF12', 'AKT1S1', 'SFMBT2', 'CYP2B6']
Refined Community 5: ['PACRG', 'PHYH', 'RBM19', 'LETMD1', 'NELFB', 'TENM3', 'MAPK3', 'EVPL', 'GPAA1', 'AMZ2', 'DIP2A', 'PFAS', 'CFD', 'TSPYL5', 'NCSTN', 'ZNF330', 'PAF1', 'OPLAH', 'MMP7', 'IFT74', 'ZNF133', 'PRG2', 'SYK', 'RTKN', 'PART1', 'WNT6', 'RHPN1', 'RECQL5', 'GPER1', 'PANX2', 'PMEPA1', 'ZNF778', 'WIPI2']
Refined Community 6: ['PRAC1', 'ETF1', 'CENPBD1P', 'DKK3', 'WASHC4', 'KRTAP9-4', 'L3MBTL2', 'SLC16A6', 'NES', 'ATP5F1B', 'PEX19', 'MFGE8', 'MARVELD2', 'SLC49A3', 'SRSF7', 'STXBP4', 'BTBD9', 'SSPN', 'PAGR1', 'CDS1', 'CDC42BPA', 'TREX1', 'KRT28', 'RNFT2', 'HOXA5', 'AAGAB', 'TMEM123', 'SUPT20H', 'VPS37C', 'EIF4A1', 'CALU', 'DIDO1']
Refined Community 7: ['NFIA', 'COL14A1', 'BRCA1', 'HES5', 'CORO1A', 'EIF3B', 'WARS1', 'NCAPD2', 'MRPS25', 'KLHL29', 'PAK2', 'PARP4', 'GPX7', 'LSM7']
Refined Community 8: ['RND1', 'AIM2', 'H2AZ1', 'ZNF20', 'SLC1A1', 'MS4A2', 'NINJ2', 'RIN1', 'ADGRB2', 'RNF43', 'BIN1', 'SCAMP1', 'DEFB1', 'LMNB1', 'AGAP2', 'PAF1', 'STARD3', 'PRKCA', 'LINC00469', 'SPIN1', 'BTBD9', 'KCNK9', 'PRKCD', 'NSD2', 'CD40LG', 'AFG2B']
Refined Community 9: ['DLL3', 'RBP4', 'ROMO1', 'SLC4A11', 'SSH3', 'MCCC1', 'APBB2', 'DSCR10']
Refined Community 10: ['C1orf122', 'GPD1', 'FUT3', 'RBM4', 'UAP1L1', 'C2CD2', 'BAIAP2L1', 'FABP5', 'ANAPC15', 'LARS1', 'AP5S1', 'PTOV1-AS2', 'PLCG2', 'FAM209A', 'SPINK5', 'SIRT5', 'CARD19', 'BRD8', 'GINS4']
Refined Community 11: ['RNFT2', 'MMP11', 'SLIT2', 'LAMB1', 'USP35', 'GSDMA', 'ZEB1', 'ZNF213', 'HPSE', 'MGRN1', 'RRAS2', 'LOXL1', 'KHSRP', 'LOX', 'DDX51', 'LIN28A']
Refined Community 12: ['RPL19', 'FRAT2', 'EGLN1', 'KLF2', 'NDUFC2', 'RASGRP1', 'HEBP2', 'TWIST1', 'BUD31', 'NOX4', 'TAGLN', 'METRNL', 'RCAN1', 'ANKIB1', 'UBLCP1', 'IFRD2', 'DLGAP4', 'SIRPAP1', 'CACNA1D', 'KLK8', 'FCGR3A', 'PBX1', 'EFCAB5', 'SAMD9', 'TCP11L1', 'ZBTB18', 'SNTG2', 'SYNJ2BP', 'APBA3', 'CREBBP', 'AKR1C3', 'POP1', 'TICAM1', 'LIN7A', 'SLC35E3', 'MAFF', 'ZFP30', 'FMNL2', 'CALB1', 'TMEM109', 'GALNT9', 'KLHL7', 'MIR31HG', 'GNA14', 'HCCS', 'NAMPT', 'PPP1R14C', 'IL17REL', 'FBXW2', 'ANKRD40', 'CIBAR1', 'PROCR', 'SFN', 'MKI67', 'SNRPA1', 'RBM33', 'AHCY', 'STAP2', 'ZNF727', 'HARS1', 'FOS', 'CDK7', 'TTLL12', 'SUMO2', 'MCM7', 'NEDD4L', 'PIERCE1', 'IGLV3-25', 'FOXA1', 'KLK6', 'HDGFL3', 'SAV1', 'ESRP2', 'ANK2', 'PKD1', 'SMARCD3', 'PSMD14', 'CCNB2', 'PIK3C2B', 'PPP1R18', 'HSPA12B', 'SNHG3', 'PRSS8', 'MSRB1', 'ITGA7', 'HLA-G', 'CD72', 'RAB11FIP4', 'LPCAT4', 'KIF3C', 'CCNYL1', 'ESPL1', 'MILR1', 'KRTAP9-3', 'TLK2', 'HOXB4', 'CXCR3', 'CAV1', 'AK5', 'STC2', 'NOG', 'CYP4B1', 'ABCA5', 'NASP', 'ATP7A', 'HSPA6', 'TSHZ2', 'ALKBH5', 'SPTLC2', 'EDEM2', 'ITPRIPL2', 'INHBC', 'HEPH', 'ADGRG2', 'BMPR1A', 'CAMP', 'ADIG', 'MLH1', 'CUL9']
Refined Community 13: ['FHL2', 'FAM200B', 'CDKAL1', 'CSNK2A2', 'TNFSF13', 'GTF3C1', 'C1R', 'MCM7', 'COL26A1', 'ATL3', 'OBI1', 'PRKD3', 'TPX2', 'CD59', 'RTN1', 'DLG5', 'ENPP5', 'PIK3R1', 'KICS2', 'BAALC', 'BICRA', 'CD300A']
Refined Community 14: ['ZNF426', 'MMP16', 'ITSN1', 'HDAC11', 'ANKRD46', 'KLHL8', 'CLEC2B', 'OAS1', 'BTN3A2', 'FCGR3B', 'MOB3B', 'ADD1', 'SSR4', 'ABCA6', 'ETFB', 'TUFM']
Refined Community 15: ['PIP4K2B', 'GNB1', 'RFNG', 'SLC46A1', 'NDUFB9', 'PTX4', 'GZMH', 'CXCL13']
Refined Community 16: ['PEX12', 'C4orf54', 'IGF2BP1', 'AGL', 'GLMN', 'SPATA7', 'PKHD1L1', 'OFD1', 'ETV1', 'PCDH18', 'CLEC11A', 'MMD', 'ADAM28', 'ACADL', 'SOX7', 'BRINP3', 'PRR14L', 'GYG1', 'KAZALD1', 'GALE', 'IFT140', 'SERPINA1', 'APBA1', 'ATAD3A', 'CLTC', 'MAP7', 'APOH', 'TOMM70', 'GPR65', 'FAT2', 'PRKX', 'ELF4', 'NRIP2', 'S100A8', 'AFF3', 'VPREB3', 'SNX3', 'DBNDD1', 'HJURP', 'PIDD1', 'PTGER3', 'ABCA5', 'ABCB1', 'AMPD2', 'BCAT1', 'MARVELD2', 'RCAN2', 'ZFP36L2', 'SMAD6', 'PLA2G4A', 'DEF6', 'SLC4A7', 'PRR13', 'CCT5', 'LY6E', 'WIPI1', 'PRRX1', 'BICC1', 'MATN2', 'BANF1', 'RGS4', 'KRTAP5-10', 'YBX1', 'EBAG9']
Refined Community 17: ['IGKV1D-8', 'FAM110C', 'GAGE1', 'CD79B', 'TXNDC16', 'PYGB', 'LINC01949', 'HOXB5', 'KCTD13', 'NFKBIZ', 'CHST8', 'IGLV4-60', 'LCP2', 'CD48', 'ID2', 'BHLHE40', 'R3HDM4', 'LGALS1', 'DMKN', 'SPRY4', 'ETF1']
Refined Community 18: ['MAN1C1', 'NOTCH3', 'LAMP3', 'BTD', 'BAG2', 'MYO7A', 'C17orf75', 'DENND2D', 'ST14', 'COPZ2', 'GABBR2', 'PLLP', 'ZNF517', 'ABCA3', 'MEOX1', 'SOX11', 'BTN3A2', 'FBXL20']
Refined Community 19: ['ZYX', 'LPCAT3', 'SYT1', 'IGKC', 'GAN', 'CHCT1', 'CALU', 'ACAA2', 'RALGAPA1', 'ADCY6', 'NDUFV1-DT', 'SOSTDC1', 'TNRC6C', 'SUSD6', 'PSMA7', 'SLC35E3', 'ASPN', 'RHO', 'PAG1', 'PPDPF', 'FLAD1', 'GPD1', 'GLI3', 'BCL11B', 'SLA', 'TBC1D16', 'KIF13B', 'ZNF84', 'CYP3A4', 'CAD', 'INAVA', 'RBP4', 'ABCD3', 'HLA-G', 'COL4A2', 'C1R', 'DENND2A', 'TF', 'NTRK2', 'MYL6B', 'MCM5', 'THOC6', 'AASS', 'ICA1', 'ATRN', 'SMAD4', 'TAGLN', 'DCST1', 'RAMAC', 'OSER1', 'ABCA9', 'CCR6', 'C2orf72', 'APBB2', 'KLHL28', 'PROCR', 'ANK3', 'TEDC2', 'SNX8', 'MRPS12', 'STAT5B', 'DDO', 'POLM', 'ARHGDIG', 'DESI2', 'FAM13B', 'SNAI3', 'LNX1', 'GALNT1', 'CD53', 'ATF5', 'TMEM147-AS1', 'FLOT2', 'GSTT2', 'NPY1R', 'FLYWCH1', 'SLC49A4', 'GPAA1', 'GPR152', 'CCL5', 'TBC1D19', 'PPP4R3B', 'UBA3', 'BIK', 'RASGRP3', 'DTX3L', 'CSTF1', 'CPPED1', 'CRISPLD1', 'CACNG8', 'SRPK3', 'ACSF3', 'ELOB', 'TRIB1', 'UCKL1', 'MSMB', 'ERGIC1', 'CRISPLD2', 'APOBEC3G', 'SHMT2', 'CTHRC1', 'EIF4E3', 'CSNK1A1L', 'KRT86', 'RFNG', 'ELL2', 'DHRS2', 'HARS1', 'PCOLCE2', 'SPCS3', 'KLRB1', 'ARHGEF40', 'ALB', 'TMEM151A', 'TPD52L2', 'IGLL3P', 'DGKD', 'XRCC4', 'ZBTB20', 'MRE11', 'PSME4', 'CASP1', 'LIMS2', 'GJC3', 'FOXO3', 'ATP6AP2', 'SETD1A', 'FCER1G', 'PRR16', 'FMNL2', 'CSF3R', 'RSAD1', 'PAPOLA', 'LINC03072', 'SCAMP1', 'HERC5', 'TXNRD1', 'NFE2L3', 'TASP1', 'GNA15', 'B4GALNT2', 'POLR1A', 'PLPBP', 'SERPINB2', 'CSRP2', 'NAP1L5', 'MARCHF8', 'MID1', 'ATP1A1', 'GALNT6', 'KRT8', 'APOBEC3B', 'CDKN3', 'ZNF358', 'RPL8', 'GATAD2A', 'NR3C2', 'CPSF6', 'ADAM8', 'PRRT3', 'ITSN1', 'SDHAF3', 'MAPK1', 'NAP1L2', 'MISP', 'ADA', 'EPB41L4A', 'HK1', 'TNPO3', 'TNFSF10']
Refined Community 20: ['SLC16A2', 'STXBP6', 'MYO18A', 'EDAR', 'PTGES', 'IKBKB', 'SUGP2', 'BRINP2', 'TRIM37', 'NPEPL1', 'NUP62', 'STAMBPL1', 'UBASH3B', 'GML', 'VPS26B', 'EPS8', 'NT5DC2', 'CLDN8', 'MDM1', 'EDN2', 'KRT27', 'PRR15', 'UGDH', 'LAMB2', 'ACTL8', 'LDLRAD4', 'SULF1', 'PRELID3B', 'TNFAIP8', 'TGFB3', 'STK36', 'UACA', 'MNDA', 'SSBP2', 'SLCO2B1', 'KCNV1', 'EFHD1', 'C3orf18', 'CD163', 'DCN', 'RGS10']
Refined Community 21: ['IGF1', 'LEPROT', 'ZNF213', 'CCNL1', 'SSBP1', 'ETF1', 'NTSR1', 'POT1', 'WDTC1', 'CXCL14', 'MICALL1', 'RTKN', 'KRTAP5-10', 'MSMB', 'NME4', 'ATG5', 'CFAP418', 'PLSCR3', 'JMJD7-PLA2G4B', 'HTT', 'SLK', 'KIF3C']
Refined Community 22: ['HDAC7', 'PTTG1', 'DESI2', 'SLC35E2B', 'FNBP4', 'WDR45B', 'PCBP3', 'BMP1', 'SIAH2', 'NDUFA4', 'NSUN5', 'BIN2', 'ANGPTL4', 'WTAP', 'HNRNPU', 'S100P', 'RDH10', 'TNFRSF12A', 'ENSG00000301761', 'YIF1A', 'CHST2', 'TUBB6', 'GPX7', 'WDR33', 'IRX3', 'LDLR', 'ZCCHC14', 'RBBP7']
Refined Community 23: ['TNFRSF10B', 'TMEM64', 'LENG8', 'ZNF367', 'PLEKHA1', 'ST3GAL2', 'CYP20A1', 'PIGU', 'RAP2A', 'HSPB8', 'MAP2K1', 'CD40LG', 'ZFHX4', 'RGS10', 'BBIP1', 'KLHL22', 'TANK', 'PARP6', 'CHRDL1', 'TMX4']
Refined Community 24: ['TOP1', 'BNC2', 'ADORA2A', 'CHCT1', 'QSOX2', 'NMI', 'CEPT1', 'OPRPN', 'MCF2', 'NQO1']
Refined Community 25: ['GLP1R', 'BIRC5', 'SF3B4', 'PDP1', 'LINC01949', 'MSX2', 'DTX4', 'HES5', 'IL13RA1', 'EPHA2', 'ARPC4', 'DOCK7', 'CYC1', 'CFAP97', 'TTF2', 'LRRC32', 'SCN8A', 'NDFIP2', 'ALAS1', 'PPP1R18', 'CETP', 'BAIAP3', 'PMAIP1', 'KDELR3', 'FEM1B', 'AMY1A', 'SLC45A4', 'HMMR', 'TBCK', 'RABEP2', 'METRN', 'CD53', 'URB2', 'MAGT1', 'SPATC1', 'LINC01128', 'SYMPK', 'METTL2A', 'DHRS7', 'RBBP8NL', 'STK26', 'FEM1C', 'CTNND1', 'TAPBPL', 'SMAD6', 'CHMP4C', 'RNF128', 'IQGAP2', 'SLC6A19', 'HMGA1', 'DNA2', 'FBLN5', 'PXDN', 'TTC28', 'TMEM87A', 'ID4', 'MGAT4A', 'YWHAE', 'ZBTB5', 'IGHA1', 'GAS1', 'CCDC80', 'E2F3', 'PCDHA9', 'VIM', 'AGAP2', 'SLC34A2', 'PLN', 'CLTC', 'CNDP2', 'FAM83E', 'QTRT1', 'PSMG1', 'MED13L', 'TLN1', 'NARF', 'CTPS1', 'SECTM1', 'YTHDF3', 'IL26', 'UBE2J1', 'SORBS1', 'TNRC6C', 'GRB7', 'HPS5', 'MED30', 'IL20RB', 'LINC00491', 'RAD1', 'HK1', 'PEX6', 'CDO1', 'AHR', 'GNG2', 'C1orf74', 'MIR1-1HG', 'PAK2', 'POLD3', 'CDT1', 'DLL3', 'DPY30', 'NUDCD1', 'TBX2', 'GNAL', 'PRSS33', 'SMCO4', 'ENSG00000293341', 'BSPRY', 'GALNS', 'CES5A', 'MAST1', 'HCP5', 'BAIAP2L1', 'LYPD3', 'TUBD1', 'RTF2', 'NRIP1', 'CSF1R', 'CIAPIN1', 'SPCS2P4', 'FGF10', 'UBQLN4', 'PSMD3', 'CAD', 'FOXA1', 'NFYB', 'FDX1', 'ELOVL4', 'OVOL1', 'SRGN', 'TMEM201', 'HSPG2', 'FRMPD1', 'DNMBP', 'TRIM3', 'SPRY1', 'ROR1', 'PPY2P', 'NABP1', 'CHRNA6', 'SMG9', 'PVR', 'RERG', 'HMCES', 'ORM2', 'HOXB4', 'SNRNP70', 'POLR3K', 'CCNL1', 'CCL3', 'UQCRB', 'GRIN2C', 'TMEFF1', 'GNB5']
Refined Community 26: ['ALDH6A1', 'SRPK1', 'NUTM2E', 'PRICKLE1', 'CYP2B6', 'RAB26', 'TAT']
Refined Community 27: ['CKMT1B', 'FNDC3B', 'CDCA3', 'MAPT', 'GM2A', 'EIF5A2', 'DNAJC13', 'SH3PXD2A', 'HRH1', 'LACTB2', 'MAP1LC3A', 'NPEPPS', 'CSTF2', 'MSX2', 'FAM106A', 'MTSS2', 'LINC03042', 'RECK', 'HLA-DOB', 'MTOR', 'TCEANC', 'DIDO1', 'PPP1R9B', 'PABPC4', 'KCNMB1', 'BSCL2', 'ACOX2', 'SRPX2', 'ZNF74', 'RALGPS2', 'SNX3', 'SNX9', 'PGA4', 'CD109', 'MCF2L-AS1', 'MRPL36', 'MPP1', 'LITAF', 'SRP19', 'GJC1', 'WIF1', 'RABEP2', 'S100A10', 'ITSN2', 'ACAA1', 'CREBBP', 'ALDOC', 'FAM217B', 'SOX18', 'TEX19', 'ABCA8', 'DCN', 'ESR2', 'KCNJ3', 'EIF5A', 'TPK1', 'ZDHHC2', 'ZNF567', 'TNFSF10', 'MEIS2', 'CA6', 'CHST3', 'CMC2', 'NINJ2', 'FAM234B', 'ZBTB10', 'OGFOD3', 'MZT2A', 'WFDC12', 'MXD4', 'ERGIC2', 'GDF5']
Refined Community 28: ['CDK16', 'KLHDC10', 'PPME1', 'RNASE4', 'DAP', 'P2RX2', 'CHST8', 'ZNF652', 'ENY2', 'TMEM74', 'PADI2', 'PROCR', 'PTOV1-AS2', 'TTLL6', 'LBR', 'EVL', 'NSMCE4A', 'ARHGDIB', 'SLC4A7', 'MED6', 'THEM4', 'H2AZ1', 'RAI2', 'LPAR2', 'MIS18A', 'PPP2R2C', 'RELB', 'CDH17', 'SEC24D', 'CCL20', 'RARRES2', 'HNRNPU']
Refined Community 29: ['RNF139', 'CALCRL', 'CHMP1B2P', 'CCDC89', 'DUSP2', 'ITPR1', 'KCNJ3']
Refined Community 30: ['ABL1', 'CACNA1G', 'NRAS', 'MARVELD3', 'SIM1', 'IGKV3-20', 'TMEFF1', 'NUP93', 'KCTD21', 'BMPR2', 'EPHA4', 'PON1', 'LY75', 'NLRP3', 'CTSC', 'ZNF142', 'TUSC2', 'JAK1', 'MRPL58', 'KCNV1', 'COLEC12', 'CLNS1A', 'UBE2C', 'TSPAN31', 'SPTLC2', 'TIMP3', 'LMBRD2', 'USE1', 'HOXB2', 'NSD2', 'FBXO43', 'GIT2', 'NGLY1', 'CXCR5', 'ATP8A2']
Refined Community 31: ['OGN', 'INPP4B', 'KLK6', 'ZNF84', 'PRKD2']
Refined Community 32: ['CLBA1', 'TRMT12', 'UCN', 'DGKE', 'EPM2AIP1', 'KHDC4', 'MLXIP', 'GSTA1', 'ZNF385D', 'FAT4', 'IL12RB2', 'ACSF2', 'PDP2', 'SLC23A2', 'SLC9A1', 'SLC16A6', 'SLC37A1', 'C17orf49', 'TRIL', 'PTGES', 'CFL2', 'SEC23B', 'RASSF2', 'PKP2', 'HLA-DQB2', 'RALGPS2', 'LILRB1', 'DLX4', 'PFKM', 'BCAP31', 'TTL', 'ZNF587', 'COG7', 'FRAT2', 'CYP3A43', 'KCTD2', 'GPR162', 'GVINP1', 'MDM2', 'IP6K2', 'ATP1A2', 'BAK1', 'B3GNT3', 'TIE1', 'GNB4', 'TMED2', 'DDIT4', 'SLC17A9', 'RBM45', 'NGFR', 'TEAD4', 'CPSF1', 'KLHL20', 'EPN3', 'TUFM', 'IFT20', 'ATP5MC1', 'HOXA9', 'PTPRCAP', 'PRC1', 'OTUD6B', 'NMT2', 'AJUBA', 'TFRC', 'HSF5', 'CD74', 'PKIA', 'KLHDC4', 'RAB37', 'TUBB3', 'NCALD', 'SERPINE2', 'ATAD1', 'GMEB2', 'IL12RB1', 'EMP2', 'SNX7', 'RDH13', 'RSKR', 'MSH2', 'HMGB2', 'BIK', 'BCAS4', 'FABP9', 'SH3GLB2', 'PRSS54', 'SLC15A3', 'AP1AR', 'ZGPAT', 'AVL9', 'CEP15', 'CCDC93', 'CHST12', 'PAH', 'BMPR2', 'CTTNBP2', 'SLC38A7', 'RAB40B', 'ENPP5', 'GREM1', 'SRSF7', 'TMIGD1', 'SYDE2', 'STK4', 'TMEM67', 'QTRT1', 'OXR1', 'AKAP9', 'PAICS', 'LIPE', 'TNIK', 'HDAC2', 'HTN1', 'MMP12', 'IKZF3', 'DLGAP4', 'PAPSS2', 'PRKACB', 'AFDN', 'ITGB6', 'UNK', 'RNF6', 'TENM3', 'DCK', 'PHB1', 'SEC31B', 'EPB41', 'HES1', 'HMGA1', 'SMAD7', 'PCYT1A', 'RASA2', 'F2RL1', 'LCP1', 'GFRA1', 'SPATA2', 'SLC11A2', 'YIPF5', 'SNRNP25', 'RASGRP2', 'BMP7', 'YWHAB', 'CCL14', 'ENTR1', 'KRT24', 'HPS5', 'FBXO46', 'DIDO1', 'CCDC77', 'LSS', 'PTOV1-AS2', 'MFAP5', 'PCGF6', 'PTK6', 'CEBPA', 'CSF2RA', 'CAP2', 'PPP2R5A', 'RNF214', 'TMEM92', 'F10', 'CAPN5', 'ZDHHC17', 'FJX1', 'FRMD4A', 'SEPHS1', 'IGHV4-34', 'FANCI', 'TOP1MT', 'ARHGEF5', 'RALY', 'ISG15', 'CCKAR', 'H4C9', 'TTC7A', 'C1orf21', 'USP21', 'PIGT', 'CHTF18', 'CDC25B', 'MYO1B', 'DVL3', 'C1RL', 'NOXA1', 'TRIP6', 'OIP5', 'KIF5C', 'PON2', 'FLT3LG', 'FGFR2', 'LFNG', 'EMG1', 'CMTR1', 'ANKRD13C-DT', 'MMP7', 'TPT1P8', 'LEF1', 'FAM242E', 'RNF151', 'CD302', 'KLC2', 'LAPTM5', 'THRA', 'CYTL1', 'CHRDL1', 'CAND1', 'AZI2', 'TIMP4', 'ABCA7', 'CAMSAP3', 'GINS4', 'CBLN1', 'TMEM74', 'MAPK12', 'TMEM138', 'ZKSCAN5', 'SLC25A44', 'CXCL8', 'WRAP53', 'MRPL38', 'CD55', 'MLLT6', 'CSNK1D', 'S100A14', 'NOX4', 'ARNT2', 'NECTIN4', 'ZDHHC11', 'BRD1', 'CYB5R1', 'FNBP4', 'CFD', 'MIS18A', 'WASHC4', 'ASNS', 'HSPBAP1', 'SOX4', 'THEM4', 'AGL', 'RND2', 'BOP1', 'SNRPC', 'FNTA', 'DCPS', 'SRL', 'REG1A', 'SUPT7L', 'DCLRE1C', 'DDAH1', 'ZBTB5', 'TBCD', 'DESI2', 'TMEM87A', 'RIF1', 'RNF139', 'PLK3', 'LMO2', 'EFCAB3', 'ARHGAP25', 'GNA14', 'JADE1', 'PTEN', 'DPY19L1', 'MAP4K3', 'F3', 'ANGPT1', 'TRMU', 'HOOK1', 'CELF3', 'TRPM3', 'SYDE1', 'EIF5AL1', 'CDK1', 'PTPRC', 'MACC1', 'ADRA2A', 'PANK1', 'FAN1', 'WBP2', 'WDR6', 'RSPH1', 'ZNF878', 'WIPF1', 'IL7', 'MRPS27', 'IGBP1', 'DOK4', 'GALK1', 'SPTBN2', 'USP7', 'GPR152', 'TPGS2', 'PARPBP', 'POLR2A', 'POLR1C', 'TMEM151A', 'TLE3', 'MCM10', 'FAM110B', 'MIR3682', 'ENSG00000278932', 'PAF1', 'TOPBP1', 'PLET1', 'CAPN15', 'SMG9', 'ANXA2', 'INPP1', 'GSTM3', 'ESM1', 'MMP2', 'TNFRSF12A', 'HBQ1', 'CD24P4', 'ARID2', 'ELAPOR1', 'HOXB1', 'MLLT11', 'HSPA2', 'HOXA5', 'IDO1', 'OGFOD3', 'SDHAF3', 'TRPC1', 'BBC3', 'RBMS1', 'SEC63', 'PDCL3', 'CMAS', 'KIF4A', 'LGALSL', 'NVL', 'SLC2A4RG', 'ZSCAN10', 'ADAM9', 'RECQL4', 'ACAT2', 'NDUFV1-DT', 'SLC39A4', 'LYL1', 'TGFBR2', 'PRKD3', 'VMP1', 'IGF2R', 'ASPH', 'PARVB', 'HELZ', 'CFLAR', 'TMEM268', 'EYA4', 'MRPL11', 'GCAT']
Refined Community 33: ['GJA4', 'L1CAM', 'CD24P4', 'CYP3A7', 'C14orf132', 'POLD3', 'CXCL10', 'ZBTB46', 'FOXN3', 'SETDB1', 'IRX3', 'TAGLN2', 'BRCA2', 'SIT1', 'MT-CO2', 'DENND2A', 'GDPD1', 'SCRG1', 'TRIP10', 'BTK', 'MTOR', 'AQP11', 'TKT', 'ZNF383', 'ARFGEF3', 'LONRF2', 'LGALSL', 'CES5A', 'NEBL', 'FGFR2', 'CACTIN', 'CHAF1A', 'CARTPT', 'NHLRC1', 'CHI3L2', 'RNF169', 'FRS2', 'PROM2', 'DNM1', 'LARP4B', 'SNCAIP', 'CCND2', 'STMN1', 'PDXDC1', 'LINS1', 'NANOG', 'IL24', 'SIRPA', 'RAB2A', 'AAR2', 'MPI', 'COTL1', 'MPP7', 'IFFO1', 'BTN3A1', 'FEZ2', 'MARCHF6', 'MAP2K6', 'MIR21', 'ITGB5', 'MDM2', 'NINJ2', 'RPSAP47', 'COL18A1-AS1', 'CD7', 'SMAD2', 'RPTOR', 'NUP107', 'RAD9A', 'CGRRF1', 'FABP9', 'DRAM2', 'CTSW', 'SYTL2', 'PLS3', 'MSN', 'LAYN', 'PLK1', 'TGFBR3', 'ATP6V0A4', 'SLC17A5', 'MMP3', 'MAPK8IP3', 'CDC25C', 'MAST1', 'NPL', 'LEPR', 'CRYGS', 'ANTXR2', 'LYVE1', 'EPHA2', 'APPL2', 'HPN', 'COL8A1', 'TRPV6', 'IRAK3', 'PNISR', 'PLAGL1', 'BAD']
Refined Community 34: ['LARS1', 'LARP6', 'JRKL', 'BIRC5', 'TMEM86B', 'YIF1A', 'EVI2A', 'CHMP4C', 'CBLC', 'AKR1C3', 'PIK3R5', 'ZFP41', 'IGHV3-20', 'ADAM33', 'CDC42BPA', 'CISH', 'MCRIP2', 'MAP7D2', 'GLP1R', 'PDGFRL', 'HAGHL', 'KRT16', 'SPRR3', 'ZNF382', 'ELF4', 'DCST2']
Refined Community 35: ['LYNX1', 'PRODH', 'MCM2', 'LRRFIP1', 'BSCL2', 'TAFAZZIN', 'GSK3B', 'IGLL3P', 'KANK3', 'CDK6', 'B3GNTL1', 'GAS8', 'GUSBP14', 'HMMR', 'ADAMTS1', 'SAMD9L', 'ZNF793']
Refined Community 36: ['DNAJB4', 'NSMAF', 'GDI2', 'PRR14L', 'ELN']
Refined Community 37: ['UBAP2L', 'LSM14B', 'CERS3', 'ALKBH5', 'AREG', 'CD69', 'VEGFD', 'HBB', 'ADGRF1']
Refined Community 38: ['HHEX', 'SLC13A2', 'THRA', 'ALDH4A1', 'ID3', 'PPP2R5A', 'RPRD2', 'RPP38', 'HEBP2', 'CYP20A1', 'TEX19', 'ARHGAP32', 'STK17B', 'CSTF1', 'ELL3', 'RRAGD', 'SLC25A44', 'CD6', 'CCDC117', 'AAMDC', 'MRFAP1L1', 'PRRT3', 'DCAF4L2', 'ZC3H3', 'ABCC4', 'COL1A1', 'NMB', 'DGKZ', 'ZNF585B', 'CHMP1B2P', 'MED24', 'PRKCQ', 'MIIP', 'NEMP1', 'SEMA5A', 'TUBG1', 'MAP3K5', 'KLK10', 'TMEM130', 'SLC27A2', 'EPB41L4A', 'CLDN1', 'ABCD3', 'GADD45G', 'TOMM34', 'GINS3', 'SLC2A1', 'PCSK6', 'CACTIN', 'KANK2', 'GZMK', 'DNAJA2', 'GRB2', 'NHERF1', 'NCF1', 'UTP25', 'WTAP', 'CHRNB2', 'EVL', 'PDCD4', 'TBL1X', 'WIF1', 'CHST15', 'DYNLRB2', 'SCRG1', 'MMP1', 'TNFRSF14', 'SDHAF2', 'BRI3', 'PRR15L', 'MPHOSPH10P1', 'AGL', 'PHGDH', 'SLC38A1', 'CTNNAL1']
Refined Community 39: ['BTBD9', 'TBC1D19', 'SCAND1', 'DDHD2', 'IL15RA', 'MDK', 'LTF', 'ESM1', 'TENT5C', 'FUT8', 'BCL6', 'RBM26', 'NECTIN2', 'AGPS', 'CEMIP', 'MAP2K2', 'NOC2L', 'CYB5R1', 'STMN3', 'GSN', 'MIR3682', 'ORM2', 'TNXB', 'POP1', 'MSH5', 'TFAP2B', 'CENPE', 'CDKAL1', 'ADGRG5', 'MYBL2', 'IL17REL', 'IGHV4-34', 'SIAH1', 'CST5', 'RAMP1', 'ZIC2', 'EIF5AL1', 'RMND1', 'CACFD1', 'HUS1', 'SERHL2', 'E2F8', 'CCKBR', 'CAND1', 'CASD1', 'SKAP1', 'TERF1', 'C1QBP', 'GPD1L', 'UGT2B11', 'ABCB1']
Refined Community 40: ['CPNE1', 'FAXC', 'RHOB', 'PCGF2', 'HOXB4', 'NUP155', 'ARHGAP5', 'C1orf131', 'NUAK1', 'GPC1-AS1', 'PEBP1', 'PTGS2', 'CSF1', 'AQP11', 'MS4A7', 'DCAF13']
Refined Community 41: ['RBP3', 'CARNS1', 'NUSAP1', 'ZMYND19', 'CAP2', 'PHB2', 'TENT5C', 'TACO1', 'TP53TG5', 'RALGPS2', 'CP', 'NCBP1', 'SOBP', 'BET1', 'IL17RB', 'HLA-DOB', 'HSD17B2', 'IL10', 'LTBP4', 'PTMA', 'MAGOHB']
Refined Community 42: ['PTPRZ1', 'EPYC', 'MLPH', 'TPR', 'CUL9', 'CARMN', 'MAP2K3', 'ADAM9', 'BAG1', 'FKBP2', 'FLG2', 'NCKAP5L', 'RNF182', 'SGO2', 'KCTD1', 'CST7']
Refined Community 43: ['RNF7', 'PDZD8', 'E2F2', 'ITGA3', 'PWWP2B', 'FNBP4', 'TUBG1', 'SPARCL1', 'ZC3HAV1L', 'SMC6', 'BNIP2', 'EIF4EBP1', 'PGA3', 'TTLL6', 'TRNP1', 'CAPNS2', 'ATXN1', 'RPUSD1', 'CCL13', 'PRKACB', 'POMK', 'RGS1', 'SLC12A1', 'LTB4R', 'NRAS', 'SRPK3', 'ZXDC', 'ETV6', 'CLIC2', 'PIP5K1A', 'RNF24', 'PITPNC1', 'MIRLET7D', 'NAP1L5', 'ELN', 'TATDN2', 'CMAS', 'CNTNAP4', 'LILRB4', 'MAP3K13', 'HOXA6', 'TCEA2', 'POLA1', 'LINC03042', 'CAMK2B', 'ACVR1', 'TG', 'GSTP1', 'ALDH3A1', 'COL8A2', 'MLLT1', 'VAMP3', 'MCM4', 'SOSTDC1', 'TBC1D17', 'DSG3', 'SCAF1', 'PKMYT1', 'VPS45', 'POLE', 'POLR2K', 'LPCAT1', 'WDR3', 'KRI1', 'MIR21', 'TMPO', 'BNC2', 'TMEM126A', 'SDC2', 'SHMT2', 'FANCF', 'AZI2', 'NECTIN2', 'ARFIP2', 'SH3BP4', 'WFIKKN1', 'TSHZ3', 'PGR', 'NEXN', 'ANP32E', 'TAF15', 'CLC', 'NFIX']
Refined Community 44: ['ENSG00000280119', 'KRT12', 'TOP2A', 'CHI3L1', 'NOB1', 'DLX4', 'LMNTD2-AS1', 'CD27', 'GDI2', 'FCGR2A', 'ACADM', 'C11orf52', 'PIR', 'FGF17', 'ZNF778', 'HELLS', 'RELL1', 'PTHLH', 'RBM39', 'TEAD3', 'XRRA1', 'CREB3L1', 'KRTAP5-7', 'ZNF74', 'YWHAE', 'UBE2Q2', 'MIR100HG', 'SELENOK', 'PI3', 'TRPV6', 'MMD', 'SERPINB1', 'CHMP4C', 'PRR14']
Refined Community 45: ['RFC1', 'NEDD4L', 'MARCHF10', 'DIP2A', 'CENPB', 'PLSCR1', 'FERMT1', 'RBFOX2', 'DIP2C', 'ST6GALNAC2', 'ZNF598', 'NR4A2']
Refined Community 46: ['ALDH1A1', 'BRAP', 'GSDME', 'GDPD5', 'ZNF611', 'DPY19L4', 'RBM47', 'FBXO17', 'ARHGEF6']
Refined Community 47: ['FAM199X', 'PBX1', 'WDTC1', 'CHAD', 'PRSS33', 'FH', 'MAGEA4', 'DUSP14', 'BUB3', 'FBXL7', 'RAI2', 'PACSIN2', 'XRCC5', 'CSPG4', 'GRK3', 'CYP2B7P', 'TUBA4A', 'ST6GAL1', 'DCBLD1', 'STK17B', 'P3H1', 'IRGQ', 'RAB2A', 'PPP4R3A', 'PRPF6', 'TMEM70', 'GHR', 'CKS2', 'ADAM10', 'PRKAR2B', 'PEF1', 'TMEM135', 'NAP1L1', 'TSPOAP1', 'HOXA3', 'RNF44', 'GAK', 'UTP25', 'PDK1', 'KCNK15', 'MRTFB', 'FOXP2', 'RGS9', 'PROCR', 'BNC1', 'ZCCHC14', 'ITM2C', 'PCDH18', 'SOCS7', 'CTSV', 'TBC1D17', 'MPHOSPH9', 'ETFB', 'TAF1A', 'PROSER1', 'SBSPON', 'CCR7', 'MTERF3', 'RBM39', 'CELSR2', 'VRK3', 'RANBP3', 'ARHGAP17', 'CEP112', 'NPIPB3', 'SAMD9', 'ZNF831', 'KDR', 'CLEC2D', 'CDH13', 'LENG8', 'BLOC1S1', 'SEC23A', 'CNOT2', 'AK2', 'PLCB4', 'ABL1', 'ITK', 'LILRA4', 'TPD52L1', 'SSTR5', 'KIT', 'ZC3H11A', 'TLN2', 'TNPO2', 'THSD4', 'STK11', 'TTC3', 'DPP3', 'CCNC', 'RCL1', 'TAF15', 'BAK1', 'PRSS22', 'SPON1', 'RABEP1', 'LALBA', 'FZD5', 'SCAND1', 'EEF1AKMT4', 'KRTAP5-10', 'SPECC1L', 'PLIN1', 'RHOV', 'CCR1', 'CD300LD-AS1', 'CD24', 'TGIF2', 'HBA1', 'FOSL1', 'TBCA', 'TREM2', 'FURIN', 'SERF1A', 'TGFB1', 'TPGS2', 'CEBPD', 'POGZ', 'DONSON', 'FNBP4', 'CDH26', 'RPS6KA5', 'MXRA8', 'GPR20', 'CD34', 'BGN', 'PRSS8', 'LSM14B', 'ZNF552', 'GTF2I', 'RTP4', 'MAP3K7CL', 'BOLA1', 'RASGEF1A', 'PRKCZ', 'APCDD1L', 'DRAM2', 'PIP', 'GABBR2', 'AFG2B', 'PALM2AKAP2', 'PGBD5', 'KIFC1', 'PGM2L1', 'HTN1', 'TCOF1', 'ERP29', 'NECTIN4', 'MOAP1', 'WLS', 'CACNB3', 'KATNAL1', 'NDRG4', 'UBE2M', 'VIPR1', 'SH2D5', 'THOC6', 'HMOX1', 'INHBA', 'PCMTD1', 'PDE4DIP', 'SMURF1', 'MIR205', 'ERO1A', 'CALCRL', 'PTBP1', 'CFAP97', 'TRDC', 'S100A2', 'WNT7A', 'KLC2', 'POLR1C', 'RAPH1', 'HOXB9', 'CDKN2C', 'CNTROB', 'EXPH5', 'PIR', 'TNFSF10', 'ADAM18', 'CYP11B2', 'ANKRD26', 'SLC15A3', 'LNX1', 'PABPC4', 'NCOA3', 'HPGDS', 'OCLN', 'SEMA3G', 'TSHZ3', 'NEFH', 'SFPQ', 'PTGR2', 'COL6A3', 'RNF43', 'LUC7L3', 'HOXB13', 'FEM1C', 'COL4A6', 'KANK1', 'CLDN4', 'BASP1', 'FYCO1', 'IDH1', 'SFT2D2', 'PRICKLE1', 'CCDC74B', 'LMBRD2', 'SNHG14', 'MXD4', 'ENSG00000278932', 'CYP1A1', 'ANKRD13C-DT', 'DZIP3', 'COL5A1', 'CXCL2', 'CDC7', 'ELP5', 'NEK2', 'DOC2B', 'IL1RAP', 'SLC48A1', 'UCHL5', 'RBL1', 'BICD1', 'NRIP1', 'CX3CR1', 'PKP1', 'SPPL2A', 'CCDC80', 'NUSAP1', 'CYP2B6', 'RFTN1', 'KAT6A', 'FN3K', 'IRS1', 'SPSB3', 'SPCS2P4', 'ZNF571', 'PGGHG', 'GNAL', 'WSB2', 'CKS1B', 'RHO', 'PDXK', 'TMEM61', 'ZFP41', 'G6PD', 'DYNLRB2', 'AGMAT', 'HIP1R', 'UPB1', 'C1QC', 'ITGA6', 'LYL1', 'MAP2K2', 'CYP3A4', 'ITFG1', 'CTHRC1', 'ZNF239', 'SOX12', 'ABCC2', 'GRB10', 'SNTB1', 'CD4', 'C1orf174', 'TUBB2A', 'CLN3', 'YTHDC2', 'HNRNPLL', 'SAT1', 'DGAT2', 'THBD', 'LGALSL', 'SLC7A1', 'BIRC7', 'RHBDF1', 'CEBPB', 'NUDT4', 'NOVA1', 'C1QTNF3', 'NRK', 'BMP4', 'WNT7B', 'COL19A1', 'DENND1B', 'CYP19A1', 'MACO1', 'TPD52L2', 'IPO5', 'RELA', 'ARFGEF2', 'LRCH4', 'RCAN1', 'ANGPT1', 'SLC35A1', 'PCK1', 'ARHGAP23', 'ACAN', 'PAQR3', 'RRP1B', 'DCLRE1C', 'ATP13A4', 'SIL1', 'ADGRG1', 'DNALI1', 'XIST', 'FCGR3A', 'COP1', 'CLASRP', 'CACNG1', 'NPM3', 'GRHL1', 'HOMER3', 'ICAM1', 'CUL2', 'PPP1R27', 'SAP18', 'ABHD13', 'FADS6', 'TNFRSF17', 'TRPM3', 'TAF1C', 'RHBG', 'HOTAIRM1', 'SLC30A8', 'PDGFA', 'CETP', 'CD79B']
Refined Community 48: ['FGF5', 'FGF20', 'RASAL1', 'WRAP53', 'NCAPD2', 'CLASRP', 'GIMAP5', 'ABCA9', 'TAF4', 'MED13']
Refined Community 49: ['TCFL5', 'L1CAM', 'EP400', 'C1QB', 'PBLD', 'ESR2', 'CLTC', 'PAF1', 'FTCD', 'SMPDL3A', 'PRRT3', 'SMIM14', 'RHPN1', 'PYCR3', 'FGFBP1', 'HACD1', 'WSB2', 'POMT1', 'JPH3', 'MEOX2', 'FAM76B', 'TBK1', 'CAMK2B', 'KRT23', 'ENTR1', 'ANKRD54', 'TFCP2L1', 'GH1', 'SLC39A3', 'ARPC1B', 'CDC42', 'INTS13', 'C14orf132', 'DHCR7', 'CYB5R3', 'SYNRG', 'P4HA3', 'CTDSP2', 'CLN8', 'PLEKHA2', 'UNC119', 'RIN1', 'HSPA6', 'PCMTD2', 'PTN', 'RBM33', 'TLE4', 'EFEMP1', 'RRN3', 'TFAP2B', 'S100A6', 'ME3', 'ILF3', 'GSK3A', 'GALNT14', 'NR1D1', 'FXYD5', 'SERPINC1', 'ADAM8', 'LBP', 'HOXB8', 'HBA2', 'CD69', 'FNDC5', 'PCOLCE2', 'WNT5A', 'ANTXR2', 'SMC6', 'ACTG2', 'PRF1', 'DHX16', 'MAPK12', 'SMAD4', 'ACOX2', 'PPP1R14B', 'BAK1', 'PNKD', 'VILL', 'TEAD3', 'VPS45', 'KIF1B', 'PRR14L', 'PLAUR', 'USH1G', 'GPC1-AS1', 'CIDEC', 'EDEM2', 'KATNAL1', 'EFNA4', 'NUDT1', 'SNRPD1', 'CD244', 'SEMA3C', 'KHSRP', 'SLC6A4', 'PLLP', 'CHRNA6', 'CALML5', 'ADAM33', 'ZNF398', 'OR2AT4', 'CALML4', 'IP6K2', 'KMO', 'SLC15A1', 'CC2D1B', 'CD1C', 'MISP', 'NEU4', 'HSD17B2', 'TLR7']
Refined Community 50: ['SIRPAP1', 'KLF10', 'LNX1', 'ASB8', 'CHAD', 'PROS1', 'STAG2', 'ACADS', 'PRDX4', 'SPON2', 'CHTF18', 'F2R', 'CERK', 'RAP2B', 'KLHL38', 'MAP7D3', 'NPAS4', 'APLP1', 'ATP6V1G1', 'ZNF469', 'IL26', 'PTGFR', 'TXNRD1', 'HPCA', 'BRD1', 'CCL17', 'CAPRIN1', 'DCX', 'RENO1', 'MTERF1', 'FNIP1', 'CDK2', 'SRCAP', 'PPP4R3A', 'PRKAR1A', 'SAMD4A', 'HCCS', 'CSE1L', 'LCP2', 'ZNRD2', 'BANF1', 'AR', 'NKAIN1', 'ELF5', 'SEMA6C', 'CYP11B2', 'AJUBA', 'SLC15A1', 'ENSG00000278932', 'POLG2', 'SMARCA2', 'ASH1L', 'SPARC', 'MBD3', 'PTPN14', 'LAPTM4B', 'LAMA2', 'DACH1', 'PIK3R1', 'SERPINF1', 'MAOA', 'ZNF213', 'ANXA13', 'CDK11B', 'RDH10', 'ATF2', 'FH', 'KASH5', 'AXIN1', 'TM4SF1', 'IQGAP2', 'FAM83A', 'HMOX1', 'LYPD3', 'PRSS8', 'TK2', 'AQP1', 'L1CAM', 'RPS6KB2', 'MCM3', 'BHLHA15', 'APPBP2', 'SLC25A10', 'TTK', 'EFCAB7', 'COL11A2', 'CYP26B1', 'RNF167', 'TEPSIN', 'COL24A1', 'VIRMA', 'GRHL2', 'FOXC1', 'SGTB', 'RESF1', 'ATP9A', 'PKP4', 'IL24', 'HLA-DQA1', 'TRAPPC10', 'ARID3A', 'HSDL2', 'LKAAEAR1', 'MYL12A', 'LHX2', 'TONSL', 'CD300LD-AS1', 'PAM', 'PPP6R2', 'AMY2B', 'IL18BP', 'CTNNAL1', 'CMTR1', 'BEX5', 'TIMM21', 'GEMIN6', 'MLLT6', 'FCER1A', 'ADAM19', 'CCDC88A', 'FBF1', 'NEK8', 'MORF4L2', 'NGFR', 'CNKSR1', 'TRIP13', 'SPEG', 'CEP192', 'CYP26A1', 'MYO5C', 'MAPK12', 'PEX12', 'TFPT', 'SHCBP1', 'POLR1HASP', 'CSF2RB', 'ARMC10', 'ZNF461', 'MCF2', 'RHEB', 'MAP3K5', 'TMEM86B', 'SLC2A1', 'UBE2S', 'PPP1R1A', 'ZNF846', 'TFPI2', 'CHAF1A', 'LMNB1', 'SMAD2', 'GNAS', 'RNF166', 'RPUSD1', 'CD164', 'TSPYL5', 'NOP16', 'HAS2', 'EN1', 'KIF18B', 'SAP30', 'SYNM', 'PRRX2', 'VNN1', 'RCBTB2', 'EWSR1', 'TMEM134', 'TOX2', 'KCTD12', 'SLURP1', 'SYT7', 'TKT', 'SRCIN1', 'COL5A2', 'PYGO2', 'COL8A2', 'AFF1', 'ABCC6', 'HRH3', 'PI15', 'DLL1', 'PRRT3', 'CENPU', 'DPH7', 'MTNAP1', 'LSM1', 'ANXA1', 'TSPAN7', 'DHX9', 'TFCP2L1', 'IL22', 'FAM217B', 'MAP6', 'FAH', 'SERPINB2', 'ABHD12', 'HLA-DMB', 'CELSR2', 'NFIX', 'GADD45A', 'BRD7', 'FLNA', 'TIE1', 'PARP4', 'CHML', 'PTPN2', 'NPBWR2', 'STK10', 'RANBP9', 'COMT', 'NAA16', 'VBP1', 'PLAT', 'SPDL1', 'NAP1L5', 'MID1', 'DHTKD1', 'CCDC47', 'PDP2', 'CERS2', 'AHR', 'RBL1', 'CYB561', 'IGHV4-61', 'CREBBP', 'CCL19', 'FGB', 'SETD6', 'IVNS1ABP', 'BCL11B', 'DBP', 'CTDSPL2', 'TRBC1', 'IFT74', 'ZFPM1', 'OFD1', 'PRR16', 'LRRN3', 'ADORA2B', 'ZFHX3', 'CST1', 'ATRNL1', 'TPPP3', 'CXCL13', 'PAX5', 'SHC2', 'ENSG00000301761', 'PPIP5K2', 'KRTCAP2', 'CARD19', 'PIK3R3', 'GNG13', 'STRADA', 'LRP5', 'MGAT3', 'CLGN', 'GABRQ', 'PTPRN2', 'FPR3', 'CPNE2', 'ECE1', 'ADIPOR2', 'URB2', 'MTUS1', 'SUCO', 'PIP4P2', 'MYT1', 'CDC6', 'ZNF704', 'PPP1R9A', 'GDF6', 'DUSP2', 'TUFM', 'GNB4', 'CHL1', 'OSGIN2', 'NDUFAF4', 'TAPBP', 'NR1H3', 'BOLA1', 'FAM200A', 'ATP8B1', 'RNU6-1016P', 'ZYX', 'NFS1', 'HS3ST1', 'PTGIS', 'PRKAG1', 'MRTO4', 'NHSL3', 'VAMP3', 'HERPUD1', 'DNAJC9', 'PLIN2', 'HOXA9', 'COL26A1', 'MARCHF8', 'TM2D1', 'ELOVL2', 'ECHDC1', 'BUD31', 'CLEC3B', 'HIPK1', 'P4HTM', 'ALOX15B', 'TTC12', 'UBBP1', 'AGAP4', 'MICALL1', 'GPC3', 'ACY3', 'SSB', 'TUBA1C', 'FKBP8', 'COL18A1-AS1', 'CLCN4', 'DRC7', 'HOXB2', 'MACIR', 'BCAP31', 'CLDN6', 'IRX2-DT', 'MBTD1', 'KIF14', 'MATK']
Refined Community 51: ['ITPR3', 'UTS2', 'RPS13', 'TATDN1', 'BTBD9', 'SSH3', 'ZMIZ2', 'P4HTM', 'DNAJA1', 'RANBP1', 'MOCOS', 'MAGOHB', 'KCNJ3']
Refined Community 52: ['KCNG1', 'THSD7A', 'CHI3L2', 'PLA2G2A', 'RHPN1', 'MRPL27', 'SDC2', 'CAPN15', 'LIME1', 'GALNS', 'SFPQ', 'PPDPF', 'SYNJ2BP', 'SNX3']
Refined Community 53: ['SPARCL1', 'KDM6A', 'FFAR2', 'ID4', 'CD14', 'HIPK2', 'TMEM204', 'C21orf58', 'TNFAIP8', 'GPR87', 'WDR11', 'MKI67', 'SETD1A', 'IRAG1-AS1', 'ZNF165', 'RIPOR3', 'LUC7L3', 'MBTD1', 'ESM1', 'SAP30BP', 'LINC03072', 'MRPS33', 'SEPTIN4', 'CDC6', 'WWC1', 'KCNMB4', 'AKR1B1', 'CHMP1A', 'MTUS1', 'CRK', 'DEGS1', 'TSPAN31', 'UCHL3', 'HSPD1', 'ECHDC1', 'BLTP2', 'FBN1', 'BEST3', 'NDUFAF4', 'MXRA7', 'ZBTB38', 'NPY1R', 'DACT3', 'TBC1D10C', 'GDI1', 'SLC17A9', 'HMGCS2', 'KIT', 'ADD3', 'HDGFL3', 'ILF2', 'RCC1', 'CCL23', 'TSC1', 'CD19', 'TCN1', 'MVD', 'CEP57', 'ITGA7', 'SMURF2', 'AKAP12', 'TLE3', 'ZEB1', 'FOXN1', 'TNIP1', 'SNORA71B', 'TNS3', 'FADS6', 'CDK1', 'ARF3', 'EXPH5', 'CHML', 'CATSPERB', 'KANSL1L', 'CACNG1', 'APOE', 'CHAF1A', 'COPZ2', 'LIX1L', 'BHLHA15', 'CDC42EP1', 'AIF1', 'RECQL4', 'SPIB', 'ARAP3', 'PIGR', 'PKP2', 'HEBP1', 'NSD2', 'GPNMB', 'CD24P4', 'LPAR2', 'KLRB1', 'RASIP1', 'BTG3', 'LAG3', 'BAD', 'BAHCC1', 'DAB2', 'TBX4', 'MAP3K12', 'ENGASE', 'CHCHD4', 'MTBP', 'ULK4', 'P3H3', 'ARHGDIG', 'VIM', 'ACSM5', 'DUOXA2', 'AP1S2', 'BMI1', 'RBP1', 'CSH1', 'NFIX', 'RTL10', 'OR4D2', 'GALK1', 'LY9', 'CEP55', 'AHNAK', 'TRPA1', 'ZNF367', 'TMEM125', 'P4HA1', 'DDHD2', 'EDIL3', 'EXOC7', 'LY6H', 'LARGE1', 'HOXA11', 'CSF3R']
Refined Community 54: ['SPOP', 'HMGB3', 'MSANTD3', 'CCNB2', 'STK3', 'FXYD3', 'GMEB2', 'PSMD3', 'GABBR1', 'OAS1', 'TESMIN', 'C1D', 'MMP7', 'FOXO4', 'CASP1', 'PRKD3', 'STYX', 'C1S', 'NFATC2', 'DUSP11', 'NOB1', 'NUAK1', 'ENG', 'CASP10', 'CD37', 'RBMS1', 'AASS', 'GPC1-AS1', 'CELF2', 'RPUSD1', 'RBM24', 'ITGB1', 'ELF3', 'SLC16A14', 'ZBTB44', 'ERICH5', 'SMPDL3B', 'JUN', 'AURKB', 'ITGAE', 'ZMAT4', 'HEXIM1', 'SLC27A6', 'ACSM1', 'SCX', 'SEC62', 'NUTM2E', 'MT1M', 'ZNF638', 'LAT2', 'ETFRF1', 'OS9', 'ALDOC', 'MPP1', 'TRGC1', 'IL4', 'C1orf74', 'CARNS1', 'HOXB4', 'SMNDC1', 'CYP2B6', 'RBL2', 'MAP3K12', 'RGPD5', 'TRDC', 'ACSL5', 'EGLN1', 'RHOBTB3', 'PDZD2', 'EXOC3', 'CLDN8', 'OR2AE1', 'GABRP', 'YPEL2', 'FGFR4', 'B3GNTL1', 'SVIL', 'AATK', 'GLIS3', 'VPS45', 'FGFR1', 'GSDMD', 'GADD45B', 'NPM3', 'ADGRL4', 'FJX1', 'TEAD3', 'BAG6', 'OGFOD3', 'OCA2', 'CYB5R3', 'ST6GAL1', 'MAGT1', 'TMEM125', 'DLL4', 'SLC2A10', 'F2R', 'MANSC1', 'ATAD3A', 'KIF2B', 'PLCL1', 'ODC1', 'EMID1', 'SHMT1', 'CSMD3', 'EMC8', 'ZBTB46', 'LBP', 'PIM1', 'DSG2', 'TCIM', 'PRKY', 'VIPR1', 'HRH1', 'SYNM', 'BET1', 'ZFTRAF1', 'MSR1', 'ACTR2', 'DUSP8', 'ATP13A3', 'ADD3', 'TNFRSF1B', 'CRX', 'FAN1', 'RNF166', 'NECTIN3', 'MICAL2', 'ERCC6L2', 'IL17RA', 'CADM1', 'ORM2', 'JAG2', 'PIP4P2', 'ADGRG3', 'B4GALNT2', 'PLPP3', 'GSK3B', 'DDX27', 'C1QA', 'CCDC88C', 'MDK', 'ADRB3', 'HS3ST6', 'CUL2', 'HLA-DRB4', 'FBXO43', 'MRGBP', 'GZMK']
Refined Community 55: ['THY1', 'PEX1', 'RGS22', 'IGLV2-14', 'MAPRE3', 'BGN', 'TRHR', 'WNT7B', 'MAFF', 'FGF10', 'SYNC', 'HNF1A', 'KIFC2', 'CARD6', 'MYO5C', 'CEP350', 'TM7SF2', 'ATAD2', 'RPUSD1', 'MTOR', 'NEXN', 'LETM1', 'STIL', 'HHAT', 'APPL2', 'CD300E', 'TNFRSF1B', 'RCAN1', 'R3HDM4', 'SH3KBP1', 'CDR2L', 'ISG15', 'BNC1', 'EPHA4', 'SETD6', 'PRKG1', 'NT5C', 'DYNC1I1', 'GPRIN2', 'DOCK10', 'CENPX', 'SUCO', 'PFKP', 'KATNB1', 'NIPAL2', 'NFKB1', 'ABHD16B', 'CHST12', 'VBP1', 'CUL7', 'SEC22C', 'MXRA7', 'KRTAP5-10', 'FGD3', 'ECHDC1', 'TNFRSF10B', 'FGFR1OP2', 'PYGL', 'NCALD', 'MCM3AP', 'BRIP1', 'MRTFB', 'MERTK', 'ZNF587', 'PHACTR2', 'ITPKB', 'METTL1', 'ZFPM1', 'JAM3', 'FGG', 'DNAJC6', 'CFAP418', 'ABCG1', 'GNAS', 'TPD52', 'UBE2Q2', 'PCMTD1', 'HLA-DOB', 'NREP', 'SLC6A15', 'SUSD5', 'BAMBI', 'FIRRM', 'PPP1R1A', 'GZMH', 'LAMA5', 'IGKV1OR1-1', 'MIR29B2', 'CYP24A1', 'C1QB', 'NORAD', 'MYO10', 'MICAL2', 'SIGLEC1', 'AGTR1', 'XRCC3', 'SOCS2', 'GTPBP2', 'MTUS1', 'CYP51A1', 'YBX1', 'DIPK1B', 'GFRA4', 'EML2', 'SLC2A10', 'ADAMTS20', 'INHBB', 'FOXA1', 'MAPK7', 'MAP4', 'INTS2', 'TPT1', 'PIGS', 'ZFHX4', 'UBAP2L', 'MPHOSPH10P1', 'SERPINB5', 'DUSP6', 'RHOT2', 'NDUFAF6', 'ZNF527', 'FOXP2', 'FAIM2', 'CLMN', 'PANX2', 'RCL1', 'CHTF18', 'PIP4P2', 'MTCL2', 'KAT7', 'ARHGAP23', 'GALNT9', 'KCTD21', 'P4HA3', 'LIMD2', 'SDF2', 'KCNS1', 'DKK3', 'STK32B', 'NFYC', 'RPL18', 'CYB561', 'NOTCH3', 'MT-ND5', 'UNC13D', 'TCF7', 'CERS3', 'PNPLA2', 'IGKV1D-39', 'MS4A2', 'B4GALT1', 'MRFAP1L1', 'EEF1AKMT4', 'KRTAP5-7', 'KRT83', 'GRIN2C', 'ATF5', 'MACO1', 'DAG1', 'CEPT1', 'GPD1', 'PDZK1IP1', 'MRPS7', 'GZMB', 'DEAF1', 'GEMIN6', 'TAOK1', 'GRID1', 'ARC', 'ST3GAL6', 'CRYBA1', 'SNRPA', 'CD7', 'LAT', 'EEIG1', 'PBLD', 'TSPAN2', 'SNX21', 'GLCCI1', 'SFT2D2', 'AP1AR', 'NOC4L', 'RNF144B', 'RBM5']
Refined Community 56: ['USP38', 'COL4A2', 'OBP2A', 'RAB2A', 'SREBF1', 'RAB37', 'ESM1', 'CES5A', 'HCFC2', 'SH3D19', 'CPA3', 'SMPDL3A', 'BTF3', 'NME4', 'NOXA1', 'TAF15', 'YEATS2', 'SLC24A1', 'UBN1', 'TIMELESS', 'RBBP7', 'KIAA0319L', 'ZFYVE16', 'SIX1', 'GPC1-AS1', 'EIF4G1', 'PGA3', 'CHMP1B2P', 'FNDC3B', 'RDH11', 'RPIA', 'TMEM134', 'RFC4', 'VAV3', 'ANTKMT', 'RAF1', 'BUB1B', 'SCG5', 'ARHGAP29', 'IKBKE', 'SPCS2', 'ITGA2', 'GFOD3P', 'TEX14', 'ZNF790', 'ZHX1', 'CDK1', 'OVOL2', 'TELO2', 'MAGEA12', 'OGN', 'NUP93', 'CTBP1', 'PDE5A', 'LIMA1', 'CEACAM5', 'SERINC2', 'S100A4', 'DUSP4', 'MAPKBP1', 'CARMN', 'MEF2D', 'RAP1BL', 'CNOT3', 'TCEAL9', 'RAI14', 'PABPC4L', 'ITPKC', 'MAPK9', 'PIEZO2', 'ZNF18', 'DKK3', 'MRPL15', 'EIF3H', 'HMCN1', 'ITPA', 'NFIA', 'NFIL3', 'NVL', 'RCBTB2', 'HLA-DQB2', 'IGKC', 'HAGH', 'PSAT1', 'B3GNT6', 'DEPDC5', 'TMEM241', 'POGLUT1', 'AK4', 'MCM4', 'HNRNPU', 'AGRN', 'CDO1', 'LMNB1', 'ADAM9', 'NPY2R', 'AP2B1', 'C1QTNF3', 'TLN2', 'PPP2R5A', 'SOX11', 'CRISP2', 'UCKL1', 'CUX1', 'UCHL3', 'LPIN1', 'LNX1', 'AR', 'IFNGR2', 'RNF7', 'EFEMP1']
Refined Community 57: ['C7', 'NSFL1C', 'GGT5', 'NCAPG', 'CHMP1B', 'JUN', 'MED30', 'L3MBTL2', 'ANKRD6', 'DACH1', 'CENPE', 'EPN3', 'ANK3', 'PLAAT4', 'PON1', 'SMPDL3A', 'RIOK3', 'C3orf52', 'ANKRD40', 'TMEM147-AS1', 'PROCR', 'SMC4', 'BBIP1', 'SIRT3', 'PGA3', 'RGS7', 'LRP6', 'VDAC3', 'SLC18B1', 'MZB1', 'C17orf58', 'MSANTD3', 'PAM', 'WARS1', 'PKP4', 'CDK2', 'SAV1', 'EME1', 'IL32', 'KLHL29', 'RAP1B', 'SPINK4', 'BMAL2', 'CCDC81', 'EXTL1', 'PXMP2', 'KCTD21', 'PLSCR1', 'ARHGAP4', 'IGFBP4', 'CALHM6', 'MADD', 'FRAT1', 'SCPEP1', 'PRG4', 'LAGE3', 'POT1', 'ERO1B', 'RIPOR2', 'TLN1', 'PDE2A', 'COPS7B', 'RIN3', 'ACTN3', 'AK5', 'NNMT', 'LBH', 'NRG1', 'ALDH3A1', 'APP', 'RBPJL', 'CIITA', 'DMKN', 'CCDC28A', 'ARC', 'ITIH4', 'TADA2B', 'IL17REL', 'RBM10', 'SPEF1', 'ITM2C', 'ACVR1B', 'SREBF1', 'DMWD', 'SPPL2A', 'DCST2', 'SLC47A2', 'ADAMTS1', 'MRPS7', 'SHC1', 'POLD4', 'XRCC5', 'DENND1B', 'OTOP2', 'CD22', 'PARVA', 'GSPT2', 'CLDN8', 'TATDN1', 'PLSCR3', 'APEX1', 'C16orf74', 'FXYD3', 'NEFH', 'FOXK2', 'CCDC93', 'TOP2A', 'CYP7A1', 'FAU', 'PDPN', 'TRAF7', 'SCGB2A2', 'ZNF34', 'KCTD3', 'CLU', 'FNIP1', 'RASSF3', 'TSR3', 'RGS2', 'HPS5', 'RPL7P9', 'TPSD1', 'DPH7', 'GTF2H2C', 'PAN2', 'PTP4A3', 'DUSP2', 'CHIC2', 'CDC25C', 'TET3', 'ANXA1', 'ZNF84', 'ENSG00000278932', 'PKP2', 'CENPU', 'EMC8', 'HPGD', 'PYGL', 'SFRP1', 'CNOT2', 'CLDND1', 'RBM15B', 'ARHGEF40', 'ZFHX3', 'RNASEH2A', 'VSTM1', 'CD4', 'APOL6', 'PPARD', 'MFGE8', 'CLMN', 'ARSJ', 'ACOT11', 'UGT2B15', 'PPP1R9A', 'CPNE2']
Refined Community 58: ['EPN1', 'IGKV1OR1-1', 'ZNF395', 'PARD6B', 'GNAI1', 'MUCL1', 'HNRNPA0', 'SCUBE1', 'VSTM1', 'DTL', 'CWH43', 'MAB21L4', 'PNOC']
Refined Community 59: ['GPX8', 'ENDOD1', 'GJC3', 'ALKBH5', 'DSG2', 'MRPL27', 'MZF1', 'LINC00511', 'ZNF260', 'GOT2', 'WIPF2', 'MBNL1', 'OCM2', 'HLA-DQA1', 'TBC1D7', 'PROM2', 'MRM2', 'SNRNP25', 'LACTB', 'VWA5A', 'MAGEA9', 'GNMT', 'NEURL1', 'DDIT4', 'USP42', 'TRMT6', 'TSC2', 'DOP1B', 'SRSF7', 'FCN2', 'MRPL11', 'TBC1D12', 'ACER3', 'LHFPL6', 'MARVELD3', 'BTG2', 'CHRD', 'RRP1B', 'GPNMB', 'ETV7', 'MAP9', 'DUSP1', 'RPP30', 'FKBP2', 'GDF6', 'CSNK2A1']
Refined Community 60: ['RTF2', 'POLA1', 'MAGOH', 'SLC24A1', 'CASP9', 'GTSE1', 'PLCH1', 'MPHOSPH9', 'MARVELD2', 'CPNE2', 'KANSL1L', 'ARSJ', 'ENY2', 'PRPF39', 'ENSG00000280119', 'CPM', 'PIDD1', 'RHOH', 'BCLAF3', 'IRS1', 'XRCC5', 'KDM6A', 'RARRES2', 'METRN', 'PARP8', 'NXN', 'COPZ2', 'GUSBP3', 'ZNF22', 'LMBRD2', 'ZNF572', 'CLSPN', 'DNAAF1', 'H19', 'CD180', 'KHSRP', 'RPS24', 'GLDC', 'MATK', 'THBS2', 'GAMT', 'MLF1', 'PER3', 'FER', 'IL17RA', 'SETMAR', 'NYNRIN', 'CD248', 'CCKAR', 'ACIN1', 'DDX27', 'DCAF8', 'CASP3', 'CTSK', 'ZC3H12C', 'CSNK1A1', 'UQCC4', 'IRX2-DT', 'SLC2A4RG', 'PDE7B', 'OR4F15', 'C6orf62', 'NFYB', 'GTF2I', 'LMTK3', 'STK26', 'SOBP', 'SNX31', 'ZNF839', 'TRIM9', 'AVIL', 'TTLL4', 'TUBB6', 'PIM3', 'THOC2', 'MACO1', 'CD86', 'RAD51AP1', 'PPP1R18', 'AQP1', 'CACNA2D3', 'TM7SF2', 'RAB6B', 'HOXA13', 'MSL1', 'CEP112', 'TNFRSF10D', 'CXCL10', 'SNX5', 'CCL3', 'RPL38', 'PLXNB1', 'KPNA2', 'TMEM61', 'TRDC', 'NCOA2', 'ZKSCAN5', 'CNOT3', 'HSF1', 'THEM6', 'IGFBP5', 'FGF7', 'PDCD4', 'NUAK1', 'OGN', 'ACSF2', 'ZACN', 'SLC8A2', 'SLC35A3', 'DNAL1', 'GPR19', 'GAS1', 'TGFBR2', 'DCAF13', 'EIF3B', 'ADISSP', 'WNT5A', 'GOT1', 'PPIP5K2', 'FAU', 'JAG2', 'P3H2', 'SNHG12', 'MKNK2', 'GOLGA3', 'TSHZ3']
Refined Community 61: ['LGALS13', 'LIN52', 'APOC1', 'CLDN12', 'RIF1', 'CACNG8', 'RIPOR2', 'LRRC14', 'GPR137C', 'TNFRSF19', 'GATA6', 'CDK2AP1', 'AMIGO2', 'XPOT', 'ACSL5', 'LBH', 'CCT5', 'ADGRG6', 'MYBL1', 'PRPF39', 'CNDP2', 'STRAP', 'IGFBP2', 'GYG1', 'GLRB', 'EFHD1', 'ZNF16', 'RAD51AP1', 'SOBP', 'LIPE', 'CYTH1', 'GDPD4']
Refined Community 62: ['KRTAP4-5', 'TRMT12', 'ADAM9', 'FPR3', 'XRCC6', 'MS4A7', 'IL1RAP', 'ZNF142', 'GPM6B', 'PALS2', 'FAAH', 'GP1BB', 'INCENP', 'DLK2', 'TRHR', 'DOK7', 'ITIH3', 'KRT25', 'COIL', 'PSME4', 'SLC13A2', 'FAM200B', 'CCDC89', 'UBAP2L', 'PLCL2', 'IFNAR2']
Refined Community 63: ['TROAP', 'TNS2', 'ACOX1', 'ELF3', 'WNT5B', 'NCALD']
Refined Community 64: ['CHEK2', 'TUBB2B', 'DUSP4', 'IFRD1', 'EFCAB3', 'DOK7', 'HOXA3', 'MAPRE2', 'PDGFA', 'GMIP', 'NEK9', 'FLT4', 'TRAF4', 'MIEN1', 'FTSJ3', 'ZNF781', 'ESRP1', 'GPR152', 'REEP1', 'GALE', 'ORMDL1', 'CALCRL', 'CD3D', 'UGDH', 'RALY', 'MOV10L1', 'PRKCQ', 'AMDHD2']
Refined Community 65: ['DLC1', 'ADGRG2', 'PIP5K1A', 'IL24', 'ABCC5', 'BICDL1', 'MRGPRF', 'ATP5F1E', 'MEOX2', 'SAMD14', 'KLHL24', 'TRMT1L', 'CSH2', 'JAM3', 'ENTR1', 'RNASE4', 'TBC1D8', 'ZNF572', 'CCDC69', 'SNX3', 'AXIN1', 'RASSF2', 'GATA6', 'HOXA2', 'RBM33', 'OSR2', 'BAALC', 'EOLA1', 'GBP1', 'GNB1', 'CCNA1', 'PUM3', 'COL6A3', 'DDAH2', 'ETFA']
Refined Community 66: ['TMEM30B', 'SEC23B', 'TP53I13', 'JTB', 'RTF2', 'NFU1', 'SORL1', 'GRIN2C', 'ADAMTS12', 'SMCO4', 'DCAF8', 'HCLS1', 'TNPO2', 'PTTG1', 'C1orf116', 'ECI2']
Refined Community 67: ['MT1H', 'XPO1', 'ECE1', 'PRKCD', 'TBCD', 'PTK7', 'SYNE3']
Refined Community 68: ['MRPL41', 'SNRPC', 'CD82', 'BIRC3', 'LYVE1', 'ALDH3B2', 'RNF144B', 'PTGER3', 'ALDH18A1', 'PPP1R16A', 'TMSB15B', 'EEF1AKMT3', 'GBP1', 'HTT', 'RFC3', 'WNT9B', 'RPSAP47', 'KLHDC4', 'IL20RB', 'HTRA1']
Refined Community 69: ['LGALS4', 'LITAF', 'SMAD9', 'SCX', 'DUSP6', 'PLCB1']
Refined Community 70: ['FNDC11', 'H2AZ1', 'LEMD2', 'PRP4K', 'TSN', 'CDH11', 'CADPS2', 'PDP1', 'TG', 'SFRP1', 'EEIG1', 'NOP16', 'CCNE1', 'BEX1', 'EFEMP1', 'KCNMB1', 'GSTZ1', 'FLAD1', 'PPP1R14B', 'ZHX2', 'TYMS', 'PMP22', 'APCDD1L', 'PLAU', 'PDZD2', 'TERT', 'IL1R2', 'PSMD1', 'IRF8', 'MIR29B1', 'POT1', 'WSB2', 'PFAS', 'AKT1S1', 'MSANTD3', 'POTEA', 'CDYL2', 'RAP2B', 'SMURF1', 'AKT1', 'GATAD1', 'EHMT2', 'DDC', 'FCN2', 'HOXB13', 'RASSF7', 'MORF4L2', 'HNMT', 'YIF1A']
Refined Community 71: ['LRRC8D', 'CDH15', 'KATNAL1', 'ZNF878', 'CCDC6']
Refined Community 72: ['SLC38A10', 'TPPP3', 'PIGQ', 'CCDC89', 'ARRDC4', 'C17orf75', 'RBM10', 'IGFBP5', 'DENND3', 'HDGFL3', 'GNA12', 'ZNF570', 'MAN1A1', 'TGFA', 'PRAC1', 'MC3R', 'GSTO2', 'SCAMP3', 'GABRP', 'ZKSCAN1', 'FNDC5', 'TSFM', 'KRTAP4-2', 'DUOX1', 'POLB', 'MAX']
Refined Community 73: ['HNRNPA0', 'ANKRD26', 'CEBPD', 'PLCL1', 'ERG', 'PANX2', 'F2RL2', 'OGN']
Refined Community 74: ['PLXNA2', 'MAP4', 'SLC35A1', 'DRG2', 'CD4', 'HELZ', 'ALDH1A3', 'LTBP4', 'FMOD', 'PIK3C2A', 'ENSG00000280119', 'PPM1B', 'FGF18', 'TIMM21', 'LEMD2', 'TFF3', 'SEPTIN11', 'FOSL2', 'PPP6R2', 'RESF1', 'BHLHE40', 'CTHRC1', 'CD48', 'CTSF', 'AFDN', 'PEDS1-UBE2V1', 'AMPD2', 'TBC1D10C', 'S100A1', 'HMGCR', 'SAA1', 'GLP1R', 'TRIP11', 'CCKBR', 'ANKRD22', 'DAGLB', 'POLK', 'NUTM2E', 'DBI', 'ROBO3', 'PIK3CD', 'NKX3-1', 'AMZ1', 'BAG1', 'ZBTB42', 'DENND5A', 'COL19A1', 'OSBPL9', 'EMCN', 'DUSP6', 'IGKV3-20', 'CD300LB', 'DYNC2H1', 'CKMT1B', 'NT5DC2', 'FLRT2', 'PPHLN1', 'MGP', 'CGA', 'RASSF7', 'RANBP9', 'CHTOP', 'IL20RB', 'GABBR1', 'BUD31', 'TICAM1', 'SCN8A', 'LRRC32', 'MSTO1', 'TXNL4A', 'VCAM1', 'GAS5', 'TAX1BP3', 'NDUFB9', 'PARP8', 'BCL2', 'NGFR', 'AQP11', 'TLN2', 'AQP9', 'CTSG', 'IFIT3', 'NFIB', 'EFNA1', 'TGFA', 'FBXW2', 'COX7B', 'GLCCI1', 'PHB1', 'LPXN', 'CNTNAP2', 'TBL1X', 'BRD8', 'PDP1', 'ASMTL-AS1', 'TMEM67', 'SMAD4', 'GET4', 'RTL10', 'RGPD5', 'MARVELD2', 'FBXL18', 'ZNF330', 'HK2', 'MTERF2', 'CCR6', 'SGCB', 'HGH1', 'ATP13A3', 'RNMT', 'POU5F1', 'FAM200B', 'FGFR1', 'DZANK1', 'ATF5', 'GZMK', 'PGA4', 'DEPDC1', 'RIMKLB', 'KLHL8', 'MYO5C', 'TTC39A', 'ZAP70', 'DNAJA4', 'SLC6A2', 'CCNC', 'NUDT15', 'ACADL', 'DCLRE1C', 'SGCA', 'MED1', 'CYP39A1', 'TBC1D7', 'NARS2', 'ABCC8', 'RNF145', 'SERBP1', 'SNAPIN', 'KRTAP5-10', 'PDCD6', 'FBLIM1', 'FBXL5', 'GABPB1', 'MAP3K7CL', 'EXOSC4', 'AHCY', 'PDGFRA', 'HLF', 'CLEC7A', 'GADD45A', 'COPZ2', 'P3H3', 'FILIP1L', 'SLC25A25-AS1', 'PNMA2', 'NES', 'CEBPA', 'TMEM92', 'GAPDHS', 'LAMP3', 'RASA4', 'TMEM123', 'RAB37', 'ABAT', 'CARTPT', 'LINC03042', 'DUS1L', 'FABP7', 'MIR31HG', 'IGHV4-61', 'PON3', 'DCLK1', 'TRAF5', 'KCNE3', 'ITSN2', 'DCAKD', 'PPP1R9A', 'KRT6A', 'GPR65', 'C8orf82', 'CRX', 'IP6K2', 'PPFIBP2', 'CAP2', 'SUPT7L', 'TRAPPC9', 'EIF4E3', 'SMIM15', 'OR4F6', 'TRMT13', 'PRDM13', 'F10', 'ADAM17', 'SESN3', 'CAVIN1', 'HOXB8', 'ULK1', 'ODR4', 'LCP1', 'SGSM3', 'DIP2C', 'LRRD1', 'UQCRH', 'LCP2', 'ASTN2', 'GPR18', 'CCDC69', 'SORL1', 'RNF112', 'PRDX4', 'MOAP1', 'NUTF2', 'CTNNBL1', 'RPS6KB1', 'ZNF165', 'SHCBP1', 'USHBP1', 'SHROOM3', 'MC3R', 'HHAT', 'IL6', 'FMNL1', 'ZNF532', 'PIN1', 'LAMB1', 'SLC27A3', 'IK', 'NUCB2', 'MMP9', 'TRIM9', 'ERN1', 'PLAAT2', 'IDH2', 'CD24', 'RAPGEF5', 'BICC1', 'DCAF11', 'DYNLRB1', 'OPLAH', 'TM2D2', 'RASGEF1A', 'ETF1', 'MGRN1', 'TROAP', 'H2AJ', 'GLDC', 'ALOX15B', 'IMPA1', 'ATF2', 'ALDOC', 'LASP1', 'ADORA2B', 'TMBIM4', 'LBH', 'XYLT1', 'PIGS', 'YEATS2', 'WDR45B', 'TOX3', 'ANKH', 'RASSF3', 'GALNT1', 'ABCF3', 'EPHA3', 'SFTPC', 'NOTCH2', 'MAPK12', 'RNF7', 'DIAPH3', 'STEAP1B', 'PCDH7', 'IGFBP6', 'IGF2BP2', 'ZFTRAF1', 'KLRB1', 'ECT2', 'ACSS3', 'NQO1', 'SPIDR', 'SEC23IP', 'ANXA2', 'GNPNAT1', 'RECQL4', 'TUFM', 'ACSL1', 'TRPS1', 'SHMT1', 'TOMM34', 'DDHD2', 'NAT2', 'LGALS4', 'TANK', 'SGMS1', 'BCKDHB', 'KANK3', 'THSD4', 'FKBP4', 'TMEM80', 'ANTXR2', 'AKAP9', 'RET', 'TIMP2', 'TAFAZZIN', 'UQCC2', 'HYAL1', 'ATXN1', 'TULP3', 'ARHGAP8', 'ATP13A4', 'SETMAR', 'EIF4H', 'MPZL2', 'PIAS1', 'PGM2', 'HOXB2', 'DNAJC21', 'UNG', 'CCDC83', 'CLEC2D', 'ITPR1', 'TMCC2', 'XCL1', 'TRBV5-4', 'SLC23A2', 'IGSF1', 'COQ7', 'MIR100HG', 'TIMP1', 'TBX2']
Refined Community 75: ['MPZL1', 'CNGB1', 'BTBD9', 'LGALS4', 'GNB2', 'TCP11L1', 'RPL30', 'COG1', 'POLQ', 'OR12D2', 'CENPI', 'CD1C', 'LITAF', 'LKAAEAR1', 'CCND2', 'ACAD8', 'EIF3B', 'RSRC1', 'ERCC2', 'LMO2', 'RNASE9', 'LY6K', 'RAD21', 'SNRNP25', 'REEP1', 'ENPEP', 'CD300C', 'DNAL1', 'POLR1A', 'GPAM', 'CASP9', 'BTD', 'HOXA5', 'RTN1', 'BCKDHB', 'KRTAP4-1', 'LRRC8D', 'RGS2']
Refined Community 76: ['UBA7', 'SORBS1', 'GLMN', 'TRDC', 'EPN2', 'MXRA7', 'SEC31B', 'SPAG1', 'SSH3', 'ANO1', 'TKFC', 'IDH2', 'GREB1', 'LCP1', 'OXTR', 'P4HB', 'BUD31', 'ICAM2', 'HLA-DMB', 'NOTCH2', 'RCAN2', 'DZIP1']
Refined Community 77: ['BLOC1S1', 'ACSF3', 'PLPBP', 'REG1A', 'WDR3', 'RAD54L', 'JAK3', 'CWH43', 'CTXN1', 'PTPN14', 'BAD', 'IFI30', 'HELLS', 'ZNF862', 'FAXC', 'ITGBL1', 'BRD1', 'ETNK2', 'DIPK1A', 'LFNG', 'HEXIM1', 'SPTAN1', 'SEC23A', 'CYLD', 'GAL', 'NAT2', 'RSKR', 'CHRNA9', 'ZIC1', 'GID8', 'TRAF4', 'SNRPB2', 'E2F5', 'XPO1', 'TSHZ2', 'OLFML3', 'MYBL2', 'KLHDC4']
Refined Community 78: ['PPP1R26', 'TCN2', 'UPF3B', 'DGKZ', 'TAPBP', 'GALNT14', 'ZNF317', 'GUSBP3', 'KDELR2', 'NSUN5', 'ADAM9', 'JPT2', 'ITIH3', 'C1QC', 'EIF3B', 'NRXN1', 'IL24', 'RASGRP3', 'GATA3-AS1', 'ARL3', 'CPA3', 'KCNK5', 'CDK7', 'ADIG', 'THBS3', 'SP2', 'GINS2', 'GRIFIN', 'FILIP1L', 'ADCY9', 'CTSB', 'BAG4']
Refined Community 79: ['ANGPTL2', 'PDK4', 'ATF2', 'CEP192', 'GFUS', 'DNM1', 'NTHL1', 'MAGT1']
Refined Community 80: ['IGLL3P', 'GINS1', 'WNT10B', 'C9orf72', 'CCDC106', 'RRP1B', 'KLF16', 'PBK', 'ACTN1', 'AP1AR', 'TRIM59', 'ENSG00000293341', 'LGALS4', 'MTERF1', 'SESN3', 'BCL11B', 'WDR6', 'MAGEA12', 'NGLY1', 'FEM1B', 'NAA16', 'PHF19', 'TYROBP', 'PARP1', 'NOC3L', 'ANKRD40', 'PDGFRA', 'FARSB', 'HYAL1', 'SAMD14', 'IGKV1OR1-1', 'PNOC', 'ZBTB20', 'MKI67', 'ATP5F1E', 'CHRNA9', 'PIK3CG', 'IKBKE', 'XYLT1', 'GAN', 'SULT2B1', 'PUS1', 'LINC00491', 'IGLV3-25', 'PPP6R2', 'FSCN2', 'EIF4A1', 'TPSB2', 'GRAMD2A', 'PPAT', 'TNS3', 'CSF2RA', 'CIITA', 'FGF2', 'H2AJ', 'MARCHF8', 'CLIP2', 'TMEM117', 'SLC6A18', 'MZF1', 'PAPPA', 'ZFYVE1', 'P4HA3', 'HOXB6', 'ZBTB8A', 'TMEM204', 'COX6C', 'DDI2', 'CYP2J2', 'TMEM126A', 'GSTA1', 'LPCAT3', 'ZNF18', 'NKX2-5', 'CCDC90B', 'CREBBP', 'CSNK2A3', 'RELA', 'MAST1', 'FZD6', 'STAU1', 'MAP4K3', 'ALG12', 'CNTROB', 'PRR13', 'ATP5MC1', 'SYDE2', 'KIAA0232', 'POLR3K', 'TOX4', 'GADD45B', 'KIFC2', 'SLC38A7', 'KLRK1', 'CDH15', 'KCNK15', 'NREP', 'UGT2B11', 'POT1', 'SGK1', 'TACC3']
Refined Community 81: ['MRPS33', 'PPL', 'CDKN2A', 'NUMA1', 'CYC1', 'PUM3', 'ALDH6A1', 'DOK4', 'KRT10-AS1', 'SLC3A2', 'NR1H3', 'CHIT1', 'HOXB13', 'PRKCA', 'FAM217B', 'DCAF4L2', 'COPS7B', 'DCLRE1C', 'YBX2', 'SMG8', 'CSNK2B', 'SCYL3', 'CTTNBP2', 'DDX17', 'FMR1', 'SLC44A4', 'SH3PXD2A', 'CABP2', 'MBP', 'ZNF124', 'PRSS22', 'ID3', 'FBXO32', 'PEX5', 'SETD1A', 'PGS1', 'BTN3A3', 'HGD', 'HCN3', 'CEPT1', 'PNOC', 'KDELR3', 'NAP1L1', 'OCA2', 'HLA-G', 'F13A1', 'TUFT1', 'ELF1', 'GPIHBP1', 'TALAM1', 'BICC1', 'ZBTB42', 'TIMM10', 'SLC29A4', 'MPG', 'GJC1', 'PTX4', 'MAPRE3', 'DMD', 'EGLN1', 'C1QTNF8', 'H2AZ2', 'MARCHF8', 'PON1', 'PRKCQ', 'GPM6B', 'HOMER3', 'FCAMR', 'LMCD1', 'NFE2L2', 'ITK', 'CHEK2', 'KCNK1', 'GABRE', 'PDGFRB', 'ZNF567', 'DENND11', 'SELENOF', 'MAP2K3', 'SCAF11', 'CHRNA6', 'OBP2B', 'CRNKL1', 'RIN3', 'ALG8', 'BAP1', 'KIF13B', 'ACOT2', 'MED24', 'MTMR2', 'SLC15A3', 'PRUNE2', 'KRTAP4-1', 'GPR146', 'HSH2D', 'DIDO1', 'ARFGAP1', 'OR12D2', 'NSUN4', 'SHMT2', 'KLF9', 'SOX9', 'ADORA2B', 'TMEM74', 'FAM199X', 'FAM20C', 'P4HA1', 'MLLT1', 'TKFC', 'TMEM259', 'POLG2', 'HTRA1', 'RYBP', 'PAXBP1', 'MYCBP2', 'CD47', 'DZANK1', 'DALRD3', 'IL17RA', 'MORF4L2', 'SREK1', 'TMED4', 'DERL1', 'FGG', 'FANCD2', 'CNTNAP2', 'WNT3', 'HOXB1', 'GHR', 'NR2F6', 'DNAJC9', 'IDI1', 'ITPRIP', 'TBC1D29P', 'THY1', 'ATAD2', 'PASK', 'NPR3', 'LZTFL1', 'GARS1', 'HSP90AA1', 'TNFRSF25', 'GALNT2', 'TMEM94', 'UCK2', 'EFCAB3', 'PPP6R3', 'MAP3K7CL', 'WDR24', 'RAMAC', 'ENTPD1', 'ZNF398', 'MAP7D2', 'CD55']
Refined Community 82: ['OSBPL2', 'HHEX', 'RAC3', 'RGCC', 'VAMP3', 'MADD', 'ISLR', 'FBXL19', 'ASH2L', 'MYL10', 'PHACTR1', 'KLHDC10', 'IRX2-DT', 'SCUBE2', 'PKD1']
Refined Community 83: ['PRSS33', 'RERG', 'NANP', 'ZNF704', 'FAM200B', 'SLC7A1', 'RNF139', 'PYGB', 'ELOVL5', 'GALNT2', 'CLK4']
Refined Community 84: ['SLC2A5', 'DECR1', 'SELENBP1', 'FAXDC2', 'PDK4', 'SNX31', 'FASN', 'BRCA1', 'ICAM1', 'PTGS2', 'CCDC57', 'ACADM', 'KREMEN2', 'PARD6B', 'LMNB1', 'TAF1A', 'GALK1', 'NRARP', 'KCTD12', 'TAPBP', 'CDC5L', 'C1orf115', 'DBI', 'ADGRF5', 'CENPV', 'FAM234A', 'PUF60', 'CCDC47', 'PGR', 'MUCL1', 'CES1', 'KCNE3', 'CORO2B', 'SGK3', 'LINC00469', 'RIPK4', 'ABAT', 'RAB35', 'JPT2', 'CASK', 'ATOSA', 'DSCC1', 'ASXL1', 'NOL8', 'CAPRIN1', 'ZNF790', 'C17orf100', 'GNLY', 'PIGF', 'FCGR3B', 'TTC23', 'VPS13B', 'IGHV3-21', 'ENSG00000301761', 'PLK2', 'CCN3', 'GLIPR1', 'TMEM154', 'TMEM98', 'CDKN3', 'WNT16', 'POLR1C', 'SCGB1D2', 'KRTAP9-2', 'NCALD', 'RTL10', 'PMAIP1', 'THUMPD1', 'EFCAB3', 'TACC1', 'CDC45', 'SPIDR', 'OPN3', 'GPT2', 'TERF1', 'FLOT2', 'CCR7', 'CD247', 'METAP2', 'RBM4', 'ROBO1', 'RFC3', 'AK2', 'NMNAT2', 'PBX1', 'PQBP1', 'SLCO5A1', 'TMEM265', 'TBC1D8', 'HSPG2', 'UBE2T', 'PKM', 'SERPING1', 'LINC00115', 'SDHA', 'HYAL1', 'ODC1', 'LRRC24', 'ABHD2', 'MLST8', 'CCR1', 'RTEL1', 'IFIH1', 'TOX', 'MED13L', 'CDH1', 'C22orf39', 'NARS2', 'PATJ', 'NUDT11', 'TBC1D10C', 'HARS1', 'SLC19A2', 'GADD45G', 'SLC16A1', 'BRF2', 'CLDN9', 'EPB41', 'MTBP', 'ANKFN1', 'PCDH18', 'TAP2', 'LRRFIP1', 'NOL3', 'STAG2', 'PDGFC', 'USP1', 'SPRY4', 'ANKRD22', 'RALB', 'MAF1', 'APLNR', 'CLIC3']
Refined Community 85: ['FGF9', 'NUDT6', 'TPD52L2', 'ZNF317', 'NANS', 'SYCP2', 'SRRM2', 'GALNT2', 'ELOVL5', 'ANPEP', 'PIERCE1', 'MRM2', 'CTAG1B', 'NTRK2', 'FKBP14', 'PMVK', 'FAM53B', 'UGT2B15', 'KCNK9', 'CXCL14', 'SYTL2', 'JAG1', 'LIMS2', 'GAS5', 'ZNF846', 'MAGI2', 'NEU3', 'GGT7', 'IGKV2D-28', 'NAF1', 'NREP', 'ECHDC1', 'MTX1', 'CHTF18', 'GID8', 'ZNF598', 'ARMT1', 'NDUFS4', 'FZD2', 'PAK2', 'CSRNP2', 'AIM2', 'GNB1', 'MMP23B', 'PPP1R16B', 'PWWP2A', 'CCN3', 'CHSY1', 'RBM5', 'PDE6G', 'ZNF567', 'RSKR']
Refined Community 86: ['RFNG', 'ENSG00000280119', 'CPSF6', 'HTT', 'IRGQ', 'TERF1', 'KCNJ2', 'CKS2', 'DPM1', 'NUMA1', 'ARMC10', 'ADAM18', 'PUS1', 'EPM2AIP1', 'CXCL8', 'HDAC10', 'PREX1', 'UGT2B11', 'USP21', 'C22orf39', 'TIMP1', 'CBS', 'C4A', 'NCSTN', 'OAZ1', 'CASK', 'SPTAN1', 'ZCCHC9', 'FAM217B', 'RDH13', 'IDO2', 'TBL1X', 'TSPYL5', 'CCNE1', 'SULT1A1', 'CD300A', 'TRIM4', 'SAT1', 'AGPS', 'CXCL9', 'TAF15', 'CD58']
Refined Community 87: ['WNT6', 'LAMA2', 'FAM83E', 'SPMAP1', 'COLEC10', 'AXIN2', 'HELZ2', 'CAPN8', 'ERBB2', 'CASP1', 'SMIM14', 'C1orf131', 'CRMA', 'RPL23AP42', 'CRTAP', 'TATDN2', 'SPTBN2', 'CCR5', 'JRK', 'COL5A1', 'TTC7A', 'CHCHD4', 'NRXN1', 'KDM5A', 'DCAKD', 'POGLUT3', 'DLEU2', 'PALS2', 'DCAF11', 'RIOK3', 'MAPRE2', 'ZFYVE1', 'ALCAM', 'TMEM94', 'UBLCP1', 'PPFIBP2', 'H1-1', 'PTGS2', 'ZNF569', 'GBE1', 'RAC3', 'TFF2', 'CMTM7', 'CHML', 'PEX6', 'CMPK1']
Refined Community 88: ['ENY2', 'MPP7', 'GSK3A', 'COMMD5', 'LGALS9', 'STAMBPL1', 'IGKV1D-13', 'DDR2', 'OBSL1', 'RNF43', 'ZNF467', 'SRP68', 'SCAMP3', 'CTTNBP2', 'PTCH1', 'SEC23B', 'PLK3', 'SPRED2', 'ARHGAP21', 'DNHD1']
Refined Community 89: ['NKX6-3', 'PRPF39', 'RIPK2', 'CCR1', 'ZNF829', 'ZNF211', 'BTG3', 'CEP131', 'CD300LD-AS1', 'EP300', 'ANGPT1', 'OSBPL9', 'ADAM15', 'AMY2B', 'SP140', 'CYRIA', 'CDH8', 'GNPTG', 'MDFIC', 'MOGS', 'EVPL', 'PRODH', 'GAK', 'CTAG1B', 'VNN2', 'BTF3', 'PBLD', 'CLDN4', 'SFT2D2', 'AHCY', 'USP18', 'LYPD1', 'SHCBP1', 'CAMTA1', 'KCNK5', 'TOGARAM1', 'ACTN3', 'FTX', 'BTN3A3', 'ANKRD46', 'ITK', 'SMC2', 'MTERF1', 'CHCHD4', 'PDP1', 'FAM217B', 'CSF2RA', 'PLIN2', 'TRPC4AP', 'GNGT2', 'STAT5A', 'ALOX15B', 'JAKMIP1', 'TMEM150C', 'GPRC5C', 'RPS20P22', 'RHPN1', 'SSR1', 'KANK1', 'CTCFL', 'ABCB1', 'NOL8', 'ANKLE2', 'CENPN', 'PDZK1IP1', 'BTN3A2', 'LZTS3', 'CNBD1', 'SPEG', 'PIK3C2A', 'CDC42EP1', 'FGG', 'TFPI', 'ERAP1', 'SELENOP', 'PAIP2', 'KICS2', 'VRK3', 'BACH1', 'RBM24', 'NNMT', 'HEPACAM2', 'MED13', 'GLRB', 'DYNC1I1', 'VCL', 'CPPED1', 'MLXIP', 'MMP2', 'CLN8', 'MRPL28', 'LRRC74A', 'TSPAN13', 'ACACB', 'RGS7', 'CLK3', 'TKFC', 'PI3', 'APLP1', 'KCNQ2', 'KRT19', 'ABAT', 'RPS24', 'CGA', 'MAD2L1', 'MIPEP', 'PPL', 'CCDC97', 'HDAC2', 'TMEM30A', 'KCNK1', 'MLST8', 'CFI', 'STIP1', 'SMG8', 'DLX5', 'CENPB', 'ART3', 'AKAP11', 'TRIP11', 'PTPN6', 'FYN', 'ITGA1', 'TPT1', 'CACHD1', 'ADCY6', 'GNG2', 'STAG2', 'MSC', 'SPINT1-AS1', 'TFAP2A', 'DSG1', 'RMND5A', 'PIERCE1', 'CFLAR', 'PTGIS', 'CSAG2', 'MYO18A', 'PYCR3', 'EFHC1', 'DHRS7', 'SOX7', 'CNNM4', 'CCDC77', 'TULP3', 'NDUFA4', 'HNRNPA2B1', 'ATL1', 'CBR3-AS1', 'RNF151', 'TMC4', 'PPP4R1', 'C1orf131', 'SIX3', 'TP53I13', 'TMEM123', 'SLC2A3']
Refined Community 90: ['CACNA2D3', 'DPP3', 'SLC25A36', 'SNCAIP', 'AHI1', 'ZFYVE1', 'TAF1D', 'P2RX2', 'PIAS1', 'MTARC1', 'ADAP1', 'MT1G', 'F8', 'EIF3H', 'PSMA5', 'RRAGD', 'IK', 'ANKRD27', 'CYTL1']
Refined Community 91: ['P4HA1', 'CLTC', 'NR4A2', 'LPXN', 'H1-1']
Refined Community 92: ['XPNPEP1', 'PSMA5', 'HAPLN1', 'CDH11', 'KIF14', 'AMY1A', 'MLLT1', 'DNAL1', 'ZNRD2', 'COL6A3', 'CDK17', 'ULK4', 'C3', 'MRPL21', 'WNT4', 'ATAD2', 'ANK1', 'BRCA2', 'DZIP1', 'ZBTB7B', 'P4HA3', 'NKD2', 'CHST3', 'MET', 'TCIRG1', 'SLC50A1']
Refined Community 93: ['SLC23A2', 'IFNGR2', 'DZANK1', 'ASB13', 'DCN', 'PPIP5K1', 'IVNS1ABP', 'CDV3', 'ATP6V0A4', 'ZNF16', 'RPS6KB1', 'CXCL8', 'DHX40']
Refined Community 94: ['PRSS16', 'RMND1', 'NKAIN1', 'PHACTR3', 'BEX5', 'SLC1A3', 'SNRPD1', 'VARS1', 'LIME1', 'AK5', 'AKR7A3', 'TPBG', 'KDELR3', 'S100A8', 'ATG12', 'MYH2', 'KCTD2', 'ATXN7', 'BRD9', 'PDXP', 'GFUS', 'LCP2', 'PTGER3', 'ACOT7', 'GADD45B', 'TRRAP']
Refined Community 95: ['CXCL10', 'FABP9', 'SMIM10L2A', 'RPE', 'COTL1', 'OPLAH', 'DDB2', 'SLC27A3']
Refined Community 96: ['OAZ3', 'BNIP3', 'HEY1', 'SELENOP', 'CALML5', 'MET', 'SV2B', 'AMH', 'MAN1C1', 'CIAPIN1', 'CHIT1', 'LDB2', 'SREBF1', 'ANGPTL4', 'NCALD', 'ADD1', 'RCBTB2', 'MCEE', 'SNAI2', 'TAC1', 'SLC50A1']
Refined Community 97: ['CYBC1', 'ITIH3', 'HEATR6', 'EMILIN3', 'CPM', 'SEC24D', 'SFRP1', 'DKK4', 'MUCL1', 'HTT', 'SP100', 'IKBIP', 'VGLL1', 'S100A16', 'ICOS', 'PALS2', 'RFC1', 'COPZ2', 'GNAO1', 'ASB4', 'SNX21']
Refined Community 98: ['CCDC102B', 'ATAT1', 'HRCT1', 'NPEPPS', 'CD86', 'C1S', 'MYEOV', 'MTCL1', 'SRSF2', 'SNX7', 'OGN', 'IRX1', 'NAV2', 'U2SURP', 'MRGBP', 'AQP9', 'EVA1C', 'OIP5', 'KDM4A', 'TIMM17A', 'AP1M2', 'CDC25B', 'SIRT1', 'ZNF638', 'ALKBH5', 'PCMTD1', 'KANSL1L', 'RBM24', 'S100P', 'NRK', 'PVALB', 'YTHDF1', 'MTERF2', 'PGGT1B', 'ORM2', 'C1orf122', 'TMEM98', 'CC2D1B']
Refined Community 99: ['UTRN', 'FBXL20', 'CA2', 'LINC01133', 'YIF1A', 'SHTN1', 'DGAT2', 'ETV5', 'DNAAF11', 'CHN2', 'ZNF43', 'PNP', 'PENK', 'NELFCD', 'PDXK', 'MFSD3', 'CIITA', 'SAMD9L', 'PRSS23', 'S100A6', 'ELF3', 'RMND1', 'UBR5', 'ABCB1', 'NUP62', 'SOCS3', 'CARNS1', 'CHRNB2', 'CCR1', 'HBM', 'PPCS', 'CDH17', 'TLE4', 'LILRB2', 'OCIAD1', 'DDX52', 'IDO2', 'ACYP1', 'SLC17A9', 'HOXA13', 'RHOT2', 'SRCIN1', 'CCL4', 'VCL', 'PCTP', 'AAMDC', 'FST', 'CYP27B1']
Refined Community 100: ['LY6D', 'GRB14', 'NSD2', 'MAN1C1', 'LEPR', 'PKM', 'BIRC3', 'TENT5C', 'TBCK', 'HERC6', 'RBP4', 'ASB8', 'TMEM79', 'RBFOX2', 'AURKB', 'S100A1', 'FBRSL1', 'TMEM41B', 'PAGR1', 'NRP1', 'CR2', 'RBM33', 'LTBP1', 'SLC11A2', 'MT1X', 'NEUROD2', 'APRT', 'GAS8', 'CRMA', 'LRRN1', 'EIF3B', 'SMPD3', 'JPH3', 'C7orf50', 'DAPK2', 'GNPNAT1', 'MRGBP', 'ZIK1', 'GJB3', 'RBBP6', 'CREB3L2']
Refined Community 101: ['LAD1', 'HK3', 'RDX', 'RRS1', 'MEMO1', 'HCFC1', 'MORF4L2']
Refined Community 102: ['ALG13', 'COL28A1', 'EFCAB5', 'SLC17A9', 'ZNF585B', 'ARHGAP5', 'ESPL1', 'DOCK5', 'SELPLG', 'ACAA1', 'KRTAP9-4', 'OAS1', 'PRSS12', 'RPS6KA5', 'NGFR', 'SPHK2', 'TOMM34', 'GFOD1', 'GLIPR2', 'NID2', 'TMEM147-AS1', 'GATA6', 'LRRN3', 'RTN4', 'JAG1', 'OMD', 'NUP62', 'AGPAT2', 'CDK2AP1', 'TM7SF2', 'KASH5', 'OPTN', 'HNMT', 'CEACAM1', 'MYB', 'DKC1', 'NAB1', 'CEBPA', 'TFPT', 'COLEC10', 'GOLT1B', 'SERF1A', 'LRRC37A3', 'MAP7D2', 'IGHV1-69', 'RIPK2']
Refined Community 103: ['MSR1', 'FBXO17', 'TSPAN10', 'CA9', 'KPNA1', 'HUS1']
Refined Community 104: ['PKN2', 'CLIP2', 'CAPN9', 'RABGAP1L', 'PEX3', 'DENND1A', 'GRB7', 'CST7', 'TRIM29', 'CRAMP1', 'HBZ', 'HHEX', 'RNF7', 'NAP1L1']
Refined Community 105: ['CLEC10A', 'CD72', 'SLC22A18', 'IRF1', 'COL6A2', 'POLE', 'XKR9', 'ADCK5', 'ZNF202', 'PLAAT2', 'CCDC97', 'MSN', 'CHTF18', 'PDGFRB', 'SMC1A', 'HNRNPA3', 'GIMAP4', 'GBP1', 'PKMYT1', 'FIRRM', 'ADGRB2', 'MCRIP2', 'C17orf100', 'RASIP1', 'ZNF623', 'ZNF250', 'TRMU', 'IL7', 'CEACAM5', 'ASH1L', 'ATAD2', 'PLAT', 'COL22A1', 'IL17RA']
Refined Community 106: ['FBN1', 'AHCY', 'B3GNT6', 'INTS1', 'RPS4XP3', 'ITPKC', 'PTP4A2', 'C3AR1', 'CCR2', 'SYNM', 'SNHG26', 'STAU2', 'GEMIN6', 'GID4', 'PIN1', 'HOXA4', 'ISG20L2', 'KCNQ2', 'PALLD', 'MBOAT1', 'GPT', 'BCAP31', 'UQCC4', 'MORC4', 'PHF11', 'PIK3R1', 'PRSS12', 'RPP38', 'LSS', 'KRT5', 'NFKB1', 'ITIH4', 'ABCC4', 'POLR2F', 'NT5C2', 'PAPSS1', 'NAV3', 'RARA', 'KRTCAP2', 'DDB2', 'F2RL2', 'STUB1', 'DZANK1', 'MTHFD1', 'ANKRD11']
Refined Community 107: ['ST6GALNAC2', 'ZFPM2', 'KATNIP', 'CXCL8', 'LRRC37A3', 'GAS1', 'ROPN1', 'SEZ6L2', 'TNK2', 'GRIN2C', 'PARPBP', 'GNG13', 'THAP1', 'UBXN2B', 'ADAM18', 'FNDC3A', 'CDH17', 'SMTN', 'ADD3', 'DHRS4-AS1', 'CAMTA1', 'ANAPC5', 'MSMB', 'C2CD2', 'CASP3', 'LRP12', 'AP1S3', 'EIF2D', 'SEMA3F', 'LACTB', 'RAB6B', 'LY75']
Refined Community 108: ['PSAP', 'H19', 'SF3B4', 'SRR', 'GPR171', 'CACNG6', 'SV2A', 'PKHD1L1', 'CCBE1', 'FABP7', 'EPHX3', 'CCL3', 'MYBL1', 'MMP25', 'UTS2', 'YBX1', 'CDH17', 'GRK5', 'PBLD', 'KCND3', 'SLC26A2', 'RMND5B']
Refined Community 109: ['CBX3', 'WFDC2', 'MTMR11', 'ADNP', 'GALNT6', 'ADAMTS20', 'CD164', 'TTC9', 'PRSS22', 'PHB2', 'ENTPD3', 'PIP', 'CAMTA1', 'EDN3', 'DOK7', 'KCTD7', 'HEPACAM2', 'PAPSS1', 'USP7', 'ABCA7', 'MED28', 'SSTR5', 'GIT2', 'DDR1', 'TALAM1', 'SEMA3A', 'GTF2H2C', 'AFG2B', 'GSDMD', 'CD247', 'LRRFIP1', 'TUBA4A', 'TMEM47', 'BMPR1A', 'TICAM1', 'CLCA2', 'BOC', 'USP3', 'ZNF148', 'ROMO1', 'RUNX1', 'IFI44', 'SLC2A8', 'ZNF367', 'CORO2B', 'MRPL12', 'PDZK1', 'CST1', 'HNF4G', 'MEF2D', 'ZBP1', 'RAPH1', 'GGA3', 'S100A10', 'CENPB', 'KRT7', 'LTO1', 'KDM7A', 'SEMA3G', 'PI4KA', 'LRBA', 'MCAM', 'CYP26B1', 'SSRP1', 'AUTS2', 'SMARCD2', 'MC3R', 'MDK', 'HOXA2', 'HBZ', 'CREB3L4', 'COX6C', 'HPCA', 'TMSB15A', 'CRABP1', 'ZNF251', 'PPP1R16A', 'C8orf76', 'COL13A1', 'EXO1', 'ZNF12', 'CYTIP', 'PSPC1', 'GTSE1', 'POLR3K', 'CASC3', 'PNP', 'SLC1A3', 'SSBP1', 'ITGB4', 'GATA6', 'CDK10', 'NOP2', 'SYT1', 'CRIP2', 'NME4', 'ODAM', 'CHMP1A', 'FGF20', 'PDX1', 'SACS', 'GALNT3', 'IDO1', 'ZSWIM8', 'STK36', 'RBM45', 'SLC39A6', 'ZNF552', 'MUC6', 'GNGT2', 'ITFG2', 'KIAA0232', 'PABPC4', 'GPR162', 'CENPN', 'FGR', 'FCER1G', 'BPNT2', 'SPRY1', 'ATP5PD', 'INIP', 'AGAP2', 'CENPU', 'ZNF316', 'HDGFL3', 'AIDA', 'SAA1', 'GH1', 'MS4A7', 'UBE2J1', 'AEBP2', 'VCL', 'NBR1', 'RBBP8', 'CYBB', 'PPP1R26', 'CD300LB', 'ZYG11B', 'APOD', 'TOMM34', 'LACTB2', 'MFNG', 'TOX2', 'AOPEP', 'GFOD3P', 'USP15', 'TAF4', 'TSEN54', 'NCAPG2', 'COL6A1', 'BAG4', 'ILF2', 'CRIPT', 'CNNM4', 'TGFBR1', 'STARD3', 'CRACD']
Refined Community 110: ['GLI3', 'SGCA', 'RNASE9', 'KCNK15', 'LYNX1', 'IFNG', 'TWIST2', 'CCS', 'RBM24', 'SLC35E2B', 'MRPL28', 'HLA-DMB', 'TPD52', 'APBB2', 'CCNDBP1', 'LSM14B', 'FBXO47', 'USP3', 'THY1', 'RSPRY1', 'UCHL3', 'RHOB', 'ADAMTS12', 'ARHGEF5', 'CYP24A1', 'STRADB', 'CASP4', 'GALNT12', 'PTPRB', 'ARL6IP6', 'SNF8', 'SHOX2', 'MAPK15', 'TRIM22', 'URB2', 'SEC63', 'BANP', 'MCMBP', 'SYNE1', 'CCDC81', 'CFL2', 'ODAM', 'GDI1', 'ROR1', 'DCK', 'TPT1', 'RAPGEFL1', 'MPV17L2', 'AKTIP', 'ZNF470', 'TAPT1', 'CHURC1', 'ABCG1', 'DUSP14', 'CHRNA9', 'SRRT', 'GEM', 'ARHGAP29', 'SDF2', 'CYP2J2', 'ZFTRAF1', 'TTC9', 'BNIP2', 'TRAF3', 'TGFBR1', 'PRKAR2B', 'SMARCE1', 'AMDHD2', 'KCNV1', 'BRAT1', 'GIT2', 'HERC6', 'SIL1', 'TBC1D16', 'MMP7', 'SH2D1A', 'NDUFB10', 'MSLN', 'ZFPM2', 'ZFP42', 'ZNF446', 'SLC7A8', 'CRNKL1', 'PRRT3', 'PEX11A', 'WNT7A', 'PKM', 'ALAS1', 'PLAC8', 'NKD2', 'TRIM3', 'EDAR', 'FBF1', 'HSD17B11', 'COL22A1', 'FRMD4A', 'SYS1', 'PYCR1', 'TRIM33', 'RFX3', 'LSM7', 'MTRF1', 'FOXH1', 'XRCC3', 'IGKV4-1', 'SESN1', 'BRD1', 'AP2B1', 'BIRC2', 'FKBP8', 'RSAD2', 'ATP7A', 'PRRT2', 'TNFRSF19', 'TAP1', 'SCG5', 'HNRNPA3', 'NXN', 'COPA', 'CKAP2L', 'MCTS1', 'PDZD8', 'ZFHX4', 'GAB2', 'NABP1', 'KCNE3', 'RIDA', 'AMY2B', 'GOLGA1', 'BAK1', 'TNFAIP3', 'NCAN', 'TSPAN14', 'LINC02693', 'GLRX', 'IL13RA1', 'MAD2L1BP', 'ZNF862', 'SSR1', 'ETV7', 'CD24P4', 'TMEM41B', 'FSTL1', 'NBN', 'GAS1', 'CXCL9', 'TAF5', 'BCAM', 'MCOLN3', 'CLEC2B', 'ADGRF5', 'NXPH3', 'TOP1', 'RB1CC1', 'METTL26', 'RHEB', 'CLCA2', 'CROT', 'FOLR2', 'PASK', 'GPR183', 'KCNJ3', 'OAZ3', 'EZH2', 'CTXN1', 'DPM3', 'TMEM134', 'CREB3L2', 'DGKZ', 'RASGEF1A', 'RASA2', 'STMN2', 'VDAC3', 'NNMT', 'PLEK', 'RND3', 'GSE1', 'MSH6', 'CDCP1', 'ZBTB16', 'PANX2', 'STAT1', 'MAGEA5P', 'SSB', 'GALNT14', 'AOAH', 'SLC34A2', 'TBC1D29P', 'MRPS27', 'SPAG1', 'TBRG1', 'TMEM191A', 'TCFL5', 'TNNT1', 'HHEX', 'PRP4K', 'TFPT', 'CDKN2C', 'MINK1', 'MOGS', 'IRS2', 'CD1C', 'CACNA1G', 'ZNF878', 'TRPC4AP', 'INTS8', 'DGKD', 'EYA1', 'ARHGDIG', 'NAPRT', 'CYP1A1', 'NET1', 'PRSS22', 'LRRC15', 'EFEMP2', 'PDIA4', 'UTS2R', 'PDLIM2', 'TNFSF13', 'CDO1', 'KRTAP9-3', 'PLN', 'BRD7', 'PLEKHO1', 'LOX', 'FZD7', 'FARSB', 'TSKU', 'SOX12', 'CCDC89', 'TFF1', 'PRRC1', 'GNGT1', 'METAP2', 'S100P', 'TRAPPC2L', 'LDLR', 'GTF3C1', 'CCDC74B', 'NAAA', 'ELL3', 'MRPS18C', 'TENT5C', 'TEAD3', 'ZNF160', 'RPL38', 'NCR3', 'DPP4', 'WDR19', 'CHAC1', 'ARHGEF12', 'BRCA1', 'LGALS3BP', 'JCAD', 'GPM6B', 'ACADSB', 'AP1S3', 'ITGAL', 'NCALD', 'FASN', 'HABP2', 'MELK', 'NLRC5', 'TNFAIP2', 'HPSE', 'DLL4', 'CACNG6', 'TUBG1', 'ACKR1', 'PUS1', 'HCP5', 'RBM48', 'RENO1', 'MRPL9', 'MCF2L-AS1', 'STOML2', 'KDM4A', 'EMILIN1', 'HLA-DRB1', 'FHL2', 'SAT1', 'PDE4DIP', 'MS4A1', 'MYLK', 'MPHOSPH6', 'PLOD3', 'ACAP1', 'BBX', 'DNAJA2', 'MTUS1', 'CST7', 'TNPO3', 'ACACB', 'ITGB7', 'YIPF6', 'GM2A', 'TMEFF1', 'SELENOP', 'IL1A', 'CAP2', 'CHMP1B2P', 'VPS13B', 'PHB2', 'ANKRD6', 'ELOB', 'ALG12', 'ING3', 'JAM3', 'NAT1', 'ALDOC', 'KCTD1', 'DCT', 'RB1', 'CD1D', 'ABHD2', 'CFAP45', 'RAD54L', 'HTRA4', 'TAF4', 'SEPTIN4', 'CTNNAL1', 'ATF1', 'SYMPK', 'TFPI2', 'UQCC2', 'CCDC83', 'REXO1', 'KCNB2', 'OTOP2', 'SLC16A6', 'SETMAR', 'CCL8', 'ERN1', 'APBA1', 'CAVIN1', 'NR2F6', 'VCAM1', 'HDAC2', 'CRISPLD1', 'FNDC4', 'THSD7A', 'ANXA1', 'ENSG00000280119', 'PPP1R26', 'ATAD3B', 'GALR2', 'TPT1P8', 'SOX7', 'CCN4', 'RHOT2', 'CEP43', 'ABCA9', 'SRD5A1', 'THBD', 'CCL7', 'NDUFA3', 'FAM107A', 'MIR21', 'TNFRSF10B', 'TREM2', 'UBAP2L', 'CDC45', 'CHAD', 'STRN3', 'SGTB', 'ADIPOR2', 'LETMD1', 'DUSP6', 'LKAAEAR1', 'FAT1', 'IFI35', 'DGAT1', 'STK3', 'NAP1L5', 'SYCP2', 'IL21R', 'IKBKB', 'PIK3C2A', 'PELI2', 'PSMG3', 'NFYC', 'CHEK1', 'METTL25B', 'MAFA', 'SRRM2', 'CA12', 'GYG1', 'GSN', 'CMIP', 'CYP11B1', 'BNC1', 'TTC17', 'MRTO4', 'GTF2H4', 'RIMS2', 'ISCU', 'HEATR3', 'COLEC12', 'GFUS', 'CFAP97', 'C3orf18', 'ZC3H18', 'ITCH', 'JMY', 'PRKD3', 'SLC4A7', 'CPM', 'AFG2B', 'DCBLD1', 'SEC14L2', 'LLGL2', 'DCTD', 'RIT1', 'MRPL27', 'HSH2D', 'ATRN', 'GALM', 'OPRL1', 'HOOK1', 'THAP2', 'CTU2', 'IFI16', 'ARID5A', 'DLG3', 'PDGFC', 'ZNF552', 'IL6', 'DGLUCY', 'FLT1', 'DONSON', 'POLR2C', 'MAPKAPK2', 'LRATD2', 'ASTN2', 'SHARPIN', 'NDEL1', 'RBM15B', 'FUT1', 'ATF2', 'MREG', 'CYP19A1', 'U2SURP', 'CA4', 'ZPR1', 'MARCHF6', 'RPSAP47', 'FAM53B', 'MIR29B2', 'TPBG', 'MRGPRD', 'LY6S-AS1', 'SRCIN1', 'CA6', 'EPB41L1', 'MAP4K1', 'RPUSD1', 'MPI', 'KLF5', 'PPP3CC', 'BAP1', 'BRAF', 'FCGR1A', 'MED13L', 'PKLR', 'PLCL2', 'ABHD11', 'CBFA2T3', 'GPRIN2', 'LBP', 'MCCC1', 'MDM2', 'RAB35', 'BSPRY', 'MYL12A', 'SEM1', 'PCBP2', 'FIGNL1', 'MED28', 'LRRC17', 'TMEM216', 'KATNA1', 'RGL2', 'PLIN2', 'HACE1', 'SRGAP2', 'POLD1', 'NCOA6', 'SLK', 'SPRED2', 'KDM5A', 'ADK', 'CHCT1', 'VAT1', 'CCL23', 'TEAD1', 'PSMB8', 'ECI2', 'ZNF703', 'MRFAP1L2', 'FBRSL1', 'LRP3', 'TP53I3', 'LIPE', 'SF1', 'P4HA1', 'IGLL3P', 'PIK3R6', 'AVL9', 'PAF1', 'CNBD2', 'TTLL12', 'DEAF1', 'ATP5F1B', 'AMZ1', 'PTPN2', 'MRPL11', 'ALKBH1', 'EID1', 'ADIG', 'TRAM2', 'CDKAL1', 'VIPR1', 'FABP7', 'MMP11', 'HNRNPH3', 'TNKS1BP1', 'COMP', 'SERPINF1', 'CHRNA6', 'NR3C2', 'KCNJ2', 'TJP3', 'SUSD6', 'RNF24', 'RNPS1', 'TMEM158', 'POLR2A', 'KPNA4', 'RAD51', 'GIGYF1', 'LASP1', 'RBM12', 'DCN', 'CALCOCO2', 'CCL21', 'NUP88', 'IGKC', 'TCEAL9', 'PRDX4', 'ABCB1', 'OR12D2', 'C1orf116', 'CLSTN2', 'IL17C', 'MAPK7', 'GLS2', 'ZNF770', 'STIP1', 'IL22', 'BOLA1', 'FAM83B', 'TNFRSF17', 'RBP1', 'RBM7', 'LOXL1', 'GNG12', 'UPP1', 'NECTIN3', 'AMH', 'DNAI7', 'DNTTIP2', 'KCNMB4', 'ACVR1', 'ZNF461', 'SNORA71B', 'MARK2', 'NAF1', 'L3MBTL2', 'GBP3', 'RNF44', 'GLRB', 'RNF112', 'PKMYT1', 'SOSTDC1', 'XPO1', 'EPPK1', 'KRT15', 'CSH2', 'PDCD5', 'CD300LD-AS1', 'THAP1', 'TRIB3', 'UTP18', 'RGS10', 'IFI44', 'KRT8', 'SORBS1', 'NUDT8', 'NDNF', 'LPAR1', 'LDLRAD4', 'LSM1', 'EML1', 'CPVL', 'CDKN1B', 'SMPDL3B', 'DNAJB4', 'DNAAF9', 'PGM2L1', 'TPD52L1', 'MRPL38', 'DNAL1', 'TRAF2', 'TBC1D1', 'CTAG1B', 'ZNF250', 'TCERG1', 'ZNF831', 'COL4A6', 'E2F5', 'ARRB1', 'FER', 'DUSP2', 'TECPR1', 'MATK', 'AEBP1', 'ITGAX', 'CYP51A1', 'CSRP2', 'DTNB-AS1', 'BICC1', 'NDUFS4', 'PIGT', 'CELSR1', 'COL5A2', 'WDR11', 'NFU1', 'LAMA3', 'ZFAND1', 'BCL11B', 'RPP30', 'RET', 'PREX1', 'IP6K2', 'LRRC8D', 'SLC27A6', 'SLC7A7', 'RBM5', 'PPP1CA', 'OMP', 'HTN1', 'PPP1R12C', 'BHLHE40', 'GRAMD2B', 'DENND2D', 'SELENOF', 'GIP', 'ANTXR2', 'MDH1', 'INTS7', 'RPL13', 'SFRP1', 'SPTLC2', 'KRT83', 'UBE2S', 'HLF', 'IRAG1-AS1', 'S100A2', 'C1D', 'CD44', 'FRMD6', 'ASCL1', 'NOVA1', 'ZNF16', 'PEX2', 'MYO9B', 'SNAPIN', 'MYNN', 'MIEN1', 'BAALC', 'PCYOX1', 'SLC8A2', 'OCIAD1', 'OSBPL9', 'MEOX2', 'SERINC2', 'PDXDC1', 'C17orf49', 'HNRNPD', 'GOLPH3', 'PPP1CB', 'CD37', 'CTNND1', 'NFKB1', 'NFYB', 'HOXB13', 'ATP8B1', 'SMPDL3A', 'PRKAG1', 'TJAP1', 'SLF2', 'CFD']
Refined Community 111: ['METTL1', 'THAP12', 'IGF2R', 'FBN2', 'SRSF11', 'HBB']
Refined Community 112: ['TMEM158', 'C21orf58', 'LMTK3', 'B3GNTL1', 'LRP12', 'FKBP2', 'ATP6V0E2', 'RALGPS2', 'VNN1', 'UNG', 'MMP19', 'PANK1', 'CYRIA', 'F8', 'TM2D2', 'HHEX', 'TRPC1', 'TMEM30A']
Refined Community 113: ['EPB41', 'CWC25', 'RNF6', 'OR4D2', 'PALLD', 'UNC5D', 'PTPRCAP', 'ENOSF1', 'ZBTB5', 'ASPSCR1', 'ZXDC', 'RSRC1', 'COL28A1', 'CHI3L1', 'ANAPC15', 'SLU7', 'RAB2A', 'SHTN1', 'FEM1B', 'PTCH1', 'SCYL2', 'SLC2A8', 'KIRREL1', 'GPAT4', 'LYPD1', 'IDH2', 'TIMELESS', 'IFT46', 'TENT5C', 'UCHL1', 'NUDT11', 'SOCS7', 'MRPS33', 'ATRNL1', 'DENND2A', 'GNS', 'CDC5L', 'TCEAL4', 'AZI2', 'ALG13', 'LZTS3', 'RTF2', 'MUC6', 'TEX2', 'EID1', 'IGHD', 'FGF21', 'TRAK2', 'DGKE', 'MIR29B2', 'COQ7', 'CRYBG2', 'KLHL8', 'CCL5', 'HMGB1P1', 'ERICH5', 'NVL', 'EFNA4', 'BBX', 'PRR15', 'KCTD12', 'ZDHHC4', 'HRK', 'CDK2AP1', 'STING1', 'DIXDC1', 'CYTH1', 'ARHGAP45', 'SEPTIN4', 'CHUK', 'ING1', 'OBP2A', 'GPT2', 'H2AC8', 'EIF4EBP3', 'PRR16', 'VWA5A', 'MC1R', 'SV2B', 'TRHR', 'ATF2', 'PERP', 'LDOC1', 'IQSEC1', 'HLA-DOB', 'DLL1', 'SOBP', 'ARPP19', 'TUBB3', 'SHC2', 'ABCC10', 'TFAP2C', 'RACGAP1', 'CRY2', 'NIPAL2', 'SPACA3', 'SLC39A3', 'ST6GALNAC2', 'DNAI2', 'ANXA5', 'GPRC5B', 'KCNS2', 'MFSD5', 'GGT5', 'ZNF398', 'KLF16', 'NXN', 'PDIA2', 'SEC23A', 'CATSPERB', 'CIC', 'SEC23B', 'NOL3', 'KLHDC4', 'BCAS1', 'OXLD1', 'DDB2', 'NRP1', 'DYRK3', 'HEPH', 'SPARCL1', 'AFMID', 'TAF1C', 'ATP6V1A', 'SLC9A1', 'BDNF', 'F13A1', 'CLDN12', 'ETFA', 'ABCC8', 'CBX1', 'WNT9B', 'MMP25', 'SLC26A2', 'MRFAP1L1', 'LIMCH1', 'VIPR1', 'SPHK2', 'CD19', 'MZT2A', 'ADORA3', 'PIM3', 'SNPH', 'ABCF3', 'SLC2A5', 'ZNF436-AS1', 'TALAM1', 'SUN1', 'SLC45A4', 'CUL2', 'MYB', 'BICRA', 'TRMT6', 'IGKV1D-13', 'ARHGAP25', 'PPP6R3', 'RTKN', 'OGFR', 'ELK3', 'ANXA13', 'TTL', 'RBM15B', 'DACH1', 'PKP2', 'FLT4', 'ELF4', 'MUC1', 'RGP1', 'PPIF', 'TNNC2', 'FKBP2', 'NKX1-2', 'PTGR2', 'NRIP2', 'SHC3', 'SIGIRR', 'MYH10', 'SLC6A4', 'HBB', 'APBA3', 'OAZ3', 'TOP3A', 'LINC02627', 'MARCO', 'PABPC4', 'ASS1', 'RNF39', 'ADGRG2', 'NR6A1', 'PIM2', 'PTP4A3', 'MMP13', 'ELOVL4', 'PHF20', 'TBX10', 'HOXB2', 'CELSR1', 'ATP8B1', 'TTLL6', 'BAIAP2L1', 'CCN3', 'PSMB2', 'ARFGEF2', 'TPT1', 'ULK4', 'KLHDC9', 'QSOX1', 'APOE', 'C7orf50', 'GPS1', 'RPL30', 'POLB', 'KCTD1', 'NPL', 'CCDC106', 'NMNAT2', 'KRTAP4-1', 'PARP1', 'CAVIN1', 'SLC35A1', 'AIDA', 'HSPG2', 'HBQ1', 'DIP2C', 'COX19', 'PON2', 'RUBCNL', 'C1QC', 'APOD', 'LEPROT', 'ADH4', 'RAPGEFL1', 'ACTA2', 'LRRC8D', 'PIP', 'SPTBN1', 'MAGEA5P', 'SLC4A3', 'BUB1', 'MAP7D2', 'NME5', 'IRS1', 'ARAP3', 'SPATC1', 'CFD', 'DVL3', 'CCR6', 'LY9', 'RHOF', 'LEFTY2', 'IGBP1', 'IL12RB1', 'HNF4G', 'ALOX15B', 'DNAJB4', 'NTRK2', 'GATA5', 'ARFGEF3', 'HMOX1', 'C7', 'ADAMTS7', 'ANTXR1', 'CACHD1', 'BAIAP3', 'MAL2', 'SLC16A7', 'MDN1', 'KLHL29', 'CD151', 'LRRN3', 'GNGT2', 'E2F3', 'PLCB2', 'HCN3', 'PVRIG', 'RPA2', 'OMD', 'UBE2V1', 'PWP1', 'NOB1', 'MRPL38', 'NR4A2', 'MORC4', 'PIK3CB', 'CNGB1', 'PDE2A', 'LAD1', 'ERI2', 'ABCC2', 'HEY2', 'ECT2', 'CD274']
Refined Community 114: ['SLC49A3', 'ADAM8', 'ADORA2B', 'PRRX1', 'DLX4', 'KDM4B', 'ROGDI', 'CNNM4', 'RPL23', 'ABCA10', 'STYK1', 'RGS22', 'EMG1', 'ESRP2', 'DRAM2', 'CLEC3B', 'MED6', 'INSM1', 'AKT1', 'ETFA', 'NAV2', 'GADD45A', 'MAP7D2', 'RBMS3', 'ZFYVE1', 'NPTX2', 'TMEM143', 'ATL1', 'MYLK3', 'POLR2F', 'MYH9', 'ZNF318', 'PNOC', 'CAV1', 'PTGDR', 'PDZD8', 'DCBLD1', 'PEX19', 'SLK', 'OASL', 'PM20D2', 'CYRIA', 'ATP6V0E2', 'MRC1', 'TNFRSF21', 'CARD19']
Refined Community 115: ['PRKD2', 'GTPBP4', 'SNHG3', 'ENPP3', 'PKM', 'VBP1', 'B2M', 'DIP2C', 'CACNA2D2', 'WDR19', 'MBTD1', 'FYCO1', 'BAZ1B', 'EPOR', 'GUCY1A1', 'TUBB', 'TNS4', 'PSMB9', 'TFPI']
Refined Community 116: ['GALNT12', 'FAM210B', 'PLD1', 'BLNK', 'NSMCE2', 'ZNF517', 'CYP24A1', 'GATA5', 'FER', 'MCAM', 'CRTAP', 'MILR1', 'CH25H', 'BUB3', 'RUBCN', 'LPIN1', 'SLC16A7', 'ZNF570', 'TENT5C', 'GFER', 'TAGLN2', 'NKAIN1', 'SOD2', 'NARF', 'THOC2', 'KIAA1217', 'PICK1', 'MFAP2', 'CYB5R1', 'PDGFC', 'ZNF24', 'GOLM1', 'DNAI2', 'STYK1', 'RAI14', 'GPR19', 'NTHL1', 'GSTZ1', 'DENND2D', 'RPP38', 'MYBL1', 'SUN1', 'CWC25', 'IMPA2', 'KHDRBS3', 'MRM2', 'TRABD', 'TBX19', 'CIBAR1', 'ECM1', 'RND2', 'GEMIN6', 'CMAS', 'DERL1', 'MAP4', 'LAMP3', 'KDM7A', 'SAMD14', 'EMC8', 'N4BP2L2', 'HSD17B4', 'ZSCAN10', 'LYPD3', 'MROH1', 'PUM3', 'KLRK1', 'AKR1A1', 'PPAT', 'SMC1A', 'MSR1', 'TP53TG5', 'TACC3', 'AKR1B1', 'DENND1B', 'GPC1-AS1', 'S100A13', 'N4BP2', 'LRP6', 'ITPK1', 'COL8A1', 'TES', 'TESC', 'JPT1', 'PMP2', 'CHASERR', 'CLIP4', 'COQ7', 'SHOX2', 'GOLPH3', 'THSD7A', 'CYP7A1', 'PHF19', 'PFDN4', 'MPZL3', 'ZNF467', 'FGFBP1', 'ZBP1', 'ARHGEF4', 'METRNL', 'SF1', 'SLC16A2', 'DACT1', 'FOXN1', 'CD4', 'ALOX5', 'IGHV3-47', 'IGLV3-19', 'ENSG00000301761', 'PPP1R14C', 'DNMBP', 'NSUN4', 'PCSK1N', 'LSM7', 'SIGIRR', 'UNCX', 'MTMR11', 'MTARC1', 'GFM2', 'XYLT2', 'LRRC17', 'RASGRP3', 'APOD', 'CLIC4', 'STRADB', 'EPB41', 'S100A4', 'PAFAH1B1', 'AGGF1', 'TOX2', 'BRINP3', 'SPARCL1', 'KRTAP9-8', 'PHF6', 'WDR6', 'POMT1', 'BPTF', 'DTNA', 'EWSR1', 'RAP1GAP2', 'RFLNB', 'TMEM204', 'ARHGAP15', 'HPX', 'SELL', 'APP', 'DLAT', 'DLG5', 'METTL9', 'PNP', 'GDPD5', 'VAT1', 'EDEM2', 'DECR2', 'CXCL14', 'RIOK3', 'IRAK2', 'SFI1', 'NOTUM', 'RFNG', 'LUC7L', 'NR1D1', 'PCGF2', 'CCKBR', 'REXO2', 'CDH3', 'GM2A', 'ST14', 'TAP2', 'ESYT2', 'NRN1', 'RASIP1', 'FLNB', 'ALDH3A2', 'TSPAN2', 'SNX5', 'PHF5A', 'PAPPA', 'ZNF527', 'KRTAP4-2', 'ZBED4', 'PLLP', 'PTPN14', 'OVOL2', 'DHX9', 'CSRNP2', 'MTF2', 'RPS4XP3', 'STK39', 'DRG2', 'SLC16A14', 'LRRC74A', 'FAM174B', 'FAU', 'HCFC2', 'SEZ6L2', 'ANKRD27', 'SNTG2', 'ARL6IP6', 'ARNT', 'CXXC5', 'OGFR', 'CELSR1', 'ISG15', 'ZNF18', 'TULP3', 'P4HA1', 'TOP6BL', 'ZFC3H1', 'CALR', 'SPP1', 'COPZ2', 'CPPED1', 'HNF1B', 'ETV7', 'ADRB2', 'ATP8B1', 'HBZ', 'PRRC2A', 'ARTN', 'VASH2', 'SYNC', 'ACADSB', 'INHBC', 'GAS5', 'TLK2', 'SPOCK1', 'MTFR2', 'CACNB1', 'NRXN1', 'ZNF692', 'EEIG1', 'SBK1', 'PBLD', 'TMEM218', 'PACRG', 'ERN1', 'BAMBI', 'PANK1', 'MDM4', 'LLGL1', 'MTSS1', 'ITGAE', 'IGKV1D-39', 'KIF13B', 'C1orf21', 'COL18A1', 'ZMAT4', 'ABCC4', 'GPX8', 'MYBL2', 'AP2B1', 'NEDD4', 'ANKRD33B', 'LAT2', 'OPN3', 'CD248', 'IL7R', 'TRPM4', 'CHMP3', 'ABCA9', 'IGHV3-7', 'SLC6A14', 'POMK', 'ARHGAP4', 'AMY1A', 'BLVRA', 'POLR2F', 'PGAP6', 'PAMR1', 'APLP1', 'DKK1', 'DNAJB4', 'RANBP3', 'HLA-DRB6', 'SPAG4', 'COMT', 'MMP1', 'IL2RG', 'PHF12', 'EGFR', 'HMCN1', 'MRPS25', 'CD83', 'TRIM3', 'TUBG1', 'IFNAR2', 'TMEM276', 'CHSY1', 'PLIN1', 'MAGI2', 'SELENOF', 'FCER1A', 'FOXM1', 'SIRT1', 'DCAF11', 'VNN2', 'ABLIM3', 'NFYC', 'ARHGEF2', 'KLF4', 'NFIX', 'MAF1', 'GJB3', 'TGFA', 'KPNB1', 'PTP4A2', 'POT1', 'ZNF398', 'ANKIB1', 'SNRPA1', 'MYH11', 'PLAC8', 'SDHAF2', 'HRH1', 'SPCS2', 'TENM3', 'CSAG2', 'TSHZ3', 'TAGLN', 'WFIKKN2', 'XRRA1', 'CABP2', 'MARCHF10', 'HAGHL', 'DTX3L', 'FAM83H', 'PEX5', 'COX11', 'PPIC', 'MBTD1', 'ZBTB43', 'XBP1', 'TRAK2', 'GGCT', 'MRTFB', 'AAMDC', 'MTMR2', 'ZMYND19', 'IGLV4-60', 'MIIP', 'UBE2F', 'ZEB2', 'CCL23', 'GAPDHS', 'PRDM14', 'PSMD1', 'CNTROB', 'BTG3', 'FKBP14', 'RAB27B', 'NCAPG2', 'HLA-DRB4', 'CSNK1A1L', 'ASPH', 'SLC25A19', 'TATDN2', 'HHIPL2', 'MAST1', 'PDSS1', 'CNR1', 'HRAS', 'TERF1', 'PDE7B', 'PTGES', 'NPY2R', 'CLPTM1L', 'AHR', 'LRATD2', 'MICALL2', 'BTBD3', 'LMBRD2', 'C2CD5', 'C6orf62', 'MCM7', 'PLXDC1', 'PRRG4', 'LRRFIP1', 'TRIM25', 'CC2D1B', 'SLF1', 'TARDBP', 'RHOD', 'SPAG5', 'HOXB8', 'KRT14', 'FAM76B', 'EIF1AX', 'TAF1C', 'DPEP1', 'METAP2', 'ZWILCH', 'DZIP1', 'LDHB', 'XKR9', 'TAF2', 'C8orf82', 'SH2D1A', 'ARMC10', 'CXCL10', 'GFPT2', 'ESRP2', 'RGL1', 'OCM2', 'KLHL22', 'MPP7', 'NOXA1', 'ETS2', 'PNLIPRP3', 'PARP8', 'POP1', 'RBM7', 'MAP2K3', 'WIPI2', 'PPIP5K2', 'SLC7A7', 'SEC23B', 'DMRT1', 'MAP3K13', 'PCNT', 'CLEC4A', 'SPDL1', 'AHI1', 'PIEZO2', 'TTC7A', 'CRISP3', 'TECPR1', 'PSAP', 'GNLY', 'TMED3', 'LINC00469', 'TRIM47', 'ETS1', 'SAMD4A', 'VAMP1', 'TUBD1', 'ANLN', 'FDFT1', 'H1-1', 'TBC1D31', 'CSTF2', 'CNN1', 'PSTPIP1', 'PCLAF', 'NSD3', 'WDR33', 'CYBC1', 'CXCR4', 'SPATA24', 'CCN2', 'TRIB1', 'STIP1', 'C3AR1', 'GZMK', 'MMD', 'MIR29B2', 'LTBP4']
Refined Community 117: ['ABCB1', 'EMC3', 'LRRN1', 'NFS1', 'NDRG2', 'OSGIN2', 'SLC24A1', 'ACADL', 'TMC6', 'RAD9A', 'RBP3', 'ADAMTS12', 'NFKBIA', 'YTHDC2', 'CHTF18', 'CKAP2L', 'MAPK7', 'KCNMB4', 'FER1L6', 'GAPDH', 'PARP4', 'TRAF5', 'TGFBR2', 'MRPS17', 'CENPA', 'OVOL2', 'LMF1', 'PAX6', 'SVEP1', 'IRX2-DT', 'EMILIN3', 'MSRB1', 'SWAP70', 'SCAF1', 'ROR1', 'FOXP2', 'CP', 'DSG2', 'RENO1', 'DDX39A', 'TMED3', 'ERCC2', 'ZNF43', 'YBX1', 'TRIM4', 'LRP5', 'SALL4', 'ADAMTS7', 'BUD31', 'C17orf75', 'OTOP2', 'S100A10', 'SYNRG', 'COQ9', 'CDKN1C', 'WIF1', 'ACTR2', 'MAP6', 'RAB3IP', 'CCL23', 'VMP1', 'SPATC1L', 'LITAF', 'C8orf82', 'PIGS', 'APC', 'PVRIG', 'CLSPN', 'SYMPK', 'LMTK2', 'SPATA33', 'FRS2', 'SLC7A11', 'PRKAA1', 'ANKRD54', 'SPINK4', 'NEDD4', 'NOX4', 'KDM4B', 'PDLIM2', 'CYB561A3', 'GPSM2', 'SKAP2', 'RAB22A', 'RAB3D', 'THUMPD1', 'RALBP1', 'EMP3', 'SNRPF', 'TPSG1', 'LDLRAD4', 'PEG3', 'C1QA', 'RNF114', 'MBD3', 'ANLN', 'PHOSPHO1', 'CD24', 'EHBP1', 'PDGFA', 'TRIB1', 'SERHL', 'IFNG', 'DCX', 'RELB', 'MNAT1', 'GM2A', 'MRPL36', 'LBP', 'TRMT1L', 'ECI2', 'HCN3', 'ARHGDIG', 'SYNM', 'ALCAM', 'HTN1', 'CCDC102B', 'BCKDHB', 'BNIP3', 'TCAF1', 'TTC9', 'IGLL3P', 'UNK', 'MGMT', 'NTHL1', 'KLF6', 'ZCCHC8', 'PVR', 'FOXN3', 'ACOT9', 'SPAG9', 'MXD4', 'GSPT2', 'MTF2', 'CD274', 'FBRSL1', 'TASP1', 'OSMR', 'PRG2', 'LCP1', 'SERINC2', 'C1QB', 'NUDT1', 'CCDC93', 'GAS8-AS1', 'MMEL1', 'PSMB6', 'MAGEA5P', 'AATK', 'ACAD10', 'UBE2W', 'CD14', 'NEXN', 'RTP4', 'ZNF7', 'GOSR1', 'TAF1', 'PNOC', 'CACYBP', 'KMO', 'ZNF585B', 'TPSD1', 'WTAP', 'SPHK1', 'SDCBP', 'COL1A1', 'HACE1', 'NF2', 'CYC1', 'MYO5C', 'IGFBP4', 'MMP19', 'CRYBA1', 'ADGRE5', 'IGHMBP2', 'CSTB', 'MAPT', 'PTGR2', 'FLYWCH2', 'PCK1', 'APBB2', 'NNMT', 'HEMK1', 'RNFT1', 'TMEM204', 'PI3', 'CXCL16', 'CCDC77', 'CACNA1D', 'COPS8', 'SOX15', 'DNAJC1', 'CDO1', 'NLRP1', 'DSE', 'ITGB2', 'COL20A1', 'TP53I3', 'TOX2', 'B3GNT6', 'CDC25C', 'GNGT2', 'VPS45', 'ZACN', 'GNA13', 'EXOC3', 'CACNA2D2', 'AMDHD2', 'KIF3B', 'CCT6P3', 'EVI5', 'FGG', 'RSAD1', 'IFT122', 'RNASE4', 'DZIP3', 'PLAAT1', 'STAP2', 'SLC35C1', 'SLC10A5', 'CASP3', 'ATL3', 'CEBPD', 'LIN28A', 'ITGB6', 'MBNL1', 'MRTFB', 'U2SURP', 'CHD7', 'EOLA1', 'LUC7L3', 'SMG5', 'CGN', 'CXCR4', 'ENSG00000278932', 'MAB21L4', 'NOTCH3', 'HERPUD1', 'RAP1GAP2', 'CASP9', 'DNAJC6', 'NFKB2', 'RAPGEF1', 'KCTD1', 'GATA3-AS1', 'CD151', 'KASH5', 'WASHC5', 'STS', 'CDC42', 'HMGA2', 'PKHD1L1', 'GLDC', 'DESI2', 'MAPKBP1', 'CTNND2', 'TMEM105', 'SNRK', 'CD36', 'TCFL5', 'GALNT2', 'THRA', 'BLOC1S6', 'TNPO3', 'RAB11FIP3', 'CDH2', 'TM7SF2', 'RAB38', 'MYO1D', 'CEP41', 'IL11', 'ACOT2', 'STRBP', 'LPL', 'SORBS1', 'NUP85', 'PRRC2C', 'DAPK2', 'OR12D2', 'POLK', 'SPATA2L', 'COMT', 'PIEZO1', 'DNAJC5', 'NUAK1', 'CFL2', 'TFF3', 'RUBCN', 'GTF2H2C', 'LRRC32', 'PPM1B', 'RPTOR', 'CLK2', 'RAB5B', 'ACOX1', 'CALD1', 'GDF7', 'LAGE3', 'IFI16', 'MICAL3', 'IGF2BP1', 'PECAM1', 'IGKV3-20', 'BLNK', 'RAC3', 'NFKBIZ', 'PKN2', 'SPIB', 'PRMT7', 'SERPINF1', 'BCL2', 'CYP2A6', 'ADAM10', 'MOB4', 'TOP3A', 'BTBD17', 'CASKIN2', 'SERTAD4', 'MSC', 'RUNX3', 'TSPYL5', 'RAP2A', 'DDHD2', 'YWHAB', 'SELE', 'PTPRC', 'GOLT1B', 'STX16', 'GLMN', 'SEMA7A', 'KRI1', 'SLC12A9', 'S100A14', 'SLX4IP', 'PRKD2', 'SPOPL', 'SCG2', 'FAM91A1', 'PSMB3', 'TMCC2', 'RIMKLB', 'KRT81', 'S100A8', 'PFN2', 'DOC2B', 'CFLAR', 'CBFA2T3', 'MED28', 'ZNF461', 'TRAPPC2', 'TBX19', 'TRIB3', 'TEAD3', 'TMEM150C', 'PIP4K2C', 'FGF5', 'CNTROB', 'CST1', 'LZTS1', 'SLC2A5', 'PRKX', 'RRAGD', 'JRK', 'IFTAP', 'DCAF4L2', 'RAB26', 'TRIM9', 'SLC26A7', 'TCF7L1', 'CPNE1', 'HCCS', 'HTRA4', 'CD1E', 'PFDN6', 'MIR100HG', 'OCM2', 'SRGAP2', 'RELN', 'FCER1G', 'CNN1', 'UGT8', 'KCNB2', 'CSF3R', 'PIDD1', 'MS4A4A', 'ATP1A1', 'ITGA3', 'PPARD', 'LYPD1', 'MINDY2', 'LINC00469', 'NOSTRIN', 'PYGB', 'GLI4', 'GDPD5', 'ELOVL2', 'TAX1BP1', 'THY1', 'DLC1', 'ARL4C', 'MCL1', 'PHB2', 'PTPN6', 'ACADS', 'RBM12B', 'SCYL3', 'CHST11', 'NKAIN1', 'PABPN1L', 'MYH9', 'ITGAX', 'ZNF789', 'SPTAN1', 'OTOP3', 'IGFBP3', 'TMEM134', 'TNIP1', 'ATP7B', 'ATP11A', 'CBLC', 'NPBWR2', 'ZBTB7B', 'CELSR2', 'APOBEC3A', 'MAPK8IP3', 'BMP1', 'HSPB8', 'LMBRD2', 'SGPL1', 'CD3D', 'PLEKHF2', 'CNDP2', 'FXR1', 'ADAP1', 'SRRT', 'PABPC1', 'SPRY2', 'EP300', 'CYTH1', 'POLR1A', 'CCDC124', 'SGCB', 'INIP', 'OAS1', 'ATP6V1G1', 'ASB1', 'WDR24', 'FJX1', 'CHAC1', 'KLC2', 'SRRM2', 'CYP27B1', 'CIDEC', 'CDC73', 'NSUN4', 'MYO1B', 'DUOX1', 'SH2D1A', 'CTNND1', 'ZNF383', 'H3C4', 'ESRP2', 'EIF3H', 'SERPINH1', 'MYRF', 'BTG3', 'ZFAND1', 'MZT2A', 'PIK3C2G', 'PFDN4', 'TPK1', 'ZNF638', 'SHC1', 'EPRS1', 'CCL3', 'PLEKHB1', 'ADAM12', 'ZFP42', 'CKS1B', 'CCDC90B', 'ZFPM1', 'PAM', 'NCF1', 'BIRC2']
Refined Community 118: ['TBC1D31', 'BICD2', 'TSTD1', 'SLC52A2', 'TDO2', 'FMNL2', 'FAM106A', 'HTRA1', 'HES1', 'STN1']
Refined Community 119: ['DTNB-AS1', 'PPP1R3C', 'CACNG1', 'HRH1', 'CHPF', 'CHI3L2', 'LCK', 'MGAT4A', 'IGFBP4', 'EHBP1', 'ZNF532', 'LAD1', 'PAPOLB', 'NEFL', 'SV2A', 'ACOT7', 'SOX11', 'PATJ', 'DMRT1', 'GPR89B', 'CNOT9', 'HDGFL3', 'NDRG1', 'SLC9A1', 'IL20RA']
Refined Community 120: ['MMD', 'HACE1', 'PRKCQ', 'NME1-NME2', 'CASKIN2', 'IL24', 'RPP38', 'MEOX1', 'SNRPA1', 'TAC4', 'TUFT1', 'TOM1L1', 'SERPINF1', 'IGFBP7', 'ASB7', 'SLC37A1', 'GALNT10', 'PRKD2', 'DPEP1', 'MTERF1', 'AP2A2', 'HBZ', 'RGS11', 'GINS4', 'POLR1C', 'AGAP2', 'TMEM138', 'MRPS30', 'DSG1', 'PM20D2', 'MAN2B2', 'ALDOC', 'ROBO3', 'GUSBP14', 'CNOT6', 'LRP6', 'BRD1', 'XPO6', 'RAB2A', 'ACP1', 'CLIP2', 'RASAL1', 'PRR11', 'DNA2', 'DLL3', 'ZC3H12C', 'BANF1', 'FAXDC2', 'PDIA3', 'RPL29P17', 'IGF1R', 'CNN1', 'GTPBP1', 'PWWP2A', 'GNGT1', 'ENO2', 'IGFBP2', 'MARVELD1', 'EIF5AL1', 'FAM83H', 'GRIFIN', 'SPARCL1', 'EIF4E3', 'PIN1', 'SERINC2', 'TEPSIN', 'SUN1', 'TPX2', 'KLK8', 'AUTS2', 'PHF21A', 'HSD17B4', 'N4BP2', 'SGO2', 'GATA3-AS1', 'CAD', 'IVD', 'NRK', 'BBOF1', 'SVIL', 'MINDY2', 'TG', 'ANKRD28', 'PPP1CB', 'NARS2', 'CARNS1', 'ELF5', 'RELN', 'ENPP2', 'ZCCHC24', 'CYP1B1', 'EMILIN2', 'ANKRD33B', 'UBQLN4', 'CR2', 'KRT27', 'HSD17B1', 'TRAM2', 'RBP3', 'MET', 'ERICH5', 'IL6', 'ATP1A1', 'PISD', 'TAOK1', 'GOT1', 'STK26', 'PNP', 'TMEM201', 'ADH1B', 'MYNN', 'LRRC40', 'SPTAN1', 'SRD5A1', 'CPT1A', 'PIK3R1', 'SCN8A', 'TNFSF10', 'EDN1', 'MARCHF9', 'S100A2', 'VIPR1', 'CSE1L', 'MSH2', 'HMCES', 'ARHGEF40']
Refined Community 121: ['FAM133B', 'ATRN', 'DCAF13', 'CRIP1', 'CDCA3', 'MARVELD2', 'UQCRB', 'PSMB8', 'CST7', 'LKAAEAR1', 'NOB1', 'TNFAIP8', 'HIF1A', 'FANCF', 'AAK1', 'DDX11', 'OPLAH', 'PDGFD', 'TYROBP', 'PAIP2', 'MUC1', 'PRRC2A', 'MGP', 'FAU', 'NUDCD1', 'B2M', 'EPM2AIP1', 'FAM114A1', 'THBD', 'MANSC1', 'XCL1', 'CD244', 'ANOS1', 'CHST2', 'LAPTM4B', 'P2RX2', 'SELE', 'TGFB1I1']
Refined Community 122: ['CCDC15', 'APRT', 'PKHD1L1', 'CSF3R', 'ZNF570', 'FEN1', 'AGRN', 'FER', 'CA5BP1', 'EIF5A', 'TRIM68', 'PLXNB2', 'RIOX2', 'ZNF529', 'TMEM35B', 'RECQL4', 'EP400', 'NUP107', 'RAMP1', 'LRP6', 'FLNB', 'VAMP5', 'GAS8', 'ODC1', 'COL8A2', 'RANBP1', 'CA5A', 'PRKG1', 'DNMT1', 'ACE', 'RGS7', 'CACNA2D3', 'LARP1', 'ACTL6A', 'KLHDC4', 'ZSCAN18', 'CTBP1', 'PAX6', 'SPINK5', 'ZNF84', 'SYDE2', 'RDH13', 'SBNO1', 'CBR1', 'PAGR1', 'RPL15', 'ERO1A', 'EIF3H', 'CCDC90B', 'GPX7', 'THBS1', 'ARPC1B', 'PUS7L', 'MPG', 'PMVK', 'FOXH1', 'REXO1', 'KRI1', 'ZNF395', 'FLI1', 'CYC1', 'PODXL', 'MYCBP2', 'TGFB3', 'FDX1', 'UGT1A10', 'SCG2', 'SS18L2', 'PHACTR2', 'ZFAND2A', 'HPS1', 'TMEM158', 'CHRDL1', 'TBX10', 'LAD1', 'ZFAND6', 'CENPA', 'PIDD1', 'PDCD6', 'EMSY', 'UTRN', 'CMPK1', 'COPS7B', 'SLC66A2', 'KIF20A', 'DBI', 'OCA2', 'MOCOS', 'NDRG4', 'IL6ST', 'MDM1', 'MAP3K13', 'EIF5AL1', 'TICAM1', 'CACNB1', 'FSTL1', 'ARRDC1', 'PTGER2', 'MAPK3', 'ZNF831', 'RSBN1L', 'CEPT1', 'MRPL41', 'C6orf132', 'CRNKL1', 'DHCR24', 'TPMT', 'REG1A', 'HPN', 'TMEFF1', 'IFTAP', 'MTF2', 'MSRB1', 'MDK', 'PLAAT3', 'GFRA4', 'BRCA1', 'LGALS3BP', 'HACD3', 'SHC3', 'TAGAP', 'NFKBIZ', 'SNED1', 'TAP1', 'RGS2', 'PEX12', 'ZNF358', 'GPSM2', 'PHF5A', 'GAS1', 'PTBP1', 'SDHAF3', 'TSEN54', 'NOTCH4', 'ETFA', 'CYB5R1', 'ESRP2', 'COL17A1', 'ENSA', 'DLG5', 'G3BP2', 'ADAMTS7', 'PTPRM', 'DNAJC13', 'NKG7', 'NPRL3', 'SLF1', 'LRP12', 'DESI1']
Refined Community 123: ['CHST2', 'IGF2BP3', 'NCAPD2', 'CD3D', 'REXO4', 'WFDC12']
Refined Community 124: ['ECE1', 'GLMN', 'CABP4', 'SMAD9', 'PACRG', 'CHST3', 'BTG3', 'GULP1', 'CSNK2A3', 'ERP29']
Refined Community 125: ['PUM1', 'TRMT12', 'THY1', 'PIK3R2', 'TARDBP', 'WDR6', 'TNFAIP2', 'MRC1', 'ARMC6', 'ANKIB1', 'TAF2', 'PDE4DIP', 'TNFRSF19', 'PHF12', 'ACKR1', 'ABHD15', 'HAUS8', 'PDIA6', 'TMEM230', 'MS4A4A', 'PCTP', 'ENOSF1', 'SERHL2', 'FAM210B', 'GPT', 'LMO4', 'S100A4', 'CSE1L', 'BBS1', 'TOMM70', 'GDI2', 'PLK3', 'TMSB15A', 'DUSP14', 'PRMT7', 'COL6A6', 'CENPN', 'PIK3R5']
Refined Community 126: ['ROR1', 'ATP6V1G1', 'DNAJA2', 'KRT4', 'PHLDB2', 'MARS1', 'SNCAIP', 'MBD4']
Refined Community 127: ['TNFSF13', 'GRIPAP1', 'BAG6', 'ACOT11', 'MYEF2', 'LYPD1', 'MCM5', 'PDCD5', 'IGHD', 'MPHOSPH9', 'NOTCH3', 'COL10A1', 'CABP4', 'OVOL1', 'PEG10', 'PCOLCE2', 'MYL12A', 'FYCO1', 'NIPAL2', 'TNF', 'UBE2J1', 'NFYB', 'LINC01355', 'DHRS7', 'DSC3', 'CHCHD4', 'RTP4', 'AZI2', 'TM6SF1', 'LMNB1', 'RAB8B', 'MTSS2', 'SAXO4', 'ELF4', 'IFIT1', 'CDYL', 'APOH', 'PPM1D', 'PTPRF', 'HOXA6', 'MORC4', 'BCAM', 'CCNB1', 'C4orf19', 'VAMP3', 'SLC12A4', 'CHTOP', 'MMP24', 'ISOC1', 'GRWD1', 'LRP1B', 'LMO4', 'MTG2', 'TNFAIP3', 'ANXA1', 'SETMAR', 'TSC1', 'COL24A1', 'RRP1', 'CYP24A1', 'ACTN1', 'DACH1', 'MT1M', 'TGFBI', 'NPAS4', 'CA6', 'USP15', 'FSTL1', 'RGS5']
Refined Community 128: ['ARHGAP21', 'WSB1', 'RPS4X', 'STYK1', 'B4GALNT2', 'ATP7A', 'LIMK2', 'PER3', 'RAC2', 'RBM15B', 'CTSK', 'CD14', 'PUS7L', 'OSBP', 'CLC', 'SUV39H1', 'CTAG1B', 'PPM1D', 'CLDN8', 'SMARCD3', 'CLIC4', 'CCT2', 'GRB10', 'ZFPM1', 'TRIP6', 'DENND3', 'ANKRD50', 'TRPC4AP', 'GPS1', 'PLXNA2', 'UBE2S', 'A2M', 'IGKV4-1', 'ATP6AP1', 'RETSAT', 'FGF23', 'MTFR1', 'TCIM', 'PRG2', 'SRI', 'SLC35E3', 'DRAM1', 'RAP1BL', 'ICAM2', 'HEATR6', 'FOSB', 'RRP1', 'DCTD', 'ANPEP', 'DNAJC21', 'NCAPD2', 'FURIN', 'KRTAP4-4', 'ZNF436-AS1', 'TACC1', 'FMNL1', 'SPON2', 'CYP3A43', 'OR4F6', 'SLC2A5', 'STAC2', 'VPS35', 'RMND5A', 'AKR7A3', 'POLR2K', 'DERL1', 'SGSH', 'TMEM64', 'FOS', 'SLC35C1', 'GABRE', 'CEBPB', 'TMEM71', 'RNF24', 'FABP7', 'TUBG1', 'RBM47', 'COMMD1', 'FRAT2', 'WNT2', 'AFDN', 'OSBPL2', 'CSE1L', 'ERCC6L2', 'CTDSPL2', 'HES5', 'DNMT1', 'HERC6', 'ZFP42', 'EFEMP1', 'SPON1', 'CWF19L1', 'CACNA2D2', 'WDR24', 'MOV10L1', 'ILF3', 'BCL2', 'CAPN8', 'FZD8', 'EIF6', 'NRIP2', 'CCNE2', 'POLR1HASP', 'KCNS1', 'PTMA', 'GNAL', 'PCLAF', 'CFAP20', 'NCAN', 'MMP12', 'EVI5', 'ECM2', 'UCHL3', 'ARFGEF3', 'SOSTDC1', 'PAFAH1B1', 'MAD1L1', 'ZSCAN10', 'C1QA', 'MIRLET7D', 'HOXB9', 'GPR146', 'DBF4', 'HBB', 'INTS8', 'LRRC37A3', 'MMP14', 'BCL11A', 'SOBP', 'EVI2A', 'PRRT3', 'TRAK2', 'POLB', 'BMI1', 'LTB4R', 'JAG1', 'CEPT1', 'PCYOX1', 'MRPL15', 'GLP1R', 'KMT2E-AS1', 'PWWP2A', 'CYP2J2', 'IFT88', 'GINS1', 'ZNF211', 'ZBP1', 'EDN1', 'SMCHD1', 'PAPOLB', 'POLR1C', 'VEGFA']
Refined Community 129: ['CD83', 'AXIN1', 'FZD3', 'PRKCQ', 'TMEFF1', 'SFN', 'ACE2', 'PIEZO2', 'THBD', 'FABP7', 'SVEP1', 'TSNARE1', 'ABCA6', 'RGS16', 'SYMPK', 'DEPDC1', 'SLC52A3', 'CD44', 'PTMA', 'PIEZO1', 'ACTL6A', 'PHF19', 'ODAM', 'NUFIP2', 'C17orf58', 'ASPN', 'DMWD', 'MCRIP2', 'TBC1D1', 'TNFRSF12A', 'ECHDC2', 'ATP2C2', 'RGS20', 'PRSS54', 'MRPL9', 'LZTS3', 'CD1E', 'MTHFD2', 'NUDT1', 'CCDC15', 'MROH1', 'CYP2B6', 'RILPL2', 'BRCA2', 'STK3', 'TAOK1', 'HOXB4', 'S100A14', 'OAS2', 'RAD9A', 'CHCT1', 'COG3', 'MINK1', 'RPGRIP1L', 'GRINA']
Refined Community 130: ['ADCY2', 'ACTA2', 'PIK3C2G', 'AMACR', 'PARP1', 'GTPBP2', 'TTYH3', 'SMARCC2', 'PRPF39', 'HOOK1', 'FOXO3', 'TNS2', 'TSC1', 'DNHD1', 'SFSWAP']
Refined Community 131: ['PIK3CG', 'OSBP', 'PIAS1', 'AQP5', 'DIPK1B', 'LINC01140', 'TEDC2', 'COQ9', 'PSIP1', 'CACNA2D2', 'DLL3', 'ZNF211', 'MRPS7', 'CD151']
Refined Community 132: ['CXCL8', 'XPO6', 'POT1', 'GP1BB', 'MS4A1', 'MEAK7', 'TEDC2', 'TMEM151A', 'CCN1', 'CCN3', 'CRIPT', 'TFAP2B', 'CARTPT', 'HBA1', 'ARL3', 'IQGAP2', 'CCDC9B', 'CHODL', 'HOMER3', 'PDIA3', 'LZTS3', 'PRSS8', 'METTL26', 'WFIKKN2', 'ZNF585A', 'PPP3CC', 'PRSS21', 'ELL2', 'PLAU', 'UBA3', 'KCTD20', 'TSPAN31', 'AVL9']
Refined Community 133: ['PDK1', 'DELE1', 'MIR29B2', 'AMZ2', 'HMMR', 'SCAMP1', 'WFDC2', 'VWA1', 'KIRREL1', 'CCDC28A', 'GABRE', 'RBM33', 'DNAAF11', 'CA6', 'TRAPPC2', 'C3orf52', 'MGMT', 'ATP6V1C1', 'CARD6', 'DZANK1', 'DUOXA2', 'FSTL1']
Refined Community 134: ['SORL1', 'STMN2', 'SPPL2B', 'SAT1', 'PNMT', 'PTPRM', 'CEP250', 'PSMG3', 'AQP3', 'UPF3B', 'CSTF2', 'ZNF789', 'MARCHF3', 'MTERF2', 'TMED4', 'SPIB', 'TGM2', 'EPB41L2', 'ZCCHC8', 'RPE', 'TOB1', 'TBC1D12', 'MED13L', 'ZFAND1', 'PKD2', 'TPM1', 'ADGRG2', 'PDCD4', 'DMAC2L', 'SEC62', 'MRGPRD', 'E2F2', 'SPIRE2', 'ALDH18A1', 'STAP1', 'FTX', 'AFDN', 'OBP2A', 'NOVA1', 'BLNK', 'SCAF11', 'AP5Z1', 'DAP', 'FSTL1', 'MTAP', 'HNF1A', 'CTBP1', 'CTNND2', 'SPX', 'PIPOX', 'BBS1', 'GNA15']
Refined Community 135: ['CD1C', 'MPHOSPH9', 'NUFIP1', 'SERHL2', 'DECR1', 'NUTM2E', 'PSMD2', 'MOCOS', 'TMEM245', 'SNPH', 'GAREM1', 'DEK', 'TAGAP', 'DYNC2H1', 'AHI1', 'RPL23A', 'TPBG', 'SEZ6', 'GNG12', 'NOL8']
Refined Community 136: ['BNIP3L', 'RSU1', 'ARHGAP45', 'KLHL22', 'GATAD2A', 'STK17A', 'SLC7A1', 'ATP6V1G1', 'LEF1', 'CCDC89', 'ARL6IP6', 'SNORA71B', 'CCNA1', 'SPRED2', 'ITGB1', 'RAB25', 'HOXB13', 'SMO', 'DNM3', 'RALGAPA1', 'RELN', 'PRAME', 'EID1', 'MPHOSPH10P1', 'ZNF251', 'ZNF444', 'DAAM1', 'SYMPK', 'AOPEP', 'C6orf62', 'LAMC2', 'HILPDA']
Refined Community 137: ['ADGRE5', 'BLTP2', 'HNRNPA3', 'BTBD17', 'CSRNP2', 'SACS', 'CCDC74B', 'FAM107A', 'LTBP3', 'RHOD', 'INHBA', 'DVL2', 'ZC3H18', 'KATNB1', 'WNT10A', 'ALG13', 'EMC10', 'POLQ', 'OLFML2B', 'ADGRL4', 'ABHD16B', 'GRB2', 'DENND5A', 'SPEG', 'VCL', 'S100A6', 'PSMC2', 'HSPB8', 'CACNG5', 'TCF7', 'CELF2', 'NAMPT', 'STAR', 'IGF2BP3', 'HLA-DMB', 'CABP2', 'CCN1', 'DGKE']
Refined Community 138: ['CEP15', 'L3MBTL2', 'CUX1', 'FAP', 'MTERF3', 'ARHGEF4', 'RALA', 'PVALB', 'RAB6B', 'USP38', 'DCST1', 'LZTR1', 'KCTD20', 'MAFA', 'CCN5', 'MRPS30', 'CRISP3', 'SRRT', 'GAB2', 'SACS', 'SSH3', 'HOXA4', 'GINS3', 'FMO2', 'TOX2', 'EOLA1', 'RDX', 'IGFBP7', 'SAV1', 'SOX15', 'PISD', 'GPX8', 'PPP1CA', 'IGKV2D-28', 'FAM174B', 'ASS1', 'TFPI2', 'AZI2', 'UBN1', 'WNT11', 'ITGB6', 'RNF167', 'RPL29P17', 'CLTC', 'TOP3A', 'DEK', 'MLKL', 'NHLRC1', 'SYNCRIP', 'CLDN6', 'PLN', 'CD151', 'ADAM19', 'RRAS2', 'DVL1', 'ARHGAP25', 'HMGCL', 'KHDC4', 'CPPED1', 'SH3KBP1', 'PPP1R9B', 'CD247', 'TANK', 'TMEM204', 'MRPL12', 'CD19', 'PDLIM2', 'IL18', 'KRT7', 'CDK12', 'TMX4', 'CFB', 'TTC28', 'CEACAM6', 'VMP1', 'NUP85', 'HBB', 'TP53TG5', 'TNFRSF17', 'RARRES1', 'ANKHD1', 'CD180', 'CTSV', 'ADGRE5', 'KCNV1', 'GJB3', 'FBXL18', 'SBNO1', 'NUFIP2', 'SALL4', 'OGG1', 'TONSL', 'SERPINA3', 'KRT18', 'STAR', 'ASPA', 'MCF2L', 'TMEM123', 'LONRF2', 'PPP6R2', 'PGA5', 'DPM3', 'BIN2', 'MKS1', 'PDK1', 'MAP7D3', 'HDAC11', 'TRMU', 'CCR2', 'NRP1', 'ZNF706', 'ZNF572', 'BCAS3', 'GZMH', 'CD36', 'RAPH1', 'ESRP2', 'HSF5', 'PLCL1', 'UHRF1', 'PTCD1', 'CLIP4', 'PCBP2', 'UGCG', 'PITRM1', 'ZNF598', 'CYP2E1', 'HPSE', 'TJP3', 'ZNF276', 'CENPX', 'NOTUM']
Refined Community 139: ['BNIP3L', 'TP53', 'SNN', 'TTLL4', 'SYDE2', 'AKAP9', 'DONSON', 'GH2', 'ANTKMT', 'JUNB', 'CARTPT', 'DSG1', 'TDRKH', 'NME2', 'TAF1D', 'PPM1D', 'NKAIN4', 'NR6A1', 'SNTG2', 'TOX2', 'IFT88', 'SAT1', 'TNFRSF4', 'TBC1D12', 'OPLAH', 'DCBLD1', 'LPAR1', 'RPL19', 'DDX11', 'RPA2', 'CNRIP1', 'PLAAT1', 'RAD54B']
 """
    
    detailed_df = run_optimized_analysis(
        input_text=input_text,
        api_key=os.environ.get("DEEPSEEK_API_KEY"), 
        detailed_csv="BreastCancer_RandomAnnotation.csv"
    )
    
    pd.set_option('display.max_colwidth', None)
    print("\nPathway Analysis Results (Clean Text):")

    main_columns = ['Community', 'Process_With_Enrichment', 'Confidence_With_Enrichment', 
                   'Process_Without_Enrichment', 'Confidence_Without_Enrichment', 
                   'Final_Process', 'Final_Confidence']
    available_main_columns = [col for col in main_columns if col in detailed_df.columns]
    print(detailed_df[available_main_columns])
    
    print(f"\nTotal communities analyzed: {len(detailed_df)}")
    print(f"Available columns: {list(detailed_df.columns)}")
    
    if 'Contributing_Genes_With_Enrichment' in detailed_df.columns:
        print(f"\nSample contributing genes (cleaned):")
        for idx, row in detailed_df.head(3).iterrows():
            print(f"Community {row['Community']}: {row.get('Contributing_Genes_With_Enrichment', 'N/A')}")

### Clean Process Name Columns — AML Annotation

Removes trailing embedded confidence-score artifacts from `Process_With_Enrichment` and
`Process_Without_Enrichment`, e.g.:

- `Name ([0.42])`
- `Name (0.68)`
- `Name (confidence score0.65)`
- `Name (confidence score: 0.65)`

Genuine parenthetical content in the middle of a name (e.g. `... (SREBP) Signaling`) is left alone,
since the pattern only matches a trailing parenthetical block that contains a number.

In [ ]:
COLUMNS_TO_CLEAN = [
    "Process_With_Enrichment",
    "Process_Without_Enrichment",
]

SCORE_SUFFIX_PATTERN = re.compile(
    r"""
    \s*                                  # leading whitespace before the paren block
    \(+                                  # one or more opening parens
    \s*
    (?:confidence\s*score\s*:?\s*)?      # optional "confidence score" / "confidence score:" label
    \[?\s*                               # optional opening bracket
    (-?\d*\.\d+|-?\d+)                   # the numeric score itself
    \s*\]?                               # optional closing bracket
    \s*
    \)+                                  # one or more closing parens
    \s*$                                 # anchored to end of string
    """,
    re.IGNORECASE | re.VERBOSE,
)
def clean_process_name(value, extract_score: bool = False):
    if not isinstance(value, str):
        return (value, None) if extract_score else value

    match = SCORE_SUFFIX_PATTERN.search(value)
    cleaned = SCORE_SUFFIX_PATTERN.sub("", value).strip()

    if extract_score:
        score = float(match.group(1)) if match else None
        return cleaned, score
    return cleaned
def clean_annotation_file(input_path: str, output_path: str, extract_scores: bool = True):
    df = pd.read_csv(input_path)
    df.columns = [c.strip() for c in df.columns]

    for col in COLUMNS_TO_CLEAN:
        if col not in df.columns:
            print(f"  Skipping \'{col}\' - column not found")
            continue

        if extract_scores:
            cleaned_and_scores = df[col].apply(lambda v: clean_process_name(v, extract_score=True))
            df[col] = cleaned_and_scores.apply(lambda t: t[0])
            score_col = f"{col}_Score_Embedded"
            df[score_col] = cleaned_and_scores.apply(lambda t: t[1])
            n_found = df[score_col].notna().sum()
            print(f"  {col}: cleaned {n_found} embedded score(s) -> new column \'{score_col}\'")
        else:
            df[col] = df[col].apply(clean_process_name)
            print(f"  {col}: cleaned")

    df.to_csv(output_path, index=False)
    print(f"Saved cleaned file to {output_path}")
    return df
INPUT_CSV = "BreastCancer_RandomAnnotation.csv"
OUTPUT_CSV = "BreastCancer_RandomAnnotation.csv"

df = clean_annotation_file(INPUT_CSV, OUTPUT_CSV)
df.head()

### Validation Pipeline

In [ ]:
import pandas as pd
import json
import os
import re
import time
import requests
from typing import List, Dict, Any, Optional, Tuple
from tqdm import tqdm

UNKNOWN_PROCESS_LABEL = "unknown process"

def _is_unknown(process_name: str) -> bool:
    """Return True if the process name is a variant of 'unknown process'."""
    return process_name.strip().lower() == UNKNOWN_PROCESS_LABEL


class GeneSetValidator:
    def __init__(
        self,
        gene_sets_path: str,
        papers_csv: str = "AML_Paper_DB.csv",
        api_key: Optional[str] = None,
        output_path: str = "validated_gene_sets.csv",
    ):
        self.gene_sets_path = gene_sets_path
        self.papers_csv     = papers_csv
        self.output_path    = output_path
        self.api_key        = api_key or os.environ.get("DEEPSEEK_API_KEY")

        if not self.api_key:
            print("Warning: No DeepSeek API key provided.")

        self.gene_sets_df      = None
        self.papers_df         = None
        self.validated_results = []

        # ── Load Clarivate JIF journal list ───────────────────────────────────
        try:
            jif_df = pd.read_csv("journals_filtered_JIF_ge_4.csv", sep=",")
            jif_df.columns = [c.strip() for c in jif_df.columns]

            self._hq_issn_set  = set()
            self._hq_eissn_set = set()
            self.TOP_JOURNALS  = []

            for _, jr in jif_df.iterrows():
                issn  = str(jr.get("ISSN",  "")).strip().replace("-", "").upper()
                eissn = str(jr.get("eISSN", "")).strip().replace("-", "").upper()
                name  = str(jr.get("Journal name", "")).strip()

                if issn  and issn  != "NAN": self._hq_issn_set.add(issn)
                if eissn and eissn != "NAN": self._hq_eissn_set.add(eissn)
                if name:                     self.TOP_JOURNALS.append(name)

            self._top_journal_set_normalized = {
                self.normalize_journal(j) for j in self.TOP_JOURNALS
            }

            print(f"Loaded {len(jif_df)} high-quality journals from Clarivate JIF CSV "
                  f"({len(self._hq_issn_set)} ISSNs, {len(self._hq_eissn_set)} eISSNs)")

        except FileNotFoundError:
            print("Warning: journals_filtered_JIF_ge_4.csv not found. No quality filter applied.")
            self._hq_issn_set                = set()
            self._hq_eissn_set               = set()
            self.TOP_JOURNALS                = []
            self._top_journal_set_normalized = set()


    def normalize_journal(self, name) -> str:
        if pd.isna(name):
            return ""
        name = str(name).lower()
        name = name.split(" : ")[0].split(" - ")[0]
        name = re.sub(r"[^\w\s]", "", name)
        name = re.sub(r"\s+", " ", name)
        return name.strip()

    def _normalize_id(self, val) -> str:
        return str(val).strip().replace("-", "").upper() if pd.notna(val) else ""

    def _is_high_quality(self, row) -> bool:
        """Match in priority order: ISSN -> eISSN -> exact normalized journal name only."""
        issn = self._normalize_id(row.get("issn"))
        if issn and issn in self._hq_issn_set:
            return True
        eissn = self._normalize_id(row.get("eissn"))
        if eissn and eissn in self._hq_eissn_set:
            return True
        norm = self.normalize_journal(row.get("journal", ""))
        if norm and norm in self._top_journal_set_normalized:
            return True
        return False

    def _load_papers_for_genes(self, genes: List[str]) -> pd.DataFrame:
        if self.papers_df is None or self.papers_df.empty:
            return pd.DataFrame()

        genes_upper = {g.upper() for g in genes if g}
        mask   = self.papers_df["gene"].str.upper().isin(genes_upper)
        subset = self.papers_df[mask].copy()

        if subset.empty:
            return subset

        if "full_text" not in subset.columns:
            subset["full_text"] = "Full text not available via PMC"

        subset["matched_journal"] = subset.apply(
            lambda row: "HQ" if self._is_high_quality(row) else None, axis=1
        )

        hq  = subset["matched_journal"].notna().sum()
        tot = len(subset)
        print(f"  Paper filter: {hq}/{tot} high-quality papers for gene set")
        return subset


    def load_data(self):
        print("Loading gene sets data...")
        self.gene_sets_df = pd.read_csv(self.gene_sets_path)
        self.gene_sets_df.columns = [col.strip() for col in self.gene_sets_df.columns]
        print(f"Loaded {len(self.gene_sets_df)} gene sets from {self.gene_sets_path}")

        print(f"Loading AML paper database from {self.papers_csv}...")
        if not os.path.exists(self.papers_csv):
            print(f"Paper CSV '{self.papers_csv}' not found.")
            self.papers_df = pd.DataFrame()
        else:
            self.papers_df = pd.read_csv(self.papers_csv, low_memory=False)
            self.papers_df.columns = [col.strip() for col in self.papers_df.columns]
            if "gene" not in self.papers_df.columns:
                print("'gene' column not found in paper CSV - gene filtering disabled.")
                self.papers_df = pd.DataFrame()
            else:
                self.papers_df["gene"] = self.papers_df["gene"].astype(str).str.strip()
                print(f"Loaded {len(self.papers_df)} papers ({self.papers_df['gene'].nunique()} unique genes)")


    def extract_genes_from_set(self, gene_set_row: pd.Series) -> List[str]:
        genes = []
        possible_gene_columns = [
            "Genes_String", "Contributing_Genes", "Genes", "Final_Contributing_Genes",
        ]
        for col in possible_gene_columns:
            if col in gene_set_row and pd.notna(gene_set_row[col]):
                gene_data = gene_set_row[col]
                if isinstance(gene_data, list):
                    genes = gene_data
                elif isinstance(gene_data, str):
                    if "," in gene_data:
                        genes = [g.strip() for g in gene_data.split(",")]
                    elif ";" in gene_data:
                        genes = [g.strip() for g in gene_data.split(";")]
                    else:
                        genes = [gene_data.strip()]
                break
        return [g for g in genes if g and g.strip()]


    def extract_gene_related_abstracts(
        self,
        genes: List[str],
        papers_df: pd.DataFrame,
        max_papers: int = 50,
    ) -> Tuple[List[Dict], List[str]]:
        """
        Filter to HQ papers only, then prioritise by gene_count descending.

        gene_count = number of query genes found in the COMBINED
        abstract + full_text corpus (never short-circuited by the gene column).
        The gene column is used only as a safety net to ensure the primary
        gene is never missed if the text search fails.

        """
        if isinstance(genes, str):
            genes = [g.strip() for g in genes.split(",")]

        genes_set = {g.upper() for g in genes if g}

        hq_papers = []

        for _, paper in papers_df.iterrows():

            # ── HQ gate ───────────────────────────────────────────────────────
            if pd.isna(paper.get("matched_journal")):
                continue

            abstract  = str(paper.get("abstract",  ""))
            full_text = str(paper.get("full_text", ""))

            if abstract  in ("nan", "Abstract not available", ""):
                abstract = ""
            if full_text in ("nan", "Full text not available via PMC", ""):
                full_text = ""

            if not abstract and not full_text:
                continue

            # ── Always scan abstract + full_text combined corpus ──────────────
            search_corpus = (abstract + " " + full_text).strip()
            mentioned_genes = [
                g for g in genes
                if g and re.search(rf"\b{re.escape(g)}\b", search_corpus, re.IGNORECASE)
            ]
            gene_from_col = str(paper.get("gene", "")).strip()
            if gene_from_col and gene_from_col.upper() in genes_set:
                if gene_from_col not in mentioned_genes:
                    mentioned_genes.append(gene_from_col)

            if not mentioned_genes:
                continue

            truncated_abstract  = (abstract[:500]  + "...") if len(abstract)  > 500  else abstract
            truncated_full_text = (full_text[:3000] + "...") if len(full_text) > 3000 else full_text

            hq_papers.append({
                "title":           paper.get("title",   "No title"),
                "authors":         paper.get("authors", "No authors"),
                "journal":         paper.get("journal", "No journal"),
                "year":            paper.get("year",    "Unknown"),
                "abstract":        truncated_abstract,
                "full_text":       truncated_full_text,
                "doi":             paper.get("doi",     "No DOI"),
                "pmid":            paper.get("pmid",    "No PMID"),
                "genes_mentioned": mentioned_genes,
                "gene_count":      len(mentioned_genes),
            })

        hq_papers.sort(key=lambda x: x["gene_count"], reverse=True)
        relevant_papers = hq_papers[:max_papers]

        top_journal_papers_used = [
            f"Paper {idx}: '{p['title']}', {p['authors']}"
            for idx, p in enumerate(relevant_papers, 1)
        ]

        return relevant_papers, top_journal_papers_used

    def create_validation_prompt(
        self,
        gene_set_id: str,
        genes: List[str],
        process_with_enrichment: str,
        confidence_with_enrichment: float,
        process_without_enrichment: str,
        confidence_without_enrichment: float,
        analysis_with_enrichment: str,
        analysis_without_enrichment: str,
        relevant_papers: List[Dict],
    ) -> str:

        genes_str      = ", ".join(genes) if isinstance(genes, list) else str(genes)
        papers_section = ""
        for i, paper in enumerate(relevant_papers, 1):
            papers_section += (
                f"PAPER {i}:\n"
                f"Title: {paper['title']}\n"
                f"Authors: {paper['authors']}\n"
                f"Journal: {paper['journal']} ({paper['year']})\n"
                f"Genes Mentioned: {', '.join(paper['genes_mentioned'])}\n"
                f"Abstract: {paper['abstract']}\n"
                f"\n"
            )

        return f"""You are a scientific expert in genomics and bioinformatics tasked with validating gene set analysis results using ONLY the provided literature.

GENE SET ID: {gene_set_id}
GENES: {genes_str}

ORIGINAL ANALYSIS WITH ENRICHMENT:
Process Name: {process_with_enrichment}
Original Confidence Score: {confidence_with_enrichment}
Analysis: {analysis_with_enrichment[:1000]}...

ORIGINAL ANALYSIS WITHOUT ENRICHMENT:
Process Name: {process_without_enrichment}
Original Confidence Score: {confidence_without_enrichment}
Analysis: {analysis_without_enrichment[:1000]}...

PROVIDED LITERATURE (USE ONLY THESE STUDIES):
{papers_section}

VALIDATION TASK:
Based STRICTLY on the provided literature above, evaluate both analyses and provide updated confidence scores.

**ABSOLUTE REQUIREMENT FOR PAPER CITATIONS:**
Whenever you reference a paper anywhere in your response, always use this exact format:
"Paper X (FirstAuthorLastName et al.)"
For example: "Paper 3 (Nakamura et al.) shows..." or "supported by Paper 1 (Chen et al.) and Paper 4 (Okafor et al.)"
Do NOT use bare numbers like "Paper 3" alone, and do NOT spell out full titles or complete author lists in-line — the first-author-et-al form is sufficient everywhere in the response.

CRITICAL REQUIREMENTS:
1. Use ONLY the provided papers - do not add external knowledge
2. For each process, check if the genes are supported by the literature
3. Provide updated confidence scores based on evidence strength
4. Select the better-supported process as the final choice
5. **MANDATORY:  When referencing papers by number anywhere in your analysis text, always use the "Paper X (FirstAuthor et al.)" format defined above — never bare numbers, never full citations..**
6. The final selected process MUST be either:
   (a) exactly one of the two original process names, OR
   (b) "Neither process" ONLY if both updated confidence scores (Updated Confidence With Enrichment, Updated Confidence Without Enrichment) are <= 0.05.

7. The final confidence MUST follow:
   - If a process is selected: Final Confidence = the updated confidence of that selected process.
   - If "Neither process" is selected: Final Confidence = max(Updated Confidence With Enrichment, Updated Confidence Without Enrichment).

FORMAT YOUR RESPONSE EXACTLY AS FOLLOWS:

VALIDATION OF ENRICHMENT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_with_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

VALIDATION OF DIRECT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_without_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

FINAL PROCESS SELECTION:
Selected Process: [Choose the better-supported process name]
Final Confidence: [The updated confidence score for your selected process]
Selection Reasoning: [Explain why this process has stronger literature support]

CONFLICT ANALYSIS:
Review all papers for contradictory evidence about gene functions, pathway assignments, or experimental results.
CONFLICTING_EVIDENCE_FOUND: [TRUE/FALSE]
CONFLICT_DESCRIPTION: [Brief description of any conflicts found, or "No conflicts detected"]

SUPPORTING CITATIONS:
[List citations in format: "Title, Authors" for papers that support the final selected process]

VALIDATION ANALYSIS TEXT:
[Comprehensive summary of all changes made, reasoning for confidence adjustments, and evidence from the provided studies that led to the final process selection.
Use "Paper X (FirstAuthor et al.)" format for any paper references.]"""

    def call_deepseek_api(self, prompt: str) -> str:
        if not self.api_key:
            raise ValueError("DeepSeek API key is required.")

        api_url = "https://api.deepseek.com/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
        }
        data = {
            "model": "deepseek-chat",
            "messages": [
                {
                    "role":    "system",
                    "content": (
                        "You are a scientific expert in genomics and bioinformatics "
                        "specializing in gene set analysis validation. "
                        "Base your analysis strictly on the provided literature."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            "temperature": 0,
            "max_tokens":  8000,
        }

        max_retries = 3
        retry_delay = 5

        for attempt in range(max_retries):
            try:
                response = requests.post(api_url, headers=headers, json=data, timeout=180)

                if response.status_code == 200:
                    return response.json()["choices"][0]["message"]["content"]

                elif response.status_code == 429:
                    wait_time = int(response.headers.get("Retry-After", retry_delay * 2))
                    print(f"Rate limited. Waiting {wait_time}s...")
                    time.sleep(wait_time)

                elif 500 <= response.status_code < 600:
                    print(f"Server error {response.status_code}. Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                    retry_delay *= 2

                else:
                    raise Exception(f"API Error {response.status_code}: {response.text}")

            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{max_retries}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"API timeout after {max_retries} attempts")

            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"Request failed after {max_retries} attempts: {e}")

        raise Exception(f"Failed to get a valid response after {max_retries} attempts")

    def parse_llm_response(self, response: str) -> Dict[str, Any]:
        results = {}

        m = re.search(
            r"VALIDATION OF ENRICHMENT ANALYSIS:.*?Updated Confidence:\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_with_enrichment_after"] = float(m.group(1))

        m = re.search(
            r"VALIDATION OF DIRECT ANALYSIS:.*?Updated Confidence:\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_without_enrichment_after"] = float(m.group(1))

        m = re.search(r"Selected Process:\s*(.+?)(?=\n|$)", response)
        if m:
            results["final_process"] = m.group(1).strip()

        m = re.search(r"Final Confidence:\s*([\d.]+)", response)
        if m:
            results["final_confidence"] = float(m.group(1))

        m = re.search(r"CONFLICTING_EVIDENCE_FOUND:\s*(TRUE|FALSE)", response, re.IGNORECASE)
        results["conflicting_evidence_found"] = (
            m.group(1).upper() == "TRUE" if m else False
        )

        m = re.search(
            r"CONFLICT_DESCRIPTION:\s*(.+?)(?=\n\n|SUPPORTING CITATIONS:|$)",
            response, re.DOTALL,
        )
        results["conflict_description"] = m.group(1).strip() if m else "No conflicts detected"

        m = re.search(
            r"SUPPORTING CITATIONS:\s*(.+?)(?=\n\n|VALIDATION ANALYSIS TEXT:|$)",
            response, re.DOTALL,
        )
        if m:
            results["supporting_citations"] = [
                line.strip()
                for line in m.group(1).strip().split("\n")
                if line.strip() and not line.strip().startswith(("-", "*"))
            ]
        else:
            results["supporting_citations"] = []

        m = re.search(r"VALIDATION ANALYSIS TEXT:\s*(.+?)$", response, re.DOTALL)
        if m:
            results["validation_analysis_text"] = m.group(1).strip()

        return results

    def validate_gene_set(self, gene_set_row: pd.Series) -> Dict[str, Any]:
        set_id = str(
            gene_set_row.get("Community", "")
            or gene_set_row.get("Set_ID", "")
            or gene_set_row.get("community", "")
            or gene_set_row.get("index", "")
        )

        genes = self.extract_genes_from_set(gene_set_row)
        if not genes:
            print(f"  No genes found for set {set_id}")
            return self._error_result(set_id, "No genes found for this set")

        process_with    = str(gene_set_row.get("Process_With_Enrichment", "")
                              or gene_set_row.get("Final_Process", ""))
        conf_with       = float(gene_set_row.get("Confidence_With_Enrichment", 0)
                                or gene_set_row.get("Final_Confidence", 0))
        process_without = str(gene_set_row.get("Process_Without_Enrichment", ""))
        conf_without    = float(gene_set_row.get("Confidence_Without_Enrichment", 0))

        analysis_with    = str(gene_set_row.get("Analysis_Text_With_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_With_Enrichment", "")
                               or gene_set_row.get("Full_Analysis", ""))
        analysis_without = str(gene_set_row.get("Analysis_Text_Without_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_Without_Enrichment", ""))

        # ── Unknown process guard ─────────────────────────────────────────────
        with_is_unknown    = _is_unknown(process_with)
        without_is_unknown = _is_unknown(process_without)

        if with_is_unknown and without_is_unknown:
            print(f"  Both processes are 'unknown process' for set {set_id} - skipping.")
            return self._both_unknown_result(set_id, genes, gene_set_row)

        if with_is_unknown:
            print(f"  'With enrichment' is unknown process for set {set_id} - zeroing out, validating direct only.")
            process_with  = UNKNOWN_PROCESS_LABEL
            conf_with     = 0.0
            analysis_with = "Unknown process - not validated."

        if without_is_unknown:
            print(f"  'Without enrichment' is unknown process for set {set_id} - zeroing out, validating enrichment only.")
            process_without  = UNKNOWN_PROCESS_LABEL
            conf_without     = 0.0
            analysis_without = "Unknown process - not validated."

        papers_df = self._load_papers_for_genes(genes)

        relevant_papers, top_journal_info = self.extract_gene_related_abstracts(
            genes, papers_df, max_papers=50
        )

        if not relevant_papers:
            print(f"  No matching high-quality papers for set {set_id}")
            return self._no_papers_result(set_id, genes, gene_set_row)

        prompt = self.create_validation_prompt(
            gene_set_id=set_id,
            genes=genes,
            process_with_enrichment=process_with,
            confidence_with_enrichment=conf_with,
            process_without_enrichment=process_without,
            confidence_without_enrichment=conf_without,
            analysis_with_enrichment=analysis_with,
            analysis_without_enrichment=analysis_without,
            relevant_papers=relevant_papers,
        )

        os.makedirs("prompts", exist_ok=True)
        with open(f"prompts/prompt_{set_id}.txt", "w", encoding="utf-8") as f:
            f.write(prompt)

        try:
            response = self.call_deepseek_api(prompt)

            os.makedirs("responses", exist_ok=True)
            with open(f"responses/response_{set_id}.txt", "w", encoding="utf-8") as f:
                f.write(response)

            parsed = self.parse_llm_response(response)

            conf_with_after    = parsed.get("confidence_with_enrichment_after",   conf_with)
            conf_without_after = parsed.get("confidence_without_enrichment_after", conf_without)
            if with_is_unknown:
                conf_with_after = 0.0
            if without_is_unknown:
                conf_without_after = 0.0

            return {
                "Set_ID":                                set_id,
                "Genes":                                 genes,
                "Process_With_Enrichment_Original":      process_with,
                "Process_Without_Enrichment_Original":   process_without,
                "Confidence_With_Enrichment_Before":     conf_with,
                "Confidence_Without_Enrichment_Before":  conf_without,
                "Confidence_With_Enrichment_After":      conf_with_after,
                "Confidence_Without_Enrichment_After":   conf_without_after,
                "Final_Process":                         parsed.get("final_process",   process_with),
                "Final_Confidence":                      parsed.get("final_confidence", conf_with),
                "Validation_Analysis_Text":              parsed.get("validation_analysis_text", "No analysis provided"),
                "Supporting_Citations":                  parsed.get("supporting_citations", []),
                "Conflicting_Evidence_Found":            parsed.get("conflicting_evidence_found", False),
                "Conflict_Description":                  parsed.get("conflict_description", "No conflicts detected"),
                "Total_Papers_Found":                    len(relevant_papers),
            }

        except Exception as e:
            print(f"  API error for set {set_id}: {e}")
            return self._error_result(set_id, f"API error: {e}", genes, gene_set_row)

    def validate_all_gene_sets(self):
        if self.gene_sets_df is None:
            self.load_data()

        results    = []
        total_sets = len(self.gene_sets_df)
        print(f"\nStarting validation of {total_sets} gene sets...")

        pbar = tqdm(
            total=total_sets,
            desc="Validating gene sets",
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
            ncols=100,
        )

        for i, (_, row) in enumerate(self.gene_sets_df.iterrows()):
            set_id = str(
                row.get("Community", "")
                or row.get("Set_ID", "")
                or row.get("community", "")
                or i
            )
            try:
                result = self.validate_gene_set(row)
                results.append(result)
            except Exception as e:
                print(f"\nUnexpected error on set {set_id}: {e}")

            pbar.update(1)
            pbar.set_postfix_str(f"Set {set_id}")

            if (i + 1) % 5 == 0:
                self.validated_results = results
                self.save_results(f"{self.output_path}.partial")

        pbar.close()
        self.validated_results = results
        self.save_results(self.output_path)
        print(f"\nValidation complete! Processed {len(results)} gene sets")
        return results

    def save_results(self, output_path: str = None):
        if not self.validated_results:
            print("No results to save")
            return

        path = output_path or self.output_path
        df   = pd.DataFrame(self.validated_results)

        required_columns = [
            "Set_ID", "Genes",
            "Process_With_Enrichment_Original", "Process_Without_Enrichment_Original",
            "Confidence_With_Enrichment_Before", "Confidence_Without_Enrichment_Before",
            "Confidence_With_Enrichment_After",  "Confidence_Without_Enrichment_After",
            "Final_Process", "Final_Confidence",
            "Validation_Analysis_Text", "Supporting_Citations",
            "Conflicting_Evidence_Found", "Conflict_Description",
            "Total_Papers_Found",
        ]
        for col in required_columns:
            if col not in df.columns:
                df[col] = None
        df = df[required_columns]
        df.to_csv(path, index=False)
        print(f"Saved {len(df)} validated results to {path}")


    def generate_summary_report(self) -> str:
        if not self.validated_results:
            return "No validation results available"

        total = len(self.validated_results)

        conflict_count     = sum(1 for r in self.validated_results if r.get("Conflicting_Evidence_Found"))
        total_paper_counts = [r.get("Total_Papers_Found", 0)           for r in self.validated_results]
        final_confidences  = [r.get("Final_Confidence", 0)             for r in self.validated_results]

        conf_delta_enrich = []
        conf_delta_direct = []
        for r in self.validated_results:
            be, ae = r.get("Confidence_With_Enrichment_Before", 0),   r.get("Confidence_With_Enrichment_After", 0)
            bd, ad = r.get("Confidence_Without_Enrichment_Before", 0), r.get("Confidence_Without_Enrichment_After", 0)
            if be > 0 and ae > 0: conf_delta_enrich.append(ae - be)
            if bd > 0 and ad > 0: conf_delta_direct.append(ad - bd)

        lines = [
            "Gene Set Validation Summary Report",
            "=" * 50,
            f"Total gene sets processed: {total}",
            "",
            "CONFLICT ANALYSIS:",
            f"  With conflicts:    {conflict_count} ({conflict_count/total*100:.1f}%)",
            f"  Without conflicts: {total - conflict_count} ({(total-conflict_count)/total*100:.1f}%)",
            "",
            "PAPER DISCOVERY:",
            f"  Avg papers/set:  {sum(total_paper_counts)/len(total_paper_counts):.2f}",
            f"  Sets with papers: {sum(1 for c in total_paper_counts if c > 0)} ({sum(1 for c in total_paper_counts if c > 0)/total*100:.1f}%)",
            "",
            "CONFIDENCE:",
            f"  Avg final confidence: {sum(final_confidences)/len(final_confidences):.2f}",
            f"  Range: {min(final_confidences):.2f} - {max(final_confidences):.2f}",
        ]
        if conf_delta_enrich:
            lines.append(f"  Avg enrichment confidence delta: {sum(conf_delta_enrich)/len(conf_delta_enrich):+.3f}")
        if conf_delta_direct:
            lines.append(f"  Avg direct confidence delta:     {sum(conf_delta_direct)/len(conf_delta_direct):+.3f}")

        return "\n".join(lines)


    def _error_result(
        self,
        set_id: str,
        error_msg: str,
        genes: List[str] = None,
        row: pd.Series = None,
    ) -> Dict[str, Any]:
        genes = genes or []
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      row.get("Process_With_Enrichment", "") if row is not None else "",
            "Process_Without_Enrichment_Original":   row.get("Process_Without_Enrichment", "") if row is not None else "",
            "Confidence_With_Enrichment_Before":     row.get("Confidence_With_Enrichment", 0) if row is not None else 0,
            "Confidence_Without_Enrichment_Before":  row.get("Confidence_Without_Enrichment", 0) if row is not None else 0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         "ERROR",
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              error_msg,
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Error occurred during validation",
            "Total_Papers_Found":                    0,
        }

    def _no_papers_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        conf_with = float(row.get("Confidence_With_Enrichment", 0) or row.get("Final_Confidence", 0))
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Process_Without_Enrichment_Original":   str(row.get("Process_Without_Enrichment", "")),
            "Confidence_With_Enrichment_Before":     conf_with,
            "Confidence_Without_Enrichment_Before":  float(row.get("Confidence_Without_Enrichment", 0)),
            "Confidence_With_Enrichment_After":      conf_with,
            "Confidence_Without_Enrichment_After":   float(row.get("Confidence_Without_Enrichment", 0)),
            "Final_Process":                         str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Final_Confidence":                      conf_with,
            "Validation_Analysis_Text":              "No high-quality papers found for this gene set",
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "No high-quality papers available",
            "Total_Papers_Found":                    0,
        }

    def _both_unknown_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      UNKNOWN_PROCESS_LABEL,
            "Process_Without_Enrichment_Original":   UNKNOWN_PROCESS_LABEL,
            "Confidence_With_Enrichment_Before":     0,
            "Confidence_Without_Enrichment_Before":  0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         UNKNOWN_PROCESS_LABEL,
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              "Both processes are unknown - validation skipped.",
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Both processes unknown - no validation performed",
            "Total_Papers_Found":                    0,
        }



def main():
    gene_sets_path = "BreastCancer_RandomAnnotation.csv"
    papers_csv     = "BC_Paper_DB.csv"
    output_path    = "BreastCancer_RandomValidation.csv"

    print("AML Gene Set Validation Pipeline")
    print("=" * 50)

    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        print("No DEEPSEEK_API_KEY env var found.")
        print("Set it with:  export DEEPSEEK_API_KEY=your_key")
        return

    print("API key found")

    validator = GeneSetValidator(
        gene_sets_path=gene_sets_path,
        papers_csv=papers_csv,
        api_key=api_key,
        output_path=output_path,
    )

    validator.load_data()

    test_mode = False

    if test_mode:
        print("\nTest mode - validating first gene set only")
        result = validator.validate_gene_set(validator.gene_sets_df.iloc[0])
        validator.validated_results = [result]
        validator.save_results("test_validation.csv")
        for k, v in result.items():
            print(f"  {k}: {len(v) if isinstance(v, list) else v}")
    else:
        validator.validate_all_gene_sets()

    report = validator.generate_summary_report()
    print("\n" + report)
    with open("validation_summary.txt", "w") as f:
        f.write(report)

    print("\nDone!")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

path = "BreastCancer_RandomValidation.csv"
df = pd.read_csv(path)

df.loc[df["Final_Confidence"] <= 0.05, "Final_Process"] = "Neither process"

df.to_csv(path, index=False)